**Step 1 — Clean environment setup + imports + seeds + project folders**

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Step 1: Environment setup (Colab) + imports + reproducibility + repo-like folders

import os, sys, json, random, time, platform
from pathlib import Path

import numpy as np
import pandas as pd

# ---- Reproducibility ----
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ---- Paths ----
ZIP_PATH = Path("/content/drive/MyDrive/archive.zip")  # your provided path
assert ZIP_PATH.exists(), f"ZIP not found at: {ZIP_PATH}"

# Project root inside Colab runtime
PROJECT_ROOT = Path("/content/fl_ids_repo_real")
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT_ROOT)

# Repo-like structure
for d in [
    "configs",
    "data/raw",
    "data/interim",
    "data/processed",
    "runs",
    "src",
    "src/data",
    "src/models",
    "src/fl",
    "src/utils",
]:
    (PROJECT_ROOT / d).mkdir(parents=True, exist_ok=True)

# Record environment metadata
env_info = {
    "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "python_version": sys.version,
    "platform": platform.platform(),
    "seed": SEED,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "zip_path": str(ZIP_PATH),
    "project_root": str(PROJECT_ROOT),
}
(PROJECT_ROOT / "runs" / "env_info.json").write_text(json.dumps(env_info, indent=2))

print("✅ Project root:", PROJECT_ROOT)
print("✅ ZIP path:", ZIP_PATH)
print("✅ Saved env info:", PROJECT_ROOT / "runs" / "env_info.json")


✅ Project root: /content/fl_ids_repo_real
✅ ZIP path: /content/drive/MyDrive/archive.zip
✅ Saved env info: /content/fl_ids_repo_real/runs/env_info.json


**Step 2 — Unzip the dataset into data/raw/ + locate the CSV**

In [3]:
# Step 2: Unzip archive.zip -> data/raw/ and locate the target CSV

import zipfile
from pathlib import Path

ZIP_PATH = Path("/content/drive/MyDrive/archive.zip")
RAW_DIR = Path("data/raw/edgeiiotset")
RAW_DIR.mkdir(parents=True, exist_ok=True)

assert ZIP_PATH.exists(), f"ZIP not found: {ZIP_PATH}"

# ---- Extract ----
with zipfile.ZipFile(ZIP_PATH, "r") as z:
    members = z.namelist()
    print(f"✅ ZIP contains {len(members)} entries")
    # Extract all
    z.extractall(RAW_DIR)

print("✅ Extracted to:", RAW_DIR.resolve())

# ---- Find candidate CSV files ----
csv_files = sorted([p for p in RAW_DIR.rglob("*.csv")])

print(f"\nFound {len(csv_files)} CSV file(s). Showing up to 20 paths:")
for p in csv_files[:20]:
    print(" -", p)

# Heuristic: prefer the known Edge-IIoTset DNN CSV name if present
preferred_names = {
    "DNN-EdgeIIoT-dataset.csv",
    "DNN_EdgeIIoT_dataset.csv",
    "DNN-EdgeIIoTset-dataset.csv",
}

target_csv = None
for p in csv_files:
    if p.name in preferred_names:
        target_csv = p
        break

# If not found by exact name, pick the largest CSV (often the main dataset)
if target_csv is None and csv_files:
    target_csv = max(csv_files, key=lambda p: p.stat().st_size)

if target_csv is None:
    raise FileNotFoundError("No CSV files found after extraction. Check the ZIP contents.")

print("\n✅ Selected target CSV:")
print(target_csv.resolve())

# Save selected path for next steps
(Path("configs") / "data_paths.json").write_text(
    json.dumps({"zip_path": str(ZIP_PATH), "raw_dir": str(RAW_DIR), "target_csv": str(target_csv)}, indent=2)
)
print("✅ Saved configs/data_paths.json")


✅ ZIP contains 52 entries
✅ Extracted to: /content/fl_ids_repo_real/data/raw/edgeiiotset

Found 26 CSV file(s). Showing up to 20 paths:
 - data/raw/edgeiiotset/Edge-IIoTset dataset/Attack traffic/Backdoor_attack.csv
 - data/raw/edgeiiotset/Edge-IIoTset dataset/Attack traffic/DDoS_HTTP_Flood_attack.csv
 - data/raw/edgeiiotset/Edge-IIoTset dataset/Attack traffic/DDoS_ICMP_Flood_attack.csv
 - data/raw/edgeiiotset/Edge-IIoTset dataset/Attack traffic/DDoS_TCP_SYN_Flood_attack.csv
 - data/raw/edgeiiotset/Edge-IIoTset dataset/Attack traffic/DDoS_UDP_Flood_attack.csv
 - data/raw/edgeiiotset/Edge-IIoTset dataset/Attack traffic/MITM_attack.csv
 - data/raw/edgeiiotset/Edge-IIoTset dataset/Attack traffic/OS_Fingerprinting_attack.csv
 - data/raw/edgeiiotset/Edge-IIoTset dataset/Attack traffic/Password_attack.csv
 - data/raw/edgeiiotset/Edge-IIoTset dataset/Attack traffic/Port_Scanning_attack.csv
 - data/raw/edgeiiotset/Edge-IIoTset dataset/Attack traffic/Ransomware_attack.csv
 - data/raw/edgeiiotse

**Step 3 — Load CSV → basic inspection → verify Attack_type + size + missingness**

In [4]:
# Step 3: Read the selected CSV and inspect the dataset

import json
import pandas as pd
from pathlib import Path

paths = json.loads(Path("configs/data_paths.json").read_text())
CSV_PATH = Path(paths["target_csv"])
assert CSV_PATH.exists(), f"CSV not found: {CSV_PATH}"

# Read (Edge-IIoTset can be large; low_memory=False helps mixed dtypes)
df = pd.read_csv(CSV_PATH, low_memory=False)

print("✅ Loaded:", CSV_PATH.name)
print("Shape:", df.shape)
print("\nColumns (first 30):")
print(list(df.columns)[:30])

# Preview
display(df.head(5))

# Check Attack_type presence
if "Attack_type" in df.columns:
    print("\nAttack_type distribution (top 20):")
    display(df["Attack_type"].value_counts().head(20))
else:
    print("\n⚠️ 'Attack_type' column not found. We'll need to identify the correct label column(s).")

# Missingness quick check
na_total = int(df.isna().sum().sum())
print(f"\nTotal missing cells: {na_total:,}")
print("Top-15 columns by missing values:")
display(df.isna().sum().sort_values(ascending=False).head(15))

# Duplicate rows count (full-row duplicates)
dup_rows = int(df.duplicated().sum())
print(f"\nDuplicate full rows: {dup_rows:,}")


✅ Loaded: DNN-EdgeIIoT-dataset.csv
Shape: (2219201, 63)

Columns (first 30):
['frame.time', 'ip.src_host', 'ip.dst_host', 'arp.dst.proto_ipv4', 'arp.opcode', 'arp.hw.size', 'arp.src.proto_ipv4', 'icmp.checksum', 'icmp.seq_le', 'icmp.transmit_timestamp', 'icmp.unused', 'http.file_data', 'http.content_length', 'http.request.uri.query', 'http.request.method', 'http.referer', 'http.request.full_uri', 'http.request.version', 'http.response', 'http.tls_port', 'tcp.ack', 'tcp.ack_raw', 'tcp.checksum', 'tcp.connection.fin', 'tcp.connection.rst', 'tcp.connection.syn', 'tcp.connection.synack', 'tcp.dstport', 'tcp.flags', 'tcp.flags.ack']


,frame.time,ip.src_host,ip.dst_host,arp.dst.proto_ipv4,arp.opcode,arp.hw.size,arp.src.proto_ipv4,icmp.checksum,icmp.seq_le,icmp.transmit_timestamp,...,mqtt.proto_len,mqtt.protoname,mqtt.topic,mqtt.topic_len,mqtt.ver,mbtcp.len,mbtcp.trans_id,mbtcp.unit_id,Attack_label,Attack_type
0,2021 11:44:10.081753000,192.168.0.128,192.168.0.101,0,0.0,0.0,0,0.0,0.0,0.0,...,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0,Normal
1,2021 11:44:10.162218000,192.168.0.101,192.168.0.128,0,0.0,0.0,0,0.0,0.0,0.0,...,4.0,MQTT,0,0.0,4.0,0.0,0.0,0.0,0,Normal
2,2021 11:44:10.162271000,192.168.0.128,192.168.0.101,0,0.0,0.0,0,0.0,0.0,0.0,...,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0,Normal
3,2021 11:44:10.162641000,192.168.0.128,192.168.0.101,0,0.0,0.0,0,0.0,0.0,0.0,...,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0,Normal
4,2021 11:44:10.166132000,192.168.0.101,192.168.0.128,0,0.0,0.0,0,0.0,0.0,0.0,...,0.0,0,Temperature_and_Humidity,24.0,0.0,0.0,0.0,0.0,0,Normal



Attack_type distribution (top 20):


,count
Attack_type,
Normal,1615643
DDoS_UDP,121568
DDoS_ICMP,116436
SQL_injection,51203
Password,50153
Vulnerability_scanner,50110
DDoS_TCP,50062
DDoS_HTTP,49911
Uploading,37634



Total missing cells: 0
Top-15 columns by missing values:


,0
frame.time,0
ip.src_host,0
ip.dst_host,0
arp.dst.proto_ipv4,0
arp.opcode,0
arp.hw.size,0
arp.src.proto_ipv4,0
icmp.checksum,0
icmp.seq_le,0
icmp.transmit_timestamp,0



Duplicate full rows: 815


**Step 4 — Cleaning: drop specified columns + drop NaN + drop duplicates + shuffle**

In [5]:
# Step 4: Cleaning (drop columns, NaN, duplicates, shuffle)

import json
import pandas as pd
from pathlib import Path
from sklearn.utils import shuffle

paths = json.loads(Path("configs/data_paths.json").read_text())
CSV_PATH = Path(paths["target_csv"])
df = pd.read_csv(CSV_PATH, low_memory=False)

print("Loaded:", CSV_PATH.name, "| shape:", df.shape)

drop_columns = [
    "frame.time", "ip.src_host", "ip.dst_host", "arp.src.proto_ipv4", "arp.dst.proto_ipv4",
    "http.file_data", "http.request.full_uri", "icmp.transmit_timestamp",
    "http.request.uri.query", "tcp.options", "tcp.payload", "tcp.srcport",
    "tcp.dstport", "udp.port", "mqtt.msg"
]

# Drop only columns that exist
existing = [c for c in drop_columns if c in df.columns]
missing = [c for c in drop_columns if c not in df.columns]

df.drop(existing, axis=1, inplace=True)

print(f"✅ Dropped {len(existing)} columns. Missing (not dropped) {len(missing)} columns.")
if missing:
    print("Missing columns:", missing)

# Drop rows with any NaN
before = df.shape[0]
df.dropna(axis=0, how="any", inplace=True)
print(f"✅ Dropped NaN rows: {before - df.shape[0]:,}")

# Drop duplicates (full-row duplicates)
before = df.shape[0]
df.drop_duplicates(subset=None, keep="first", inplace=True)
print(f"✅ Dropped duplicate rows: {before - df.shape[0]:,}")

# Shuffle
df = shuffle(df, random_state=42).reset_index(drop=True)
print("✅ Shuffled dataset.")

print("\nNew shape:", df.shape)

# Attack_type distribution
if "Attack_type" in df.columns:
    print("\nAttack_type distribution (top 20):")
    display(df["Attack_type"].value_counts().head(20))
else:
    print("\n⚠️ 'Attack_type' not found after cleaning. We must identify the label column.")

# Save cleaned interim file
out_path = Path("data/interim/edgeiiotset_cleaned.csv")
out_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(out_path, index=False)
print("\n✅ Saved cleaned CSV to:", out_path.resolve())


Loaded: DNN-EdgeIIoT-dataset.csv | shape: (2219201, 63)
✅ Dropped 15 columns. Missing (not dropped) 0 columns.
✅ Dropped NaN rows: 0
✅ Dropped duplicate rows: 309,530
✅ Shuffled dataset.

New shape: (1909671, 48)

Attack_type distribution (top 20):


,count
Attack_type,
Normal,1363998
DDoS_UDP,121567
DDoS_ICMP,67939
SQL_injection,50826
DDoS_TCP,50062
Vulnerability_scanner,50026
Password,49933
DDoS_HTTP,48544
Uploading,36807



✅ Saved cleaned CSV to: /content/fl_ids_repo_real/data/interim/edgeiiotset_cleaned.csv


**Step 5 — Build labels + one-hot encode categoricals + save encoded arrays for FL**

This will:

* load data/interim/edgeiiotset_cleaned.csv

* create binary Label from Attack_type (Normal → 0, else → 1)

* one-hot encode categorical features (your listed columns if present, otherwise it falls back to all object columns)

* save X, y, attack_type, feature_names into data/processed/edgeiiotset_encoded.npz + metadata JSON

In [6]:
# Step 5: Label mapping + categorical encoding (one-hot) + save processed arrays

import json
import numpy as np
import pandas as pd
from pathlib import Path

CLEAN_PATH = Path("data/interim/edgeiiotset_cleaned.csv")
assert CLEAN_PATH.exists(), f"Missing cleaned file: {CLEAN_PATH}"

df = pd.read_csv(CLEAN_PATH, low_memory=False)
print("✅ Loaded cleaned CSV:", CLEAN_PATH.name, "| shape:", df.shape)

# ---- 1) Build binary label y from Attack_type ----
assert "Attack_type" in df.columns, "Expected 'Attack_type' column for label mapping."
attack_type = df["Attack_type"].astype(str)

# Map: Normal -> 0, everything else -> 1
y = (attack_type != "Normal").astype(np.int64).to_numpy()

print("Label distribution (0=benign, 1=intrusion):")
print(pd.Series(y).value_counts())

# ---- 2) Prepare feature dataframe (drop labels) ----
feature_df = df.drop(columns=["Attack_type"])

# ---- 3) One-hot encode categoricals ----
# Prefer your specified columns, but only those that exist.
preferred_cat_cols = [
    "http.request.method",
    "http.referer",
    "http.request.version",
    "dns.qry.name.len",
    "mqtt.conack.flags",
    "mqtt.protoname",
    "mqtt.topic",
]

present_preferred = [c for c in preferred_cat_cols if c in feature_df.columns]
object_cols = [c for c in feature_df.columns if feature_df[c].dtype == "object"]

# If any preferred cols exist, encode them + any other object cols (safer for fixed-dim)
# If none exist, encode all object cols.
if present_preferred:
    cat_cols = sorted(set(present_preferred + object_cols))
    print("✅ Encoding preferred categorical cols (plus other object cols).")
else:
    cat_cols = object_cols
    print("✅ Preferred categorical cols not found; encoding all object cols.")

print("Categorical columns to encode:", cat_cols)

# One-hot encode; keep all levels (drop_first=False) to avoid information loss
X_df = pd.get_dummies(feature_df, columns=cat_cols, drop_first=False)

# ---- 4) Ensure numeric + no NaNs ----
if X_df.isna().any().any():
    n_nan = int(X_df.isna().sum().sum())
    raise ValueError(f"Found NaNs in encoded features: total NaNs={n_nan}")

X = X_df.to_numpy(dtype=np.float32)

print("\n✅ Encoded feature matrix ready:")
print("X shape:", X.shape, "| y shape:", y.shape)

# ---- 5) Save processed artifacts ----
PROC_DIR = Path("data/processed")
PROC_DIR.mkdir(parents=True, exist_ok=True)

npz_path = PROC_DIR / "edgeiiotset_encoded.npz"
meta_path = PROC_DIR / "edgeiiotset_encoded_meta.json"

np.savez_compressed(
    npz_path,
    X=X,
    y=y,
    attack_type=attack_type.to_numpy(),
)

meta = {
    "source_clean_csv": str(CLEAN_PATH),
    "n_rows": int(X.shape[0]),
    "n_features": int(X.shape[1]),
    "feature_names": list(X_df.columns),
    "encoded_categorical_columns": cat_cols,
    "label_definition": "Label=1 if Attack_type!='Normal' else 0",
}

meta_path.write_text(json.dumps(meta, indent=2))

print("\n✅ Saved:")
print(" -", npz_path.resolve())
print(" -", meta_path.resolve())


✅ Loaded cleaned CSV: edgeiiotset_cleaned.csv | shape: (1909671, 48)
Label distribution (0=benign, 1=intrusion):
0    1363998
1     545673
Name: count, dtype: int64
✅ Encoding preferred categorical cols (plus other object cols).
Categorical columns to encode: ['dns.qry.name.len', 'http.referer', 'http.request.method', 'http.request.version', 'mqtt.conack.flags', 'mqtt.protoname', 'mqtt.topic']

✅ Encoded feature matrix ready:
X shape: (1909671, 96) | y shape: (1909671,)

✅ Saved:
 - /content/fl_ids_repo_real/data/processed/edgeiiotset_encoded.npz
 - /content/fl_ids_repo_real/data/processed/edgeiiotset_encoded_meta.json


**Step 6 — Dirichlet(𝛼) non-IID partition into K=10 silos + 70/15/15 split within each silo**

This creates splits for 𝛼∈{0.1,0.3,0.5,1.0} and saves them to data/processed/.

In [7]:
# Step 6: Dirichlet non-IID partition (K=10) + per-silo 70/15/15 split (saved to disk)

import json
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split

SEED = 42
rng = np.random.default_rng(SEED)

K = 10
alpha_grid = [0.1, 0.3, 0.5, 1.0]
split = {"train": 0.70, "val": 0.15, "test": 0.15}

ENC_PATH = Path("data/processed/edgeiiotset_encoded.npz")
META_PATH = Path("data/processed/edgeiiotset_encoded_meta.json")
assert ENC_PATH.exists(), f"Missing encoded dataset: {ENC_PATH}"
assert META_PATH.exists(), f"Missing meta: {META_PATH}"

data = np.load(ENC_PATH, allow_pickle=True)
X = data["X"]
y = data["y"].astype(int)

print("✅ Loaded encoded dataset:", ENC_PATH.name)
print("X shape:", X.shape, "| y shape:", y.shape)

OUT_DIR = Path("data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

def dirichlet_partition_indices(y, K, alpha, seed=42, min_per_silo=50, max_tries=500):
    """
    Label-skew Dirichlet partition with best-effort fallback.
    For each class c: sample proportions p_c ~ Dirichlet(alpha,...,alpha) over K silos,
    then split class indices accordingly.
    """
    rng_local = np.random.default_rng(seed)
    classes = np.unique(y)

    best_silos = None
    best_min_size = -1

    for _ in range(max_tries):
        silos = [[] for _ in range(K)]

        for c in classes:
            idx_c = np.where(y == c)[0]
            rng_local.shuffle(idx_c)

            props = rng_local.dirichlet(alpha=np.full(K, alpha))
            counts = rng_local.multinomial(len(idx_c), props)

            start = 0
            for k in range(K):
                cnt = counts[k]
                if cnt > 0:
                    silos[k].extend(idx_c[start:start+cnt].tolist())
                start += cnt

        sizes = np.array([len(s) for s in silos], dtype=int)
        min_size = int(sizes.min())

        if min_size > best_min_size:
            best_min_size = min_size
            best_silos = [np.array(s, dtype=np.int64) for s in silos]

        if min_size >= min_per_silo:
            for k in range(K):
                rng_local.shuffle(silos[k])
            return [np.array(s, dtype=np.int64) for s in silos], True, best_min_size

    # fallback: return best we saw
    return best_silos, False, best_min_size

def split_silo_indices(idx, y, seed=42, split=(0.70, 0.15, 0.15)):
    """
    Split a silo's indices into train/val/test with stratification when feasible.
    """
    idx = np.array(idx, dtype=np.int64)
    y_s = y[idx]

    # If too small or single-class, do a simple non-stratified split
    if len(idx) < 30 or len(np.unique(y_s)) < 2:
        n = len(idx)
        n_train = max(1, int(split[0]*n))
        n_val   = max(0, int(split[1]*n))
        perm = np.random.default_rng(seed).permutation(idx)
        tr = perm[:n_train]
        va = perm[n_train:n_train+n_val]
        te = perm[n_train+n_val:]
        return tr, va, te

    # train vs temp
    test_val_frac = split[1] + split[2]
    strat_ok = (len(np.unique(y_s)) > 1) and (np.min(np.bincount(y_s)) >= 2)

    tr, temp = train_test_split(
        idx,
        test_size=test_val_frac,
        random_state=seed,
        shuffle=True,
        stratify=y_s if strat_ok else None
    )

    # val vs test
    y_temp = y[temp]
    val_frac_of_temp = split[1] / (split[1] + split[2])
    strat_ok2 = (len(np.unique(y_temp)) > 1) and (np.min(np.bincount(y_temp)) >= 2)

    va, te = train_test_split(
        temp,
        test_size=(1.0 - val_frac_of_temp),
        random_state=seed,
        shuffle=True,
        stratify=y_temp if strat_ok2 else None
    )
    return tr, va, te

# For alpha=0.1 you might get tiny silos; allow smaller min_per_silo
def min_per_silo_for_alpha(a):
    return 50 if a >= 0.5 else 10

all_summaries = {}

for a in alpha_grid:
    min_per = min_per_silo_for_alpha(a)
    silos, met_constraint, best_min = dirichlet_partition_indices(
        y, K=K, alpha=a, seed=SEED, min_per_silo=min_per, max_tries=500
    )

    splits = {"train": [], "val": [], "test": []}
    silo_stats = []

    for k in range(K):
        idx_k = silos[k]
        tr, va, te = split_silo_indices(idx_k, y, seed=SEED, split=(split["train"], split["val"], split["test"]))

        splits["train"].append(tr)
        splits["val"].append(va)
        splits["test"].append(te)

        yk = y[idx_k]
        n0 = int((yk == 0).sum())
        n1 = int((yk == 1).sum())
        silo_stats.append({
            "silo": k,
            "n_total": int(len(idx_k)),
            "n_benign_0": n0,
            "n_intrusion_1": n1,
            "intrusion_rate": float(n1 / max(1, len(idx_k))),
            "n_train": int(len(tr)),
            "n_val": int(len(va)),
            "n_test": int(len(te)),
        })

    out_npz = OUT_DIR / f"splits_edgeiiotset_dirichlet_alpha_{a}.npz"
    np.savez_compressed(
        out_npz,
        train=np.array(splits["train"], dtype=object),
        val=np.array(splits["val"], dtype=object),
        test=np.array(splits["test"], dtype=object),
        alpha=np.array([a], dtype=np.float32),
        K=np.array([K], dtype=np.int32),
        seed=np.array([SEED], dtype=np.int32),
    )

    out_json = OUT_DIR / f"splits_edgeiiotset_dirichlet_alpha_{a}_summary.json"
    summary = {
        "dataset": "Edge-IIoTset (encoded)",
        "alpha": a,
        "K": K,
        "seed": SEED,
        "split_ratios": split,
        "min_per_silo_target": min_per,
        "met_min_per_silo_target": bool(met_constraint),
        "best_min_silo_size_seen": int(best_min),
        "silo_stats": silo_stats,
    }
    out_json.write_text(json.dumps(summary, indent=2))

    sizes = [s["n_total"] for s in silo_stats]
    all_summaries[str(a)] = {
        "npz": str(out_npz),
        "summary": str(out_json),
        "min_silo_size": int(min(sizes)),
        "max_silo_size": int(max(sizes)),
        "met_min_target": bool(met_constraint),
    }

    print(f"✅ alpha={a}: min/max silo size = {min(sizes)}/{max(sizes)} | met_min_target={met_constraint}")

combined_path = OUT_DIR / "splits_edgeiiotset_index.json"
combined_path.write_text(json.dumps(all_summaries, indent=2))
print("✅ Wrote:", combined_path)


✅ Loaded encoded dataset: edgeiiotset_encoded.npz
X shape: (1909671, 96) | y shape: (1909671,)
✅ alpha=0.1: min/max silo size = 331/1108901 | met_min_target=True
✅ alpha=0.3: min/max silo size = 26428/880046 | met_min_target=True
✅ alpha=0.5: min/max silo size = 10530/830587 | met_min_target=True
✅ alpha=1.0: min/max silo size = 48639/526937 | met_min_target=True
✅ Wrote: data/processed/splits_edgeiiotset_index.json


**Step 7 — Per-silo local normalization + PyTorch DataLoaders (choose one 𝛼)**

It loads your saved Dirichlet splits, computes (𝜇𝑘,𝜎𝑘) from each silo’s train split only, applies it to that silo’s train/val/test, and saves the norm stats.

In [8]:
# Step 7: Build per-silo loaders with LOCAL normalization (train-only stats) for a chosen alpha

import json
import numpy as np
from pathlib import Path

import torch
from torch.utils.data import Dataset, DataLoader

# ---- Choose alpha here ----
ALPHA = 0.3  # 0.1 / 0.3 / 0.5 / 1.0
BATCH_SIZE = 256
EPS = 1e-8

ENC_PATH = Path("data/processed/edgeiiotset_encoded.npz")
SPL_PATH = Path(f"data/processed/splits_edgeiiotset_dirichlet_alpha_{ALPHA}.npz")
assert ENC_PATH.exists(), f"Missing: {ENC_PATH}"
assert SPL_PATH.exists(), f"Missing: {SPL_PATH}"

data = np.load(ENC_PATH, allow_pickle=True)
X = data["X"].astype(np.float32)
y = data["y"].astype(np.int64)

spl = np.load(SPL_PATH, allow_pickle=True)
train_idxs = spl["train"]
val_idxs = spl["val"]
test_idxs = spl["test"]
K = int(spl["K"][0])

print("✅ Loaded encoded X,y:", X.shape, y.shape)
print("✅ Loaded splits:", SPL_PATH.name, "| K =", K, "| alpha =", ALPHA)

class TabularDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).long()
    def __len__(self): return self.X.shape[0]
    def __getitem__(self, i): return self.X[i], self.y[i]

def fit_local_norm(X_train):
    mu = X_train.mean(axis=0, dtype=np.float64)
    sigma = X_train.std(axis=0, dtype=np.float64)
    return mu.astype(np.float32), sigma.astype(np.float32)

def apply_local_norm(X_part, mu, sigma, eps=1e-8):
    return (X_part - mu) / (sigma + eps)

silo_data = []
mu_list, sigma_list = [], []
sizes = []

for k in range(K):
    tr = np.array(train_idxs[k], dtype=np.int64)
    va = np.array(val_idxs[k], dtype=np.int64)
    te = np.array(test_idxs[k], dtype=np.int64)

    X_tr, y_tr = X[tr], y[tr]
    X_va, y_va = X[va], y[va]
    X_te, y_te = X[te], y[te]

    mu_k, sigma_k = fit_local_norm(X_tr)

    X_tr_n = apply_local_norm(X_tr, mu_k, sigma_k, EPS)
    X_va_n = apply_local_norm(X_va, mu_k, sigma_k, EPS)
    X_te_n = apply_local_norm(X_te, mu_k, sigma_k, EPS)

    silo_data.append({
        "train_loader": DataLoader(TabularDataset(X_tr_n, y_tr), batch_size=BATCH_SIZE, shuffle=True),
        "val_loader":   DataLoader(TabularDataset(X_va_n, y_va), batch_size=BATCH_SIZE, shuffle=False),
        "test_loader":  DataLoader(TabularDataset(X_te_n, y_te), batch_size=BATCH_SIZE, shuffle=False),
        "n_train": int(len(tr)),
        "n_val": int(len(va)),
        "n_test": int(len(te)),
    })

    mu_list.append(mu_k)
    sigma_list.append(sigma_k)
    sizes.append((len(tr), len(va), len(te)))

# Save norm stats for reproducibility
out_norm = Path(f"data/processed/local_norm_edgeiiotset_alpha_{ALPHA}.npz")
np.savez_compressed(
    out_norm,
    mu=np.stack(mu_list, axis=0),
    sigma=np.stack(sigma_list, axis=0),
    alpha=np.array([ALPHA], dtype=np.float32),
    K=np.array([K], dtype=np.int32),
    eps=np.array([EPS], dtype=np.float32),
)

print("✅ Saved local norm stats:", out_norm)
print("Per-silo (train,val,test) sizes (first 5):", sizes[:5])
print("Feature dim d =", X.shape[1])

# Keep these in memory for next steps
GLOBAL_D = X.shape[1]
GLOBAL_K = K


✅ Loaded encoded X,y: (1909671, 96) (1909671,)
✅ Loaded splits: splits_edgeiiotset_dirichlet_alpha_0.3.npz | K = 10 | alpha = 0.3
✅ Saved local norm stats: data/processed/local_norm_edgeiiotset_alpha_0.3.npz
Per-silo (train,val,test) sizes (first 5): [(31113, 6667, 6668), (47670, 10215, 10216), (18499, 3964, 3965), (60411, 12945, 12946), (616032, 132006, 132008)]
Feature dim d = 96


**Step 8 — Define the MLP IDS (BN disabled) + weighted BCE + evaluation helpers**

In [9]:
# Step 8: MLP IDS (FL-stable: no BatchNorm) + weighted BCE + eval utilities (Proposed Methodology MLP Model)

import numpy as np
import torch
import torch.nn as nn

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("✅ Using device:", DEVICE)

# ---- Baseline model hyperparams (from your table) ----
HIDDEN = (256, 128, 64)
DROPOUT = 0.30
BATCH_NORM = False   # keep False for FL stability

# Weighted loss (lambda1=2.0, lambda0=1.0) -> pos_weight = 2.0
POS_WEIGHT = torch.tensor([2.0], device=DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=POS_WEIGHT)

class MLPIDS(nn.Module):
    """
    Lightweight MLP IDS for tabular flow/session features.
    Outputs logits (sigmoid -> probability).
    """
    def __init__(self, input_dim, hidden=HIDDEN, dropout=DROPOUT, batch_norm=BATCH_NORM):
        super().__init__()
        layers = []
        in_dim = input_dim

        for h in hidden:
            layers.append(nn.Linear(in_dim, h))
            if batch_norm:
                layers.append(nn.BatchNorm1d(h))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            in_dim = h

        self.net = nn.Sequential(*layers)
        self.out = nn.Linear(in_dim, 1)

    def forward(self, x):
        return self.out(self.net(x)).squeeze(-1)  # logits

@torch.no_grad()
def eval_on_loader(model, loader, threshold=0.5):
    model.eval()
    probs_list, y_list = [], []
    total_loss, total_n = 0.0, 0

    for xb, yb in loader:
        xb = xb.to(DEVICE)
        yb_f = yb.float().to(DEVICE)
        logits = model(xb)
        loss = criterion(logits, yb_f)

        bs = xb.size(0)
        total_loss += float(loss.item()) * bs
        total_n += bs

        probs_list.append(torch.sigmoid(logits).cpu().numpy())
        y_list.append(yb.cpu().numpy())

    probs = np.concatenate(probs_list)
    y_true = np.concatenate(y_list).astype(int)
    y_pred = (probs >= threshold).astype(int)

    tp = int(((y_pred==1)&(y_true==1)).sum())
    tn = int(((y_pred==0)&(y_true==0)).sum())
    fp = int(((y_pred==1)&(y_true==0)).sum())
    fn = int(((y_pred==0)&(y_true==1)).sum())

    precision = tp / (tp + fp + 1e-12)
    recall    = tp / (tp + fn + 1e-12)
    f1        = 2*precision*recall / (precision + recall + 1e-12)
    acc       = (tp + tn) / max(1, (tp+tn+fp+fn))

    return {
        "loss": float(total_loss / max(1,total_n)),
        "acc": float(acc),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "pred_pos_rate": float(y_pred.mean()),
        "tp": tp, "tn": tn, "fp": fp, "fn": fn
    }

@torch.no_grad()
def eval_global_union(model, silo_data, split="val", threshold=0.5):
    # Union-of-silos evaluation (global metrics)
    model.eval()
    probs_all, y_all_list = [], []
    total_loss, total_n = 0.0, 0

    for k in range(len(silo_data)):
        loader = silo_data[k]["val_loader"] if split == "val" else silo_data[k]["test_loader"]
        for xb, yb in loader:
            xb = xb.to(DEVICE)
            yb_f = yb.float().to(DEVICE)
            logits = model(xb)
            loss = criterion(logits, yb_f)
            bs = xb.size(0)

            total_loss += float(loss.item()) * bs
            total_n += bs
            probs_all.append(torch.sigmoid(logits).cpu().numpy())
            y_all_list.append(yb.cpu().numpy())

    probs = np.concatenate(probs_all)
    y_true = np.concatenate(y_all_list).astype(int)
    y_pred = (probs >= threshold).astype(int)

    tp = int(((y_pred==1)&(y_true==1)).sum())
    tn = int(((y_pred==0)&(y_true==0)).sum())
    fp = int(((y_pred==1)&(y_true==0)).sum())
    fn = int(((y_pred==0)&(y_true==1)).sum())

    precision = tp / (tp + fp + 1e-12)
    recall    = tp / (tp + fn + 1e-12)
    f1        = 2*precision*recall / (precision + recall + 1e-12)
    acc       = (tp + tn) / max(1, (tp+tn+fp+fn))

    return {
        "loss": float(total_loss / max(1,total_n)),
        "acc": float(acc),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "pred_pos_rate": float(y_pred.mean()),
        "tp": tp, "tn": tn, "fp": fp, "fn": fn
    }

print("✅ Model + evaluation utilities ready.")


✅ Using device: cpu
✅ Model + evaluation utilities ready.


In [10]:
# Step 8A — Logistic Regression IDS (FL-compatible) + weighted BCE + evaluation helpers (Can configure for other Models. Example for Regression)

import numpy as np
import torch
import torch.nn as nn

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("✅ Device:", DEVICE)

# -----------------------------
# Model: Logistic Regression IDS
# -----------------------------
class LogisticIDS(nn.Module):
    """
    Logistic regression for binary intrusion detection.
    Outputs logits (NOT sigmoid).
    """
    def __init__(self, input_dim: int):
        super().__init__()
        self.linear = nn.Linear(input_dim, 1)

    def forward(self, x):
        return self.linear(x).squeeze(-1)  # logits


# -----------------------------
# Loss: Weighted BCE (imbalance)
# -----------------------------
POS_WEIGHT = 2.0  # pos_weight ~= lambda1/lambda0 in your weighted CE
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([POS_WEIGHT], device=DEVICE))


# -----------------------------
# Metrics / Evaluation helpers
# -----------------------------
@torch.no_grad()
def eval_loader(model: nn.Module, loader, threshold=0.5):
    model.eval()
    total_loss, n = 0.0, 0
    tp = tn = fp = fn = 0

    for xb, yb in loader:
        xb = xb.to(DEVICE)
        yb = yb.float().to(DEVICE)

        logits = model(xb)
        loss = criterion(logits, yb)

        probs = torch.sigmoid(logits)
        pred = (probs >= threshold).long()
        y = yb.long()

        tp += int(((pred == 1) & (y == 1)).sum().item())
        tn += int(((pred == 0) & (y == 0)).sum().item())
        fp += int(((pred == 1) & (y == 0)).sum().item())
        fn += int(((pred == 0) & (y == 1)).sum().item())

        bsz = xb.size(0)
        total_loss += float(loss.item()) * bsz
        n += bsz

    precision = tp / (tp + fp + 1e-12)
    recall = tp / (tp + fn + 1e-12)
    f1 = 2 * precision * recall / (precision + recall + 1e-12)
    acc = (tp + tn) / max(1, (tp + tn + fp + fn))
    fpr = fp / (fp + tn + 1e-12)

    return {
        "loss": total_loss / max(1, n),
        "acc": float(acc),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "fpr": float(fpr),
        "tp": tp, "tn": tn, "fp": fp, "fn": fn,
        "pred_pos_rate": float((tp + fp) / max(1, (tp + tn + fp + fn))),
        "n": int(n),
    }


@torch.no_grad()
def collect_probs(model: nn.Module, loader):
    model.eval()
    probs_list, y_list = [], []
    for xb, yb in loader:
        xb = xb.to(DEVICE)
        logits = model(xb)
        probs_list.append(torch.sigmoid(logits).cpu().numpy())
        y_list.append(yb.numpy())
    return np.concatenate(probs_list).astype(np.float64), np.concatenate(y_list).astype(int)


print("✅ Step 8A ready: LogisticIDS + weighted BCE + eval helpers.")


✅ Device: cpu
✅ Step 8A ready: LogisticIDS + weighted BCE + eval helpers.


**Step 9 — FedProx local training + update protection + multi-round FedAvg FL on real Edge-IIoTset**

In [11]:
# Step 9: FL training loop (FedAvg baseline) with FedProx clients + clipped updates

import time, copy, json
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---- Assumes Step 7 created: silo_data, GLOBAL_D, GLOBAL_K ----
assert "silo_data" in globals(), "Run Step 7 first (build silo_data)."
d = int(GLOBAL_D)
K = int(GLOBAL_K)

# ---- Training config (baseline-ish; adjust later) ----
CFG = {
    "seed": 42,
    "alpha": 0.3,
    "rounds": 10,
    "dropout_q": 0.2,
    "participation_min": 1,

    "local_epochs": 2,
    "fedprox_mu": 1e-3,
    "lr": 3e-4,
    "weight_decay": 1e-4,
    "grad_clip": 1.0,

    "server_gamma": 1.0,

    "clip_C": 1.0,      # update clipping
    "dp_sigma": 0.0,    # DP noise disabled; set to 0.4 to enable

    "threshold": 0.5,
}

rng = np.random.default_rng(CFG["seed"])

# ---- Model ----
global_model = MLPIDS(input_dim=d).to(DEVICE)

# ---- Helpers: FedProx training ----
def get_param_vector(model):
    return torch.cat([p.view(-1) for p in model.parameters()])

def fedprox_penalty(model, w0_vec):
    w_vec = get_param_vector(model)
    return 0.5 * torch.sum((w_vec - w0_vec) ** 2)

def train_one_client_fedprox(global_model, train_loader, cfg, seed_offset=0):
    torch.manual_seed(cfg["seed"] + seed_offset)
    model = copy.deepcopy(global_model).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])

    with torch.no_grad():
        w0 = get_param_vector(model).detach()

    model.train()
    for _ in range(cfg["local_epochs"]):
        for xb, yb in train_loader:
            xb = xb.to(DEVICE)
            yb = yb.float().to(DEVICE)

            opt.zero_grad()
            logits = model(xb)
            base = criterion(logits, yb)
            prox = cfg["fedprox_mu"] * fedprox_penalty(model, w0)
            loss = base + prox
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["grad_clip"])
            opt.step()

    return model

# ---- Helpers: delta, clipping, FedAvg ----
def state_dict_to_vector(sd):
    return torch.cat([v.detach().reshape(-1).float().cpu() for v in sd.values() if torch.is_tensor(v)])

def vector_to_state_dict_like(vec, ref_sd):
    out, off = {}, 0
    for k, v in ref_sd.items():
        if torch.is_tensor(v):
            n = v.numel()
            out[k] = vec[off:off+n].view_as(v).to(v.dtype)
            off += n
        else:
            out[k] = v
    return out

def compute_delta(global_model, local_model):
    g_sd = global_model.state_dict()
    l_sd = local_model.state_dict()
    return state_dict_to_vector(l_sd) - state_dict_to_vector(g_sd), g_sd

def clip_and_noise(delta_vec, C=1.0, sigma=0.0, seed=42):
    delta = delta_vec.clone()
    norm = torch.linalg.norm(delta, ord=2)
    scale = max(1.0, float(norm.item()) / float(C))
    delta = delta / scale
    if sigma and sigma > 0.0:
        gen = torch.Generator(device="cpu").manual_seed(int(seed))
        delta = delta + torch.normal(0.0, float(sigma), size=delta.shape, generator=gen)
    return delta

def payload_bytes(delta_vec): return int(delta_vec.numel() * 4)

def agg_fedavg(deltas, weights):
    W = float(np.sum(weights))
    out = torch.zeros_like(deltas[0])
    for d, w in zip(deltas, weights):
        out += d * (float(w) / W)
    return out

def apply_delta_to_model(global_model, delta_vec, ref_sd, gamma=1.0):
    g_sd = global_model.state_dict()
    delta_sd = vector_to_state_dict_like(delta_vec, ref_sd)
    new_sd = {}
    for k, v in g_sd.items():
        if torch.is_tensor(v):
            new_sd[k] = v + gamma * delta_sd[k].to(v.device)
        else:
            new_sd[k] = v
    global_model.load_state_dict(new_sd)
    return global_model

# ---- Communication sizing ----
param_count = sum(p.numel() for p in global_model.parameters())
model_bytes = int(param_count * 4)
downlink_bytes_per_round = int(K * model_bytes)

# ---- Run directory ----
run_id = time.strftime("run_edgeiiotset_%Y%m%d_%H%M%S", time.gmtime())
RUN_DIR = Path("runs") / run_id
RUN_DIR.mkdir(parents=True, exist_ok=True)
(RUN_DIR / "config.json").write_text(json.dumps(CFG, indent=2))

rows = []
for t in range(CFG["rounds"]):
    # participation with dropout
    participants = [k for k in range(K) if rng.uniform() > CFG["dropout_q"]]
    if len(participants) < CFG["participation_min"]:
        participants = list(rng.choice(K, size=CFG["participation_min"], replace=False))

    pre_val = eval_global_union(global_model, silo_data, split="val", threshold=CFG["threshold"])

    t0 = time.time()
    deltas, weights = [], []
    uplink_bytes = 0
    ref_sd = None

    for k in participants:
        local_model = train_one_client_fedprox(global_model, silo_data[k]["train_loader"], CFG, seed_offset=1000*t + k)
        delta_vec, ref_sd = compute_delta(global_model, local_model)
        protected = clip_and_noise(delta_vec, C=CFG["clip_C"], sigma=CFG["dp_sigma"], seed=CFG["seed"] + 1000*t + k)
        deltas.append(protected)
        weights.append(silo_data[k]["n_train"])
        uplink_bytes += payload_bytes(protected)

    agg_delta = agg_fedavg(deltas, weights)
    global_model = apply_delta_to_model(global_model, agg_delta, ref_sd, gamma=CFG["server_gamma"])

    round_sec = time.time() - t0
    post_val = eval_global_union(global_model, silo_data, split="val", threshold=CFG["threshold"])

    rows.append({
        "round": t,
        "participants": len(participants),
        "pre_val_f1": pre_val["f1"],
        "post_val_f1": post_val["f1"],
        "pre_pred_pos_rate": pre_val["pred_pos_rate"],
        "post_pred_pos_rate": post_val["pred_pos_rate"],
        "uplink_MB": uplink_bytes / 1e6,
        "downlink_MB": downlink_bytes_per_round / 1e6,
        "round_seconds": round_sec,
    })

    print(f"Round {t}: m={len(participants)} | val_f1 {pre_val['f1']:.4f}->{post_val['f1']:.4f} | "
          f"pred+ {pre_val['pred_pos_rate']:.3f}->{post_val['pred_pos_rate']:.3f} | "
          f"up={uplink_bytes/1e6:.3f}MB down={downlink_bytes_per_round/1e6:.3f}MB sec={round_sec:.2f}")

# Final test (global union)
final_test = eval_global_union(global_model, silo_data, split="test", threshold=CFG["threshold"])

# Save artifacts
import pandas as pd
pd.DataFrame(rows).to_csv(RUN_DIR / "round_log.csv", index=False)
(RUN_DIR / "final_test.json").write_text(json.dumps(final_test, indent=2))
torch.save(global_model.state_dict(), RUN_DIR / "global_model_final.pt")

print("\n✅ Saved run artifacts to:", RUN_DIR)
print(" - round_log.csv")
print(" - final_test.json")
print(" - global_model_final.pt")
print("\n📌 Final global TEST:", final_test)


Round 0: m=8 | val_f1 0.4445->0.4692 | pred+ 1.000->0.932 | up=2.114MB down=2.642MB sec=54.58
Round 1: m=9 | val_f1 0.4692->0.5446 | pred+ 0.932->0.107 | up=2.378MB down=2.642MB sec=103.97
Round 2: m=7 | val_f1 0.5446->0.0000 | pred+ 0.107->0.000 | up=1.849MB down=2.642MB sec=81.47
Round 3: m=8 | val_f1 0.0000->0.0000 | pred+ 0.000->0.000 | up=2.114MB down=2.642MB sec=97.80
Round 4: m=9 | val_f1 0.0000->0.0000 | pred+ 0.000->0.000 | up=2.378MB down=2.642MB sec=113.62
Round 5: m=6 | val_f1 0.0000->0.0000 | pred+ 0.000->0.000 | up=1.585MB down=2.642MB sec=89.95
Round 6: m=9 | val_f1 0.0000->0.0000 | pred+ 0.000->0.000 | up=2.378MB down=2.642MB sec=98.95
Round 7: m=9 | val_f1 0.0000->0.6445 | pred+ 0.000->0.136 | up=2.378MB down=2.642MB sec=60.96
Round 8: m=6 | val_f1 0.6445->0.6890 | pred+ 0.136->0.542 | up=1.585MB down=2.642MB sec=35.26
Round 9: m=7 | val_f1 0.6890->0.8024 | pred+ 0.542->0.192 | up=1.849MB down=2.642MB sec=80.90

✅ Saved run artifacts to: runs/run_edgeiiotset_20261001_0

**Step 10 — Per-silo threshold calibration 𝜃𝑘 on validation + per-silo & global test metrics**

In [12]:
# Step 10: Calibrate theta_k per silo using VAL, then evaluate per-silo & global TEST

import json
import numpy as np
import torch
from pathlib import Path

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---- Use the most recent run from Step 9 (or set manually) ----
# If you want to set manually: RUN_DIR = Path("runs/<your_run_id>")
runs = sorted(Path("runs").glob("run_edgeiiotset_*"))
assert len(runs) > 0, "No run_edgeiiotset_* found. Run Step 9 first."
RUN_DIR = runs[-1]
print("✅ Using RUN_DIR:", RUN_DIR)

cfg = json.loads((RUN_DIR / "config.json").read_text())
threshold_default = float(cfg.get("threshold", 0.5))

# ---- Calibration mode ----
CALIBRATION_MODE = "fpr"     # "fpr" or "f1"
TARGET_FPR = 0.05            # used only if CALIBRATION_MODE == "fpr"

assert "silo_data" in globals(), "Run Step 7 first (build silo_data)."
d = int(GLOBAL_D)
K = int(GLOBAL_K)

# ---- Load model weights from Step 9 ----
model = MLPIDS(input_dim=d).to(DEVICE)
model.load_state_dict(torch.load(RUN_DIR / "global_model_final.pt", map_location=DEVICE))
model.eval()

@torch.no_grad()
def collect_probs(loader):
    probs_list, y_list = [], []
    for xb, yb in loader:
        xb = xb.to(DEVICE)
        logits = model(xb)
        probs_list.append(torch.sigmoid(logits).cpu().numpy())
        y_list.append(yb.cpu().numpy())
    probs = np.concatenate(probs_list).astype(np.float64)
    y_true = np.concatenate(y_list).astype(int)
    return probs, y_true

def metrics_at_threshold(probs, y_true, thr):
    y_pred = (probs >= thr).astype(int)
    tp = int(((y_pred==1)&(y_true==1)).sum())
    tn = int(((y_pred==0)&(y_true==0)).sum())
    fp = int(((y_pred==1)&(y_true==0)).sum())
    fn = int(((y_pred==0)&(y_true==1)).sum())

    precision = tp / (tp + fp + 1e-12)
    recall    = tp / (tp + fn + 1e-12)
    f1        = 2*precision*recall / (precision + recall + 1e-12)
    acc       = (tp + tn) / max(1, (tp+tn+fp+fn))
    fpr       = fp / (fp + tn + 1e-12)

    return {
        "acc": float(acc),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "fpr": float(fpr),
        "pred_pos_rate": float(y_pred.mean()),
        "tp": tp, "tn": tn, "fp": fp, "fn": fn
    }

def calibrate_theta_fpr(probs_val, y_val, target_fpr=0.05):
    benign_probs = probs_val[y_val == 0]
    if benign_probs.size == 0:
        return threshold_default
    # Threshold at (1-target_fpr) quantile of benign probs
    return float(np.quantile(benign_probs, 1.0 - target_fpr))

def calibrate_theta_max_f1(probs_val, y_val):
    grid = np.linspace(0.01, 0.99, 99)
    best_thr, best_f1 = threshold_default, -1.0
    for thr in grid:
        m = metrics_at_threshold(probs_val, y_val, thr)
        if m["f1"] > best_f1:
            best_f1 = m["f1"]
            best_thr = float(thr)
    return best_thr

# ---- Per-silo calibration + evaluation ----
thetas = []
per_silo_val = []
per_silo_test = []

for k in range(K):
    probs_val, y_val = collect_probs(silo_data[k]["val_loader"])
    if CALIBRATION_MODE == "fpr":
        theta_k = calibrate_theta_fpr(probs_val, y_val, target_fpr=TARGET_FPR)
    else:
        theta_k = calibrate_theta_max_f1(probs_val, y_val)

    thetas.append(theta_k)
    per_silo_val.append(metrics_at_threshold(probs_val, y_val, theta_k))

    probs_test, y_test = collect_probs(silo_data[k]["test_loader"])
    per_silo_test.append(metrics_at_threshold(probs_test, y_test, theta_k))

# ---- Global union metrics (sum confusion across silos) ----
tp = sum(m["tp"] for m in per_silo_test)
tn = sum(m["tn"] for m in per_silo_test)
fp = sum(m["fp"] for m in per_silo_test)
fn = sum(m["fn"] for m in per_silo_test)

precision = tp / (tp + fp + 1e-12)
recall    = tp / (tp + fn + 1e-12)
f1        = 2*precision*recall / (precision + recall + 1e-12)
acc       = (tp + tn) / max(1, tp+tn+fp+fn)
fpr       = fp / (fp + tn + 1e-12)

global_metrics = {
    "acc": float(acc),
    "precision": float(precision),
    "recall": float(recall),
    "f1": float(f1),
    "fpr": float(fpr),
    "tp": int(tp), "tn": int(tn), "fp": int(fp), "fn": int(fn),
}

print(f"✅ Calibration mode: {CALIBRATION_MODE} | target_fpr={TARGET_FPR if CALIBRATION_MODE=='fpr' else 'N/A'}")
print("Theta_k (first 5):", [round(t, 4) for t in thetas[:5]])
print("\n📌 Global TEST with per-silo theta_k:", global_metrics)

# ---- Per-silo table ----
import pandas as pd
df = pd.DataFrame([{
    "silo": k,
    "theta_k": thetas[k],
    "val_f1": per_silo_val[k]["f1"],
    "test_f1": per_silo_test[k]["f1"],
    "test_precision": per_silo_test[k]["precision"],
    "test_recall": per_silo_test[k]["recall"],
    "test_fpr": per_silo_test[k]["fpr"],
    "pred_pos_rate": per_silo_test[k]["pred_pos_rate"],
    "n_test": silo_data[k]["n_test"],
} for k in range(K)])
display(df)

# ---- Save artifacts ----
out = RUN_DIR / f"theta_calibration_{CALIBRATION_MODE}.json"
out.write_text(json.dumps({
    "calibration_mode": CALIBRATION_MODE,
    "target_fpr": TARGET_FPR,
    "thetas": thetas,
    "global_test": global_metrics,
    "per_silo_val": per_silo_val,
    "per_silo_test": per_silo_test,
}, indent=2))
print("✅ Saved:", out)


✅ Using RUN_DIR: runs/run_edgeiiotset_20261001_093908
✅ Calibration mode: fpr | target_fpr=0.05
Theta_k (first 5): [0.0, 0.0, 0.051, 0.0, 0.0513]

📌 Global TEST with per-silo theta_k: {'acc': 0.9642599063733824, 'precision': 0.8889371639602455, 'recall': 0.9998411786983984, 'f1': 0.9411331777039371, 'fpr': 0.049974585052100645, 'tp': 81840, 'tn': 194379, 'fp': 10225, 'fn': 13}


,silo,theta_k,val_f1,test_f1,test_precision,test_recall,test_fpr,pred_pos_rate,n_test
0,0,3.267214e-06,0.973752,0.976872,0.954789,1.000000,0.043956,0.504199,6668
1,1,3.114551e-20,0.999088,0.999443,0.998987,0.999899,0.028490,0.966523,10216
2,2,5.100373e-02,0.000000,0.000000,0.000000,0.000000,0.049180,0.049180,3965
3,3,1.588848e-08,0.990487,0.989440,0.979408,0.999680,0.055136,0.738993,12946
4,4,5.127764e-02,0.000000,0.000000,0.000000,0.000000,0.049906,0.049906,132008
5,5,3.663909e-04,0.910490,0.904941,0.826994,0.999111,0.053381,0.245778,16580
6,6,2.457788e-18,0.998808,0.998808,0.997768,0.999851,0.049261,0.958562,28042
7,7,5.419723e-08,0.988195,0.987894,0.976162,0.999912,0.051456,0.694674,33515
8,8,4.877480e-02,0.002215,0.003415,0.001710,1.000000,0.048604,0.048683,36029
9,9,0.000000e+00,0.999074,0.999074,0.998150,1.000000,1.000000,1.000000,6488


✅ Saved: runs/run_edgeiiotset_20261001_093908/theta_calibration_fpr.json


**Next step (Step 10.1) — Fix calibration: enforce sane 𝜃𝑘 and guardrails**

In [13]:
# Step 10.1: Robust per-silo theta calibration with bounds + fallbacks + sanity checks

import json
import numpy as np
import torch
from pathlib import Path

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Use latest run from Step 9
runs = sorted(Path("runs").glob("run_edgeiiotset_*"))
assert len(runs) > 0, "No run_edgeiiotset_* found. Run Step 9 first."
RUN_DIR = runs[-1]
print("✅ Using RUN_DIR:", RUN_DIR)

cfg = json.loads((RUN_DIR / "config.json").read_text())
default_theta = float(cfg.get("threshold", 0.5))

assert "silo_data" in globals(), "Run Step 7 first (build silo_data)."
d = int(GLOBAL_D); K = int(GLOBAL_K)

# Load trained model
model = MLPIDS(input_dim=d).to(DEVICE)
model.load_state_dict(torch.load(RUN_DIR / "global_model_final.pt", map_location=DEVICE))
model.eval()

# Calibration settings
TARGET_FPR = 0.05
THETA_MIN = 0.05
THETA_MAX = 0.95
MIN_BENIGN_VAL = 200          # if fewer benign val samples, fallback
MAX_PRED_POS = 0.90           # sanity upper bound
MIN_PRED_POS = 0.01           # sanity lower bound

@torch.no_grad()
def collect_probs(loader):
    probs_list, y_list = [], []
    for xb, yb in loader:
        xb = xb.to(DEVICE)
        probs_list.append(torch.sigmoid(model(xb)).cpu().numpy())
        y_list.append(yb.cpu().numpy())
    return np.concatenate(probs_list).astype(np.float64), np.concatenate(y_list).astype(int)

def metrics_at_threshold(probs, y_true, thr):
    y_pred = (probs >= thr).astype(int)
    tp = int(((y_pred==1)&(y_true==1)).sum())
    tn = int(((y_pred==0)&(y_true==0)).sum())
    fp = int(((y_pred==1)&(y_true==0)).sum())
    fn = int(((y_pred==0)&(y_true==1)).sum())
    precision = tp / (tp + fp + 1e-12)
    recall    = tp / (tp + fn + 1e-12)
    f1        = 2*precision*recall / (precision + recall + 1e-12)
    acc       = (tp + tn) / max(1, tp+tn+fp+fn)
    fpr       = fp / (fp + tn + 1e-12)
    return {
        "acc": float(acc), "precision": float(precision), "recall": float(recall), "f1": float(f1),
        "fpr": float(fpr), "pred_pos_rate": float(y_pred.mean()),
        "tp": tp, "tn": tn, "fp": fp, "fn": fn
    }

def robust_theta_from_fpr(probs_val, y_val, target_fpr):
    benign = probs_val[y_val == 0]
    if benign.size < MIN_BENIGN_VAL:
        return default_theta, f"fallback_default (benign_val={benign.size})"
    thr = float(np.quantile(benign, 1.0 - target_fpr))
    thr = float(np.clip(thr, THETA_MIN, THETA_MAX))
    return thr, "quantile_benign"

# Calibrate per silo with sanity checks
thetas = []
notes = []
per_silo_test = []

for k in range(K):
    probs_val, y_val = collect_probs(silo_data[k]["val_loader"])
    theta_k, note = robust_theta_from_fpr(probs_val, y_val, TARGET_FPR)

    # sanity check on val predicted positive rate
    m_val = metrics_at_threshold(probs_val, y_val, theta_k)
    if m_val["pred_pos_rate"] > MAX_PRED_POS:
        theta_k = min(THETA_MAX, max(theta_k, 0.80))
        note += " | adjusted_high_pred_pos"
    elif m_val["pred_pos_rate"] < MIN_PRED_POS:
        theta_k = max(THETA_MIN, min(theta_k, 0.20))
        note += " | adjusted_low_pred_pos"

    thetas.append(theta_k)
    notes.append(note)

    probs_test, y_test = collect_probs(silo_data[k]["test_loader"])
    per_silo_test.append(metrics_at_threshold(probs_test, y_test, theta_k))

# Global union
tp = sum(m["tp"] for m in per_silo_test)
tn = sum(m["tn"] for m in per_silo_test)
fp = sum(m["fp"] for m in per_silo_test)
fn = sum(m["fn"] for m in per_silo_test)
precision = tp / (tp + fp + 1e-12)
recall    = tp / (tp + fn + 1e-12)
f1        = 2*precision*recall / (precision + recall + 1e-12)
acc       = (tp + tn) / max(1, tp+tn+fp+fn)
fpr       = fp / (fp + tn + 1e-12)

global_metrics = {
    "acc": float(acc), "precision": float(precision), "recall": float(recall),
    "f1": float(f1), "fpr": float(fpr),
    "tp": int(tp), "tn": int(tn), "fp": int(fp), "fn": int(fn),
}

print("\n📌 Global TEST with robust per-silo theta_k:", global_metrics)

# Per-silo summary table
import pandas as pd
df = pd.DataFrame([{
    "silo": k,
    "theta_k": thetas[k],
    "note": notes[k],
    "test_f1": per_silo_test[k]["f1"],
    "test_precision": per_silo_test[k]["precision"],
    "test_recall": per_silo_test[k]["recall"],
    "test_fpr": per_silo_test[k]["fpr"],
    "pred_pos_rate": per_silo_test[k]["pred_pos_rate"],
    "n_test": silo_data[k]["n_test"],
} for k in range(K)])
display(df)

# Save artifact
out = RUN_DIR / "theta_calibration_fpr_robust.json"
out.write_text(json.dumps({
    "target_fpr": TARGET_FPR,
    "theta_min": THETA_MIN,
    "theta_max": THETA_MAX,
    "min_benign_val": MIN_BENIGN_VAL,
    "thetas": thetas,
    "notes": notes,
    "global_test": global_metrics,
    "per_silo_test": per_silo_test,
}, indent=2))
print("✅ Saved:", out)


✅ Using RUN_DIR: runs/run_edgeiiotset_20261001_093908

📌 Global TEST with robust per-silo theta_k: {'acc': 0.8214636053578722, 'precision': 0.8256627783669142, 'recall': 0.47560871318094633, 'f1': 0.6035518553827444, 'fpr': 0.04017516764090633, 'tp': 38930, 'tn': 196384, 'fp': 8220, 'fn': 42923}


,silo,theta_k,note,test_f1,test_precision,test_recall,test_fpr,pred_pos_rate,n_test
0,0,0.050000,quantile_benign,0.999377,1.000000,0.998754,0.000000,0.480804,6668
1,1,0.800000,quantile_benign | adjusted_high_pred_pos,0.000000,0.000000,0.000000,0.000000,0.000000,10216
2,2,0.051004,quantile_benign,0.000000,0.000000,0.000000,0.049180,0.049180,3965
3,3,0.050000,quantile_benign,0.999787,1.000000,0.999573,0.000000,0.723698,12946
4,4,0.051278,quantile_benign,0.000000,0.000000,0.000000,0.049906,0.049906,132008
5,5,0.050000,quantile_benign,0.999555,1.000000,0.999111,0.000000,0.203257,16580
6,6,0.800000,quantile_benign | adjusted_high_pred_pos,0.008907,1.000000,0.004474,0.000000,0.004279,28042
7,7,0.050000,quantile_benign,0.999890,1.000000,0.999780,0.000000,0.678025,33515
8,8,0.050000,quantile_benign,0.004158,0.002083,1.000000,0.039888,0.039968,36029
9,9,0.500000,fallback_default (benign_val=12),0.041730,1.000000,0.021309,0.000000,0.021270,6488


✅ Saved: runs/run_edgeiiotset_20261001_093908/theta_calibration_fpr_robust.json


**Next step — Step 11: FedAvg vs TrimMean𝛽 on real Edge-IIoTset + robust 𝜃𝑘 evaluation**

In [14]:
# Step 11: Controlled FedAvg vs TrimMean(beta) comparison on REAL Edge-IIoTset
#          + robust per-silo theta_k evaluation (FPR-target with guardrails)

import time, copy, json
import numpy as np
import torch
from pathlib import Path

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert "silo_data" in globals(), "Run Step 7 first."
d = int(GLOBAL_D); K = int(GLOBAL_K)

# -----------------------------
# Shared config for both runs
# -----------------------------
BASE = {
    "seed": 42,
    "alpha": 0.3,
    "rounds": 10,
    "dropout_q": 0.2,
    "participation_min": 6,   # IMPORTANT: ensure TrimMean has enough participants

    "local_epochs": 2,
    "fedprox_mu": 1e-3,
    "lr": 3e-4,
    "weight_decay": 1e-4,
    "grad_clip": 1.0,

    "server_gamma": 1.0,

    "clip_C": 1.0,
    "dp_sigma": 0.0,

    "threshold": 0.5,
}

TRIM_BETA = 0.10  # you can tune later

# -----------------------------
# Local training (FedProx)
# -----------------------------
def get_param_vector(model):
    return torch.cat([p.view(-1) for p in model.parameters()])

def fedprox_penalty(model, w0_vec):
    w_vec = get_param_vector(model)
    return 0.5 * torch.sum((w_vec - w0_vec) ** 2)

def train_one_client_fedprox(global_model, train_loader, cfg, seed_offset=0):
    torch.manual_seed(cfg["seed"] + seed_offset)
    model = copy.deepcopy(global_model).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])

    with torch.no_grad():
        w0 = get_param_vector(model).detach()

    model.train()
    for _ in range(cfg["local_epochs"]):
        for xb, yb in train_loader:
            xb = xb.to(DEVICE)
            yb = yb.float().to(DEVICE)

            opt.zero_grad()
            logits = model(xb)
            base = criterion(logits, yb)
            prox = cfg["fedprox_mu"] * fedprox_penalty(model, w0)
            loss = base + prox
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["grad_clip"])
            opt.step()
    return model

# -----------------------------
# Delta/protection/aggregation
# -----------------------------
def state_dict_to_vector(sd):
    return torch.cat([v.detach().reshape(-1).float().cpu() for v in sd.values() if torch.is_tensor(v)])

def vector_to_state_dict_like(vec, ref_sd):
    out, off = {}, 0
    for k, v in ref_sd.items():
        if torch.is_tensor(v):
            n = v.numel()
            out[k] = vec[off:off+n].view_as(v).to(v.dtype)
            off += n
        else:
            out[k] = v
    return out

def compute_delta(global_model, local_model):
    g_sd = global_model.state_dict()
    l_sd = local_model.state_dict()
    return state_dict_to_vector(l_sd) - state_dict_to_vector(g_sd), g_sd

def clip_and_noise(delta_vec, C=1.0, sigma=0.0, seed=42):
    delta = delta_vec.clone()
    norm = torch.linalg.norm(delta, ord=2)
    scale = max(1.0, float(norm.item()) / float(C))
    delta = delta / scale
    if sigma and sigma > 0.0:
        gen = torch.Generator(device="cpu").manual_seed(int(seed))
        delta = delta + torch.normal(0.0, float(sigma), size=delta.shape, generator=gen)
    return delta

def agg_fedavg(deltas, weights):
    W = float(np.sum(weights))
    out = torch.zeros_like(deltas[0])
    for d, w in zip(deltas, weights):
        out += d * (float(w) / W)
    return out

def agg_trimmed_mean(deltas, beta=0.10):
    m = len(deltas)
    X = torch.stack(deltas, dim=0)  # [m,P]
    trim = int(np.floor(beta * m))
    if 2 * trim >= m:
        # Not enough participants to trim safely
        return torch.mean(X, dim=0)
    Xs, _ = torch.sort(X, dim=0)
    Xk = Xs[trim:m-trim, :]
    return torch.mean(Xk, dim=0)

def apply_delta_to_model(global_model, delta_vec, ref_sd, gamma=1.0):
    g_sd = global_model.state_dict()
    delta_sd = vector_to_state_dict_like(delta_vec, ref_sd)
    new_sd = {}
    for k, v in g_sd.items():
        if torch.is_tensor(v):
            new_sd[k] = v + gamma * delta_sd[k].to(v.device)
        else:
            new_sd[k] = v
    global_model.load_state_dict(new_sd)
    return global_model

# -----------------------------
# Robust theta evaluation (same logic as Step 10.1)
# -----------------------------
TARGET_FPR = 0.05
THETA_MIN = 0.05
THETA_MAX = 0.95
MIN_BENIGN_VAL = 200
MAX_PRED_POS = 0.90
MIN_PRED_POS = 0.01

@torch.no_grad()
def collect_probs(model, loader):
    probs_list, y_list = [], []
    for xb, yb in loader:
        xb = xb.to(DEVICE)
        probs_list.append(torch.sigmoid(model(xb)).cpu().numpy())
        y_list.append(yb.cpu().numpy())
    return np.concatenate(probs_list).astype(np.float64), np.concatenate(y_list).astype(int)

def metrics_at_threshold(probs, y_true, thr):
    y_pred = (probs >= thr).astype(int)
    tp = int(((y_pred==1)&(y_true==1)).sum())
    tn = int(((y_pred==0)&(y_true==0)).sum())
    fp = int(((y_pred==1)&(y_true==0)).sum())
    fn = int(((y_pred==0)&(y_true==1)).sum())
    precision = tp / (tp + fp + 1e-12)
    recall    = tp / (tp + fn + 1e-12)
    f1        = 2*precision*recall / (precision + recall + 1e-12)
    acc       = (tp + tn) / max(1, tp+tn+fp+fn)
    fpr       = fp / (fp + tn + 1e-12)
    return {"acc":acc,"precision":precision,"recall":recall,"f1":f1,"fpr":fpr,
            "tp":tp,"tn":tn,"fp":fp,"fn":fn,"pred_pos_rate":float(y_pred.mean())}

def robust_theta_from_fpr(probs_val, y_val, default_theta):
    benign = probs_val[y_val == 0]
    if benign.size < MIN_BENIGN_VAL:
        return default_theta, f"fallback_default (benign_val={benign.size})"
    thr = float(np.quantile(benign, 1.0 - TARGET_FPR))
    thr = float(np.clip(thr, THETA_MIN, THETA_MAX))
    return thr, "quantile_benign"

def eval_with_robust_thetas(model, default_theta):
    thetas, notes, per_silo = [], [], []
    for k in range(K):
        probs_val, y_val = collect_probs(model, silo_data[k]["val_loader"])
        theta_k, note = robust_theta_from_fpr(probs_val, y_val, default_theta)
        m_val = metrics_at_threshold(probs_val, y_val, theta_k)
        if m_val["pred_pos_rate"] > MAX_PRED_POS:
            theta_k = min(THETA_MAX, max(theta_k, 0.80))
            note += " | adjusted_high_pred_pos"
        elif m_val["pred_pos_rate"] < MIN_PRED_POS:
            theta_k = max(THETA_MIN, min(theta_k, 0.20))
            note += " | adjusted_low_pred_pos"

        probs_test, y_test = collect_probs(model, silo_data[k]["test_loader"])
        m_test = metrics_at_threshold(probs_test, y_test, theta_k)

        thetas.append(theta_k); notes.append(note); per_silo.append(m_test)

    tp = sum(m["tp"] for m in per_silo)
    tn = sum(m["tn"] for m in per_silo)
    fp = sum(m["fp"] for m in per_silo)
    fn = sum(m["fn"] for m in per_silo)
    precision = tp / (tp + fp + 1e-12)
    recall    = tp / (tp + fn + 1e-12)
    f1        = 2*precision*recall / (precision + recall + 1e-12)
    acc       = (tp + tn) / max(1, tp+tn+fp+fn)
    fpr       = fp / (fp + tn + 1e-12)

    global_m = {"acc":float(acc), "precision":float(precision), "recall":float(recall),
                "f1":float(f1), "fpr":float(fpr), "tp":int(tp), "tn":int(tn), "fp":int(fp), "fn":int(fn)}
    return global_m, thetas, notes

# -----------------------------
# Simulator (runs one aggregator)
# -----------------------------
def run_fl(aggregator="fedavg"):
    cfg = dict(BASE)
    cfg["aggregator"] = aggregator
    cfg["trim_beta"] = TRIM_BETA

    rng = np.random.default_rng(cfg["seed"])
    model = MLPIDS(input_dim=d).to(DEVICE)

    run_id = time.strftime(f"run_real_cmp_{aggregator}_%Y%m%d_%H%M%S", time.gmtime())
    RUN_DIR = Path("runs") / run_id
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    (RUN_DIR / "config.json").write_text(json.dumps(cfg, indent=2))

    for t in range(cfg["rounds"]):
        participants = [k for k in range(K) if rng.uniform() > cfg["dropout_q"]]
        if len(participants) < cfg["participation_min"]:
            participants = list(rng.choice(K, size=cfg["participation_min"], replace=False))

        deltas, weights = [], []
        ref_sd = None
        for k in participants:
            local_model = train_one_client_fedprox(model, silo_data[k]["train_loader"], cfg, seed_offset=1000*t + k)
            delta_vec, ref_sd = compute_delta(model, local_model)
            protected = clip_and_noise(delta_vec, C=cfg["clip_C"], sigma=cfg["dp_sigma"], seed=cfg["seed"] + 1000*t + k)
            deltas.append(protected)
            weights.append(silo_data[k]["n_train"])

        if aggregator == "fedavg":
            agg_delta = agg_fedavg(deltas, weights)
        else:
            agg_delta = agg_trimmed_mean(deltas, beta=cfg["trim_beta"])

        model = apply_delta_to_model(model, agg_delta, ref_sd, gamma=cfg["server_gamma"])
        print(f"{aggregator} | round {t} done | participants={len(participants)}")

    # Evaluate with robust theta_k
    global_m, thetas, notes = eval_with_robust_thetas(model, default_theta=cfg["threshold"])

    (RUN_DIR / "global_test_robust_theta.json").write_text(json.dumps(global_m, indent=2))
    torch.save(model.state_dict(), RUN_DIR / "global_model_final.pt")

    return RUN_DIR, global_m

# -----------------------------
# Run both and compare
# -----------------------------
fed_dir, fed_m = run_fl("fedavg")
trim_dir, trim_m = run_fl("trimmean")

print("\n✅ Done.")
print("FedAvg run   :", fed_dir)
print("TrimMean run :", trim_dir)
print("\nFinal GLOBAL TEST (robust theta_k):")
print("FedAvg  :", fed_m)
print("TrimMean:", trim_m)


fedavg | round 0 done | participants=8
fedavg | round 1 done | participants=9
fedavg | round 2 done | participants=7
fedavg | round 3 done | participants=8
fedavg | round 4 done | participants=9
fedavg | round 5 done | participants=6
fedavg | round 6 done | participants=9
fedavg | round 7 done | participants=9
fedavg | round 8 done | participants=6
fedavg | round 9 done | participants=7
trimmean | round 0 done | participants=8
trimmean | round 1 done | participants=9
trimmean | round 2 done | participants=7
trimmean | round 3 done | participants=8
trimmean | round 4 done | participants=9
trimmean | round 5 done | participants=6
trimmean | round 6 done | participants=9
trimmean | round 7 done | participants=9
trimmean | round 8 done | participants=6
trimmean | round 9 done | participants=7

✅ Done.
FedAvg run   : runs/run_real_cmp_fedavg_20261001_095417
TrimMean run : runs/run_real_cmp_trimmean_20261001_100743

Final GLOBAL TEST (robust theta_k):
FedAvg  : {'acc': 0.9401341213515466, 'p

**Next step — Step 12: Add FL-stage adversary simulation (poisoning / backdoor-ready scaffold)**

In [15]:
# Step 12: Adversarial FL simulation (poisoning via sign-flip) + compare FedAvg vs TrimMean

import time, copy, json
import numpy as np
import torch
from pathlib import Path

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert "silo_data" in globals(), "Run Step 7 first."
d = int(GLOBAL_D); K = int(GLOBAL_K)

# -----------------------------
# Config
# -----------------------------
CFG = {
    "seed": 42,
    "rounds": 10,
    "dropout_q": 0.2,
    "participation_min": 6,   # keep >=6 so TrimMean has enough participants

    "local_epochs": 2,
    "fedprox_mu": 1e-3,
    "lr": 3e-4,
    "weight_decay": 1e-4,
    "grad_clip": 1.0,

    "server_gamma": 1.0,
    "clip_C": 1.0,
    "dp_sigma": 0.0,

    "threshold": 0.5,

    # adversary
    "malicious_frac": 0.2,    # alpha in your threat model (e.g., 0.1, 0.2, 0.3)
    "poison_scale": 5.0,      # scale for flipped update (try 1.0, 5.0, 10.0)
}

TRIM_BETA = 0.10

# -----------------------------
# Local training (FedProx)
# -----------------------------
def get_param_vector(model):
    return torch.cat([p.view(-1) for p in model.parameters()])

def fedprox_penalty(model, w0_vec):
    w_vec = get_param_vector(model)
    return 0.5 * torch.sum((w_vec - w0_vec) ** 2)

def train_one_client_fedprox(global_model, train_loader, cfg, seed_offset=0):
    torch.manual_seed(cfg["seed"] + seed_offset)
    model = copy.deepcopy(global_model).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])

    with torch.no_grad():
        w0 = get_param_vector(model).detach()

    model.train()
    for _ in range(cfg["local_epochs"]):
        for xb, yb in train_loader:
            xb = xb.to(DEVICE)
            yb = yb.float().to(DEVICE)
            opt.zero_grad()
            logits = model(xb)
            base = criterion(logits, yb)
            prox = cfg["fedprox_mu"] * fedprox_penalty(model, w0)
            loss = base + prox
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["grad_clip"])
            opt.step()
    return model

# -----------------------------
# Delta/protection/aggregation
# -----------------------------
def state_dict_to_vector(sd):
    return torch.cat([v.detach().reshape(-1).float().cpu() for v in sd.values() if torch.is_tensor(v)])

def vector_to_state_dict_like(vec, ref_sd):
    out, off = {}, 0
    for k, v in ref_sd.items():
        if torch.is_tensor(v):
            n = v.numel()
            out[k] = vec[off:off+n].view_as(v).to(v.dtype)
            off += n
        else:
            out[k] = v
    return out

def compute_delta(global_model, local_model):
    g_sd = global_model.state_dict()
    l_sd = local_model.state_dict()
    return state_dict_to_vector(l_sd) - state_dict_to_vector(g_sd), g_sd

def clip_and_noise(delta_vec, C=1.0, sigma=0.0, seed=42):
    delta = delta_vec.clone()
    norm = torch.linalg.norm(delta, ord=2)
    scale = max(1.0, float(norm.item()) / float(C))
    delta = delta / scale
    if sigma and sigma > 0.0:
        gen = torch.Generator(device="cpu").manual_seed(int(seed))
        delta = delta + torch.normal(0.0, float(sigma), size=delta.shape, generator=gen)
    return delta

def agg_fedavg(deltas, weights):
    W = float(np.sum(weights))
    out = torch.zeros_like(deltas[0])
    for dlt, w in zip(deltas, weights):
        out += dlt * (float(w) / W)
    return out

def agg_trimmed_mean(deltas, beta=0.10):
    m = len(deltas)
    X = torch.stack(deltas, dim=0)
    trim = int(np.floor(beta * m))
    if 2 * trim >= m:
        return torch.mean(X, dim=0)
    Xs, _ = torch.sort(X, dim=0)
    Xk = Xs[trim:m-trim, :]
    return torch.mean(Xk, dim=0)

def apply_delta_to_model(global_model, delta_vec, ref_sd, gamma=1.0):
    g_sd = global_model.state_dict()
    delta_sd = vector_to_state_dict_like(delta_vec, ref_sd)
    new_sd = {}
    for k, v in g_sd.items():
        if torch.is_tensor(v):
            new_sd[k] = v + gamma * delta_sd[k].to(v.device)
        else:
            new_sd[k] = v
    global_model.load_state_dict(new_sd)
    return global_model

# -----------------------------
# Robust theta eval (reuse from Step 11)
# -----------------------------
TARGET_FPR = 0.05
THETA_MIN = 0.05
THETA_MAX = 0.95
MIN_BENIGN_VAL = 200
MAX_PRED_POS = 0.90
MIN_PRED_POS = 0.01

@torch.no_grad()
def collect_probs(model, loader):
    probs_list, y_list = [], []
    for xb, yb in loader:
        xb = xb.to(DEVICE)
        probs_list.append(torch.sigmoid(model(xb)).cpu().numpy())
        y_list.append(yb.cpu().numpy())
    return np.concatenate(probs_list).astype(np.float64), np.concatenate(y_list).astype(int)

def metrics_at_threshold(probs, y_true, thr):
    y_pred = (probs >= thr).astype(int)
    tp = int(((y_pred==1)&(y_true==1)).sum())
    tn = int(((y_pred==0)&(y_true==0)).sum())
    fp = int(((y_pred==1)&(y_true==0)).sum())
    fn = int(((y_pred==0)&(y_true==1)).sum())
    precision = tp / (tp + fp + 1e-12)
    recall    = tp / (tp + fn + 1e-12)
    f1        = 2*precision*recall / (precision + recall + 1e-12)
    acc       = (tp + tn) / max(1, tp+tn+fp+fn)
    fpr       = fp / (fp + tn + 1e-12)
    return {"acc":float(acc),"precision":float(precision),"recall":float(recall),"f1":float(f1),"fpr":float(fpr),
            "tp":tp,"tn":tn,"fp":fp,"fn":fn,"pred_pos_rate":float(y_pred.mean())}

def robust_theta_from_fpr(probs_val, y_val, default_theta):
    benign = probs_val[y_val == 0]
    if benign.size < MIN_BENIGN_VAL:
        return default_theta, f"fallback_default (benign_val={benign.size})"
    thr = float(np.quantile(benign, 1.0 - TARGET_FPR))
    thr = float(np.clip(thr, THETA_MIN, THETA_MAX))
    return thr, "quantile_benign"

def eval_with_robust_thetas(model, default_theta):
    per_silo = []
    for k in range(K):
        probs_val, y_val = collect_probs(model, silo_data[k]["val_loader"])
        theta_k, note = robust_theta_from_fpr(probs_val, y_val, default_theta)
        m_val = metrics_at_threshold(probs_val, y_val, theta_k)
        if m_val["pred_pos_rate"] > MAX_PRED_POS:
            theta_k = min(THETA_MAX, max(theta_k, 0.80))
        elif m_val["pred_pos_rate"] < MIN_PRED_POS:
            theta_k = max(THETA_MIN, min(theta_k, 0.20))

        probs_test, y_test = collect_probs(model, silo_data[k]["test_loader"])
        per_silo.append(metrics_at_threshold(probs_test, y_test, theta_k))

    tp = sum(m["tp"] for m in per_silo); tn = sum(m["tn"] for m in per_silo)
    fp = sum(m["fp"] for m in per_silo); fn = sum(m["fn"] for m in per_silo)
    precision = tp / (tp + fp + 1e-12)
    recall    = tp / (tp + fn + 1e-12)
    f1        = 2*precision*recall / (precision + recall + 1e-12)
    acc       = (tp + tn) / max(1, tp+tn+fp+fn)
    fpr       = fp / (fp + tn + 1e-12)
    return {"acc":acc,"precision":precision,"recall":recall,"f1":f1,"fpr":fpr,"tp":tp,"tn":tn,"fp":fp,"fn":fn}

# -----------------------------
# Run FL with attack
# -----------------------------
def run_fl(aggregator="fedavg"):
    cfg = dict(CFG)
    cfg["aggregator"] = aggregator
    cfg["trim_beta"] = TRIM_BETA

    rng = np.random.default_rng(cfg["seed"])
    model = MLPIDS(input_dim=d).to(DEVICE)

    for t in range(cfg["rounds"]):
        participants = [k for k in range(K) if rng.uniform() > cfg["dropout_q"]]
        if len(participants) < cfg["participation_min"]:
            participants = list(rng.choice(K, size=cfg["participation_min"], replace=False))

        # choose malicious subset among participants
        m_count = int(np.floor(cfg["malicious_frac"] * len(participants)))
        malicious = set(rng.choice(participants, size=m_count, replace=False).tolist()) if m_count > 0 else set()

        deltas, weights = [], []
        ref_sd = None

        for k in participants:
            local_model = train_one_client_fedprox(model, silo_data[k]["train_loader"], cfg, seed_offset=1000*t + k)
            delta_vec, ref_sd = compute_delta(model, local_model)
            protected = clip_and_noise(delta_vec, C=cfg["clip_C"], sigma=cfg["dp_sigma"], seed=cfg["seed"] + 1000*t + k)

            # poisoning: sign-flip + scale
            if k in malicious:
                protected = -cfg["poison_scale"] * protected

            deltas.append(protected)
            weights.append(silo_data[k]["n_train"])

        if aggregator == "fedavg":
            agg_delta = agg_fedavg(deltas, weights)
        else:
            agg_delta = agg_trimmed_mean(deltas, beta=cfg["trim_beta"])

        model = apply_delta_to_model(model, agg_delta, ref_sd, gamma=cfg["server_gamma"])
        print(f"{aggregator} | round {t} done | participants={len(participants)} | malicious={len(malicious)}")

    return eval_with_robust_thetas(model, default_theta=cfg["threshold"])

fed_m = run_fl("fedavg")
trim_m = run_fl("trimmean")

print("\n✅ Final GLOBAL TEST under poisoning (robust theta_k):")
print("FedAvg  :", {k: float(v) if isinstance(v, (np.floating,float)) else int(v) for k,v in fed_m.items()})
print("TrimMean:", {k: float(v) if isinstance(v, (np.floating,float)) else int(v) for k,v in trim_m.items()})


fedavg | round 0 done | participants=8 | malicious=1
fedavg | round 1 done | participants=9 | malicious=1
fedavg | round 2 done | participants=7 | malicious=1
fedavg | round 3 done | participants=8 | malicious=1
fedavg | round 4 done | participants=7 | malicious=1
fedavg | round 5 done | participants=8 | malicious=1
fedavg | round 6 done | participants=9 | malicious=1
fedavg | round 7 done | participants=8 | malicious=1
fedavg | round 8 done | participants=6 | malicious=1
fedavg | round 9 done | participants=8 | malicious=1
trimmean | round 0 done | participants=8 | malicious=1
trimmean | round 1 done | participants=9 | malicious=1
trimmean | round 2 done | participants=7 | malicious=1
trimmean | round 3 done | participants=8 | malicious=1
trimmean | round 4 done | participants=7 | malicious=1
trimmean | round 5 done | participants=8 | malicious=1
trimmean | round 6 done | participants=9 | malicious=1
trimmean | round 7 done | participants=8 | malicious=1
trimmean | round 8 done | part

**Next step — Step 13: Robustness sweep over malicious fraction
𝛼 and poison strength**

In [16]:
# Step 13: Robustness sweep (malicious_frac x poison_scale) for FedAvg vs TrimMean

import copy
import numpy as np
import pandas as pd
import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert "silo_data" in globals(), "Run Step 7 first."
d = int(GLOBAL_D); K = int(GLOBAL_K)

# Base training config (keep same as Step 12 for apples-to-apples)
BASE = {
    "seed": 42,
    "rounds": 10,
    "dropout_q": 0.2,
    "participation_min": 6,

    "local_epochs": 2,
    "fedprox_mu": 1e-3,
    "lr": 3e-4,
    "weight_decay": 1e-4,
    "grad_clip": 1.0,

    "server_gamma": 1.0,
    "clip_C": 1.0,
    "dp_sigma": 0.0,

    "threshold": 0.5,
}
TRIM_BETA = 0.10

# Reuse helpers from Step 12 if present; otherwise fail loudly
needed = [
    "train_one_client_fedprox", "compute_delta", "clip_and_noise",
    "agg_fedavg", "agg_trimmed_mean", "apply_delta_to_model",
    "eval_with_robust_thetas"
]
for fn in needed:
    assert fn in globals(), f"Missing helper '{fn}'. Re-run Step 12 cell first."

def run_poisoned(aggregator, malicious_frac, poison_scale):
    cfg = dict(BASE)
    cfg["malicious_frac"] = float(malicious_frac)
    cfg["poison_scale"] = float(poison_scale)

    rng = np.random.default_rng(cfg["seed"])
    model = MLPIDS(input_dim=d).to(DEVICE)

    for t in range(cfg["rounds"]):
        participants = [k for k in range(K) if rng.uniform() > cfg["dropout_q"]]
        if len(participants) < cfg["participation_min"]:
            participants = list(rng.choice(K, size=cfg["participation_min"], replace=False))

        m_count = int(np.floor(cfg["malicious_frac"] * len(participants)))
        malicious = set(rng.choice(participants, size=m_count, replace=False).tolist()) if m_count > 0 else set()

        deltas, weights = [], []
        ref_sd = None

        for k in participants:
            local_model = train_one_client_fedprox(model, silo_data[k]["train_loader"], cfg, seed_offset=1000*t + k)
            delta_vec, ref_sd = compute_delta(model, local_model)
            protected = clip_and_noise(delta_vec, C=cfg["clip_C"], sigma=cfg["dp_sigma"], seed=cfg["seed"] + 1000*t + k)
            if k in malicious:
                protected = -cfg["poison_scale"] * protected
            deltas.append(protected)
            weights.append(silo_data[k]["n_train"])

        if aggregator == "fedavg":
            agg_delta = agg_fedavg(deltas, weights)
        else:
            agg_delta = agg_trimmed_mean(deltas, beta=TRIM_BETA)

        model = apply_delta_to_model(model, agg_delta, ref_sd, gamma=cfg["server_gamma"])

    metrics = eval_with_robust_thetas(model, default_theta=cfg["threshold"])
    return metrics

alpha_grid = [0.0, 0.1, 0.2, 0.3]
scale_grid = [1.0, 5.0, 10.0]

rows = []
for a in alpha_grid:
    for s in scale_grid:
        for agg in ["fedavg", "trimmean"]:
            m = run_poisoned(agg, malicious_frac=a, poison_scale=s)
            rows.append({
                "malicious_frac": a,
                "poison_scale": s,
                "aggregator": agg,
                "acc": float(m["acc"]),
                "precision": float(m["precision"]),
                "recall": float(m["recall"]),
                "f1": float(m["f1"]),
                "fpr": float(m["fpr"]),
                "tp": int(m["tp"]), "tn": int(m["tn"]), "fp": int(m["fp"]), "fn": int(m["fn"]),
            })
        print(f"✅ Done: alpha={a} scale={s}")

df = pd.DataFrame(rows).sort_values(["malicious_frac", "poison_scale", "aggregator"])
display(df)

# Save sweep results
out_path = Path("runs") / f"poison_sweep_{time.strftime('%Y%m%d_%H%M%S', time.gmtime())}.csv"
df.to_csv(out_path, index=False)
print("✅ Saved sweep table:", out_path)


✅ Done: alpha=0.0 scale=1.0
✅ Done: alpha=0.0 scale=5.0
✅ Done: alpha=0.0 scale=10.0
✅ Done: alpha=0.1 scale=1.0
✅ Done: alpha=0.1 scale=5.0
✅ Done: alpha=0.1 scale=10.0
✅ Done: alpha=0.2 scale=1.0
✅ Done: alpha=0.2 scale=5.0
✅ Done: alpha=0.2 scale=10.0
✅ Done: alpha=0.3 scale=1.0
✅ Done: alpha=0.3 scale=5.0
✅ Done: alpha=0.3 scale=10.0


,malicious_frac,poison_scale,aggregator,acc,precision,recall,f1,fpr,tp,tn,fp,fn
0,0.0,1.0,fedavg,0.934276,0.892119,0.875912,0.883941,0.042375,71696,195934,8670,10157
1,0.0,1.0,trimmean,0.919325,0.781064,0.997178,0.875989,0.111821,81622,181725,22879,231
2,0.0,5.0,fedavg,0.934276,0.892119,0.875912,0.883941,0.042375,71696,195934,8670,10157
3,0.0,5.0,trimmean,0.919325,0.781064,0.997178,0.875989,0.111821,81622,181725,22879,231
4,0.0,10.0,fedavg,0.934276,0.892119,0.875912,0.883941,0.042375,71696,195934,8670,10157
5,0.0,10.0,trimmean,0.919325,0.781064,0.997178,0.875989,0.111821,81622,181725,22879,231
6,0.1,1.0,fedavg,0.934276,0.892119,0.875912,0.883941,0.042375,71696,195934,8670,10157
7,0.1,1.0,trimmean,0.919325,0.781064,0.997178,0.875989,0.111821,81622,181725,22879,231
8,0.1,5.0,fedavg,0.934276,0.892119,0.875912,0.883941,0.042375,71696,195934,8670,10157
9,0.1,5.0,trimmean,0.919325,0.781064,0.997178,0.875989,0.111821,81622,181725,22879,231


✅ Saved sweep table: runs/poison_sweep_20261001_161631.csv


**Next step — Step 14: Create a clean repo structure + write reusable modules + single entrypoint script**

In [17]:
# Step 14 (FIXED): Generate clean repo code (src/, scripts/, configs/, README) safely

from pathlib import Path
import json

ROOT = Path(".").resolve()

# Create folders
for p in [
    ROOT / "src",
    ROOT / "src" / "data",
    ROOT / "src" / "models",
    ROOT / "src" / "fl",
    ROOT / "src" / "eval",
    ROOT / "scripts",
    ROOT / "configs",
]:
    p.mkdir(parents=True, exist_ok=True)

# Add __init__.py files
for p in [
    ROOT / "src" / "__init__.py",
    ROOT / "src" / "data" / "__init__.py",
    ROOT / "src" / "models" / "__init__.py",
    ROOT / "src" / "fl" / "__init__.py",
    ROOT / "src" / "eval" / "__init__.py",
]:
    p.write_text("")

# -------------------------
# src/models/mlp.py
# -------------------------
mlp_py = "\n".join([
"import torch",
"import torch.nn as nn",
"",
"class MLPIDS(nn.Module):",
"    \"\"\"Lightweight MLP IDS for tabular flow/session features. Outputs logits.\"\"\"",
"",
"    def __init__(self, input_dim: int, hidden=(256,128,64), dropout=0.30, batch_norm=False):",
"        super().__init__()",
"        layers = []",
"        in_dim = input_dim",
"        for h in hidden:",
"            layers.append(nn.Linear(in_dim, h))",
"            if batch_norm:",
"                layers.append(nn.BatchNorm1d(h))",
"            layers.append(nn.ReLU())",
"            layers.append(nn.Dropout(dropout))",
"            in_dim = h",
"        self.net = nn.Sequential(*layers)",
"        self.out = nn.Linear(in_dim, 1)",
"",
"    def forward(self, x):",
"        return self.out(self.net(x)).squeeze(-1)",
"",
])
(ROOT / "src/models/mlp.py").write_text(mlp_py)

# -------------------------
# src/data/preprocess_edgeiiotset.py
# -------------------------
preprocess_py = "\n".join([
"import json",
"from pathlib import Path",
"import numpy as np",
"import pandas as pd",
"",
"DROP_COLUMNS_DEFAULT = [",
"    \"frame.time\", \"ip.src_host\", \"ip.dst_host\", \"arp.src.proto_ipv4\", \"arp.dst.proto_ipv4\",",
"    \"http.file_data\", \"http.request.full_uri\", \"icmp.transmit_timestamp\",",
"    \"http.request.uri.query\", \"tcp.options\", \"tcp.payload\", \"tcp.srcport\",",
"    \"tcp.dstport\", \"udp.port\", \"mqtt.msg\"",
"]",
"",
"PREFERRED_CAT_COLS = [",
"    \"http.request.method\", \"http.referer\", \"http.request.version\", \"dns.qry.name.len\",",
"    \"mqtt.conack.flags\", \"mqtt.protoname\", \"mqtt.topic\"",
"]",
"",
"def preprocess_edgeiiotset(csv_path: str, out_npz: str, out_meta: str, drop_columns=None):",
"    csv_path = Path(csv_path)",
"    out_npz = Path(out_npz); out_npz.parent.mkdir(parents=True, exist_ok=True)",
"    out_meta = Path(out_meta); out_meta.parent.mkdir(parents=True, exist_ok=True)",
"",
"    df = pd.read_csv(csv_path, low_memory=False)",
"",
"    drop_columns = DROP_COLUMNS_DEFAULT if drop_columns is None else drop_columns",
"    existing = [c for c in drop_columns if c in df.columns]",
"    df.drop(existing, axis=1, inplace=True)",
"",
"    df.dropna(axis=0, how=\"any\", inplace=True)",
"    df.drop_duplicates(keep=\"first\", inplace=True)",
"    df = df.sample(frac=1.0, random_state=42).reset_index(drop=True)",
"",
"    if \"Attack_type\" not in df.columns:",
"        raise ValueError(\"Expected 'Attack_type' column in Edge-IIoTset CSV.\")",
"",
"    attack_type = df[\"Attack_type\"].astype(str).to_numpy()",
"    y = (attack_type != \"Normal\").astype(np.int64)",
"",
"    feature_df = df.drop(columns=[\"Attack_type\"])",
"",
"    preferred_present = [c for c in PREFERRED_CAT_COLS if c in feature_df.columns]",
"    object_cols = [c for c in feature_df.columns if feature_df[c].dtype == \"object\"]",
"    if preferred_present:",
"        cat_cols = sorted(set(preferred_present + object_cols))",
"    else:",
"        cat_cols = object_cols",
"",
"    X_df = pd.get_dummies(feature_df, columns=cat_cols, drop_first=False)",
"    if X_df.isna().any().any():",
"        raise ValueError(\"NaNs found after encoding; check preprocessing pipeline.\")",
"",
"    X = X_df.to_numpy(dtype=np.float32)",
"    np.savez_compressed(out_npz, X=X, y=y, attack_type=attack_type)",
"",
"    meta = {",
"        \"csv_path\": str(csv_path),",
"        \"n_rows\": int(X.shape[0]),",
"        \"n_features\": int(X.shape[1]),",
"        \"feature_names\": list(X_df.columns),",
"        \"encoded_categorical_columns\": cat_cols,",
"        \"label_definition\": \"Label=1 if Attack_type!='Normal' else 0\",",
"        \"dropped_columns\": existing,",
"    }",
"    out_meta.write_text(json.dumps(meta, indent=2))",
"    return str(out_npz), str(out_meta)",
"",
])
(ROOT / "src/data/preprocess_edgeiiotset.py").write_text(preprocess_py)

# -------------------------
# src/data/splits.py
# -------------------------
splits_py = "\n".join([
"import json",
"import numpy as np",
"from pathlib import Path",
"from sklearn.model_selection import train_test_split",
"",
"def dirichlet_partition_indices(y, K, alpha, seed=42, min_per_silo=10, max_tries=500):",
"    rng = np.random.default_rng(seed)",
"    classes = np.unique(y)",
"    best_silos = None",
"    best_min_size = -1",
"    for _ in range(max_tries):",
"        silos = [[] for _ in range(K)]",
"        for c in classes:",
"            idx_c = np.where(y == c)[0]",
"            rng.shuffle(idx_c)",
"            props = rng.dirichlet(alpha=np.full(K, alpha))",
"            counts = rng.multinomial(len(idx_c), props)",
"            start = 0",
"            for k in range(K):",
"                cnt = counts[k]",
"                if cnt > 0:",
"                    silos[k].extend(idx_c[start:start+cnt].tolist())",
"                start += cnt",
"        sizes = np.array([len(s) for s in silos], dtype=int)",
"        min_size = int(sizes.min())",
"        if min_size > best_min_size:",
"            best_min_size = min_size",
"            best_silos = [np.array(s, dtype=np.int64) for s in silos]",
"        if min_size >= min_per_silo:",
"            for k in range(K):",
"                rng.shuffle(silos[k])",
"            return [np.array(s, dtype=np.int64) for s in silos], True, best_min_size",
"    return best_silos, False, best_min_size",
"",
"def split_silo_indices(idx, y, seed=42, ratios=(0.70,0.15,0.15)):",
"    idx = np.array(idx, dtype=np.int64)",
"    y_s = y[idx]",
"    if len(idx) < 30 or len(np.unique(y_s)) < 2:",
"        n = len(idx)",
"        n_train = max(1, int(ratios[0]*n))",
"        n_val = max(0, int(ratios[1]*n))",
"        perm = np.random.default_rng(seed).permutation(idx)",
"        tr = perm[:n_train]",
"        va = perm[n_train:n_train+n_val]",
"        te = perm[n_train+n_val:]",
"        return tr, va, te",
"    test_val_frac = ratios[1] + ratios[2]",
"    strat_ok = (len(np.unique(y_s)) > 1) and (np.min(np.bincount(y_s)) >= 2)",
"    tr, temp = train_test_split(idx, test_size=test_val_frac, random_state=seed, shuffle=True,",
"                                stratify=y_s if strat_ok else None)",
"    y_temp = y[temp]",
"    val_frac_of_temp = ratios[1] / (ratios[1] + ratios[2])",
"    strat_ok2 = (len(np.unique(y_temp)) > 1) and (np.min(np.bincount(y_temp)) >= 2)",
"    va, te = train_test_split(temp, test_size=(1.0 - val_frac_of_temp), random_state=seed, shuffle=True,",
"                              stratify=y_temp if strat_ok2 else None)",
"    return tr, va, te",
"",
"def make_splits_and_save(encoded_npz, out_npz, out_summary_json, K=10, alpha=0.3, seed=42,",
"                         ratios=(0.70,0.15,0.15), min_per_silo=10):",
"    data = np.load(encoded_npz, allow_pickle=True)",
"    y = data[\"y\"].astype(int)",
"    silos, met, best_min = dirichlet_partition_indices(y, K=K, alpha=alpha, seed=seed, min_per_silo=min_per_silo)",
"    splits = {\"train\": [], \"val\": [], \"test\": []}",
"    stats = []",
"    for k in range(K):",
"        tr, va, te = split_silo_indices(silos[k], y, seed=seed, ratios=ratios)",
"        splits[\"train\"].append(tr); splits[\"val\"].append(va); splits[\"test\"].append(te)",
"        yk = y[silos[k]]",
"        stats.append({",
"            \"silo\": k,",
"            \"n_total\": int(len(silos[k])),",
"            \"n0\": int((yk==0).sum()),",
"            \"n1\": int((yk==1).sum()),",
"            \"intrusion_rate\": float((yk==1).mean()) if len(yk) else 0.0,",
"            \"n_train\": int(len(tr)),",
"            \"n_val\": int(len(va)),",
"            \"n_test\": int(len(te)),",
"        })",
"    out_npz = Path(out_npz); out_npz.parent.mkdir(parents=True, exist_ok=True)",
"    np.savez_compressed(out_npz,",
"        train=np.array(splits[\"train\"], dtype=object),",
"        val=np.array(splits[\"val\"], dtype=object),",
"        test=np.array(splits[\"test\"], dtype=object),",
"        alpha=np.array([alpha], dtype=np.float32),",
"        K=np.array([K], dtype=np.int32),",
"        seed=np.array([seed], dtype=np.int32),",
"    )",
"    out_summary_json = Path(out_summary_json); out_summary_json.parent.mkdir(parents=True, exist_ok=True)",
"    out_summary_json.write_text(json.dumps({",
"        \"alpha\": alpha, \"K\": K, \"seed\": seed, \"ratios\": ratios,",
"        \"min_per_silo_target\": min_per_silo,",
"        \"met_min_per_silo_target\": bool(met),",
"        \"best_min_silo_size_seen\": int(best_min),",
"        \"silo_stats\": stats",
"    }, indent=2))",
"    return str(out_npz), str(out_summary_json)",
"",
])
(ROOT / "src/data/splits.py").write_text(splits_py)

# -------------------------
# src/data/loaders.py
# -------------------------
loaders_py = "\n".join([
"import numpy as np",
"import torch",
"from torch.utils.data import Dataset, DataLoader",
"",
"class TabularDataset(Dataset):",
"    def __init__(self, X, y):",
"        self.X = torch.from_numpy(X).float()",
"        self.y = torch.from_numpy(y).long()",
"    def __len__(self): return self.X.shape[0]",
"    def __getitem__(self, i): return self.X[i], self.y[i]",
"",
"def fit_local_norm(X_train):",
"    mu = X_train.mean(axis=0, dtype=np.float64)",
"    sigma = X_train.std(axis=0, dtype=np.float64)",
"    return mu.astype(np.float32), sigma.astype(np.float32)",
"",
"def apply_local_norm(X, mu, sigma, eps=1e-8):",
"    return (X - mu) / (sigma + eps)",
"",
"def build_silo_loaders(encoded_npz, splits_npz, batch_size=256, eps=1e-8):",
"    data = np.load(encoded_npz, allow_pickle=True)",
"    X = data[\"X\"].astype(np.float32)",
"    y = data[\"y\"].astype(np.int64)",
"",
"    spl = np.load(splits_npz, allow_pickle=True)",
"    train_idxs = spl[\"train\"]; val_idxs = spl[\"val\"]; test_idxs = spl[\"test\"]",
"    K = int(spl[\"K\"][0])",
"",
"    silo = []",
"    for k in range(K):",
"        tr = np.array(train_idxs[k], dtype=np.int64)",
"        va = np.array(val_idxs[k], dtype=np.int64)",
"        te = np.array(test_idxs[k], dtype=np.int64)",
"",
"        X_tr, y_tr = X[tr], y[tr]",
"        X_va, y_va = X[va], y[va]",
"        X_te, y_te = X[te], y[te]",
"",
"        mu_k, sigma_k = fit_local_norm(X_tr)",
"        X_tr = apply_local_norm(X_tr, mu_k, sigma_k, eps)",
"        X_va = apply_local_norm(X_va, mu_k, sigma_k, eps)",
"        X_te = apply_local_norm(X_te, mu_k, sigma_k, eps)",
"",
"        silo.append({",
"            \"train_loader\": DataLoader(TabularDataset(X_tr, y_tr), batch_size=batch_size, shuffle=True),",
"            \"val_loader\":   DataLoader(TabularDataset(X_va, y_va), batch_size=batch_size, shuffle=False),",
"            \"test_loader\":  DataLoader(TabularDataset(X_te, y_te), batch_size=batch_size, shuffle=False),",
"            \"n_train\": int(len(tr)), \"n_val\": int(len(va)), \"n_test\": int(len(te)),",
"        })",
"    return silo, X.shape[1], K",
"",
])
(ROOT / "src/data/loaders.py").write_text(loaders_py)

# -------------------------
# src/fl/aggregators.py
# -------------------------
aggs_py = "\n".join([
"import numpy as np",
"import torch",
"",
"def agg_fedavg(deltas, weights):",
"    W = float(np.sum(weights))",
"    out = torch.zeros_like(deltas[0])",
"    for d, w in zip(deltas, weights):",
"        out += d * (float(w) / W)",
"    return out",
"",
"def agg_trimmed_mean(deltas, beta=0.10):",
"    m = len(deltas)",
"    X = torch.stack(deltas, dim=0)",
"    trim = int(np.floor(beta * m))",
"    if 2 * trim >= m:",
"        return torch.mean(X, dim=0)",
"    Xs, _ = torch.sort(X, dim=0)",
"    Xk = Xs[trim:m-trim, :]",
"    return torch.mean(Xk, dim=0)",
"",
])
(ROOT / "src/fl/aggregators.py").write_text(aggs_py)

# -------------------------
# src/fl/attacks.py
# -------------------------
(ROOT / "src/fl/attacks.py").write_text("def signflip_poison(delta_vec, scale=5.0):\n    return -scale * delta_vec\n")

# -------------------------
# src/fl/client.py
# -------------------------
client_py = "\n".join([
"import copy",
"import torch",
"",
"def get_param_vector(model):",
"    return torch.cat([p.view(-1) for p in model.parameters()])",
"",
"def fedprox_penalty(model, w0_vec):",
"    w_vec = get_param_vector(model)",
"    return 0.5 * torch.sum((w_vec - w0_vec) ** 2)",
"",
"def train_one_client_fedprox(global_model, train_loader, criterion, device, cfg, seed_offset=0):",
"    torch.manual_seed(cfg[\"seed\"] + seed_offset)",
"    model = copy.deepcopy(global_model).to(device)",
"    opt = torch.optim.Adam(model.parameters(), lr=cfg[\"lr\"], weight_decay=cfg[\"weight_decay\"])",
"    with torch.no_grad():",
"        w0 = get_param_vector(model).detach()",
"    model.train()",
"    for _ in range(cfg[\"local_epochs\"]):",
"        for xb, yb in train_loader:",
"            xb = xb.to(device)",
"            yb = yb.float().to(device)",
"            opt.zero_grad()",
"            logits = model(xb)",
"            base = criterion(logits, yb)",
"            prox = cfg[\"fedprox_mu\"] * fedprox_penalty(model, w0)",
"            loss = base + prox",
"            loss.backward()",
"            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg[\"grad_clip\"])",
"            opt.step()",
"    return model",
"",
])
(ROOT / "src/fl/client.py").write_text(client_py)

# -------------------------
# src/fl/utils.py
# -------------------------
utils_py = "\n".join([
"import torch",
"",
"def state_dict_to_vector(sd):",
"    return torch.cat([v.detach().reshape(-1).float().cpu() for v in sd.values() if torch.is_tensor(v)])",
"",
"def vector_to_state_dict_like(vec, ref_sd):",
"    out, off = {}, 0",
"    for k, v in ref_sd.items():",
"        if torch.is_tensor(v):",
"            n = v.numel()",
"            out[k] = vec[off:off+n].view_as(v).to(v.dtype)",
"            off += n",
"        else:",
"            out[k] = v",
"    return out",
"",
"def compute_delta(global_model, local_model):",
"    g_sd = global_model.state_dict()",
"    l_sd = local_model.state_dict()",
"    return state_dict_to_vector(l_sd) - state_dict_to_vector(g_sd), g_sd",
"",
"def clip_and_noise(delta_vec, C=1.0, sigma=0.0, seed=42):",
"    delta = delta_vec.clone()",
"    norm = torch.linalg.norm(delta, ord=2)",
"    scale = max(1.0, float(norm.item()) / float(C))",
"    delta = delta / scale",
"    if sigma and sigma > 0.0:",
"        gen = torch.Generator(device=\"cpu\").manual_seed(int(seed))",
"        delta = delta + torch.normal(0.0, float(sigma), size=delta.shape, generator=gen)",
"    return delta",
"",
"def apply_delta_to_model(global_model, delta_vec, ref_sd, gamma=1.0):",
"    g_sd = global_model.state_dict()",
"    delta_sd = vector_to_state_dict_like(delta_vec, ref_sd)",
"    new_sd = {}",
"    for k, v in g_sd.items():",
"        if torch.is_tensor(v):",
"            new_sd[k] = v + gamma * delta_sd[k].to(v.device)",
"        else:",
"            new_sd[k] = v",
"    global_model.load_state_dict(new_sd)",
"    return global_model",
"",
])
(ROOT / "src/fl/utils.py").write_text(utils_py)

# -------------------------
# src/eval/thresholds.py
# -------------------------
thresholds_py = "\n".join([
"import numpy as np",
"import torch",
"",
"def metrics_at_threshold(probs, y_true, thr):",
"    y_pred = (probs >= thr).astype(int)",
"    tp = int(((y_pred==1)&(y_true==1)).sum())",
"    tn = int(((y_pred==0)&(y_true==0)).sum())",
"    fp = int(((y_pred==1)&(y_true==0)).sum())",
"    fn = int(((y_pred==0)&(y_true==1)).sum())",
"    precision = tp / (tp + fp + 1e-12)",
"    recall    = tp / (tp + fn + 1e-12)",
"    f1        = 2*precision*recall / (precision + recall + 1e-12)",
"    acc       = (tp + tn) / max(1, tp+tn+fp+fn)",
"    fpr       = fp / (fp + tn + 1e-12)",
"    return {\"acc\":acc,\"precision\":precision,\"recall\":recall,\"f1\":f1,\"fpr\":fpr,",
"            \"tp\":tp,\"tn\":tn,\"fp\":fp,\"fn\":fn,\"pred_pos_rate\":float(y_pred.mean())}",
"",
"@torch.no_grad()",
"def collect_probs(model, loader, device):",
"    probs_list, y_list = [], []",
"    for xb, yb in loader:",
"        xb = xb.to(device)",
"        probs_list.append(torch.sigmoid(model(xb)).cpu().numpy())",
"        y_list.append(yb.cpu().numpy())",
"    return np.concatenate(probs_list).astype(np.float64), np.concatenate(y_list).astype(int)",
"",
"def robust_theta_from_fpr(probs_val, y_val, default_theta, target_fpr=0.05, theta_min=0.05, theta_max=0.95, min_benign_val=200):",
"    benign = probs_val[y_val == 0]",
"    if benign.size < min_benign_val:",
"        return default_theta, f\"fallback_default (benign_val={benign.size})\"",
"    thr = float(np.quantile(benign, 1.0 - target_fpr))",
"    thr = float(np.clip(thr, theta_min, theta_max))",
"    return thr, \"quantile_benign\"",
"",
])
(ROOT / "src/eval/thresholds.py").write_text(thresholds_py)

# -------------------------
# scripts/run_experiment.py
# -------------------------
run_script = "\n".join([
"import argparse, json, time",
"from pathlib import Path",
"import numpy as np",
"import torch",
"import torch.nn as nn",
"",
"from src.data.loaders import build_silo_loaders",
"from src.models.mlp import MLPIDS",
"from src.fl.client import train_one_client_fedprox",
"from src.fl.aggregators import agg_fedavg, agg_trimmed_mean",
"from src.fl.attacks import signflip_poison",
"from src.fl.utils import compute_delta, clip_and_noise, apply_delta_to_model",
"from src.eval.thresholds import collect_probs, metrics_at_threshold, robust_theta_from_fpr",
"",
"def eval_with_robust_thetas(model, silo_data, device, default_theta, calib):",
"    K = len(silo_data)",
"    per_silo = []",
"    for k in range(K):",
"        probs_val, y_val = collect_probs(model, silo_data[k]['val_loader'], device)",
"        theta_k, _ = robust_theta_from_fpr(",
"            probs_val, y_val, default_theta,",
"            target_fpr=calib['target_fpr'], theta_min=calib['theta_min'], theta_max=calib['theta_max'],",
"            min_benign_val=calib['min_benign_val']",
"        )",
"        m_val = metrics_at_threshold(probs_val, y_val, theta_k)",
"        if m_val['pred_pos_rate'] > calib['max_pred_pos']:",
"            theta_k = min(calib['theta_max'], max(theta_k, 0.80))",
"        elif m_val['pred_pos_rate'] < calib['min_pred_pos']:",
"            theta_k = max(calib['theta_min'], min(theta_k, 0.20))",
"",
"        probs_test, y_test = collect_probs(model, silo_data[k]['test_loader'], device)",
"        per_silo.append(metrics_at_threshold(probs_test, y_test, theta_k))",
"",
"    tp = sum(m['tp'] for m in per_silo); tn = sum(m['tn'] for m in per_silo)",
"    fp = sum(m['fp'] for m in per_silo); fn = sum(m['fn'] for m in per_silo)",
"    precision = tp / (tp + fp + 1e-12)",
"    recall    = tp / (tp + fn + 1e-12)",
"    f1        = 2*precision*recall / (precision + recall + 1e-12)",
"    acc       = (tp + tn) / max(1, tp+tn+fp+fn)",
"    fpr       = fp / (fp + tn + 1e-12)",
"    return {'acc':float(acc),'precision':float(precision),'recall':float(recall),'f1':float(f1),",
"            'fpr':float(fpr),'tp':int(tp),'tn':int(tn),'fp':int(fp),'fn':int(fn)}",
"",
"def main():",
"    ap = argparse.ArgumentParser()",
"    ap.add_argument('--config', type=str, required=True)",
"    args = ap.parse_args()",
"",
"    cfg = json.loads(Path(args.config).read_text())",
"    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')",
"",
"    run_id = time.strftime('run_repo_%Y%m%d_%H%M%S', time.gmtime())",
"    run_dir = Path('runs') / run_id",
"    run_dir.mkdir(parents=True, exist_ok=True)",
"    (run_dir / 'config.json').write_text(json.dumps(cfg, indent=2))",
"",
"    silo_data, d, K = build_silo_loaders(cfg['encoded_npz'], cfg['splits_npz'], batch_size=cfg['batch_size'])",
"",
"    model = MLPIDS(d, hidden=tuple(cfg['hidden']), dropout=cfg['dropout'], batch_norm=cfg['batch_norm']).to(device)",
"    pos_weight = torch.tensor([cfg['pos_weight']], device=device)",
"    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)",
"",
"    rng = np.random.default_rng(cfg['seed'])",
"",
"    for t in range(cfg['rounds']):",
"        participants = [k for k in range(K) if rng.uniform() > cfg['dropout_q']]",
"        if len(participants) < cfg['participation_min']:",
"            participants = list(rng.choice(K, size=cfg['participation_min'], replace=False))",
"",
"        m_count = int(np.floor(cfg['malicious_frac'] * len(participants)))",
"        malicious = set(rng.choice(participants, size=m_count, replace=False).tolist()) if m_count > 0 else set()",
"",
"        deltas, weights = [], []",
"        ref_sd = None",
"        for k in participants:",
"            local_model = train_one_client_fedprox(model, silo_data[k]['train_loader'], criterion, device, cfg, seed_offset=1000*t + k)",
"            delta_vec, ref_sd = compute_delta(model, local_model)",
"            protected = clip_and_noise(delta_vec, C=cfg['clip_C'], sigma=cfg['dp_sigma'], seed=cfg['seed'] + 1000*t + k)",
"            if k in malicious:",
"                protected = signflip_poison(protected, scale=cfg['poison_scale'])",
"            deltas.append(protected)",
"            weights.append(silo_data[k]['n_train'])",
"",
"        if cfg['aggregator'] == 'fedavg':",
"            agg_delta = agg_fedavg(deltas, weights)",
"        else:",
"            agg_delta = agg_trimmed_mean(deltas, beta=cfg['trim_beta'])",
"",
"        model = apply_delta_to_model(model, agg_delta, ref_sd, gamma=cfg['server_gamma'])",
"        print(f\"{cfg['aggregator']} round {t} | participants={len(participants)} malicious={len(malicious)}\")",
"",
"    metrics = eval_with_robust_thetas(model, silo_data, device, cfg['threshold'], cfg['calibration'])",
"    (run_dir / 'final_global_test.json').write_text(json.dumps(metrics, indent=2))",
"    torch.save(model.state_dict(), run_dir / 'model_final.pt')",
"    print('✅ Final metrics:', metrics)",
"    print('✅ Saved run:', run_dir)",
"",
"if __name__ == '__main__':",
"    main()",
"",
])
(ROOT / "scripts/run_experiment.py").write_text(run_script)

# -------------------------
# configs/base.json
# -------------------------
base_cfg = {
    "seed": 42,
    "encoded_npz": "data/processed/edgeiiotset_encoded.npz",
    "splits_npz": "data/processed/splits_edgeiiotset_dirichlet_alpha_0.3.npz",
    "batch_size": 256,

    "hidden": [256, 128, 64],
    "dropout": 0.30,
    "batch_norm": False,
    "pos_weight": 2.0,

    "rounds": 10,
    "dropout_q": 0.2,
    "participation_min": 6,

    "local_epochs": 2,
    "fedprox_mu": 1e-3,
    "lr": 3e-4,
    "weight_decay": 1e-4,
    "grad_clip": 1.0,

    "server_gamma": 1.0,

    "clip_C": 1.0,
    "dp_sigma": 0.0,

    "threshold": 0.5,

    "aggregator": "fedavg",      # "fedavg" or "trimmean"
    "trim_beta": 0.10,

    "malicious_frac": 0.2,
    "poison_scale": 5.0,

    "calibration": {
        "target_fpr": 0.05,
        "theta_min": 0.05,
        "theta_max": 0.95,
        "min_benign_val": 200,
        "max_pred_pos": 0.90,
        "min_pred_pos": 0.01
    }
}
(ROOT / "configs/base.json").write_text(json.dumps(base_cfg, indent=2))

# -------------------------
# README.md
# -------------------------
readme = "\n".join([
"# Cross-silo FL IDS (Edge-IIoTset)",
"",
"Minimal, reproducible code for:",
"- preprocessing Edge-IIoTset CSV into encoded NPZ",
"- Dirichlet non-IID partitioning (K=10)",
"- FedProx local training + FedAvg / TrimMean aggregation",
"- sign-flip poisoning robustness evaluation",
"- robust per-silo threshold calibration (FPR-target)",
"",
"## Run",
"",
"After you have generated `data/processed/edgeiiotset_encoded.npz` and a splits file:",
"",
"```bash",
"python scripts/run_experiment.py --config configs/base.json",
"```",
"",
"Artifacts are saved under `runs/run_repo_*`.",
"",
])
(ROOT / "README.md").write_text(readme)

print("✅ Repo files created successfully (fixed Step 14).")
print("Check: src/, scripts/, configs/, README.md")


✅ Repo files created successfully (fixed Step 14).
Check: src/, scripts/, configs/, README.md


**Step 15 — Add ready-to-run configs + a sweep runner that aggregates results into one CSV**

In [18]:
# Step 15: Create experiment configs + sweep runner script (aggregates to one CSV)

from pathlib import Path
import json

ROOT = Path(".").resolve()
CFG_DIR = ROOT / "configs"
SCR_DIR = ROOT / "scripts"
CFG_DIR.mkdir(parents=True, exist_ok=True)
SCR_DIR.mkdir(parents=True, exist_ok=True)

base = json.loads((CFG_DIR / "base.json").read_text())

def write_cfg(name, updates):
    cfg = dict(base)
    # deep update for nested calibration if present
    if "calibration" in updates:
        cal = dict(cfg["calibration"])
        cal.update(updates["calibration"])
        updates = dict(updates)
        updates["calibration"] = cal
    cfg.update(updates)
    (CFG_DIR / name).write_text(json.dumps(cfg, indent=2))

# Clean baselines
write_cfg("clean_fedavg.json", {
    "aggregator": "fedavg",
    "malicious_frac": 0.0,
    "poison_scale": 1.0,
})
write_cfg("clean_trimmean.json", {
    "aggregator": "trimmean",
    "trim_beta": 0.10,
    "malicious_frac": 0.0,
    "poison_scale": 1.0,
})

# Example attacked configs
write_cfg("poison_alpha20_scale5_fedavg.json", {
    "aggregator": "fedavg",
    "malicious_frac": 0.2,
    "poison_scale": 5.0,
})
write_cfg("poison_alpha20_scale5_trimmean.json", {
    "aggregator": "trimmean",
    "trim_beta": 0.10,
    "malicious_frac": 0.2,
    "poison_scale": 5.0,
})

# Sweep spec (grid)
sweep = {
    "alpha_grid": [0.0, 0.1, 0.2, 0.3],
    "scale_grid": [1.0, 5.0, 10.0],
    "aggregators": ["fedavg", "trimmean"],
    "trim_beta": 0.10,
    "base_config": "configs/base.json",
    "out_csv": "runs/sweep_repo_results.csv"
}
(CFG_DIR / "sweep_poison.json").write_text(json.dumps(sweep, indent=2))

# scripts/run_sweep.py
run_sweep = "\n".join([
"import argparse, json, subprocess, sys",
"from pathlib import Path",
"import pandas as pd",
"",
"def main():",
"    ap = argparse.ArgumentParser()",
"    ap.add_argument('--sweep', type=str, required=True, help='configs/sweep_poison.json')",
"    args = ap.parse_args()",
"",
"    sweep = json.loads(Path(args.sweep).read_text())",
"    base = json.loads(Path(sweep['base_config']).read_text())",
"    out_csv = Path(sweep['out_csv'])",
"    out_csv.parent.mkdir(parents=True, exist_ok=True)",
"",
"    rows = []",
"    tmp_cfg_dir = Path('configs/_generated')",
"    tmp_cfg_dir.mkdir(parents=True, exist_ok=True)",
"",
"    for a in sweep['alpha_grid']:",
"        for s in sweep['scale_grid']:",
"            for agg in sweep['aggregators']:",
"                cfg = dict(base)",
"                cfg['malicious_frac'] = float(a)",
"                cfg['poison_scale'] = float(s)",
"                cfg['aggregator'] = agg",
"                if agg == 'trimmean':",
"                    cfg['trim_beta'] = float(sweep.get('trim_beta', 0.10))",
"",
"                cfg_path = tmp_cfg_dir / f'cfg_a{a}_s{s}_{agg}.json'",
"                cfg_path.write_text(json.dumps(cfg, indent=2))",
"",
"                # Run experiment",
"                cmd = [sys.executable, 'scripts/run_experiment.py', '--config', str(cfg_path)]",
"                print('RUN:', ' '.join(cmd))",
"                subprocess.run(cmd, check=True)",
"",
"    # Collect results from runs/run_repo_*",
"    run_dirs = sorted(Path('runs').glob('run_repo_*'))",
"    for rd in run_dirs:",
"        cfg_path = rd / 'config.json'",
"        res_path = rd / 'final_global_test.json'",
"        if not (cfg_path.exists() and res_path.exists()):",
"            continue",
"        cfg = json.loads(cfg_path.read_text())",
"        res = json.loads(res_path.read_text())",
"        rows.append({",
"            'run_dir': str(rd),",
"            'aggregator': cfg.get('aggregator'),",
"            'trim_beta': cfg.get('trim_beta', None),",
"            'malicious_frac': cfg.get('malicious_frac'),",
"            'poison_scale': cfg.get('poison_scale'),",
"            'acc': res.get('acc'),",
"            'precision': res.get('precision'),",
"            'recall': res.get('recall'),",
"            'f1': res.get('f1'),",
"            'fpr': res.get('fpr'),",
"            'tp': res.get('tp'),",
"            'tn': res.get('tn'),",
"            'fp': res.get('fp'),",
"            'fn': res.get('fn'),",
"        })",
"",
"    df = pd.DataFrame(rows)",
"    df.to_csv(out_csv, index=False)",
"    print('✅ Wrote:', out_csv, '| rows:', len(df))",
"",
"if __name__ == '__main__':",
"    main()",
""
])
(SCR_DIR / "run_sweep.py").write_text(run_sweep)

print("✅ Created configs:")
print(" - configs/clean_fedavg.json")
print(" - configs/clean_trimmean.json")
print(" - configs/poison_alpha20_scale5_fedavg.json")
print(" - configs/poison_alpha20_scale5_trimmean.json")
print(" - configs/sweep_poison.json")
print("✅ Created sweep runner: scripts/run_sweep.py")


✅ Created configs:
 - configs/clean_fedavg.json
 - configs/clean_trimmean.json
 - configs/poison_alpha20_scale5_fedavg.json
 - configs/poison_alpha20_scale5_trimmean.json
 - configs/sweep_poison.json
✅ Created sweep runner: scripts/run_sweep.py


**Step 16 — Add a dataset-from-Drive unzip + CSV discovery script**

In [19]:
# Step 16: Create a script to unzip your private Drive dataset locally and locate the CSV

from pathlib import Path
import json

ROOT = Path(".").resolve()
(ROOT / "scripts").mkdir(parents=True, exist_ok=True)
(ROOT / "configs").mkdir(parents=True, exist_ok=True)
(ROOT / "data" / "raw").mkdir(parents=True, exist_ok=True)
(ROOT / "data" / "processed").mkdir(parents=True, exist_ok=True)

prepare_py = "\n".join([
"import argparse, zipfile, os",
"from pathlib import Path",
"",
"def find_csvs(root: Path):",
"    return sorted([p for p in root.rglob('*.csv') if p.is_file()])",
"",
"def main():",
"    ap = argparse.ArgumentParser()",
"    ap.add_argument('--zip_path', type=str, required=True, help='Path to archive.zip (Drive-mounted)')",
"    ap.add_argument('--out_dir', type=str, default='data/raw/edgeiiotset', help='Where to unzip')",
"    ap.add_argument('--pick_contains', type=str, default='DNN-EdgeIIoT-dataset.csv',",
"                    help='Preferred CSV filename substring to auto-select')",
"    args = ap.parse_args()",
"",
"    zip_path = Path(args.zip_path)",
"    out_dir = Path(args.out_dir)",
"    out_dir.mkdir(parents=True, exist_ok=True)",
"",
"    if not zip_path.exists():",
"        raise FileNotFoundError(f'Zip not found: {zip_path}')",
"",
"    # Unzip",
"    with zipfile.ZipFile(zip_path, 'r') as zf:",
"        zf.extractall(out_dir)",
"",
"    csvs = find_csvs(out_dir)",
"    if not csvs:",
"        raise RuntimeError(f'No CSV files found after unzip under: {out_dir}')",
"",
"    # Auto-pick",
"    chosen = None",
"    for p in csvs:",
"        if args.pick_contains in p.name:",
"            chosen = p",
"            break",
"    if chosen is None:",
"        # fallback: largest CSV",
"        chosen = max(csvs, key=lambda p: p.stat().st_size)",
"",
"    print('✅ Unzipped to:', out_dir)",
"    print('✅ Found CSVs:', len(csvs))",
"    print('✅ Selected CSV:', chosen)",
"    print(str(chosen))",
"",
"if __name__ == '__main__':",
"    main()",
""
])
(Path("scripts/prepare_data_from_zip.py")).write_text(prepare_py)

cfg = {
  "zip_path": "/content/drive/MyDrive/archive.zip",
  "out_dir": "data/raw/edgeiiotset",
  "pick_contains": "DNN-EdgeIIoT-dataset.csv"
}
(Path("configs/data_from_zip.json")).write_text(json.dumps(cfg, indent=2))

print("✅ Created:")
print(" - scripts/prepare_data_from_zip.py")
print(" - configs/data_from_zip.json")


✅ Created:
 - scripts/prepare_data_from_zip.py
 - configs/data_from_zip.json


**Step 17 — Preprocess the unzipped CSV → build encoded NPZ → create Dirichlet splits**

In [20]:
# Step 17: Auto-find CSV in unzipped folder -> preprocess -> create Dirichlet splits

from pathlib import Path
import json

from src.data.preprocess_edgeiiotset import preprocess_edgeiiotset
from src.data.splits import make_splits_and_save

# Load the zip/unzip config from Step 16
cfg_zip = json.loads(Path("configs/data_from_zip.json").read_text())
RAW_DIR = Path(cfg_zip["out_dir"])
PICK = cfg_zip.get("pick_contains", "DNN-EdgeIIoT-dataset.csv")

assert RAW_DIR.exists(), f"Raw unzip folder not found: {RAW_DIR}. Run Step 16 unzip first."

# Find CSVs
csvs = sorted([p for p in RAW_DIR.rglob("*.csv") if p.is_file()])
assert len(csvs) > 0, f"No CSV files found under {RAW_DIR}"

# Pick preferred (by name substring) else largest
chosen = None
for p in csvs:
    if PICK in p.name:
        chosen = p
        break
if chosen is None:
    chosen = max(csvs, key=lambda p: p.stat().st_size)

print("✅ Found CSVs:", len(csvs))
print("✅ Selected CSV:", chosen)

# Output paths
ENC_NPZ  = Path("data/processed/edgeiiotset_encoded.npz")
ENC_META = Path("data/processed/edgeiiotset_encoded_meta.json")

SPLITS_NPZ = Path("data/processed/splits_edgeiiotset_dirichlet_alpha_0.3.npz")
SPLITS_SUM = Path("data/processed/splits_edgeiiotset_dirichlet_alpha_0.3_summary.json")

# 1) Preprocess (CSV -> encoded NPZ)
preprocess_edgeiiotset(
    csv_path=str(chosen),
    out_npz=str(ENC_NPZ),
    out_meta=str(ENC_META),
)
print("✅ Wrote:", ENC_NPZ)
print("✅ Wrote:", ENC_META)

# 2) Create non-IID Dirichlet partition + per-silo 70/15/15 splits
make_splits_and_save(
    encoded_npz=str(ENC_NPZ),
    out_npz=str(SPLITS_NPZ),
    out_summary_json=str(SPLITS_SUM),
    K=10,
    alpha=0.3,
    seed=42,
    ratios=(0.70, 0.15, 0.15),
    min_per_silo=10,
)
print("✅ Wrote:", SPLITS_NPZ)
print("✅ Wrote:", SPLITS_SUM)

# Quick sanity: show where to point configs/base.json
print("\n📌 Update your configs to use:")
print("encoded_npz =", str(ENC_NPZ))
print("splits_npz  =", str(SPLITS_NPZ))


✅ Found CSVs: 26
✅ Selected CSV: data/raw/edgeiiotset/Edge-IIoTset dataset/Selected dataset for ML and DL/DNN-EdgeIIoT-dataset.csv
✅ Wrote: data/processed/edgeiiotset_encoded.npz
✅ Wrote: data/processed/edgeiiotset_encoded_meta.json
✅ Wrote: data/processed/splits_edgeiiotset_dirichlet_alpha_0.3.npz
✅ Wrote: data/processed/splits_edgeiiotset_dirichlet_alpha_0.3_summary.json

📌 Update your configs to use:
encoded_npz = data/processed/edgeiiotset_encoded.npz
splits_npz  = data/processed/splits_edgeiiotset_dirichlet_alpha_0.3.npz


**Step 18 — Run the repo entrypoint (clean FedAvg + clean TrimMean) and verify artifacts**

In [21]:
!PYTHONPATH=. python scripts/run_experiment.py --config configs/clean_fedavg.json


fedavg round 0 | participants=8 malicious=0
fedavg round 1 | participants=9 malicious=0
fedavg round 2 | participants=7 malicious=0
fedavg round 3 | participants=8 malicious=0
fedavg round 4 | participants=9 malicious=0
fedavg round 5 | participants=6 malicious=0
fedavg round 6 | participants=9 malicious=0
fedavg round 7 | participants=9 malicious=0
fedavg round 8 | participants=6 malicious=0
fedavg round 9 | participants=7 malicious=0
✅ Final metrics: {'acc': 0.9264357303190357, 'precision': 0.8998158137087225, 'recall': 0.83558330177269, 'f1': 0.8665108353440716, 'fpr': 0.03721823620261579, 'tp': 68395, 'tn': 196989, 'fp': 7615, 'fn': 13458}
✅ Saved run: runs/run_repo_20261001_161812


In [22]:
!PYTHONPATH=. python scripts/run_experiment.py --config configs/clean_trimmean.json

trimmean round 0 | participants=8 malicious=0
trimmean round 1 | participants=9 malicious=0
trimmean round 2 | participants=7 malicious=0
trimmean round 3 | participants=8 malicious=0
trimmean round 4 | participants=9 malicious=0
trimmean round 5 | participants=6 malicious=0
trimmean round 6 | participants=9 malicious=0
trimmean round 7 | participants=9 malicious=0
trimmean round 8 | participants=6 malicious=0
trimmean round 9 | participants=7 malicious=0
✅ Final metrics: {'acc': 0.9151565505468534, 'precision': 0.7722254283308578, 'recall': 0.997214518710371, 'f1': 0.8704159868619583, 'fpr': 0.11767120877402201, 'tp': 81625, 'tn': 180528, 'fp': 24076, 'fn': 228}
✅ Saved run: runs/run_repo_20261001_163230


**Verification cell**

In [23]:
from pathlib import Path
import json

runs = sorted(Path("runs").glob("run_repo_*"))
print("✅ Found run_repo_*:", len(runs))
for rd in runs[-2:]:
    res = json.loads((rd / "final_global_test.json").read_text())
    cfg = json.loads((rd / "config.json").read_text())
    print("\nRun:", rd)
    print(" aggregator:", cfg.get("aggregator"), "malicious_frac:", cfg.get("malicious_frac"), "poison_scale:", cfg.get("poison_scale"))
    print(" metrics:", {k: res[k] for k in ["acc","precision","recall","f1","fpr"]})


✅ Found run_repo_*: 2

Run: runs/run_repo_20261001_161812
 aggregator: fedavg malicious_frac: 0.0 poison_scale: 1.0
 metrics: {'acc': 0.9264357303190357, 'precision': 0.8998158137087225, 'recall': 0.83558330177269, 'f1': 0.8665108353440716, 'fpr': 0.03721823620261579}

Run: runs/run_repo_20261001_163230
 aggregator: trimmean malicious_frac: 0.0 poison_scale: 1.0
 metrics: {'acc': 0.9151565505468534, 'precision': 0.7722254283308578, 'recall': 0.997214518710371, 'f1': 0.8704159868619583, 'fpr': 0.11767120877402201}


**Step19: Create a separate reviewer-experiment script**

In [24]:
# ============================================================
# STEP 19.1 — Prepare reviewer-specific DP–TrimMean experiment
# ============================================================
# This cell DOES NOT modify the original experiment script.
# It creates a separate script/config space for Reviewer 2,
# Comment 2 interaction analysis.

from pathlib import Path
import json
import shutil

ROOT = Path(".").resolve()

SRC_SCRIPT = ROOT / "scripts" / "run_experiment.py"
NEW_SCRIPT = ROOT / "scripts" / "run_dp_trimmean_interaction.py"

BASE_CFG = ROOT / "configs" / "base.json"
REVIEW_CFG_DIR = ROOT / "configs" / "reviewer2_dp_trimmean"

REVIEW_CFG_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. Verify that the original reproducible pipeline exists
# ------------------------------------------------------------
required_files = [
    SRC_SCRIPT,
    BASE_CFG,
    ROOT / "src" / "fl" / "aggregators.py",
    ROOT / "src" / "fl" / "attacks.py",
    ROOT / "src" / "fl" / "utils.py",
    ROOT / "src" / "fl" / "client.py",
]

missing = [str(p) for p in required_files if not p.exists()]

if missing:
    raise FileNotFoundError(
        "The following required pipeline files are missing:\n"
        + "\n".join(missing)
    )

print("✅ Existing FL pipeline files verified.")

# ------------------------------------------------------------
# 2. Copy the current experiment runner
#    Original scripts/run_experiment.py remains unchanged.
# ------------------------------------------------------------
shutil.copy2(SRC_SCRIPT, NEW_SCRIPT)

print(f"✅ Reviewer-specific runner created:\n   {NEW_SCRIPT}")

# ------------------------------------------------------------
# 3. Load and inspect the current baseline configuration
# ------------------------------------------------------------
base_cfg = json.loads(BASE_CFG.read_text())

important_keys = [
    "seed",
    "encoded_npz",
    "splits_npz",
    "batch_size",
    "rounds",
    "local_epochs",
    "fedprox_mu",
    "lr",
    "clip_C",
    "dp_sigma",
    "aggregator",
    "trim_beta",
    "malicious_frac",
    "poison_scale",
]

print("\nCurrent baseline configuration:")
for key in important_keys:
    print(f"  {key:18s}: {base_cfg.get(key)}")

# ------------------------------------------------------------
# 4. Define the fixed Reviewer-2 interaction protocol
# ------------------------------------------------------------
review_protocol = {
    "sigma_grid": [0.0, 0.1, 0.2, 0.4, 0.6],

    # Same repeated-run seeds for every sigma value.
    "seeds": [42, 43, 44, 45, 46],

    # Parameters that must stay fixed during the interaction test.
    "aggregator": "trimmean",
    "trim_beta": 0.10,
    "clip_C": 1.0,
    "fedprox_mu": 1e-3,
    "malicious_frac": 0.20,
    "poison_scale": 5.0,

    # Important:
    # only sigma and replicate seed will vary.
    "vary_only": [
        "dp_sigma",
        "seed"
    ]
}

protocol_path = REVIEW_CFG_DIR / "interaction_protocol.json"
protocol_path.write_text(
    json.dumps(review_protocol, indent=2)
)

print(f"\n✅ Reviewer interaction protocol saved:\n   {protocol_path}")

print("\nSigma grid:", review_protocol["sigma_grid"])
print("Repeated-run seeds:", review_protocol["seeds"])

print("\nIMPORTANT:")
print("The original run_experiment.py has NOT been modified.")
print("All Reviewer-2 changes will be made only in:")
print("scripts/run_dp_trimmean_interaction.py")

✅ Existing FL pipeline files verified.
✅ Reviewer-specific runner created:
   /content/fl_ids_repo_real/scripts/run_dp_trimmean_interaction.py

Current baseline configuration:
  seed              : 42
  encoded_npz       : data/processed/edgeiiotset_encoded.npz
  splits_npz        : data/processed/splits_edgeiiotset_dirichlet_alpha_0.3.npz
  batch_size        : 256
  rounds            : 10
  local_epochs      : 2
  fedprox_mu        : 0.001
  lr                : 0.0003
  clip_C            : 1.0
  dp_sigma          : 0.0
  aggregator        : fedavg
  trim_beta         : 0.1
  malicious_frac    : 0.2
  poison_scale      : 5.0

✅ Reviewer interaction protocol saved:
   /content/fl_ids_repo_real/configs/reviewer2_dp_trimmean/interaction_protocol.json

Sigma grid: [0.0, 0.1, 0.2, 0.4, 0.6]
Repeated-run seeds: [42, 43, 44, 45, 46]

IMPORTANT:
The original run_experiment.py has NOT been modified.
All Reviewer-2 changes will be made only in:
scripts/run_dp_trimmean_interaction.py


**Step 19.2 — Add DP–TrimMean interaction diagnostics**

In [25]:
# ============================================================
# STEP 19.2 — Add DP–TrimMean interaction diagnostics
#             to reviewer-specific experiment runner
# ============================================================

from pathlib import Path

ROOT = Path(".").resolve()

REVIEW_SCRIPT = (
    ROOT / "scripts" / "run_dp_trimmean_interaction.py"
)

assert REVIEW_SCRIPT.exists(), (
    "Reviewer-specific script not found. "
    "Run Step 19.1 first."
)

text = REVIEW_SCRIPT.read_text()

# ------------------------------------------------------------
# 1. Add diagnostic helper:
#    TrimMean + malicious-coordinate retention
# ------------------------------------------------------------

marker = (
    "def eval_with_robust_thetas("
)

diagnostic_helper = r'''
def trimmean_with_retention(
    deltas,
    client_ids,
    malicious_ids,
    beta=0.10
):
    """
    Coordinate-wise TrimMean plus malicious-coordinate
    retention ratio.

    R_mal =
        number of malicious coordinate values surviving trim
        -----------------------------------------------------
        total malicious coordinate values submitted

    Returns
    -------
    agg_delta : torch.Tensor
        Coordinate-wise trimmed-mean aggregate.

    retention : float
        Fraction of malicious coordinate values retained
        after trimming.
    """

    X = torch.stack(deltas, dim=0)
    n_clients = X.shape[0]

    trim = int(np.floor(beta * n_clients))

    # Safety fallback
    if 2 * trim >= n_clients:
        return torch.mean(X, dim=0), 1.0

    # Sort values separately for every model coordinate.
    X_sorted, order = torch.sort(X, dim=0)

    # Keep central client values.
    kept_values = X_sorted[
        trim:n_clients-trim, :
    ]

    kept_indices = order[
        trim:n_clients-trim, :
    ]

    agg_delta = torch.mean(
        kept_values,
        dim=0
    )

    # Boolean mapping from client position -> malicious/honest
    malicious_mask = torch.tensor(
        [
            cid in malicious_ids
            for cid in client_ids
        ],
        dtype=torch.bool
    )

    n_malicious = int(
        malicious_mask.sum().item()
    )

    if n_malicious == 0:
        retention = 0.0

    else:
        # For every retained coordinate value,
        # identify whether it came from a malicious client.
        retained_malicious = malicious_mask[
            kept_indices
        ]

        retained_count = int(
            retained_malicious.sum().item()
        )

        submitted_count = (
            n_malicious * X.shape[1]
        )

        retention = (
            retained_count /
            max(submitted_count, 1)
        )

    return agg_delta, float(retention)


def honest_update_dispersion(
    deltas,
    client_ids,
    malicious_ids
):
    """
    Mean coordinate-wise variance among protected
    updates submitted by honest clients.

    Implements D_hon for one communication round.
    """

    honest_updates = [
        deltas[i]
        for i, cid in enumerate(client_ids)
        if cid not in malicious_ids
    ]

    if len(honest_updates) < 2:
        return 0.0

    X_honest = torch.stack(
        honest_updates,
        dim=0
    )

    # Variance across honest clients for each coordinate,
    # followed by mean across all model coordinates.
    coordinate_var = torch.var(
        X_honest,
        dim=0,
        unbiased=False
    )

    return float(
        coordinate_var.mean().item()
    )


'''

if "def trimmean_with_retention(" not in text:
    assert marker in text, (
        "Could not locate insertion point "
        "before eval_with_robust_thetas()."
    )

    text = text.replace(
        marker,
        diagnostic_helper + marker,
        1
    )

    print(
        "✅ Added TrimMean retention and "
        "honest-dispersion helpers."
    )
else:
    print(
        "ℹ️ Diagnostic helper already present; "
        "skipping insertion."
    )


# ------------------------------------------------------------
# 2. Initialize per-round diagnostic storage
# ------------------------------------------------------------

old = """    rng = np.random.default_rng(cfg['seed'])

    for t in range(cfg['rounds']):"""

new = """    rng = np.random.default_rng(cfg['seed'])

    # Reviewer-2 interaction diagnostics
    round_d_hon = []
    round_r_mal = []

    for t in range(cfg['rounds']):"""

if old in text:
    text = text.replace(
        old,
        new,
        1
    )
    print(
        "✅ Added per-round diagnostic storage."
    )
elif "round_d_hon = []" in text:
    print(
        "ℹ️ Diagnostic storage already present."
    )
else:
    raise RuntimeError(
        "Could not locate RNG/round-loop block."
    )


# ------------------------------------------------------------
# 3. Track client identities alongside their updates
# ------------------------------------------------------------

old = """        deltas, weights = [], []
        ref_sd = None"""

new = """        deltas, weights = [], []
        client_ids = []
        ref_sd = None"""

if old in text:
    text = text.replace(
        old,
        new,
        1
    )
    print(
        "✅ Added client-ID tracking."
    )
elif "client_ids = []" in text:
    print(
        "ℹ️ Client-ID tracking already present."
    )
else:
    raise RuntimeError(
        "Could not locate deltas/weights block."
    )


# ------------------------------------------------------------
# 4. Save each participating client's ID
# ------------------------------------------------------------

old = """            deltas.append(protected)
            weights.append(silo_data[k]['n_train'])"""

new = """            deltas.append(protected)
            weights.append(silo_data[k]['n_train'])
            client_ids.append(k)"""

if old in text:
    text = text.replace(
        old,
        new,
        1
    )
    print(
        "✅ Client IDs are now stored with updates."
    )
elif "client_ids.append(k)" in text:
    print(
        "ℹ️ Client IDs already appended."
    )
else:
    raise RuntimeError(
        "Could not locate update append block."
    )


# ------------------------------------------------------------
# 5. Replace aggregation block with diagnostic-aware version
# ------------------------------------------------------------

old = """        if cfg['aggregator'] == 'fedavg':
            agg_delta = agg_fedavg(deltas, weights)
        else:
            agg_delta = agg_trimmed_mean(deltas, beta=cfg['trim_beta'])"""

new = """        # ------------------------------------------------
        # DP--robust aggregation interaction diagnostics
        # ------------------------------------------------

        d_hon = honest_update_dispersion(
            deltas=deltas,
            client_ids=client_ids,
            malicious_ids=malicious
        )

        round_d_hon.append(d_hon)

        if cfg['aggregator'] == 'fedavg':
            agg_delta = agg_fedavg(
                deltas,
                weights
            )

            # Retention is defined specifically for
            # coordinate-wise trimming.
            r_mal = float('nan')

        else:
            agg_delta, r_mal = (
                trimmean_with_retention(
                    deltas=deltas,
                    client_ids=client_ids,
                    malicious_ids=malicious,
                    beta=cfg['trim_beta']
                )
            )

        round_r_mal.append(r_mal)"""

if old in text:
    text = text.replace(
        old,
        new,
        1
    )
    print(
        "✅ Aggregation now records D_hon and R_mal."
    )
elif "honest_update_dispersion(" in text and \
     "round_r_mal.append" in text:
    print(
        "ℹ️ Diagnostic-aware aggregation already present."
    )
else:
    raise RuntimeError(
        "Could not locate original aggregation block."
    )


# ------------------------------------------------------------
# 6. Add final averaged diagnostics to output JSON
# ------------------------------------------------------------

old = """    metrics = eval_with_robust_thetas(model, silo_data, device, cfg['threshold'], cfg['calibration'])
    (run_dir / 'final_global_test.json').write_text(json.dumps(metrics, indent=2))"""

new = """    metrics = eval_with_robust_thetas(
        model,
        silo_data,
        device,
        cfg['threshold'],
        cfg['calibration']
    )

    # Average interaction diagnostics across rounds.
    metrics['D_hon'] = float(
        np.mean(round_d_hon)
    ) if round_d_hon else None

    valid_r_mal = [
        x for x in round_r_mal
        if not np.isnan(x)
    ]

    metrics['R_mal'] = float(
        np.mean(valid_r_mal)
    ) if valid_r_mal else None

    metrics['D_hon_per_round'] = [
        float(x)
        for x in round_d_hon
    ]

    metrics['R_mal_per_round'] = [
        (
            None
            if np.isnan(x)
            else float(x)
        )
        for x in round_r_mal
    ]

    (run_dir / 'final_global_test.json').write_text(
        json.dumps(metrics, indent=2)
    )"""

if old in text:
    text = text.replace(
        old,
        new,
        1
    )
    print(
        "✅ Final JSON will include D_hon and R_mal."
    )
elif "metrics['D_hon']" in text:
    print(
        "ℹ️ Final diagnostic outputs already present."
    )
else:
    raise RuntimeError(
        "Could not locate final metric-saving block."
    )


# ------------------------------------------------------------
# 7. Save modified reviewer-specific runner
# ------------------------------------------------------------

REVIEW_SCRIPT.write_text(text)

print(
    "\n✅ Step 19.2 completed successfully."
)

print(
    "\nModified file:"
)
print(
    REVIEW_SCRIPT
)

print(
    "\nNew outputs that will be saved per run:"
)
print(
    "  D_hon             -> mean honest-update dispersion"
)
print(
    "  R_mal             -> mean malicious-coordinate retention"
)
print(
    "  D_hon_per_round   -> round-level dispersion"
)
print(
    "  R_mal_per_round   -> round-level retention"
)

print(
    "\nOriginal scripts/run_experiment.py remains unchanged."
)

✅ Added TrimMean retention and honest-dispersion helpers.
✅ Added per-round diagnostic storage.
✅ Added client-ID tracking.
✅ Client IDs are now stored with updates.
✅ Aggregation now records D_hon and R_mal.
✅ Final JSON will include D_hon and R_mal.

✅ Step 19.2 completed successfully.

Modified file:
/content/fl_ids_repo_real/scripts/run_dp_trimmean_interaction.py

New outputs that will be saved per run:
  D_hon             -> mean honest-update dispersion
  R_mal             -> mean malicious-coordinate retention
  D_hon_per_round   -> round-level dispersion
  R_mal_per_round   -> round-level retention

Original scripts/run_experiment.py remains unchanged.


**Step 19.3 — Validate the reviewer-specific script**

In [26]:
# ============================================================
# STEP 19.3 — Structural validation of reviewer experiment
# ============================================================

from pathlib import Path
import py_compile
import ast
import re

ROOT = Path(".").resolve()

REVIEW_SCRIPT = (
    ROOT / "scripts" / "run_dp_trimmean_interaction.py"
)

ORIGINAL_SCRIPT = (
    ROOT / "scripts" / "run_experiment.py"
)

assert REVIEW_SCRIPT.exists(), (
    "Reviewer-specific script not found."
)

assert ORIGINAL_SCRIPT.exists(), (
    "Original experiment script not found."
)

print("Reviewer script:")
print(REVIEW_SCRIPT)

# ------------------------------------------------------------
# 1. Python syntax validation
# ------------------------------------------------------------

try:
    py_compile.compile(
        str(REVIEW_SCRIPT),
        doraise=True
    )
    print("\n✅ Python syntax check passed.")
except py_compile.PyCompileError as e:
    print("\n❌ Python syntax error:")
    raise e


# ------------------------------------------------------------
# 2. AST validation
# ------------------------------------------------------------

source = REVIEW_SCRIPT.read_text()

try:
    tree = ast.parse(source)
    print("✅ AST parsing passed.")
except SyntaxError as e:
    print("❌ AST parsing failed.")
    raise e


# ------------------------------------------------------------
# 3. Confirm required reviewer functions exist
# ------------------------------------------------------------

function_names = {
    node.name
    for node in ast.walk(tree)
    if isinstance(
        node,
        (ast.FunctionDef, ast.AsyncFunctionDef)
    )
}

required_functions = [
    "trimmean_with_retention",
    "honest_update_dispersion",
]

print("\nRequired diagnostic functions:")

for fn in required_functions:
    if fn in function_names:
        print(f"  ✅ {fn}")
    else:
        print(f"  ❌ {fn} MISSING")


# ------------------------------------------------------------
# 4. Confirm diagnostic outputs exist
# ------------------------------------------------------------

required_markers = [
    "round_d_hon",
    "round_r_mal",
    "metrics['D_hon']",
    "metrics['R_mal']",
    "D_hon_per_round",
    "R_mal_per_round",
]

print("\nRequired diagnostic markers:")

for marker in required_markers:
    if marker in source:
        print(f"  ✅ {marker}")
    else:
        print(f"  ❌ {marker} MISSING")


# ------------------------------------------------------------
# 5. Check important FL pipeline operations
# ------------------------------------------------------------

pipeline_markers = {
    "FedProx local training": "fedprox",
    "Clipping + DP noise": "clip_and_noise",
    "Poisoning attack": "signflip_poison",
    "TrimMean aggregation": "trimmean_with_retention",
}

print("\nPipeline components:")

lower_source = source.lower()

for name, marker in pipeline_markers.items():
    if marker.lower() in lower_source:
        print(f"  ✅ {name}")
    else:
        print(f"  ❌ {name} not found")


# ------------------------------------------------------------
# 6. CRITICAL: inspect attack/protection ordering
# ------------------------------------------------------------

clip_positions = [
    m.start()
    for m in re.finditer(
        r"clip_and_noise\s*\(",
        source
    )
]

attack_positions = [
    m.start()
    for m in re.finditer(
        r"signflip_poison\s*\(",
        source
    )
]

print("\nCritical client-update ordering:")

if not clip_positions:
    print("  ❌ No clip_and_noise() call found.")

if not attack_positions:
    print("  ❌ No signflip_poison() call found.")

if clip_positions and attack_positions:

    # Use the final calls because earlier occurrences
    # may be imports or helper definitions.
    clip_pos = clip_positions[-1]
    attack_pos = attack_positions[-1]

    if attack_pos < clip_pos:
        print(
            "  ✅ Attack occurs BEFORE clipping/noise."
        )
        print(
            "     Current order:"
        )
        print(
            "     local update -> attack -> "
            "clip/noise -> aggregation"
        )
    else:
        print(
            "  ⚠️ Attack occurs AFTER clipping/noise."
        )
        print(
            "     Current order:"
        )
        print(
            "     local update -> clip/noise -> "
            "attack -> aggregation"
        )
        print(
            "\n     This must be corrected before "
            "the DP--TrimMean sensitivity experiment."
        )


# ------------------------------------------------------------
# 7. Check TrimMean comes after protected updates
# ------------------------------------------------------------

agg_positions = [
    m.start()
    for m in re.finditer(
        r"trimmean_with_retention\s*\(",
        source
    )
]

if clip_positions and agg_positions:

    clip_pos = clip_positions[-1]
    agg_pos = agg_positions[-1]

    print("\nProtection -> aggregation ordering:")

    if clip_pos < agg_pos:
        print(
            "  ✅ Protected updates are processed "
            "before TrimMean."
        )
    else:
        print(
            "  ❌ Unexpected aggregation ordering."
        )


# ------------------------------------------------------------
# 8. Confirm original script remains separate
# ------------------------------------------------------------

if REVIEW_SCRIPT.resolve() != ORIGINAL_SCRIPT.resolve():
    print(
        "\n✅ Reviewer runner is separate from "
        "the original runner."
    )
else:
    print(
        "\n❌ Reviewer runner points to original file!"
    )


# ------------------------------------------------------------
# 9. Print relevant client-update code region
# ------------------------------------------------------------

lines = source.splitlines()

interesting_terms = [
    "compute_delta(",
    "signflip_poison(",
    "clip_and_noise(",
    "deltas.append(",
]

indices = []

for i, line in enumerate(lines):
    if any(
        term in line
        for term in interesting_terms
    ):
        indices.append(i)

if indices:

    start = max(
        min(indices) - 8,
        0
    )

    end = min(
        max(indices) + 12,
        len(lines)
    )

    print(
        "\n========== CLIENT UPDATE CODE =========="
    )

    for i in range(start, end):
        print(
            f"{i+1:4d}: {lines[i]}"
        )

    print(
        "========================================"
    )


print(
    "\n✅ Step 19.3 validation finished."
)

print(
    "\nDo NOT run the sigma sweep yet."
)

print(
    "Send the complete output of this cell "
    "before proceeding to Step 19.4."
)

Reviewer script:
/content/fl_ids_repo_real/scripts/run_dp_trimmean_interaction.py

✅ Python syntax check passed.
✅ AST parsing passed.

Required diagnostic functions:
  ✅ trimmean_with_retention
  ✅ honest_update_dispersion

Required diagnostic markers:
  ✅ round_d_hon
  ✅ round_r_mal
  ✅ metrics['D_hon']
  ✅ metrics['R_mal']
  ✅ D_hon_per_round
  ✅ R_mal_per_round

Pipeline components:
  ✅ FedProx local training
  ✅ Clipping + DP noise
  ✅ Poisoning attack
  ✅ TrimMean aggregation

Critical client-update ordering:
  ⚠️ Attack occurs AFTER clipping/noise.
     Current order:
     local update -> clip/noise -> attack -> aggregation

     This must be corrected before the DP--TrimMean sensitivity experiment.

Protection -> aggregation ordering:
  ✅ Protected updates are processed before TrimMean.

✅ Reviewer runner is separate from the original runner.

========== CLIENT UPDATE CODE ==========
 204:         m_count = int(np.floor(cfg['malicious_frac'] * len(participants)))
 205:         

**Step 19.4 — Correct the attack/protection order**

In [27]:
# ============================================================
# STEP 19.4 — Correct client-update ordering
# Reviewer-specific runner ONLY
#
# Required order:
# local update
#   -> malicious manipulation (if malicious)
#   -> L2 clipping
#   -> Gaussian noise
#   -> TrimMean
# ============================================================

from pathlib import Path
import py_compile
import re

ROOT = Path(".").resolve()

REVIEW_SCRIPT = (
    ROOT / "scripts" / "run_dp_trimmean_interaction.py"
)

assert REVIEW_SCRIPT.exists(), (
    "Reviewer-specific script not found. "
    "Run Steps 19.1–19.3 first."
)

text = REVIEW_SCRIPT.read_text()

# ------------------------------------------------------------
# 1. Locate the current incorrect ordering
# ------------------------------------------------------------

old_block = """            delta_vec, ref_sd = compute_delta(model, local_model)
            protected = clip_and_noise(delta_vec, C=cfg['clip_C'], sigma=cfg['dp_sigma'], seed=cfg['seed'] + 1000*t + k)
            if k in malicious:
                protected = signflip_poison(protected, scale=cfg['poison_scale'])
            deltas.append(protected)"""

new_block = """            delta_vec, ref_sd = compute_delta(model, local_model)

            # ------------------------------------------------
            # Reviewer-2 interaction pipeline
            #
            # local update
            #   -> adversarial manipulation (if malicious)
            #   -> L2 clipping
            #   -> Gaussian perturbation
            #   -> server aggregation
            # ------------------------------------------------

            if k in malicious:
                delta_vec = signflip_poison(
                    delta_vec,
                    scale=cfg['poison_scale']
                )

            protected = clip_and_noise(
                delta_vec,
                C=cfg['clip_C'],
                sigma=cfg['dp_sigma'],
                seed=cfg['seed'] + 1000*t + k
            )

            deltas.append(protected)"""

if old_block in text:
    text = text.replace(
        old_block,
        new_block,
        1
    )

    REVIEW_SCRIPT.write_text(text)

    print(
        "✅ Corrected attack/protection ordering."
    )

else:
    # Check whether correction was already applied
    attack_pattern = (
        "delta_vec = signflip_poison("
    )

    if attack_pattern in text:
        print(
            "ℹ️ Corrected ordering appears to "
            "already be present."
        )
    else:
        raise RuntimeError(
            "Could not locate the expected "
            "client-update block."
        )


# ------------------------------------------------------------
# 2. Syntax validation after modification
# ------------------------------------------------------------

try:
    py_compile.compile(
        str(REVIEW_SCRIPT),
        doraise=True
    )

    print(
        "✅ Python syntax check passed "
        "after modification."
    )

except py_compile.PyCompileError as e:
    print(
        "❌ Syntax error after modification."
    )
    raise e


# ------------------------------------------------------------
# 3. Verify operation ordering in the actual code
# ------------------------------------------------------------

source = REVIEW_SCRIPT.read_text()

# Locate the actual execution calls.
attack_calls = [
    m.start()
    for m in re.finditer(
        r"delta_vec\s*=\s*signflip_poison\s*\(",
        source
    )
]

clip_calls = [
    m.start()
    for m in re.finditer(
        r"protected\s*=\s*clip_and_noise\s*\(",
        source
    )
]

agg_calls = [
    m.start()
    for m in re.finditer(
        r"trimmean_with_retention\s*\(",
        source
    )
]

print(
    "\nPipeline-order validation:"
)

if attack_calls and clip_calls:

    attack_pos = attack_calls[-1]
    clip_pos = clip_calls[-1]

    if attack_pos < clip_pos:
        print(
            "  ✅ malicious manipulation "
            "occurs before clipping/noise"
        )
    else:
        print(
            "  ❌ malicious manipulation still "
            "occurs after clipping/noise"
        )

else:
    print(
        "  ❌ Could not identify attack or "
        "protection execution call."
    )


if clip_calls and agg_calls:

    clip_pos = clip_calls[-1]
    agg_pos = agg_calls[-1]

    if clip_pos < agg_pos:
        print(
            "  ✅ clipping/noise occurs "
            "before TrimMean aggregation"
        )
    else:
        print(
            "  ❌ unexpected aggregation order"
        )


# ------------------------------------------------------------
# 4. Print the modified client-update block
# ------------------------------------------------------------

lines = source.splitlines()

target_idx = None

for i, line in enumerate(lines):
    if (
        "delta_vec, ref_sd = compute_delta"
        in line
    ):
        target_idx = i
        break

if target_idx is not None:

    start = max(
        target_idx - 2,
        0
    )

    end = min(
        target_idx + 35,
        len(lines)
    )

    print(
        "\n========== MODIFIED UPDATE PIPELINE =========="
    )

    for i in range(start, end):
        print(
            f"{i+1:4d}: {lines[i]}"
        )

    print(
        "=============================================="
    )


print(
    "\n✅ Step 19.4 completed."
)

print(
    "\nExpected reviewer-experiment pipeline:"
)

print(
    "local training"
    " -> model delta"
    " -> sign-flip manipulation (malicious clients only)"
    " -> L2 clipping"
    " -> Gaussian noise"
    " -> D_hon/R_mal diagnostics"
    " -> TrimMean"
)

print(
    "\nDo NOT run the full sigma sweep yet."
)

print(
    "Send the complete output of this cell "
    "before Step 19.5."
)

✅ Corrected attack/protection ordering.
✅ Python syntax check passed after modification.

Pipeline-order validation:
  ✅ malicious manipulation occurs before clipping/noise
  ✅ clipping/noise occurs before TrimMean aggregation

========== MODIFIED UPDATE PIPELINE ==========
 210:         for k in participants:
 211:             local_model = train_one_client_fedprox(model, silo_data[k]['train_loader'], criterion, device, cfg, seed_offset=1000*t + k)
 212:             delta_vec, ref_sd = compute_delta(model, local_model)
 213: 
 214:             # ------------------------------------------------
 215:             # Reviewer-2 interaction pipeline
 216:             #
 217:             # local update
 218:             #   -> adversarial manipulation (if malicious)
 219:             #   -> L2 clipping
 220:             #   -> Gaussian perturbation
 221:             #   -> server aggregation
 222:             # ------------------------------------------------
 223: 
 224:             if k i

**Step 19.5 — One-round smoke test**

In [28]:
# ============================================================
# STEP 19.5 — One-round smoke test
# DP–TrimMean interaction diagnostics
#
# Purpose:
#   1. Run ONLY one FL round
#   2. Use TrimMean
#   3. Use sigma = 0.4
#   4. Use malicious_frac = 0.20
#   5. Verify D_hon and R_mal are produced correctly
#
# This is NOT a manuscript result.
# It is only a structural/numerical validation.
# ============================================================

from pathlib import Path
import json
import subprocess
import sys
import os
import math
import time

ROOT = Path(".").resolve()

BASE_CFG = ROOT / "configs" / "base.json"

REVIEW_SCRIPT = (
    ROOT / "scripts" / "run_dp_trimmean_interaction.py"
)

REVIEW_CFG_DIR = (
    ROOT / "configs" / "reviewer2_dp_trimmean"
)

RUNS_DIR = ROOT / "runs"

# ------------------------------------------------------------
# 1. Verify required files
# ------------------------------------------------------------

assert BASE_CFG.exists(), (
    f"Missing base config: {BASE_CFG}"
)

assert REVIEW_SCRIPT.exists(), (
    f"Missing reviewer script: {REVIEW_SCRIPT}"
)

REVIEW_CFG_DIR.mkdir(
    parents=True,
    exist_ok=True
)

RUNS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# 2. Load baseline configuration
# ------------------------------------------------------------

cfg = json.loads(
    BASE_CFG.read_text()
)

# ------------------------------------------------------------
# 3. Create ONE-ROUND smoke-test configuration
# ------------------------------------------------------------

cfg["seed"] = 42

cfg["rounds"] = 1

cfg["aggregator"] = "trimmean"

cfg["trim_beta"] = 0.10

cfg["clip_C"] = 1.0

cfg["dp_sigma"] = 0.4

cfg["malicious_frac"] = 0.20

cfg["poison_scale"] = 5.0

cfg["fedprox_mu"] = 1e-3

SMOKE_CFG = (
    REVIEW_CFG_DIR /
    "smoke_sigma_0.4.json"
)

SMOKE_CFG.write_text(
    json.dumps(
        cfg,
        indent=2
    )
)

print(
    "✅ Smoke-test configuration created:"
)

print(
    SMOKE_CFG
)

print(
    "\nSmoke-test parameters:"
)

for key in [
    "seed",
    "rounds",
    "aggregator",
    "trim_beta",
    "clip_C",
    "dp_sigma",
    "malicious_frac",
    "poison_scale",
    "fedprox_mu",
    "lr",
    "local_epochs",
]:
    print(
        f"  {key:18s}: {cfg.get(key)}"
    )


# ------------------------------------------------------------
# 4. Verify dataset files BEFORE training
# ------------------------------------------------------------

encoded_path = (
    ROOT / cfg["encoded_npz"]
)

splits_path = (
    ROOT / cfg["splits_npz"]
)

assert encoded_path.exists(), (
    f"Encoded dataset missing: {encoded_path}"
)

assert splits_path.exists(), (
    f"Split file missing: {splits_path}"
)

print(
    "\n✅ Dataset and split files verified."
)


# ------------------------------------------------------------
# 5. Record current run directories
# ------------------------------------------------------------

before_runs = set(
    RUNS_DIR.glob("run_repo_*")
)

# Small pause protects against timestamp collision
# with a run created immediately before this cell.
time.sleep(1.1)


# ------------------------------------------------------------
# 6. Execute reviewer-specific runner
# ------------------------------------------------------------

cmd = [
    sys.executable,
    str(REVIEW_SCRIPT),
    "--config",
    str(SMOKE_CFG),
]

env = os.environ.copy()

# Ensure repository modules under src/ are importable.
existing_pythonpath = env.get(
    "PYTHONPATH",
    ""
)

env["PYTHONPATH"] = (
    str(ROOT)
    if not existing_pythonpath
    else str(ROOT) + os.pathsep + existing_pythonpath
)

print(
    "\nRunning one-round smoke test..."
)

print(
    "Command:"
)

print(
    " ".join(cmd)
)

completed = subprocess.run(
    cmd,
    cwd=str(ROOT),
    env=env,
    text=True,
    capture_output=True
)

print(
    "\n========== RUN STDOUT =========="
)

print(
    completed.stdout
)

if completed.stderr.strip():
    print(
        "\n========== RUN STDERR =========="
    )
    print(
        completed.stderr
    )

if completed.returncode != 0:
    raise RuntimeError(
        "Smoke test failed. "
        f"Return code = {completed.returncode}"
    )

print(
    "✅ Reviewer-specific runner finished."
)


# ------------------------------------------------------------
# 7. Locate the newly generated run directory
# ------------------------------------------------------------

after_runs = set(
    RUNS_DIR.glob("run_repo_*")
)

new_runs = sorted(
    after_runs - before_runs,
    key=lambda p: p.stat().st_mtime
)

if new_runs:
    RUN_DIR = new_runs[-1]

else:
    # Safe fallback: newest run directory
    candidates = sorted(
        RUNS_DIR.glob("run_repo_*"),
        key=lambda p: p.stat().st_mtime
    )

    if not candidates:
        raise RuntimeError(
            "No run_repo_* directory was generated."
        )

    RUN_DIR = candidates[-1]


print(
    "\n✅ Smoke-test run directory:"
)

print(
    RUN_DIR
)


# ------------------------------------------------------------
# 8. Load generated metrics
# ------------------------------------------------------------

RESULT_FILE = (
    RUN_DIR /
    "final_global_test.json"
)

RUN_CONFIG_FILE = (
    RUN_DIR /
    "config.json"
)

assert RESULT_FILE.exists(), (
    f"Missing result file: {RESULT_FILE}"
)

assert RUN_CONFIG_FILE.exists(), (
    f"Missing saved config: {RUN_CONFIG_FILE}"
)

metrics = json.loads(
    RESULT_FILE.read_text()
)

saved_cfg = json.loads(
    RUN_CONFIG_FILE.read_text()
)


# ------------------------------------------------------------
# 9. Confirm the correct experiment was actually executed
# ------------------------------------------------------------

assert saved_cfg["rounds"] == 1

assert saved_cfg["aggregator"] == "trimmean"

assert abs(
    saved_cfg["dp_sigma"] - 0.4
) < 1e-12

assert abs(
    saved_cfg["trim_beta"] - 0.10
) < 1e-12

assert abs(
    saved_cfg["malicious_frac"] - 0.20
) < 1e-12

print(
    "\n✅ Saved configuration matches "
    "the intended smoke test."
)


# ------------------------------------------------------------
# 10. Verify required diagnostic outputs
# ------------------------------------------------------------

required_outputs = [
    "D_hon",
    "R_mal",
    "D_hon_per_round",
    "R_mal_per_round",
]

print(
    "\nDiagnostic output check:"
)

for key in required_outputs:
    if key in metrics:
        print(
            f"  ✅ {key}"
        )
    else:
        print(
            f"  ❌ {key} MISSING"
        )

missing = [
    key
    for key in required_outputs
    if key not in metrics
]

if missing:
    raise RuntimeError(
        "Missing diagnostic outputs: "
        + ", ".join(missing)
    )


# ------------------------------------------------------------
# 11. Extract diagnostic values
# ------------------------------------------------------------

D_hon = metrics["D_hon"]

R_mal = metrics["R_mal"]

D_round = metrics[
    "D_hon_per_round"
]

R_round = metrics[
    "R_mal_per_round"
]


print(
    "\n========== SMOKE-TEST DIAGNOSTICS =========="
)

print(
    f"D_hon = {D_hon}"
)

print(
    f"R_mal = {R_mal}"
)

print(
    f"D_hon_per_round = {D_round}"
)

print(
    f"R_mal_per_round = {R_round}"
)

print(
    "============================================"
)


# ------------------------------------------------------------
# 12. Numerical validity checks
# ------------------------------------------------------------

assert D_hon is not None, (
    "D_hon is None."
)

assert math.isfinite(
    float(D_hon)
), (
    "D_hon is not finite."
)

assert float(D_hon) >= 0.0, (
    "D_hon must be non-negative."
)


assert R_mal is not None, (
    "R_mal is None."
)

assert math.isfinite(
    float(R_mal)
), (
    "R_mal is not finite."
)

assert (
    0.0 <= float(R_mal) <= 1.0
), (
    "R_mal must lie in [0, 1]."
)


assert len(D_round) == 1, (
    "Expected exactly one D_hon value "
    "for the one-round smoke test."
)

assert len(R_round) == 1, (
    "Expected exactly one R_mal value "
    "for the one-round smoke test."
)


print(
    "\n✅ D_hon is numerically valid."
)

print(
    "✅ R_mal is numerically valid."
)

print(
    "✅ Per-round arrays contain exactly "
    "one value."
)


# ------------------------------------------------------------
# 13. Verify ordinary IDS metrics still exist
# ------------------------------------------------------------

standard_metrics = [
    "acc",
    "precision",
    "recall",
    "f1",
    "fpr",
]

print(
    "\nStandard IDS metrics:"
)

for key in standard_metrics:

    value = metrics.get(
        key,
        None
    )

    print(
        f"  {key:10s}: {value}"
    )

    if value is None:
        raise RuntimeError(
            f"Missing standard metric: {key}"
        )

    if not math.isfinite(
        float(value)
    ):
        raise RuntimeError(
            f"Non-finite standard metric: {key}"
        )


# ------------------------------------------------------------
# 14. Final smoke-test status
# ------------------------------------------------------------

print(
    "\n============================================"
)

print(
    "✅ STEP 19.5 SMOKE TEST PASSED"
)

print(
    "============================================"
)

print(
    "\nThe reviewer-specific runner successfully:"
)

print(
    "  ✓ trained for one FL round"
)

print(
    "  ✓ used TrimMean"
)

print(
    "  ✓ used sigma = 0.4"
)

print(
    "  ✓ used malicious_frac = 0.20"
)

print(
    "  ✓ generated D_hon"
)

print(
    "  ✓ generated R_mal"
)

print(
    "  ✓ preserved standard IDS metrics"
)

print(
    "\nDo NOT start the full sigma sweep yet."
)

print(
    "Send the complete output of Step 19.5 "
    "before proceeding to Step 19.6."
)

✅ Smoke-test configuration created:
/content/fl_ids_repo_real/configs/reviewer2_dp_trimmean/smoke_sigma_0.4.json

Smoke-test parameters:
  seed              : 42
  rounds            : 1
  aggregator        : trimmean
  trim_beta         : 0.1
  clip_C            : 1.0
  dp_sigma          : 0.4
  malicious_frac    : 0.2
  poison_scale      : 5.0
  fedprox_mu        : 0.001
  lr                : 0.0003
  local_epochs      : 2

✅ Dataset and split files verified.

Running one-round smoke test...
Command:
/usr/bin/python3 /content/fl_ids_repo_real/scripts/run_dp_trimmean_interaction.py --config /content/fl_ids_repo_real/configs/reviewer2_dp_trimmean/smoke_sigma_0.4.json

========== RUN STDOUT ==========
trimmean round 0 | participants=8 malicious=1
✅ Final metrics: {'acc': 0.6257483671196724, 'precision': 0.35982352160643993, 'recall': 0.39755415195533456, 'f1': 0.37774901473636774, 'fpr': 0.2829612324294735, 'tp': 32541, 'tn': 146709, 'fp': 57895, 'fn': 49312, 'D_hon': 0.1368943750858307,

**Step 19.6 — Identify and fix participation to \(K=10\)**

In [29]:
# ============================================================
# STEP 19.6 — Audit participant selection
#
# Goal:
#   Determine exactly why 8/10 silos participate and identify
#   the configuration parameter controlling participation.
#
# DO NOT run the sigma sweep yet.
# ============================================================

from pathlib import Path
import json
import re

ROOT = Path(".").resolve()

BASE_CFG = ROOT / "configs" / "base.json"

REVIEW_SCRIPT = (
    ROOT / "scripts" / "run_dp_trimmean_interaction.py"
)

assert BASE_CFG.exists()
assert REVIEW_SCRIPT.exists()

cfg = json.loads(
    BASE_CFG.read_text()
)

source = REVIEW_SCRIPT.read_text()


# ------------------------------------------------------------
# 1. Show all configuration entries related to participation
# ------------------------------------------------------------

print("========== PARTICIPATION-RELATED CONFIG ==========")

candidate_terms = [
    "particip",
    "client",
    "silo",
    "fraction",
    "frac",
    "sample",
]

found_cfg = {}

for key, value in cfg.items():

    key_lower = key.lower()

    if any(
        term in key_lower
        for term in candidate_terms
    ):
        found_cfg[key] = value
        print(
            f"{key:30s}: {value}"
        )

if not found_cfg:
    print(
        "No obvious participation-related config key found."
    )


# ------------------------------------------------------------
# 2. Print participant-selection code from reviewer runner
# ------------------------------------------------------------

lines = source.splitlines()

print(
    "\n========== PARTICIPANT-SELECTION CODE =========="
)

matching_indices = []

search_terms = [
    "participants",
    "participation",
    "choice(",
    "num_part",
    "n_part",
    "client_frac",
    "participation_frac",
]

for i, line in enumerate(lines):

    if any(
        term.lower() in line.lower()
        for term in search_terms
    ):
        matching_indices.append(i)

if matching_indices:

    start = max(
        min(matching_indices) - 8,
        0
    )

    end = min(
        max(matching_indices) + 12,
        len(lines)
    )

    for i in range(start, end):
        print(
            f"{i+1:4d}: {lines[i]}"
        )

else:
    print(
        "No participant-selection lines identified."
    )

print(
    "================================================="
)


# ------------------------------------------------------------
# 3. Detect cfg[...] keys referenced near participant selection
# ------------------------------------------------------------

participant_region = "\n".join(
    lines[
        max(min(matching_indices) - 15, 0)
        :
        min(max(matching_indices) + 20, len(lines))
    ]
) if matching_indices else source

cfg_keys = sorted(
    set(
        re.findall(
            r"""cfg\[['"]([^'"]+)['"]\]""",
            participant_region
        )
    )
)

print(
    "\nConfiguration keys used around "
    "participant selection:"
)

for key in cfg_keys:
    print(
        f"  {key}: {cfg.get(key, '<not in base config>')}"
    )


# ------------------------------------------------------------
# 4. Audit current TrimMean arithmetic
# ------------------------------------------------------------

K = 10
beta = 0.10

observed_participants = 8
observed_malicious = 1

trim_observed = int(
    beta * observed_participants
)

trim_full = int(
    beta * K
)

mal_observed = int(
    0.20 * observed_participants
)

mal_full = int(
    0.20 * K
)

print(
    "\n========== ARITHMETIC AUDIT =========="
)

print(
    f"Observed participating clients: "
    f"{observed_participants}"
)

print(
    f"beta = {beta}"
)

print(
    f"floor(beta * 8)  = {trim_observed}"
)

print(
    f"floor(beta * 10) = {trim_full}"
)

print()

print(
    f"floor(0.20 * 8)  = {mal_observed}"
)

print(
    f"floor(0.20 * 10) = {mal_full}"
)

print(
    "======================================"
)


# ------------------------------------------------------------
# 5. Interpretation
# ------------------------------------------------------------

if trim_observed == 0:

    print(
        "\n⚠️ CRITICAL:"
    )

    print(
        "With 8 participants and beta=0.10, "
        "TrimMean removes ZERO values from each tail."
    )

    print(
        "Therefore the current 8-client reviewer run "
        "does not actually exercise robust trimming."
    )


if trim_full == 1:

    print(
        "\n✅ With full K=10 participation:"
    )

    print(
        "TrimMean removes 1 smallest and "
        "1 largest coordinate value."
    )

    print(
        "alpha_mal=0.20 gives exactly 2 malicious clients."
    )


print(
    "\nDo NOT modify anything yet."
)

print(
    "Send the complete output of Step 19.6."
)

========== PARTICIPATION-RELATED CONFIG ==========
participation_min             : 6
malicious_frac                : 0.2

========== PARTICIPANT-SELECTION CODE ==========
 192: 
 193:     rng = np.random.default_rng(cfg['seed'])
 194: 
 195:     # Reviewer-2 interaction diagnostics
 196:     round_d_hon = []
 197:     round_r_mal = []
 198: 
 199:     for t in range(cfg['rounds']):
 200:         participants = [k for k in range(K) if rng.uniform() > cfg['dropout_q']]
 201:         if len(participants) < cfg['participation_min']:
 202:             participants = list(rng.choice(K, size=cfg['participation_min'], replace=False))
 203: 
 204:         m_count = int(np.floor(cfg['malicious_frac'] * len(participants)))
 205:         malicious = set(rng.choice(participants, size=m_count, replace=False).tolist()) if m_count > 0 else set()
 206: 
 207:         deltas, weights = [], []
 208:         client_ids = []
 209:         ref_sd = None
 210:         for k in participants:
 211:            

**Step 19.7 — Force full \(K=10\) participation and rerun the smoke test**

In [30]:
# ============================================================
# STEP 19.7 — Force full K=10 participation
#              and rerun one-round smoke test
#
# Reviewer-specific experiment ONLY.
#
# Desired configuration:
#   K = 10
#   participants = 10
#   malicious_frac = 0.20 -> 2 malicious clients
#   beta = 0.10 -> trim 1 value from each tail
# ============================================================

from pathlib import Path
import json
import subprocess
import sys
import os
import math
import time

ROOT = Path(".").resolve()

BASE_CFG = ROOT / "configs" / "base.json"

REVIEW_SCRIPT = (
    ROOT / "scripts" / "run_dp_trimmean_interaction.py"
)

REVIEW_CFG_DIR = (
    ROOT / "configs" / "reviewer2_dp_trimmean"
)

RUNS_DIR = ROOT / "runs"

assert BASE_CFG.exists()
assert REVIEW_SCRIPT.exists()

REVIEW_CFG_DIR.mkdir(
    parents=True,
    exist_ok=True
)

RUNS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# 1. Load original baseline configuration
# ------------------------------------------------------------

cfg = json.loads(
    BASE_CFG.read_text()
)


# ------------------------------------------------------------
# 2. Reviewer-specific full-participation settings
# ------------------------------------------------------------

cfg["seed"] = 42

cfg["rounds"] = 1

cfg["aggregator"] = "trimmean"

cfg["trim_beta"] = 0.10

cfg["clip_C"] = 1.0

cfg["dp_sigma"] = 0.4

cfg["malicious_frac"] = 0.20

cfg["poison_scale"] = 5.0

cfg["fedprox_mu"] = 1e-3


# ------------------------------------------------------------
# CRITICAL:
# Force all K=10 clients to participate.
#
# dropout_q = 0 disables normal simulated dropout.
# participation_min = 10 guarantees all clients even if
# participant selection returns fewer than 10.
# ------------------------------------------------------------

cfg["dropout_q"] = 0.0

cfg["participation_min"] = 10


# ------------------------------------------------------------
# 3. Save reviewer-specific full-participation config
# ------------------------------------------------------------

FULL_SMOKE_CFG = (
    REVIEW_CFG_DIR /
    "smoke_fullK_sigma_0.4.json"
)

FULL_SMOKE_CFG.write_text(
    json.dumps(
        cfg,
        indent=2
    )
)

print(
    "✅ Full-participation smoke config created:"
)

print(
    FULL_SMOKE_CFG
)


# ------------------------------------------------------------
# 4. Print critical parameters
# ------------------------------------------------------------

print(
    "\nCritical reviewer-experiment parameters:"
)

critical_keys = [
    "seed",
    "rounds",
    "dropout_q",
    "participation_min",
    "aggregator",
    "trim_beta",
    "clip_C",
    "dp_sigma",
    "malicious_frac",
    "poison_scale",
    "fedprox_mu",
    "lr",
    "local_epochs",
]

for key in critical_keys:
    print(
        f"  {key:20s}: {cfg.get(key)}"
    )


# ------------------------------------------------------------
# 5. Expected arithmetic
# ------------------------------------------------------------

K = 10

expected_participants = 10

expected_malicious = int(
    math.floor(
        cfg["malicious_frac"]
        * expected_participants
    )
)

expected_trim = int(
    math.floor(
        cfg["trim_beta"]
        * expected_participants
    )
)

print(
    "\nExpected arithmetic:"
)

print(
    f"  available silos      = {K}"
)

print(
    f"  expected participants = {expected_participants}"
)

print(
    f"  expected malicious    = {expected_malicious}"
)

print(
    f"  trim per tail         = {expected_trim}"
)

assert expected_malicious == 2

assert expected_trim == 1


# ------------------------------------------------------------
# 6. Verify dataset files
# ------------------------------------------------------------

encoded_path = (
    ROOT / cfg["encoded_npz"]
)

splits_path = (
    ROOT / cfg["splits_npz"]
)

assert encoded_path.exists()

assert splits_path.exists()

print(
    "\n✅ Dataset files verified."
)


# ------------------------------------------------------------
# 7. Record current runs
# ------------------------------------------------------------

before_runs = set(
    RUNS_DIR.glob("run_repo_*")
)

time.sleep(1.1)


# ------------------------------------------------------------
# 8. Execute one-round full-participation smoke test
# ------------------------------------------------------------

cmd = [
    sys.executable,
    str(REVIEW_SCRIPT),
    "--config",
    str(FULL_SMOKE_CFG),
]

env = os.environ.copy()

existing_pythonpath = env.get(
    "PYTHONPATH",
    ""
)

env["PYTHONPATH"] = (
    str(ROOT)
    if not existing_pythonpath
    else str(ROOT)
    + os.pathsep
    + existing_pythonpath
)

print(
    "\nRunning full-participation smoke test..."
)

print(
    "Command:"
)

print(
    " ".join(cmd)
)

completed = subprocess.run(
    cmd,
    cwd=str(ROOT),
    env=env,
    text=True,
    capture_output=True
)

print(
    "\n========== RUN STDOUT =========="
)

print(
    completed.stdout
)

if completed.stderr.strip():

    print(
        "\n========== RUN STDERR =========="
    )

    print(
        completed.stderr
    )

if completed.returncode != 0:

    raise RuntimeError(
        "Full-participation smoke test failed. "
        f"Return code = {completed.returncode}"
    )

print(
    "✅ Reviewer-specific runner finished."
)


# ------------------------------------------------------------
# 9. Locate the new run directory
# ------------------------------------------------------------

after_runs = set(
    RUNS_DIR.glob("run_repo_*")
)

new_runs = sorted(
    after_runs - before_runs,
    key=lambda p: p.stat().st_mtime
)

if new_runs:

    RUN_DIR = new_runs[-1]

else:

    candidates = sorted(
        RUNS_DIR.glob("run_repo_*"),
        key=lambda p: p.stat().st_mtime
    )

    if not candidates:

        raise RuntimeError(
            "No run directory generated."
        )

    RUN_DIR = candidates[-1]


print(
    "\n✅ New run directory:"
)

print(
    RUN_DIR
)


# ------------------------------------------------------------
# 10. Load results and saved config
# ------------------------------------------------------------

RESULT_FILE = (
    RUN_DIR /
    "final_global_test.json"
)

CONFIG_FILE = (
    RUN_DIR /
    "config.json"
)

assert RESULT_FILE.exists()

assert CONFIG_FILE.exists()

metrics = json.loads(
    RESULT_FILE.read_text()
)

saved_cfg = json.loads(
    CONFIG_FILE.read_text()
)


# ------------------------------------------------------------
# 11. Verify the full-participation configuration
# ------------------------------------------------------------

assert saved_cfg["dropout_q"] == 0.0

assert saved_cfg["participation_min"] == 10

assert saved_cfg["aggregator"] == "trimmean"

assert abs(
    saved_cfg["trim_beta"] - 0.10
) < 1e-12

assert abs(
    saved_cfg["malicious_frac"] - 0.20
) < 1e-12

assert abs(
    saved_cfg["dp_sigma"] - 0.4
) < 1e-12


print(
    "\n✅ Saved configuration confirms "
    "full-participation reviewer setup."
)


# ------------------------------------------------------------
# 12. Verify interaction diagnostics
# ------------------------------------------------------------

required_outputs = [
    "D_hon",
    "R_mal",
    "D_hon_per_round",
    "R_mal_per_round",
]

print(
    "\nInteraction diagnostics:"
)

for key in required_outputs:

    if key not in metrics:

        raise RuntimeError(
            f"Missing diagnostic: {key}"
        )

    print(
        f"  {key:20s}: {metrics[key]}"
    )


# ------------------------------------------------------------
# 13. Numerical checks
# ------------------------------------------------------------

D_hon = float(
    metrics["D_hon"]
)

R_mal = float(
    metrics["R_mal"]
)

assert math.isfinite(D_hon)

assert D_hon >= 0.0

assert math.isfinite(R_mal)

assert 0.0 <= R_mal <= 1.0


# ------------------------------------------------------------
# 14. Verify standard metrics remain available
# ------------------------------------------------------------

print(
    "\nStandard IDS metrics:"
)

for key in [
    "acc",
    "precision",
    "recall",
    "f1",
    "fpr",
]:

    value = metrics.get(
        key
    )

    if value is None:

        raise RuntimeError(
            f"Missing standard metric: {key}"
        )

    print(
        f"  {key:10s}: {value}"
    )


# ------------------------------------------------------------
# 15. Final validation summary
# ------------------------------------------------------------

print(
    "\n============================================"
)

print(
    "✅ STEP 19.7 FULL-PARTICIPATION TEST PASSED"
)

print(
    "============================================"
)

print(
    "\nReviewer-specific interaction configuration:"
)

print(
    "  K available              = 10"
)

print(
    "  participation            = full"
)

print(
    "  expected participants    = 10"
)

print(
    "  malicious fraction       = 0.20"
)

print(
    "  expected malicious       = 2"
)

print(
    "  TrimMean beta            = 0.10"
)

print(
    "  trim count per tail      = 1"
)

print(
    "  sigma                    = 0.4"
)

print(
    f"  D_hon                    = {D_hon}"
)

print(
    f"  R_mal                    = {R_mal}"
)

print(
    "\nIMPORTANT:"
)

print(
    "Check the RUN STDOUT above."
)

print(
    "It must contain:"
)

print(
    "participants=10 malicious=2"
)

print(
    "\nR_mal does NOT need to be below 1.0."
)

print(
    "A value of 1.0 can still occur if none of the "
    "malicious coordinate values falls in a trimmed tail."
)

print(
    "The critical requirement is that with 10 clients "
    "beta=0.10 now produces an actual trim count of 1 "
    "from each tail."
)

print(
    "\nDo NOT launch the full sigma sweep yet."
)

print(
    "Send the complete Step 19.7 output first."
)

✅ Full-participation smoke config created:
/content/fl_ids_repo_real/configs/reviewer2_dp_trimmean/smoke_fullK_sigma_0.4.json

Critical reviewer-experiment parameters:
  seed                : 42
  rounds              : 1
  dropout_q           : 0.0
  participation_min   : 10
  aggregator          : trimmean
  trim_beta           : 0.1
  clip_C              : 1.0
  dp_sigma            : 0.4
  malicious_frac      : 0.2
  poison_scale        : 5.0
  fedprox_mu          : 0.001
  lr                  : 0.0003
  local_epochs        : 2

Expected arithmetic:
  available silos      = 10
  expected participants = 10
  expected malicious    = 2
  trim per tail         = 1

✅ Dataset files verified.

Running full-participation smoke test...
Command:
/usr/bin/python3 /content/fl_ids_repo_real/scripts/run_dp_trimmean_interaction.py --config /content/fl_ids_repo_real/configs/reviewer2_dp_trimmean/smoke_fullK_sigma_0.4.json

========== RUN STDOUT ==========
trimmean round 0 | participants=10 maliciou

**Step 19.8 — Record the actual trim count per round**

In [31]:
# ============================================================
# STEP 19.8 — Add explicit per-round TrimMean audit fields
#
# Adds:
#   participants_per_round
#   malicious_per_round
#   trim_count_per_round
#
# Reviewer-specific runner ONLY.
# ============================================================

from pathlib import Path
import py_compile
import json
import subprocess
import sys
import os
import time

ROOT = Path(".").resolve()

REVIEW_SCRIPT = (
    ROOT / "scripts" / "run_dp_trimmean_interaction.py"
)

SMOKE_CFG = (
    ROOT /
    "configs" /
    "reviewer2_dp_trimmean" /
    "smoke_fullK_sigma_0.4.json"
)

RUNS_DIR = ROOT / "runs"

assert REVIEW_SCRIPT.exists()
assert SMOKE_CFG.exists()

text = REVIEW_SCRIPT.read_text()


# ------------------------------------------------------------
# 1. Add audit-storage arrays
# ------------------------------------------------------------

old = """    # Reviewer-2 interaction diagnostics
    round_d_hon = []
    round_r_mal = []

    for t in range(cfg['rounds']):"""

new = """    # Reviewer-2 interaction diagnostics
    round_d_hon = []
    round_r_mal = []

    # Explicit aggregation-audit information
    round_participants = []
    round_malicious = []
    round_trim_count = []

    for t in range(cfg['rounds']):"""

if old in text:
    text = text.replace(
        old,
        new,
        1
    )

    print(
        "✅ Added per-round aggregation audit storage."
    )

elif "round_trim_count = []" in text:
    print(
        "ℹ️ Aggregation audit storage already exists."
    )

else:
    raise RuntimeError(
        "Could not locate diagnostic initialization block."
    )


# ------------------------------------------------------------
# 2. Record participant and malicious-client counts
#
# Insert immediately after malicious-client selection.
# ------------------------------------------------------------

old = """        m_count = int(np.floor(cfg['malicious_frac'] * len(participants)))
        malicious = set(rng.choice(participants, size=m_count, replace=False).tolist()) if m_count > 0 else set()

        deltas, weights = [], []"""

new = """        m_count = int(np.floor(cfg['malicious_frac'] * len(participants)))
        malicious = set(rng.choice(participants, size=m_count, replace=False).tolist()) if m_count > 0 else set()

        # Reviewer-2 aggregation audit
        round_participants.append(
            int(len(participants))
        )

        round_malicious.append(
            int(len(malicious))
        )

        deltas, weights = [], []"""

if old in text:
    text = text.replace(
        old,
        new,
        1
    )

    print(
        "✅ Added participant/malicious count logging."
    )

elif "round_participants.append" in text:
    print(
        "ℹ️ Participant-count logging already exists."
    )

else:
    raise RuntimeError(
        "Could not locate malicious-client selection block."
    )


# ------------------------------------------------------------
# 3. Explicitly compute actual TrimMean trim count
#
# Insert before D_hon calculation.
# ------------------------------------------------------------

old = """        # ------------------------------------------------
        # DP--robust aggregation interaction diagnostics
        # ------------------------------------------------

        d_hon = honest_update_dispersion("""

new = """        # ------------------------------------------------
        # DP--robust aggregation interaction diagnostics
        # ------------------------------------------------

        # Actual number of client values removed from
        # EACH tail of EACH coordinate by TrimMean.
        if cfg['aggregator'] == 'trimmean':
            trim_count = int(
                np.floor(
                    cfg['trim_beta'] * len(deltas)
                )
            )
        else:
            trim_count = 0

        round_trim_count.append(
            int(trim_count)
        )

        d_hon = honest_update_dispersion("""

if old in text:
    text = text.replace(
        old,
        new,
        1
    )

    print(
        "✅ Added explicit TrimMean trim-count calculation."
    )

elif "round_trim_count.append" in text:
    print(
        "ℹ️ Trim-count calculation already exists."
    )

else:
    raise RuntimeError(
        "Could not locate diagnostic block."
    )


# ------------------------------------------------------------
# 4. Save audit fields in final_global_test.json
# ------------------------------------------------------------

old = """    metrics['R_mal_per_round'] = [
        (
            None
            if np.isnan(x)
            else float(x)
        )
        for x in round_r_mal
    ]

    (run_dir / 'final_global_test.json').write_text("""

new = """    metrics['R_mal_per_round'] = [
        (
            None
            if np.isnan(x)
            else float(x)
        )
        for x in round_r_mal
    ]

    # Explicit aggregation-audit information
    metrics['participants_per_round'] = [
        int(x)
        for x in round_participants
    ]

    metrics['malicious_per_round'] = [
        int(x)
        for x in round_malicious
    ]

    metrics['trim_count_per_round'] = [
        int(x)
        for x in round_trim_count
    ]

    (run_dir / 'final_global_test.json').write_text("""

if old in text:
    text = text.replace(
        old,
        new,
        1
    )

    print(
        "✅ Added aggregation audit fields to result JSON."
    )

elif "metrics['trim_count_per_round']" in text:
    print(
        "ℹ️ Result audit fields already exist."
    )

else:
    raise RuntimeError(
        "Could not locate final result-saving block."
    )


# ------------------------------------------------------------
# 5. Save modified reviewer runner
# ------------------------------------------------------------

REVIEW_SCRIPT.write_text(text)


# ------------------------------------------------------------
# 6. Syntax validation
# ------------------------------------------------------------

py_compile.compile(
    str(REVIEW_SCRIPT),
    doraise=True
)

print(
    "✅ Python syntax check passed."
)


# ------------------------------------------------------------
# 7. Re-run ONE round with the existing full-K smoke config
# ------------------------------------------------------------

before_runs = set(
    RUNS_DIR.glob("run_repo_*")
)

time.sleep(1.1)

cmd = [
    sys.executable,
    str(REVIEW_SCRIPT),
    "--config",
    str(SMOKE_CFG),
]

env = os.environ.copy()

existing_pythonpath = env.get(
    "PYTHONPATH",
    ""
)

env["PYTHONPATH"] = (
    str(ROOT)
    if not existing_pythonpath
    else str(ROOT)
    + os.pathsep
    + existing_pythonpath
)

print(
    "\nRunning audited one-round smoke test..."
)

completed = subprocess.run(
    cmd,
    cwd=str(ROOT),
    env=env,
    text=True,
    capture_output=True
)

print(
    "\n========== RUN STDOUT =========="
)

print(
    completed.stdout
)

if completed.stderr.strip():
    print(
        "\n========== RUN STDERR =========="
    )
    print(
        completed.stderr
    )

if completed.returncode != 0:
    raise RuntimeError(
        "Audited smoke test failed."
    )


# ------------------------------------------------------------
# 8. Locate new run
# ------------------------------------------------------------

after_runs = set(
    RUNS_DIR.glob("run_repo_*")
)

new_runs = sorted(
    after_runs - before_runs,
    key=lambda p: p.stat().st_mtime
)

if not new_runs:
    raise RuntimeError(
        "Could not identify new run directory."
    )

RUN_DIR = new_runs[-1]

RESULT_FILE = (
    RUN_DIR /
    "final_global_test.json"
)

assert RESULT_FILE.exists()

metrics = json.loads(
    RESULT_FILE.read_text()
)


# ------------------------------------------------------------
# 9. Print and validate aggregation audit
# ------------------------------------------------------------

print(
    "\n========== AGGREGATION AUDIT =========="
)

print(
    "participants_per_round =",
    metrics.get(
        "participants_per_round"
    )
)

print(
    "malicious_per_round =",
    metrics.get(
        "malicious_per_round"
    )
)

print(
    "trim_count_per_round =",
    metrics.get(
        "trim_count_per_round"
    )
)

print(
    "D_hon_per_round =",
    metrics.get(
        "D_hon_per_round"
    )
)

print(
    "R_mal_per_round =",
    metrics.get(
        "R_mal_per_round"
    )
)

print(
    "======================================="
)


# ------------------------------------------------------------
# 10. Exact reviewer-configuration checks
# ------------------------------------------------------------

assert metrics[
    "participants_per_round"
] == [10], (
    "Expected 10 participants."
)

assert metrics[
    "malicious_per_round"
] == [2], (
    "Expected 2 malicious clients."
)

assert metrics[
    "trim_count_per_round"
] == [1], (
    "Expected TrimMean to trim "
    "one value per tail."
)


print(
    "\n✅ participants = 10 confirmed."
)

print(
    "✅ malicious clients = 2 confirmed."
)

print(
    "✅ TrimMean trim count = 1 per tail confirmed."
)

print(
    "\n============================================"
)

print(
    "✅ STEP 19.8 AGGREGATION AUDIT PASSED"
)

print(
    "============================================"
)

print(
    "\nDo NOT start the full sigma sweep yet."
)

print(
    "Send the complete Step 19.8 output."
)

✅ Added per-round aggregation audit storage.
✅ Added participant/malicious count logging.
✅ Added explicit TrimMean trim-count calculation.
✅ Added aggregation audit fields to result JSON.
✅ Python syntax check passed.

Running audited one-round smoke test...

========== RUN STDOUT ==========
trimmean round 0 | participants=10 malicious=2
✅ Final metrics: {'acc': 0.7017318480609656, 'precision': 0.430237984134391, 'recall': 0.13516914468620575, 'f1': 0.20570982346178607, 'fpr': 0.07161150319641844, 'tp': 11064, 'tn': 189952, 'fp': 14652, 'fn': 70789, 'D_hon': 0.13974608480930328, 'R_mal': 0.7989447228572726, 'D_hon_per_round': [0.13974608480930328], 'R_mal_per_round': [0.7989447228572726], 'participants_per_round': [10], 'malicious_per_round': [2], 'trim_count_per_round': [1]}
✅ Saved run: runs/run_repo_20261001_165012


========== AGGREGATION AUDIT ==========
participants_per_round = [10]
malicious_per_round = [2]
trim_count_per_round = [1]
D_hon_per_round = [0.13974608480930328]
R_ma

**Step 19.9 — Log exact client IDs and prepare the paired sweep**

In [32]:
# ============================================================
# STEP 19.9 — Make the DP sweep explicitly paired
#
# Part A:
#   Save exact participant IDs and malicious-client IDs
#   for every FL round.
#
# Part B:
#   Generate the complete paired sigma × seed configuration
#   set WITHOUT running expensive training yet.
#
# Desired design:
#   sigma = [0.0, 0.1, 0.2, 0.4, 0.6]
#   seeds = [42, 43, 44, 45, 46]
#
# For a given seed:
#   - same data splits
#   - same participants
#   - same malicious clients
#   - same FedProx/local-training randomness
#   - same clipping and TrimMean
#   - only sigma changes
# ============================================================

from pathlib import Path
import json
import py_compile
import copy

ROOT = Path(".").resolve()

REVIEW_SCRIPT = (
    ROOT / "scripts" / "run_dp_trimmean_interaction.py"
)

BASE_CFG = (
    ROOT / "configs" / "base.json"
)

REVIEW_CFG_DIR = (
    ROOT / "configs" / "reviewer2_dp_trimmean"
)

SWEEP_CFG_DIR = (
    REVIEW_CFG_DIR / "paired_sigma_configs"
)

assert REVIEW_SCRIPT.exists()
assert BASE_CFG.exists()

REVIEW_CFG_DIR.mkdir(
    parents=True,
    exist_ok=True
)

SWEEP_CFG_DIR.mkdir(
    parents=True,
    exist_ok=True
)

text = REVIEW_SCRIPT.read_text()


# ============================================================
# PART A — Save exact client IDs per round
# ============================================================

# ------------------------------------------------------------
# 1. Add storage for participant IDs and malicious IDs
# ------------------------------------------------------------

old = """    # Explicit aggregation-audit information
    round_participants = []
    round_malicious = []
    round_trim_count = []

    for t in range(cfg['rounds']):"""

new = """    # Explicit aggregation-audit information
    round_participants = []
    round_malicious = []
    round_trim_count = []

    # Exact identities used for paired experimental auditing
    round_participant_ids = []
    round_malicious_ids = []

    for t in range(cfg['rounds']):"""

if old in text:
    text = text.replace(
        old,
        new,
        1
    )
    print(
        "✅ Added storage for exact participant/malicious IDs."
    )

elif "round_malicious_ids = []" in text:
    print(
        "ℹ️ Exact client-ID storage already exists."
    )

else:
    raise RuntimeError(
        "Could not locate aggregation-audit initialization."
    )


# ------------------------------------------------------------
# 2. Save exact client IDs immediately after attacker selection
# ------------------------------------------------------------

old = """        round_malicious.append(
            int(len(malicious))
        )

        deltas, weights = [], []"""

new = """        round_malicious.append(
            int(len(malicious))
        )

        # Exact identities are stored to verify that
        # paired sigma conditions use the same clients.
        round_participant_ids.append(
            [int(x) for x in sorted(participants)]
        )

        round_malicious_ids.append(
            [int(x) for x in sorted(malicious)]
        )

        deltas, weights = [], []"""

if old in text:
    text = text.replace(
        old,
        new,
        1
    )
    print(
        "✅ Added exact client-ID logging."
    )

elif "round_malicious_ids.append" in text:
    print(
        "ℹ️ Exact client-ID logging already exists."
    )

else:
    raise RuntimeError(
        "Could not locate participant audit block."
    )


# ------------------------------------------------------------
# 3. Save exact IDs into final_global_test.json
# ------------------------------------------------------------

old = """    metrics['trim_count_per_round'] = [
        int(x)
        for x in round_trim_count
    ]

    (run_dir / 'final_global_test.json').write_text("""

new = """    metrics['trim_count_per_round'] = [
        int(x)
        for x in round_trim_count
    ]

    metrics['participant_ids_per_round'] = [
        [int(v) for v in row]
        for row in round_participant_ids
    ]

    metrics['malicious_ids_per_round'] = [
        [int(v) for v in row]
        for row in round_malicious_ids
    ]

    (run_dir / 'final_global_test.json').write_text("""

if old in text:
    text = text.replace(
        old,
        new,
        1
    )
    print(
        "✅ Added exact client IDs to result JSON."
    )

elif "metrics['malicious_ids_per_round']" in text:
    print(
        "ℹ️ Exact client IDs already saved in result JSON."
    )

else:
    raise RuntimeError(
        "Could not locate result-audit block."
    )


# ------------------------------------------------------------
# 4. Save and syntax-check reviewer runner
# ------------------------------------------------------------

REVIEW_SCRIPT.write_text(text)

py_compile.compile(
    str(REVIEW_SCRIPT),
    doraise=True
)

print(
    "✅ Reviewer script syntax check passed."
)


# ============================================================
# PART B — Generate paired experiment configurations
# ============================================================

base_cfg = json.loads(
    BASE_CFG.read_text()
)

sigma_grid = [
    0.0,
    0.1,
    0.2,
    0.4,
    0.6
]

seed_grid = [
    42,
    43,
    44,
    45,
    46
]


# ------------------------------------------------------------
# 5. Define parameters that MUST remain fixed
# ------------------------------------------------------------

fixed_parameters = {
    "rounds": 10,
    "local_epochs": 2,
    "fedprox_mu": 1e-3,
    "clip_C": 1.0,
    "aggregator": "trimmean",
    "trim_beta": 0.10,
    "malicious_frac": 0.20,
    "poison_scale": 5.0,

    # Full K=10 participation
    "dropout_q": 0.0,
    "participation_min": 10,
}


# ------------------------------------------------------------
# 6. Generate all 25 configs
# ------------------------------------------------------------

manifest = []

for seed in seed_grid:

    for sigma in sigma_grid:

        cfg = copy.deepcopy(
            base_cfg
        )

        # Apply fixed reviewer-experiment settings
        for key, value in fixed_parameters.items():
            cfg[key] = value

        # Paired dimensions
        cfg["seed"] = int(seed)
        cfg["dp_sigma"] = float(sigma)

        # File-safe sigma label
        sigma_label = (
            str(sigma)
            .replace(".", "p")
        )

        cfg_name = (
            f"seed_{seed}_sigma_{sigma_label}.json"
        )

        cfg_path = (
            SWEEP_CFG_DIR /
            cfg_name
        )

        cfg_path.write_text(
            json.dumps(
                cfg,
                indent=2
            )
        )

        manifest.append(
            {
                "seed": int(seed),
                "sigma": float(sigma),
                "config": str(cfg_path),
            }
        )


# ------------------------------------------------------------
# 7. Save experiment manifest
# ------------------------------------------------------------

MANIFEST_PATH = (
    REVIEW_CFG_DIR /
    "paired_sigma_manifest.json"
)

MANIFEST_PATH.write_text(
    json.dumps(
        manifest,
        indent=2
    )
)

print(
    "\n✅ Generated paired sigma-sweep configurations."
)

print(
    f"Number of configs = {len(manifest)}"
)

assert len(manifest) == 25


# ============================================================
# PART C — Structural pairing audit
# ============================================================

# ------------------------------------------------------------
# 8. Parameters allowed to differ
# ------------------------------------------------------------

allowed_to_vary = {
    "seed",
    "dp_sigma",
}


# ------------------------------------------------------------
# 9. For EACH seed, verify sigma configs are identical
#    except for dp_sigma
# ------------------------------------------------------------

print(
    "\n========== PAIRED CONFIG AUDIT =========="
)

for seed in seed_grid:

    seed_configs = []

    for sigma in sigma_grid:

        sigma_label = (
            str(sigma)
            .replace(".", "p")
        )

        cfg_path = (
            SWEEP_CFG_DIR /
            f"seed_{seed}_sigma_{sigma_label}.json"
        )

        cfg = json.loads(
            cfg_path.read_text()
        )

        seed_configs.append(
            (
                sigma,
                cfg
            )
        )

    reference_sigma, reference_cfg = (
        seed_configs[0]
    )

    for sigma, cfg in seed_configs[1:]:

        keys = (
            set(reference_cfg.keys())
            | set(cfg.keys())
        )

        differing_keys = []

        for key in sorted(keys):

            if (
                reference_cfg.get(key)
                != cfg.get(key)
            ):
                differing_keys.append(
                    key
                )

        # Within the same seed, ONLY dp_sigma should differ.
        unexpected = [
            key
            for key in differing_keys
            if key != "dp_sigma"
        ]

        if unexpected:
            raise RuntimeError(
                f"Seed {seed}: unexpected differences "
                f"between sigma={reference_sigma} "
                f"and sigma={sigma}: {unexpected}"
            )

    print(
        f"✅ Seed {seed}: sigma configs differ "
        f"only in dp_sigma."
    )


print(
    "========================================="
)


# ------------------------------------------------------------
# 10. Verify critical fixed values in all 25 configs
# ------------------------------------------------------------

for item in manifest:

    cfg = json.loads(
        Path(item["config"]).read_text()
    )

    assert cfg["rounds"] == 10

    assert cfg["local_epochs"] == 2

    assert abs(
        cfg["fedprox_mu"] - 1e-3
    ) < 1e-12

    assert abs(
        cfg["clip_C"] - 1.0
    ) < 1e-12

    assert cfg["aggregator"] == "trimmean"

    assert abs(
        cfg["trim_beta"] - 0.10
    ) < 1e-12

    assert abs(
        cfg["malicious_frac"] - 0.20
    ) < 1e-12

    assert cfg["dropout_q"] == 0.0

    assert cfg["participation_min"] == 10


print(
    "\n✅ All 25 configs use the same "
    "reviewer-experiment protocol."
)


# ------------------------------------------------------------
# 11. Print the complete experimental design
# ------------------------------------------------------------

print(
    "\n========== EXPERIMENT DESIGN =========="
)

print(
    "Sigma grid:",
    sigma_grid
)

print(
    "Seed grid:",
    seed_grid
)

print(
    "Total runs:",
    len(manifest)
)

print(
    "\nFixed parameters:"
)

for key, value in fixed_parameters.items():
    print(
        f"  {key:20s}: {value}"
    )

print(
    "\nBaseline learning rate retained from implementation:"
)

print(
    f"  lr = {base_cfg.get('lr')}"
)

print(
    "\nDataset:"
)

print(
    f"  {base_cfg.get('encoded_npz')}"
)

print(
    "\nNon-IID split:"
)

print(
    f"  {base_cfg.get('splits_npz')}"
)

print(
    "======================================="
)


# ------------------------------------------------------------
# 12. Print first five manifest entries
# ------------------------------------------------------------

print(
    "\nFirst five paired configurations:"
)

for item in manifest[:5]:
    print(
        item
    )


print(
    "\n============================================"
)

print(
    "✅ STEP 19.9 PAIRED CONFIGURATION PASSED"
)

print(
    "============================================"
)

print(
    "\nNo full experiment has been launched yet."
)

print(
    "\nNext validation must confirm that, for the "
    "same seed, different sigma values actually "
    "select identical malicious-client IDs."
)

✅ Added storage for exact participant/malicious IDs.
✅ Added exact client-ID logging.
✅ Added exact client IDs to result JSON.
✅ Reviewer script syntax check passed.

✅ Generated paired sigma-sweep configurations.
Number of configs = 25

========== PAIRED CONFIG AUDIT ==========
✅ Seed 42: sigma configs differ only in dp_sigma.
✅ Seed 43: sigma configs differ only in dp_sigma.
✅ Seed 44: sigma configs differ only in dp_sigma.
✅ Seed 45: sigma configs differ only in dp_sigma.
✅ Seed 46: sigma configs differ only in dp_sigma.

✅ All 25 configs use the same reviewer-experiment protocol.

========== EXPERIMENT DESIGN ==========
Sigma grid: [0.0, 0.1, 0.2, 0.4, 0.6]
Seed grid: [42, 43, 44, 45, 46]
Total runs: 25

Fixed parameters:
  rounds              : 10
  local_epochs        : 2
  fedprox_mu          : 0.001
  clip_C              : 1.0
  aggregator          : trimmean
  trim_beta           : 0.1
  malicious_frac      : 0.2
  poison_scale        : 5.0
  dropout_q           : 0.0
  partic

**Step 19.10 — Runtime pairing validation**

In [33]:
# ============================================================
# STEP 19.10 — Runtime pairing validation
#
# Compare:
#   seed = 42, sigma = 0.0
#   seed = 42, sigma = 0.6
#
# Both runs use ONE round only for validation.
#
# We verify:
#   ✓ identical participant IDs
#   ✓ identical malicious-client IDs
#   ✓ identical trim count
#   ✓ same fixed experiment parameters
#   ✓ sigma is the intended changing factor
#
# This is NOT a manuscript result.
# ============================================================

from pathlib import Path
import json
import copy
import subprocess
import sys
import os
import time
import math

ROOT = Path(".").resolve()

REVIEW_SCRIPT = (
    ROOT / "scripts" / "run_dp_trimmean_interaction.py"
)

CFG_DIR = (
    ROOT /
    "configs" /
    "reviewer2_dp_trimmean" /
    "paired_sigma_configs"
)

VALIDATION_DIR = (
    ROOT /
    "configs" /
    "reviewer2_dp_trimmean" /
    "runtime_pair_validation"
)

RUNS_DIR = ROOT / "runs"

VALIDATION_DIR.mkdir(
    parents=True,
    exist_ok=True
)

assert REVIEW_SCRIPT.exists()


# ------------------------------------------------------------
# 1. Load seed-42 sigma endpoints
# ------------------------------------------------------------

CFG_SIGMA_0 = (
    CFG_DIR /
    "seed_42_sigma_0p0.json"
)

CFG_SIGMA_06 = (
    CFG_DIR /
    "seed_42_sigma_0p6.json"
)

assert CFG_SIGMA_0.exists()
assert CFG_SIGMA_06.exists()

cfg0 = json.loads(
    CFG_SIGMA_0.read_text()
)

cfg06 = json.loads(
    CFG_SIGMA_06.read_text()
)


# ------------------------------------------------------------
# 2. Convert both to ONE-round validation configs
#
# All scientific parameters remain unchanged.
# Only rounds=1 is used to make this check inexpensive.
# ------------------------------------------------------------

cfg0_test = copy.deepcopy(cfg0)
cfg06_test = copy.deepcopy(cfg06)

cfg0_test["rounds"] = 1
cfg06_test["rounds"] = 1

TEST_CFG_0 = (
    VALIDATION_DIR /
    "seed42_sigma0_runtime_test.json"
)

TEST_CFG_06 = (
    VALIDATION_DIR /
    "seed42_sigma06_runtime_test.json"
)

TEST_CFG_0.write_text(
    json.dumps(
        cfg0_test,
        indent=2
    )
)

TEST_CFG_06.write_text(
    json.dumps(
        cfg06_test,
        indent=2
    )
)

print(
    "✅ One-round paired validation configs created."
)

print(
    f"  sigma=0.0: {TEST_CFG_0}"
)

print(
    f"  sigma=0.6: {TEST_CFG_06}"
)


# ------------------------------------------------------------
# 3. Confirm that the two validation configs differ ONLY
#    in dp_sigma
# ------------------------------------------------------------

all_keys = (
    set(cfg0_test.keys())
    | set(cfg06_test.keys())
)

differences = []

for key in sorted(all_keys):

    if (
        cfg0_test.get(key)
        != cfg06_test.get(key)
    ):
        differences.append(
            (
                key,
                cfg0_test.get(key),
                cfg06_test.get(key)
            )
        )

print(
    "\nConfiguration differences:"
)

for item in differences:
    print(
        f"  {item[0]}: "
        f"{item[1]} -> {item[2]}"
    )

unexpected = [
    item
    for item in differences
    if item[0] != "dp_sigma"
]

assert not unexpected, (
    "Unexpected differences between paired configs: "
    f"{unexpected}"
)

print(
    "\n✅ Paired configs differ only in dp_sigma."
)


# ------------------------------------------------------------
# 4. Helper to execute one reviewer run and retrieve results
# ------------------------------------------------------------

def run_validation(config_path):

    before = set(
        RUNS_DIR.glob("run_repo_*")
    )

    # Protect timestamp-based output names.
    time.sleep(1.2)

    cmd = [
        sys.executable,
        str(REVIEW_SCRIPT),
        "--config",
        str(config_path),
    ]

    env = os.environ.copy()

    existing_pythonpath = env.get(
        "PYTHONPATH",
        ""
    )

    env["PYTHONPATH"] = (
        str(ROOT)
        if not existing_pythonpath
        else str(ROOT)
        + os.pathsep
        + existing_pythonpath
    )

    completed = subprocess.run(
        cmd,
        cwd=str(ROOT),
        env=env,
        text=True,
        capture_output=True
    )

    print(
        "\nCommand:"
    )

    print(
        " ".join(cmd)
    )

    print(
        "\nSTDOUT:"
    )

    print(
        completed.stdout
    )

    if completed.stderr.strip():

        print(
            "\nSTDERR:"
        )

        print(
            completed.stderr
        )

    if completed.returncode != 0:

        raise RuntimeError(
            f"Validation run failed for "
            f"{config_path.name}"
        )

    after = set(
        RUNS_DIR.glob("run_repo_*")
    )

    new_runs = sorted(
        after - before,
        key=lambda p: p.stat().st_mtime
    )

    if not new_runs:

        raise RuntimeError(
            "Could not identify generated run."
        )

    run_dir = new_runs[-1]

    result_file = (
        run_dir /
        "final_global_test.json"
    )

    saved_config_file = (
        run_dir /
        "config.json"
    )

    assert result_file.exists()
    assert saved_config_file.exists()

    result = json.loads(
        result_file.read_text()
    )

    saved_config = json.loads(
        saved_config_file.read_text()
    )

    return (
        run_dir,
        result,
        saved_config
    )


# ------------------------------------------------------------
# 5. Run sigma = 0.0
# ------------------------------------------------------------

print(
    "\n============================================"
)

print(
    "Running seed=42, sigma=0.0"
)

print(
    "============================================"
)

run0, result0, saved0 = (
    run_validation(
        TEST_CFG_0
    )
)


# ------------------------------------------------------------
# 6. Run sigma = 0.6
# ------------------------------------------------------------

print(
    "\n============================================"
)

print(
    "Running seed=42, sigma=0.6"
)

print(
    "============================================"
)

run06, result06, saved06 = (
    run_validation(
        TEST_CFG_06
    )
)


# ------------------------------------------------------------
# 7. Extract runtime pairing information
# ------------------------------------------------------------

participants0 = result0[
    "participant_ids_per_round"
]

participants06 = result06[
    "participant_ids_per_round"
]

malicious0 = result0[
    "malicious_ids_per_round"
]

malicious06 = result06[
    "malicious_ids_per_round"
]

trim0 = result0[
    "trim_count_per_round"
]

trim06 = result06[
    "trim_count_per_round"
]


# ------------------------------------------------------------
# 8. Print runtime comparison
# ------------------------------------------------------------

print(
    "\n========== RUNTIME PAIRING AUDIT =========="
)

print(
    "\nSigma = 0.0"
)

print(
    "participant IDs:",
    participants0
)

print(
    "malicious IDs:",
    malicious0
)

print(
    "trim count:",
    trim0
)

print(
    "D_hon:",
    result0["D_hon"]
)

print(
    "R_mal:",
    result0["R_mal"]
)


print(
    "\nSigma = 0.6"
)

print(
    "participant IDs:",
    participants06
)

print(
    "malicious IDs:",
    malicious06
)

print(
    "trim count:",
    trim06
)

print(
    "D_hon:",
    result06["D_hon"]
)

print(
    "R_mal:",
    result06["R_mal"]
)

print(
    "============================================"
)


# ------------------------------------------------------------
# 9. Exact pairing assertions
# ------------------------------------------------------------

assert participants0 == participants06, (
    "Participant identities differ across sigma."
)

assert malicious0 == malicious06, (
    "Malicious-client identities differ across sigma."
)

assert trim0 == trim06, (
    "Trim count differs across sigma."
)

assert participants0 == [
    list(range(10))
], (
    "Expected all ten silos to participate."
)

assert len(
    malicious0[0]
) == 2, (
    "Expected exactly two malicious clients."
)

assert trim0 == [1], (
    "Expected one value trimmed per tail."
)


print(
    "\n✅ Participant identities are identical."
)

print(
    "✅ Malicious-client identities are identical."
)

print(
    "✅ Trim counts are identical."
)

print(
    "✅ Full K=10 participation confirmed."
)

print(
    "✅ Two malicious clients confirmed."
)


# ------------------------------------------------------------
# 10. Verify intended sigma settings
# ------------------------------------------------------------

assert abs(
    float(saved0["dp_sigma"]) - 0.0
) < 1e-12

assert abs(
    float(saved06["dp_sigma"]) - 0.6
) < 1e-12

print(
    "✅ Correct sigma values confirmed."
)


# ------------------------------------------------------------
# 11. Compare D_hon
#
# Do NOT require a specific scientific trend from one round.
# We only verify that both values are finite and measurable.
# ------------------------------------------------------------

d0 = float(
    result0["D_hon"]
)

d06 = float(
    result06["D_hon"]
)

assert math.isfinite(d0)
assert math.isfinite(d06)

assert d0 >= 0.0
assert d06 >= 0.0

print(
    "\nHonest-update dispersion comparison:"
)

print(
    f"  D_hon(sigma=0.0) = {d0}"
)

print(
    f"  D_hon(sigma=0.6) = {d06}"
)

print(
    f"  Difference         = {d06 - d0}"
)

if d06 > d0:

    print(
        "  ✅ Dispersion increased at sigma=0.6 "
        "in this validation run."
    )

else:

    print(
        "  ⚠️ Dispersion did not increase in this "
        "single-round validation."
    )

    print(
        "     Do not infer a scientific conclusion "
        "from this one-round test."
    )


# ------------------------------------------------------------
# 12. Compare malicious retention
# ------------------------------------------------------------

r0 = float(
    result0["R_mal"]
)

r06 = float(
    result06["R_mal"]
)

assert 0.0 <= r0 <= 1.0
assert 0.0 <= r06 <= 1.0

print(
    "\nMalicious-coordinate retention:"
)

print(
    f"  R_mal(sigma=0.0) = {r0}"
)

print(
    f"  R_mal(sigma=0.6) = {r06}"
)

print(
    f"  Difference         = {r06 - r0}"
)


# ------------------------------------------------------------
# 13. Final validation
# ------------------------------------------------------------

print(
    "\n============================================"
)

print(
    "✅ STEP 19.10 RUNTIME PAIRING PASSED"
)

print(
    "============================================"
)

print(
    "\nFor seed 42, sigma=0.0 and sigma=0.6 used:"
)

print(
    "  ✓ identical participant IDs"
)

print(
    "  ✓ identical malicious-client IDs"
)

print(
    "  ✓ identical TrimMean trimming"
)

print(
    "  ✓ identical full-participation protocol"
)

print(
    "  ✓ different Gaussian noise scales only"
)

print(
    "\nDo NOT launch the 25-run sweep yet."
)

print(
    "Send the complete Step 19.10 output."
)

✅ One-round paired validation configs created.
  sigma=0.0: /content/fl_ids_repo_real/configs/reviewer2_dp_trimmean/runtime_pair_validation/seed42_sigma0_runtime_test.json
  sigma=0.6: /content/fl_ids_repo_real/configs/reviewer2_dp_trimmean/runtime_pair_validation/seed42_sigma06_runtime_test.json

Configuration differences:
  dp_sigma: 0.0 -> 0.6

✅ Paired configs differ only in dp_sigma.

Running seed=42, sigma=0.0

Command:
/usr/bin/python3 /content/fl_ids_repo_real/scripts/run_dp_trimmean_interaction.py --config /content/fl_ids_repo_real/configs/reviewer2_dp_trimmean/runtime_pair_validation/seed42_sigma0_runtime_test.json

STDOUT:
trimmean round 0 | participants=10 malicious=2
✅ Final metrics: {'acc': 0.8706367796911928, 'precision': 0.8425084870171575, 'recall': 0.6730968932110002, 'f1': 0.7483344312612636, 'fpr': 0.050336259310668416, 'tp': 55095, 'tn': 194305, 'fp': 10299, 'fn': 26758, 'D_hon': 6.309205673460383e-06, 'R_mal': 0.5946645672152493, 'D_hon_per_round': [6.309205673460

In [ ]:
# ============================================================
# STEP 19.11 — FINAL DP–TrimMean paired sensitivity sweep
#
# 5 sigma values × 5 seeds = 25 runs
#
# Outputs:
#   runs/reviewer2_dp_trimmean/raw_results.csv
#   runs/reviewer2_dp_trimmean/summary_results.csv
#
# IMPORTANT:
# Current implementation uses sign-flip poisoning.
# Therefore F1 is reported as F1_under_poisoning,
# NOT backdoor ASR.
# ============================================================

from pathlib import Path
import json
import subprocess
import sys
import os
import time
import math
import pandas as pd
import numpy as np

ROOT = Path(".").resolve()

REVIEW_SCRIPT = (
    ROOT / "scripts" / "run_dp_trimmean_interaction.py"
)

MANIFEST_PATH = (
    ROOT /
    "configs" /
    "reviewer2_dp_trimmean" /
    "paired_sigma_manifest.json"
)

RUNS_DIR = ROOT / "runs"

FINAL_DIR = (
    RUNS_DIR /
    "reviewer2_dp_trimmean"
)

FINAL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

assert REVIEW_SCRIPT.exists()
assert MANIFEST_PATH.exists()

manifest = json.loads(
    MANIFEST_PATH.read_text()
)

assert len(manifest) == 25

print(
    f"✅ Loaded {len(manifest)} paired configurations."
)


# ------------------------------------------------------------
# Helper: execute one experiment
# ------------------------------------------------------------

def run_one_experiment(item, run_index, total_runs):

    seed = int(item["seed"])
    sigma = float(item["sigma"])
    config_path = Path(item["config"])

    print(
        "\n================================================"
    )

    print(
        f"RUN {run_index}/{total_runs}"
    )

    print(
        f"seed={seed} | sigma={sigma}"
    )

    print(
        "================================================"
    )

    before = set(
        RUNS_DIR.glob("run_repo_*")
    )

    time.sleep(1.1)

    cmd = [
        sys.executable,
        str(REVIEW_SCRIPT),
        "--config",
        str(config_path),
    ]

    env = os.environ.copy()

    existing_pythonpath = env.get(
        "PYTHONPATH",
        ""
    )

    env["PYTHONPATH"] = (
        str(ROOT)
        if not existing_pythonpath
        else str(ROOT)
        + os.pathsep
        + existing_pythonpath
    )

    completed = subprocess.run(
        cmd,
        cwd=str(ROOT),
        env=env,
        text=True,
        capture_output=True
    )

    if completed.returncode != 0:

        print(
            completed.stdout
        )

        print(
            completed.stderr
        )

        raise RuntimeError(
            f"Run failed: seed={seed}, sigma={sigma}"
        )

    after = set(
        RUNS_DIR.glob("run_repo_*")
    )

    new_runs = sorted(
        after - before,
        key=lambda p: p.stat().st_mtime
    )

    if not new_runs:

        raise RuntimeError(
            "Could not locate generated run directory."
        )

    run_dir = new_runs[-1]

    result_file = (
        run_dir /
        "final_global_test.json"
    )

    saved_config_file = (
        run_dir /
        "config.json"
    )

    assert result_file.exists()
    assert saved_config_file.exists()

    result = json.loads(
        result_file.read_text()
    )

    saved_cfg = json.loads(
        saved_config_file.read_text()
    )

    # --------------------------------------------------------
    # Validate intended configuration
    # --------------------------------------------------------

    assert saved_cfg["rounds"] == 10

    assert saved_cfg["aggregator"] == "trimmean"

    assert saved_cfg["participation_min"] == 10

    assert saved_cfg["dropout_q"] == 0.0

    assert abs(
        saved_cfg["malicious_frac"] - 0.20
    ) < 1e-12

    assert abs(
        saved_cfg["trim_beta"] - 0.10
    ) < 1e-12

    assert abs(
        saved_cfg["dp_sigma"] - sigma
    ) < 1e-12


    # --------------------------------------------------------
    # Validate all 10 rounds
    # --------------------------------------------------------

    participants = result[
        "participants_per_round"
    ]

    malicious_counts = result[
        "malicious_per_round"
    ]

    trim_counts = result[
        "trim_count_per_round"
    ]

    assert len(participants) == 10
    assert len(malicious_counts) == 10
    assert len(trim_counts) == 10

    assert all(
        x == 10
        for x in participants
    )

    assert all(
        x == 2
        for x in malicious_counts
    )

    assert all(
        x == 1
        for x in trim_counts
    )


    # --------------------------------------------------------
    # Extract only paper-relevant outputs
    # --------------------------------------------------------

    row = {
        "seed": seed,
        "sigma": sigma,

        # This model was trained under sign-flip poisoning.
        "F1_under_poisoning": float(
            result["f1"]
        ),

        "FPR_under_poisoning": float(
            result["fpr"]
        ),

        "D_hon": float(
            result["D_hon"]
        ),

        "R_mal": float(
            result["R_mal"]
        ),

        "participant_ids_per_round":
            json.dumps(
                result[
                    "participant_ids_per_round"
                ]
            ),

        "malicious_ids_per_round":
            json.dumps(
                result[
                    "malicious_ids_per_round"
                ]
            ),

        "run_dir": str(
            run_dir
        )
    }

    print(
        f"✅ F1={row['F1_under_poisoning']:.4f}"
        f" | FPR={row['FPR_under_poisoning']:.4f}"
        f" | D_hon={row['D_hon']:.6f}"
        f" | R_mal={row['R_mal']:.4f}"
    )

    return row


# ------------------------------------------------------------
# Run all 25 experiments
# ------------------------------------------------------------

rows = []

total_runs = len(manifest)

for idx, item in enumerate(
    manifest,
    start=1
):

    row = run_one_experiment(
        item,
        idx,
        total_runs
    )

    rows.append(
        row
    )

    # Save continuously so results survive interruptions.
    pd.DataFrame(
        rows
    ).to_csv(
        FINAL_DIR /
        "raw_results_partial.csv",
        index=False
    )


# ------------------------------------------------------------
# Create raw-results table
# ------------------------------------------------------------

raw_df = pd.DataFrame(
    rows
)

raw_path = (
    FINAL_DIR /
    "raw_results.csv"
)

raw_df.to_csv(
    raw_path,
    index=False
)

print(
    "\n✅ Raw results saved:"
)

print(
    raw_path
)


# ============================================================
# PAIRED-RUNTIME AUDIT
# ============================================================

print(
    "\n========== PAIRED RUNTIME AUDIT =========="
)

for seed in sorted(
    raw_df["seed"].unique()
):

    seed_df = (
        raw_df[
            raw_df["seed"] == seed
        ]
        .sort_values(
            "sigma"
        )
    )

    participant_patterns = (
        seed_df[
            "participant_ids_per_round"
        ]
        .unique()
    )

    malicious_patterns = (
        seed_df[
            "malicious_ids_per_round"
        ]
        .unique()
    )

    assert len(
        participant_patterns
    ) == 1, (
        f"Participant IDs changed across sigma "
        f"for seed {seed}."
    )

    assert len(
        malicious_patterns
    ) == 1, (
        f"Malicious IDs changed across sigma "
        f"for seed {seed}."
    )

    print(
        f"✅ Seed {seed}: identical participant "
        f"and malicious-client sequences across sigma."
    )

print(
    "=========================================="
)


# ============================================================
# SUMMARY: mean ± standard deviation across 5 seeds
# ============================================================

metrics_to_summarize = [
    "F1_under_poisoning",
    "FPR_under_poisoning",
    "D_hon",
    "R_mal",
]

summary_rows = []

for sigma in sorted(
    raw_df["sigma"].unique()
):

    sub = raw_df[
        raw_df["sigma"] == sigma
    ]

    row = {
        "sigma": sigma,
        "n_seeds": len(sub)
    }

    for metric in metrics_to_summarize:

        values = sub[
            metric
        ].astype(float)

        row[
            f"{metric}_mean"
        ] = float(
            values.mean()
        )

        row[
            f"{metric}_std"
        ] = float(
            values.std(
                ddof=1
            )
        )

    summary_rows.append(
        row
    )


summary_df = pd.DataFrame(
    summary_rows
)

summary_path = (
    FINAL_DIR /
    "summary_results.csv"
)

summary_df.to_csv(
    summary_path,
    index=False
)


# ------------------------------------------------------------
# Print compact paper-oriented summary
# ------------------------------------------------------------

print(
    "\n========== FINAL SIGMA SUMMARY =========="
)

for _, row in summary_df.iterrows():

    print(
        f"\nsigma = {row['sigma']:.1f}"
    )

    print(
        "  F1 under poisoning = "
        f"{row['F1_under_poisoning_mean']:.4f}"
        " ± "
        f"{row['F1_under_poisoning_std']:.4f}"
    )

    print(
        "  FPR               = "
        f"{row['FPR_under_poisoning_mean']:.4f}"
        " ± "
        f"{row['FPR_under_poisoning_std']:.4f}"
    )

    print(
        "  D_hon             = "
        f"{row['D_hon_mean']:.6f}"
        " ± "
        f"{row['D_hon_std']:.6f}"
    )

    print(
        "  R_mal             = "
        f"{row['R_mal_mean']:.4f}"
        " ± "
        f"{row['R_mal_std']:.4f}"
    )

print(
    "\n========================================="
)


# ------------------------------------------------------------
# Simple trend diagnostics
# ------------------------------------------------------------

sigma0 = summary_df[
    summary_df["sigma"] == 0.0
].iloc[0]

sigma06 = summary_df[
    summary_df["sigma"] == 0.6
].iloc[0]

print(
    "\nEndpoint changes: sigma=0.0 -> sigma=0.6"
)

print(
    "D_hon change =",
    sigma06["D_hon_mean"]
    - sigma0["D_hon_mean"]
)

print(
    "R_mal change =",
    sigma06["R_mal_mean"]
    - sigma0["R_mal_mean"]
)

print(
    "F1 change =",
    sigma06["F1_under_poisoning_mean"]
    - sigma0["F1_under_poisoning_mean"]
)

print(
    "FPR change =",
    sigma06["FPR_under_poisoning_mean"]
    - sigma0["FPR_under_poisoning_mean"]
)


# ------------------------------------------------------------
# Final status
# ------------------------------------------------------------

print(
    "\n============================================"
)

print(
    "✅ STEP 19.11 FINAL 25-RUN SWEEP COMPLETED"
)

print(
    "============================================"
)

print(
    "\nFiles:"
)

print(
    "Raw results:"
)

print(
    raw_path
)

print(
    "\nSummary results:"
)

print(
    summary_path
)

print(
    "\nSend me the FINAL SIGMA SUMMARY section."
)

✅ Loaded 25 paired configurations.

RUN 1/25
seed=42 | sigma=0.0
✅ F1=0.6446 | FPR=0.4411 | D_hon=0.000007 | R_mal=0.6314

RUN 2/25
seed=42 | sigma=0.1
✅ F1=0.7490 | FPR=0.2490 | D_hon=0.008755 | R_mal=0.7996

RUN 3/25
seed=42 | sigma=0.2
✅ F1=0.6374 | FPR=0.3460 | D_hon=0.035000 | R_mal=0.7997

RUN 4/25
seed=42 | sigma=0.4
✅ F1=0.5011 | FPR=0.4390 | D_hon=0.139981 | R_mal=0.7999

RUN 5/25
seed=42 | sigma=0.6
✅ F1=0.4483 | FPR=0.4080 | D_hon=0.314950 | R_mal=0.7998

RUN 6/25
seed=43 | sigma=0.0
✅ F1=0.9325 | FPR=0.0438 | D_hon=0.000007 | R_mal=0.6295

RUN 7/25
seed=43 | sigma=0.1
✅ F1=0.8353 | FPR=0.0470 | D_hon=0.008750 | R_mal=0.7994

RUN 8/25
seed=43 | sigma=0.2
✅ F1=0.6654 | FPR=0.1816 | D_hon=0.034983 | R_mal=0.7995

RUN 9/25
seed=43 | sigma=0.4
✅ F1=0.4250 | FPR=0.2970 | D_hon=0.139916 | R_mal=0.7997

RUN 10/25
seed=43 | sigma=0.6
✅ F1=0.3716 | FPR=0.2473 | D_hon=0.314805 | R_mal=0.7997

RUN 11/25
seed=44 | sigma=0.0
✅ F1=0.8835 | FPR=0.0419 | D_hon=0.000007 | R_mal=0.6208

RUN 1

In [1]:
# ============================================================
# STEP 20.1 — Download ToN-IoT and X-IIoTID
#
# Purpose:
#   1. Download public dataset copies.
#   2. Preserve raw files unchanged.
#   3. Copy them into this repository under data/raw/.
#   4. Save download provenance for reproducibility.
#
# NO preprocessing is performed in this step.
# ============================================================

from pathlib import Path
import subprocess
import sys
import shutil
import json
import hashlib
import os
import time

ROOT = Path(".").resolve()

RAW_ROOT = ROOT / "data" / "raw"

TON_DIR = RAW_ROOT / "ton_iot"
XIIOT_DIR = RAW_ROOT / "xiiotid"

TON_DIR.mkdir(
    parents=True,
    exist_ok=True
)

XIIOT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Repository root:")
print(ROOT)

print("\nRaw dataset directory:")
print(RAW_ROOT)


# ============================================================
# 1. Install KaggleHub
# ============================================================

print(
    "\nInstalling/verifying kagglehub..."
)

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "kagglehub"
    ]
)

import kagglehub

print(
    "✅ kagglehub ready."
)


# ============================================================
# 2. Dataset handles
#
# X-IIoTID:
#   Author-provided Kaggle dataset.
#
# ToN-IoT:
#   Public mirror of the ToN-IoT dataset.
#
# Official ToN-IoT provenance is retained separately below.
# ============================================================

DATASETS = {
    "ton_iot": {
        "kaggle_handle":
            "richardjacob04/ton-iot-datasets",

        "target_dir":
            TON_DIR,

        "official_source":
            "https://research.unsw.edu.au/projects/toniot-datasets",

        "description":
            "ToN-IoT public dataset copy; official dataset source is UNSW Canberra."
    },

    "xiiotid": {
        "kaggle_handle":
            "munaalhawawreh/xiiotid-iiot-intrusion-dataset",

        "target_dir":
            XIIOT_DIR,

        "official_source":
            "https://github.com/Alhawawreh/X-IIoTID",

        "description":
            "X-IIoTID public dataset released by the dataset author."
    }
}


# ============================================================
# 3. Utility: SHA256
#
# We hash files so the final repository can record exactly
# which raw files generated the reported results.
# ============================================================

def sha256_file(path, chunk_size=1024 * 1024):

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                chunk_size
            )

            if not chunk:
                break

            h.update(
                chunk
            )

    return h.hexdigest()


# ============================================================
# 4. Utility: recursively copy KaggleHub download
# ============================================================

def copy_downloaded_dataset(
    source_dir,
    target_dir
):

    source_dir = Path(
        source_dir
    )

    target_dir = Path(
        target_dir
    )

    target_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    copied = []

    for src in source_dir.rglob("*"):

        if not src.is_file():
            continue

        rel = src.relative_to(
            source_dir
        )

        dst = (
            target_dir /
            rel
        )

        dst.parent.mkdir(
            parents=True,
            exist_ok=True
        )

        # Do not repeatedly copy an unchanged large file.
        if (
            dst.exists()
            and dst.stat().st_size
            == src.stat().st_size
        ):
            print(
                f"  already present: {rel}"
            )

        else:

            shutil.copy2(
                src,
                dst
            )

            print(
                f"  copied: {rel}"
            )

        copied.append(
            dst
        )

    return copied


# ============================================================
# 5. Download both datasets
# ============================================================

download_records = {}

for dataset_name, info in DATASETS.items():

    print(
        "\n================================================"
    )

    print(
        f"Downloading: {dataset_name}"
    )

    print(
        f"Kaggle handle: {info['kaggle_handle']}"
    )

    print(
        "================================================"
    )

    try:

        cache_path = (
            kagglehub.dataset_download(
                info["kaggle_handle"]
            )
        )

    except Exception as e:

        print(
            "\n❌ KaggleHub download failed."
        )

        print(
            "Error:"
        )

        print(
            repr(e)
        )

        print(
            "\nIf Kaggle requests authentication, "
            "open Kaggle -> Settings -> API -> "
            "Create New Token, upload kaggle.json "
            "to the Colab session, and rerun this cell."
        )

        raise


    print(
        "\nDownloaded to KaggleHub cache:"
    )

    print(
        cache_path
    )


    # --------------------------------------------------------
    # Copy into repository
    # --------------------------------------------------------

    copied_files = copy_downloaded_dataset(
        cache_path,
        info["target_dir"]
    )

    if not copied_files:

        raise RuntimeError(
            f"No files downloaded for {dataset_name}"
        )

    download_records[
        dataset_name
    ] = copied_files

    print(
        f"\n✅ {dataset_name}: "
        f"{len(copied_files)} files saved."
    )


# ============================================================
# 6. Inventory the downloaded files
# ============================================================

def inventory_dataset(
    dataset_name,
    dataset_dir
):

    files = sorted(
        [
            p
            for p in dataset_dir.rglob("*")
            if p.is_file()
        ]
    )

    print(
        "\n================================================"
    )

    print(
        f"{dataset_name.upper()} FILE INVENTORY"
    )

    print(
        "================================================"
    )

    rows = []

    for p in files:

        size_mb = (
            p.stat().st_size
            / (1024 ** 2)
        )

        row = {
            "path":
                str(
                    p.relative_to(ROOT)
                ),

            "size_bytes":
                int(
                    p.stat().st_size
                ),

            "size_mb":
                round(
                    size_mb,
                    3
                )
        }

        rows.append(
            row
        )

        print(
            f"{row['path']} "
            f"({size_mb:.2f} MB)"
        )

    return rows


ton_inventory = inventory_dataset(
    "ton_iot",
    TON_DIR
)

xiiot_inventory = inventory_dataset(
    "xiiotid",
    XIIOT_DIR
)


# ============================================================
# 7. Identify tabular dataset files
# ============================================================

TABULAR_EXTENSIONS = {
    ".csv",
    ".parquet",
    ".xlsx",
    ".xls",
}

ton_tabular = sorted(
    [
        p
        for p in TON_DIR.rglob("*")
        if (
            p.is_file()
            and p.suffix.lower()
            in TABULAR_EXTENSIONS
        )
    ]
)

xiiot_tabular = sorted(
    [
        p
        for p in XIIOT_DIR.rglob("*")
        if (
            p.is_file()
            and p.suffix.lower()
            in TABULAR_EXTENSIONS
        )
    ]
)


print(
    "\n========== ToN-IoT TABULAR FILES =========="
)

for p in ton_tabular:

    print(
        p.relative_to(ROOT)
    )


print(
    "\n========== X-IIoTID TABULAR FILES =========="
)

for p in xiiot_tabular:

    print(
        p.relative_to(ROOT)
    )


if len(ton_tabular) == 0:

    raise RuntimeError(
        "No ToN-IoT CSV/Parquet/Excel files were found."
    )


if len(xiiot_tabular) == 0:

    raise RuntimeError(
        "No X-IIoTID CSV/Parquet/Excel files were found."
    )


# ============================================================
# 8. Create reproducibility provenance metadata
#
# Hash only the tabular files we may use for experiments.
# ============================================================

print(
    "\nComputing SHA256 hashes..."
)

provenance = {
    "download_time_utc":
        time.strftime(
            "%Y-%m-%dT%H:%M:%SZ",
            time.gmtime()
        ),

    "datasets": {}
}


for dataset_name, info, tabular_files in [

    (
        "ton_iot",
        DATASETS["ton_iot"],
        ton_tabular
    ),

    (
        "xiiotid",
        DATASETS["xiiotid"],
        xiiot_tabular
    ),
]:

    file_metadata = []

    for p in tabular_files:

        print(
            f"  hashing {p.name}..."
        )

        file_metadata.append(
            {
                "relative_path":
                    str(
                        p.relative_to(ROOT)
                    ),

                "size_bytes":
                    int(
                        p.stat().st_size
                    ),

                "sha256":
                    sha256_file(
                        p
                    )
            }
        )

    provenance[
        "datasets"
    ][
        dataset_name
    ] = {

        "kaggle_handle":
            info[
                "kaggle_handle"
            ],

        "official_source":
            info[
                "official_source"
            ],

        "description":
            info[
                "description"
            ],

        "files":
            file_metadata
    }


PROVENANCE_FILE = (
    RAW_ROOT /
    "dataset_provenance.json"
)

PROVENANCE_FILE.write_text(
    json.dumps(
        provenance,
        indent=2
    )
)

print(
    "\n✅ Dataset provenance saved:"
)

print(
    PROVENANCE_FILE
)


# ============================================================
# 9. Inspect ONLY headers / tiny samples
#
# We do not load the complete datasets yet.
# ============================================================

import pandas as pd


def inspect_tabular_file(
    path
):

    print(
        "\n----------------------------------------"
    )

    print(
        "File:"
    )

    print(
        path.relative_to(ROOT)
    )

    try:

        suffix = (
            path.suffix.lower()
        )

        if suffix == ".csv":

            sample = pd.read_csv(
                path,
                nrows=3,
                low_memory=False
            )

        elif suffix == ".parquet":

            sample = pd.read_parquet(
                path
            ).head(3)

        elif suffix in {
            ".xlsx",
            ".xls",
        }:

            sample = pd.read_excel(
                path,
                nrows=3
            )

        else:

            return

        print(
            f"Columns = {len(sample.columns)}"
        )

        print(
            list(
                sample.columns
            )
        )

        print(
            "\nSample:"
        )

        print(
            sample.head(2).to_string(
                index=False
            )
        )

    except Exception as e:

        print(
            "⚠️ Inspection failed:"
        )

        print(
            repr(e)
        )


print(
    "\n\n========== ToN-IoT HEADER INSPECTION =========="
)

for p in ton_tabular[:10]:

    inspect_tabular_file(
        p
    )


print(
    "\n\n========== X-IIoTID HEADER INSPECTION =========="
)

for p in xiiot_tabular[:5]:

    inspect_tabular_file(
        p
    )


# ============================================================
# 10. Final status
# ============================================================

print(
    "\n============================================"
)

print(
    "✅ STEP 20.1 DATASET DOWNLOAD COMPLETED"
)

print(
    "============================================"
)

print(
    "\nSaved locations:"
)

print(
    f"ToN-IoT : {TON_DIR}"
)

print(
    f"X-IIoTID: {XIIOT_DIR}"
)

print(
    "\nProvenance:"
)

print(
    PROVENANCE_FILE
)

print(
    "\nToN-IoT tabular files:",
    len(ton_tabular)
)

print(
    "X-IIoTID tabular files:",
    len(xiiot_tabular)
)

print(
    "\nDo NOT preprocess yet."
)

print(
    "Send me the output showing:"
)

print(
    "  1. ToN-IoT tabular filenames"
)

print(
    "  2. ToN-IoT columns"
)

print(
    "  3. X-IIoTID filename"
)

print(
    "  4. X-IIoTID columns"
)

Repository root:
/content

Raw dataset directory:
/content/data/raw

Installing/verifying kagglehub...
✅ kagglehub ready.

Downloading: ton_iot
Kaggle handle: richardjacob04/ton-iot-datasets
Using Colab cache for faster access to the 'ton-iot-datasets' dataset.

Downloaded to KaggleHub cache:
/kaggle/input/ton-iot-datasets
  copied: train_test_network.csv

✅ ton_iot: 1 files saved.

Downloading: xiiotid
Kaggle handle: munaalhawawreh/xiiotid-iiot-intrusion-dataset
Using Colab cache for faster access to the 'xiiotid-iiot-intrusion-dataset' dataset.

Downloaded to KaggleHub cache:
/kaggle/input/xiiotid-iiot-intrusion-dataset
  copied: X-IIoTID dataset.csv

✅ xiiotid: 1 files saved.

TON_IOT FILE INVENTORY
data/raw/ton_iot/train_test_network.csv (28.52 MB)

XIIOTID FILE INVENTORY
data/raw/xiiotid/X-IIoTID dataset.csv (338.85 MB)

========== ToN-IoT TABULAR FILES ==========
data/raw/ton_iot/train_test_network.csv

========== X-IIoTID TABULAR FILES ==========
data/raw/xiiotid/X-IIoTID datase

**Step 20.2 — Move datasets into repo and preprocess ToN-IoT**

In [5]:
# ============================================================
# STEP 20.2A — Persist repository + datasets in Google Drive
#
# Persistent location:
#   /content/drive/MyDrive/fl_ids_repo_real
#
# Compatibility path:
#   /content/fl_ids_repo_real
#
# Existing notebook code can continue using:
#   /content/fl_ids_repo_real
# ============================================================

from pathlib import Path
import shutil
import os
import subprocess
import sys

# ------------------------------------------------------------
# 1. Mount Google Drive
# ------------------------------------------------------------

try:
    from google.colab import drive

    drive.mount(
        "/content/drive",
        force_remount=False
    )

except Exception as e:
    print(
        "Drive may already be mounted:",
        e
    )


# ------------------------------------------------------------
# 2. Define paths
# ------------------------------------------------------------

DRIVE_REPO = Path(
    "/content/drive/MyDrive/fl_ids_repo_real"
)

COMPAT_REPO = Path(
    "/content/fl_ids_repo_real"
)

TEMP_RAW = Path(
    "/content/data/raw"
)

GITHUB_URL = (
    "https://github.com/"
    "iqbalmansoor1986-hue/"
    "fl-iiot-intrusion-detection.git"
)


print(
    "\nPersistent repository:"
)
print(
    DRIVE_REPO
)

print(
    "\nCompatibility path:"
)
print(
    COMPAT_REPO
)


# ------------------------------------------------------------
# 3. Restore repository into Google Drive if needed
# ------------------------------------------------------------

if not DRIVE_REPO.exists():

    print(
        "\nRepository not yet present in Drive."
    )

    print(
        "Cloning GitHub repository into Drive..."
    )

    subprocess.check_call(
        [
            "git",
            "clone",
            GITHUB_URL,
            str(DRIVE_REPO)
        ]
    )

    print(
        "✅ Repository cloned into Google Drive."
    )

else:

    print(
        "\n✅ Persistent repository already exists in Drive."
    )


# ------------------------------------------------------------
# 4. Ensure expected repository directories exist
# ------------------------------------------------------------

(DRIVE_REPO / "data" / "raw").mkdir(
    parents=True,
    exist_ok=True
)

(DRIVE_REPO / "data" / "processed").mkdir(
    parents=True,
    exist_ok=True
)

(DRIVE_REPO / "runs").mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# 5. Copy downloaded ToN-IoT into persistent repository
# ------------------------------------------------------------

SRC_TON = (
    TEMP_RAW /
    "ton_iot" /
    "train_test_network.csv"
)

DST_TON = (
    DRIVE_REPO /
    "data" /
    "raw" /
    "ton_iot" /
    "train_test_network.csv"
)

DST_TON.parent.mkdir(
    parents=True,
    exist_ok=True
)

if SRC_TON.exists():

    if not DST_TON.exists():

        print(
            "\nCopying ToN-IoT to Google Drive..."
        )

        shutil.copy2(
            SRC_TON,
            DST_TON
        )

        print(
            "✅ ToN-IoT saved persistently."
        )

    else:

        print(
            "✅ ToN-IoT already exists in Drive."
        )

else:

    print(
        "⚠️ Temporary ToN-IoT file not found:",
        SRC_TON
    )


# ------------------------------------------------------------
# 6. Copy downloaded X-IIoTID into persistent repository
# ------------------------------------------------------------

SRC_XIIOT = (
    TEMP_RAW /
    "xiiotid" /
    "X-IIoTID dataset.csv"
)

DST_XIIOT = (
    DRIVE_REPO /
    "data" /
    "raw" /
    "xiiotid" /
    "X-IIoTID dataset.csv"
)

DST_XIIOT.parent.mkdir(
    parents=True,
    exist_ok=True
)

if SRC_XIIOT.exists():

    if not DST_XIIOT.exists():

        print(
            "\nCopying X-IIoTID to Google Drive..."
        )

        shutil.copy2(
            SRC_XIIOT,
            DST_XIIOT
        )

        print(
            "✅ X-IIoTID saved persistently."
        )

    else:

        print(
            "✅ X-IIoTID already exists in Drive."
        )

else:

    print(
        "⚠️ Temporary X-IIoTID file not found:",
        SRC_XIIOT
    )


# ------------------------------------------------------------
# 7. Copy dataset provenance metadata
# ------------------------------------------------------------

SRC_PROV = (
    TEMP_RAW /
    "dataset_provenance.json"
)

DST_PROV = (
    DRIVE_REPO /
    "data" /
    "raw" /
    "dataset_provenance.json"
)

if SRC_PROV.exists():

    shutil.copy2(
        SRC_PROV,
        DST_PROV
    )

    print(
        "✅ Dataset provenance copied to Drive."
    )


# ------------------------------------------------------------
# 8. Create compatibility symlink
#
# This is the important part.
#
# Every existing cell can continue using:
#     /content/fl_ids_repo_real
#
# while actual files live persistently in Drive.
# ------------------------------------------------------------

if COMPAT_REPO.is_symlink():

    COMPAT_REPO.unlink()

elif COMPAT_REPO.exists():

    print(
        "\n⚠️ Existing non-symlink repository found at:"
    )
    print(
        COMPAT_REPO
    )

    backup = Path(
        "/content/fl_ids_repo_real_runtime_backup"
    )

    if backup.exists():

        if backup.is_dir():
            shutil.rmtree(
                backup
            )
        else:
            backup.unlink()

    shutil.move(
        str(COMPAT_REPO),
        str(backup)
    )

    print(
        "Moved existing runtime repository to:"
    )

    print(
        backup
    )


os.symlink(
    DRIVE_REPO,
    COMPAT_REPO,
    target_is_directory=True
)


print(
    "\n✅ Compatibility symbolic link created:"
)

print(
    f"{COMPAT_REPO} -> {DRIVE_REPO}"
)


# ------------------------------------------------------------
# 9. Verify compatibility path
# ------------------------------------------------------------

assert COMPAT_REPO.exists()

assert (
    COMPAT_REPO /
    "data" /
    "raw" /
    "ton_iot" /
    "train_test_network.csv"
).exists(), (
    "ToN-IoT not visible through compatibility path."
)

assert (
    COMPAT_REPO /
    "data" /
    "raw" /
    "xiiotid" /
    "X-IIoTID dataset.csv"
).exists(), (
    "X-IIoTID not visible through compatibility path."
)


print(
    "\n========== PERSISTENCE CHECK =========="
)

print(
    "Repository:"
)
print(
    COMPAT_REPO.resolve()
)

print(
    "\nToN-IoT:"
)
print(
    COMPAT_REPO /
    "data/raw/ton_iot/train_test_network.csv"
)

print(
    "\nX-IIoTID:"
)
print(
    COMPAT_REPO /
    "data/raw/xiiotid/X-IIoTID dataset.csv"
)

print(
    "\nGit repository:",
    (COMPAT_REPO / ".git").exists()
)

print(
    "======================================="
)


print(
    "\n✅ STEP 20.2A COMPLETED"
)

print(
    "\nFrom now on continue using exactly:"
)

print(
    "/content/fl_ids_repo_real"
)

print(
    "\nNo changes are required in the previous code."
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Persistent repository:
/content/drive/MyDrive/fl_ids_repo_real

Compatibility path:
/content/fl_ids_repo_real

Repository not yet present in Drive.
Cloning GitHub repository into Drive...
✅ Repository cloned into Google Drive.

Copying ToN-IoT to Google Drive...
✅ ToN-IoT saved persistently.

Copying X-IIoTID to Google Drive...
✅ X-IIoTID saved persistently.
✅ Dataset provenance copied to Drive.

✅ Compatibility symbolic link created:
/content/fl_ids_repo_real -> /content/drive/MyDrive/fl_ids_repo_real

========== PERSISTENCE CHECK ==========
Repository:
/content/drive/MyDrive/fl_ids_repo_real

ToN-IoT:
/content/fl_ids_repo_real/data/raw/ton_iot/train_test_network.csv

X-IIoTID:
/content/fl_ids_repo_real/data/raw/xiiotid/X-IIoTID dataset.csv

Git repository: True

✅ STEP 20.2A COMPLETED

From now on continue using exactly:
/content/fl_ids_repo_real

No chang

In [7]:
# ============================================================
# STEP 20.2 — Preprocess ToN-IoT + create K=10 Dirichlet split
#
# Input:
#   /content/fl_ids_repo_real/data/raw/ton_iot/
#       train_test_network.csv
#
# Outputs:
#   data/processed/ton_iot_encoded.npz
#   data/processed/ton_iot_encoded_meta.json
#   data/processed/splits_ton_iot_dirichlet_alpha_0.3.npz
#
# Also creates:
#   src/data/preprocess_toniot.py
# ============================================================

from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd

REPO = Path("/content/fl_ids_repo_real")

RAW_FILE = (
    REPO /
    "data/raw/ton_iot/train_test_network.csv"
)

PROC_DIR = (
    REPO /
    "data/processed"
)

SRC_DATA_DIR = (
    REPO /
    "src/data"
)

PROC_DIR.mkdir(
    parents=True,
    exist_ok=True
)

SRC_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

assert RAW_FILE.exists(), (
    f"Missing ToN-IoT file: {RAW_FILE}"
)

print("✅ ToN-IoT raw file found:")
print(RAW_FILE)


# ============================================================
# 1. Create reusable ToN-IoT preprocessing module
# ============================================================

PREPROCESS_FILE = (
    SRC_DATA_DIR /
    "preprocess_toniot.py"
)

preprocess_code = r'''
from pathlib import Path
import json

import numpy as np
import pandas as pd


def preprocess_toniot(
    csv_path,
    out_npz,
    out_meta,
    seed=42,
    high_cardinality_threshold=100,
):
    """
    Preprocess ToN-IoT Network for binary IDS training.

    Source labels:
        label = 0 -> benign
        label = 1 -> intrusion

    The 'type' column is stored as metadata only and is not
    included in the model features.
    """

    csv_path = Path(csv_path)
    out_npz = Path(out_npz)
    out_meta = Path(out_meta)

    df = pd.read_csv(
        csv_path,
        low_memory=False
    )

    original_rows = int(len(df))

    # --------------------------------------------------------
    # Validate labels
    # --------------------------------------------------------

    if "label" not in df.columns:
        raise ValueError(
            "ToN-IoT source does not contain 'label'."
        )

    if "type" not in df.columns:
        raise ValueError(
            "ToN-IoT source does not contain 'type'."
        )

    y = pd.to_numeric(
        df["label"],
        errors="raise"
    ).astype(np.int64)

    unique_y = sorted(
        y.unique().tolist()
    )

    if not set(unique_y).issubset({0, 1}):
        raise ValueError(
            f"Unexpected label values: {unique_y}"
        )

    attack_type = (
        df["type"]
        .fillna("unknown")
        .astype(str)
    )

    # --------------------------------------------------------
    # Remove labels from features
    # --------------------------------------------------------

    X_df = df.drop(
        columns=[
            "label",
            "type",
        ]
    ).copy()

    # --------------------------------------------------------
    # Remove identifiers / free-text fields
    #
    # These fields can encode host identity or have extremely
    # high cardinality and are not suitable direct IDS inputs.
    # --------------------------------------------------------

    explicit_drop = [
        "src_ip",
        "dst_ip",
        "dns_query",
        "ssl_subject",
        "ssl_issuer",
        "http_uri",
        "http_user_agent",
    ]

    explicit_drop = [
        c
        for c in explicit_drop
        if c in X_df.columns
    ]

    X_df.drop(
        columns=explicit_drop,
        inplace=True
    )

    # --------------------------------------------------------
    # Identify categorical columns
    # --------------------------------------------------------

    categorical = [
        c
        for c in X_df.columns
        if (
            X_df[c].dtype == "object"
            or str(X_df[c].dtype).startswith("string")
        )
    ]

    # --------------------------------------------------------
    # Drop remaining very-high-cardinality categorical fields
    # --------------------------------------------------------

    high_cardinality_drop = []

    category_counts = {}

    for col in categorical:

        n_unique = int(
            X_df[col]
            .fillna("__MISSING__")
            .astype(str)
            .nunique()
        )

        category_counts[col] = n_unique

        if n_unique > high_cardinality_threshold:
            high_cardinality_drop.append(col)

    if high_cardinality_drop:

        X_df.drop(
            columns=high_cardinality_drop,
            inplace=True
        )

    # Recompute categorical columns after dropping.
    categorical = [
        c
        for c in X_df.columns
        if (
            X_df[c].dtype == "object"
            or str(X_df[c].dtype).startswith("string")
        )
    ]

    numeric = [
        c
        for c in X_df.columns
        if c not in categorical
    ]

    # --------------------------------------------------------
    # Clean numeric values
    # --------------------------------------------------------

    for col in numeric:

        X_df[col] = pd.to_numeric(
            X_df[col],
            errors="coerce"
        )

        X_df[col] = X_df[col].replace(
            [np.inf, -np.inf],
            np.nan
        )

        if X_df[col].isna().any():

            median = X_df[col].median()

            if pd.isna(median):
                median = 0.0

            X_df[col] = (
                X_df[col]
                .fillna(median)
            )

    # --------------------------------------------------------
    # Clean categorical values
    # --------------------------------------------------------

    for col in categorical:

        X_df[col] = (
            X_df[col]
            .fillna("__MISSING__")
            .astype(str)
        )

    # --------------------------------------------------------
    # One-hot encode low-cardinality categorical variables
    # --------------------------------------------------------

    X_df = pd.get_dummies(
        X_df,
        columns=categorical,
        drop_first=False,
        dtype=np.uint8
    )

    X_df = X_df.replace(
        [np.inf, -np.inf],
        np.nan
    )

    if X_df.isna().any().any():

        bad = X_df.columns[
            X_df.isna().any()
        ].tolist()

        raise ValueError(
            f"NaNs remain in features: {bad[:20]}"
        )

    # --------------------------------------------------------
    # Convert to arrays
    # --------------------------------------------------------

    X = X_df.to_numpy(
        dtype=np.float32
    )

    y = y.to_numpy(
        dtype=np.int64
    )

    attack_type = attack_type.to_numpy()

    # --------------------------------------------------------
    # Deterministic shuffle
    # --------------------------------------------------------

    rng = np.random.default_rng(seed)

    order = rng.permutation(
        len(y)
    )

    X = X[order]
    y = y[order]
    attack_type = attack_type[order]

    # --------------------------------------------------------
    # Save
    # --------------------------------------------------------

    np.savez_compressed(
        out_npz,
        X=X,
        y=y,
        attack_type=attack_type,
    )

    meta = {
        "dataset": "ToN-IoT Network",
        "source_file": str(csv_path),
        "original_rows": original_rows,
        "final_rows": int(len(y)),
        "n_features": int(X.shape[1]),
        "label_column": "label",
        "label_mapping": {
            "0": "benign",
            "1": "intrusion"
        },
        "benign_count": int((y == 0).sum()),
        "intrusion_count": int((y == 1).sum()),
        "intrusion_fraction": float(y.mean()),
        "explicit_dropped_columns": explicit_drop,
        "high_cardinality_dropped_columns":
            high_cardinality_drop,
        "categorical_unique_counts":
            category_counts,
        "feature_names":
            list(X_df.columns),
        "seed": int(seed),
    }

    Path(out_meta).write_text(
        json.dumps(
            meta,
            indent=2
        )
    )

    return meta
'''

PREPROCESS_FILE.write_text(
    preprocess_code
)

print("\n✅ Created:")
print(PREPROCESS_FILE)


# ============================================================
# 2. Import preprocessor
# ============================================================

if str(REPO) not in sys.path:
    sys.path.insert(
        0,
        str(REPO)
    )

from src.data.preprocess_toniot import (
    preprocess_toniot
)


# ============================================================
# 3. Preprocess ToN-IoT
# ============================================================

ENCODED_FILE = (
    PROC_DIR /
    "ton_iot_encoded.npz"
)

META_FILE = (
    PROC_DIR /
    "ton_iot_encoded_meta.json"
)

print(
    "\nPreprocessing ToN-IoT..."
)

meta = preprocess_toniot(
    csv_path=RAW_FILE,
    out_npz=ENCODED_FILE,
    out_meta=META_FILE,
    seed=42,
    high_cardinality_threshold=100,
)

print(
    "✅ Encoding complete."
)


# ============================================================
# 4. Validate encoded dataset
# ============================================================

data = np.load(
    ENCODED_FILE,
    allow_pickle=True
)

X = data["X"]
y = data["y"]
attack_type = data["attack_type"]

assert X.ndim == 2
assert y.ndim == 1
assert len(X) == len(y)
assert set(np.unique(y)) == {0, 1}
assert np.isfinite(X).all()

print(
    "\n========== TON-IOT ENCODED DATA =========="
)

print(
    "records:",
    len(y)
)

print(
    "features:",
    X.shape[1]
)

print(
    "benign:",
    int((y == 0).sum())
)

print(
    "intrusion:",
    int((y == 1).sum())
)

print(
    "intrusion fraction:",
    round(
        float(y.mean()),
        4
    )
)

print(
    "attack types:",
    sorted(
        np.unique(
            attack_type
        ).tolist()
    )
)

print(
    "dropped identifiers:",
    meta[
        "explicit_dropped_columns"
    ]
)

print(
    "dropped high-cardinality:",
    meta[
        "high_cardinality_dropped_columns"
    ]
)

print(
    "=========================================="
)


# ============================================================
# 5. Create Dirichlet K=10, alpha=0.3 partition
#
# Partition is label-driven.
# Each sample belongs to exactly one silo.
# ============================================================

K = 10
ALPHA = 0.3
SEED = 42

TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

rng = np.random.default_rng(
    SEED
)

silo_indices = [
    []
    for _ in range(K)
]

# ------------------------------------------------------------
# Partition each class separately using Dirichlet proportions
# ------------------------------------------------------------

for cls in [0, 1]:

    cls_idx = np.where(
        y == cls
    )[0]

    rng.shuffle(
        cls_idx
    )

    proportions = rng.dirichlet(
        np.full(
            K,
            ALPHA
        )
    )

    counts = rng.multinomial(
        len(cls_idx),
        proportions
    )

    start = 0

    for k, count in enumerate(
        counts
    ):

        end = start + count

        silo_indices[k].extend(
            cls_idx[
                start:end
            ].tolist()
        )

        start = end


# ============================================================
# 6. Create 70/15/15 split inside each silo
# ============================================================

train_splits = []
val_splits = []
test_splits = []

for k in range(K):

    idx = np.asarray(
        silo_indices[k],
        dtype=np.int64
    )

    rng.shuffle(
        idx
    )

    n = len(idx)

    if n < 10:
        raise RuntimeError(
            f"Silo {k} has too few samples: {n}"
        )

    n_train = int(
        np.floor(
            TRAIN_RATIO * n
        )
    )

    n_val = int(
        np.floor(
            VAL_RATIO * n
        )
    )

    train_idx = idx[
        :n_train
    ]

    val_idx = idx[
        n_train:
        n_train + n_val
    ]

    test_idx = idx[
        n_train + n_val:
    ]

    train_splits.append(
        train_idx
    )

    val_splits.append(
        val_idx
    )

    test_splits.append(
        test_idx
    )


# ============================================================
# 7. Save split in repository-compatible NPZ format
# ============================================================

SPLIT_FILE = (
    PROC_DIR /
    "splits_ton_iot_dirichlet_alpha_0.3.npz"
)

np.savez_compressed(
    SPLIT_FILE,

    train=np.array(
        train_splits,
        dtype=object
    ),

    val=np.array(
        val_splits,
        dtype=object
    ),

    test=np.array(
        test_splits,
        dtype=object
    ),

    K=np.array(
        [K],
        dtype=np.int64
    ),

    alpha=np.array(
        [ALPHA],
        dtype=np.float32
    ),

    seed=np.array(
        [SEED],
        dtype=np.int64
    ),
)


# ============================================================
# 8. Audit split
# ============================================================

print(
    "\n========== TON-IOT SILO AUDIT =========="
)

all_assigned = []

for k in range(K):

    combined = np.concatenate(
        [
            train_splits[k],
            val_splits[k],
            test_splits[k],
        ]
    )

    all_assigned.extend(
        combined.tolist()
    )

    silo_y = y[
        combined
    ]

    print(
        f"Silo {k:2d} | "
        f"total={len(combined):6d} | "
        f"train={len(train_splits[k]):6d} | "
        f"val={len(val_splits[k]):6d} | "
        f"test={len(test_splits[k]):6d} | "
        f"intrusion_rate={silo_y.mean():.4f}"
    )


all_assigned = np.asarray(
    all_assigned,
    dtype=np.int64
)

assert len(all_assigned) == len(y), (
    "Some samples were not assigned."
)

assert len(
    np.unique(
        all_assigned
    )
) == len(y), (
    "Duplicate assignment detected."
)

print(
    "\n✅ Every ToN-IoT record assigned exactly once."
)


# ============================================================
# 9. Save split summary
# ============================================================

summary = {
    "dataset":
        "ToN-IoT Network",

    "K":
        K,

    "alpha":
        ALPHA,

    "seed":
        SEED,

    "split_ratios": {
        "train":
            TRAIN_RATIO,

        "validation":
            VAL_RATIO,

        "test":
            TEST_RATIO,
    },

    "silos": []
}

for k in range(K):

    combined = np.concatenate(
        [
            train_splits[k],
            val_splits[k],
            test_splits[k],
        ]
    )

    summary["silos"].append(
        {
            "silo":
                k,

            "total":
                int(
                    len(combined)
                ),

            "train":
                int(
                    len(train_splits[k])
                ),

            "validation":
                int(
                    len(val_splits[k])
                ),

            "test":
                int(
                    len(test_splits[k])
                ),

            "intrusion_fraction":
                float(
                    y[
                        combined
                    ].mean()
                ),
        }
    )


SUMMARY_FILE = (
    PROC_DIR /
    "splits_ton_iot_dirichlet_alpha_0.3_summary.json"
)

SUMMARY_FILE.write_text(
    json.dumps(
        summary,
        indent=2
    )
)


# ============================================================
# 10. Final status
# ============================================================

print(
    "\n============================================"
)

print(
    "✅ STEP 20.2 ToN-IoT PREPROCESSING PASSED"
)

print(
    "============================================"
)

print(
    "\nEncoded dataset:"
)

print(
    ENCODED_FILE
)

print(
    "\nMetadata:"
)

print(
    META_FILE
)

print(
    "\nDirichlet split:"
)

print(
    SPLIT_FILE
)

print(
    "\nSplit summary:"
)

print(
    SUMMARY_FILE
)

print(
    "\nReusable preprocessor:"
)

print(
    PREPROCESS_FILE
)

print(
    "\nFinal:")
print(
    f"  records   = {len(y)}"
)
print(
    f"  input_dim = {X.shape[1]}"
)
print(
    f"  K         = {K}"
)
print(
    f"  alpha     = {ALPHA}"
)

✅ ToN-IoT raw file found:
/content/fl_ids_repo_real/data/raw/ton_iot/train_test_network.csv

✅ Created:
/content/fl_ids_repo_real/src/data/preprocess_toniot.py

Preprocessing ToN-IoT...
✅ Encoding complete.

========== TON-IOT ENCODED DATA ==========
records: 211043
features: 115
benign: 50000
intrusion: 161043
intrusion fraction: 0.7631
attack types: ['backdoor', 'ddos', 'dos', 'injection', 'mitm', 'normal', 'password', 'ransomware', 'scanning', 'xss']
dropped identifiers: ['src_ip', 'dst_ip', 'dns_query', 'ssl_subject', 'ssl_issuer', 'http_uri', 'http_user_agent']
dropped high-cardinality: []

========== TON-IOT SILO AUDIT ==========
Silo  0 | total= 93789 | train= 65652 | val= 14068 | test= 14069 | intrusion_rate=0.9932
Silo  1 | total= 25047 | train= 17532 | val=  3757 | test=  3758 | intrusion_rate=0.0884
Silo  2 | total= 11461 | train=  8022 | val=  1719 | test=  1720 | intrusion_rate=0.9559
Silo  3 | total=   678 | train=   474 | val=   101 | test=   103 | intrusion_rate=0.2094


**Step 20.3 — Preprocess X-IIoTID and create the same \(K=10,\alpha=0.3\) pipeline**

In [9]:
# ============================================================
# STEP 20.3 — X-IIoTID preprocessing + K=10 Dirichlet split
#
# Binary target:
#   class3 = Normal -> 0
#   class3 = Attack -> 1
#
# class1/class2 are retained only as attack metadata and are
# NOT used as model features.
#
# Outputs:
#   data/processed/xiiotid_encoded.npz
#   data/processed/xiiotid_encoded_meta.json
#   data/processed/splits_xiiotid_dirichlet_alpha_0.3.npz
#   data/processed/splits_xiiotid_dirichlet_alpha_0.3_summary.json
#
# Creates:
#   src/data/preprocess_xiiotid.py
# ============================================================

from pathlib import Path
import json
import sys
import gc
import numpy as np
import pandas as pd

REPO = Path("/content/fl_ids_repo_real")

RAW_FILE = (
    REPO /
    "data/raw/xiiotid/X-IIoTID dataset.csv"
)

PROC_DIR = (
    REPO /
    "data/processed"
)

SRC_DATA_DIR = (
    REPO /
    "src/data"
)

PROC_DIR.mkdir(
    parents=True,
    exist_ok=True
)

SRC_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

assert RAW_FILE.exists(), (
    f"Missing X-IIoTID file: {RAW_FILE}"
)

print("✅ X-IIoTID raw file found:")
print(RAW_FILE)


# ============================================================
# 1. First validate class3 BEFORE preprocessing
# ============================================================

label_check = pd.read_csv(
    RAW_FILE,
    usecols=[
        "class1",
        "class2",
        "class3",
    ],
    low_memory=False
)

print(
    "\n========== X-IIoTID LABEL AUDIT =========="
)

print(
    "class3 values:"
)

print(
    label_check["class3"]
    .astype(str)
    .str.strip()
    .value_counts(dropna=False)
)

print(
    "\nclass2 top categories:"
)

print(
    label_check["class2"]
    .astype(str)
    .str.strip()
    .value_counts(dropna=False)
    .head(20)
)

normalized_class3 = set(
    label_check["class3"]
    .astype(str)
    .str.strip()
    .str.lower()
    .unique()
    .tolist()
)

assert normalized_class3.issubset(
    {"normal", "attack"}
), (
    "Unexpected class3 labels: "
    f"{sorted(normalized_class3)}"
)

print(
    "\n✅ class3 confirmed as binary Normal/Attack."
)

del label_check
gc.collect()


# ============================================================
# 2. Create reusable preprocessing module
# ============================================================

PREPROCESS_FILE = (
    SRC_DATA_DIR /
    "preprocess_xiiotid.py"
)

preprocess_code = r'''
from pathlib import Path
import json

import numpy as np
import pandas as pd


def preprocess_xiiotid(
    csv_path,
    out_npz,
    out_meta,
    seed=42,
    high_cardinality_threshold=100,
):
    """
    Preprocess X-IIoTID for binary intrusion detection.

    Binary target:
        class3 == Normal -> 0
        class3 == Attack -> 1

    class1 and class2 are stored only as attack metadata.
    They are NOT model features.
    """

    csv_path = Path(csv_path)
    out_npz = Path(out_npz)
    out_meta = Path(out_meta)

    df = pd.read_csv(
        csv_path,
        low_memory=False
    )

    original_rows = int(len(df))
    original_columns = list(df.columns)

    required = {
        "class1",
        "class2",
        "class3",
    }

    missing = (
        required -
        set(df.columns)
    )

    if missing:
        raise ValueError(
            "Missing required X-IIoTID columns: "
            + str(sorted(missing))
        )

    # --------------------------------------------------------
    # Extract labels / attack metadata
    # --------------------------------------------------------

    class3 = (
        df.pop("class3")
        .astype(str)
        .str.strip()
        .str.lower()
    )

    unique_class3 = sorted(
        class3.unique().tolist()
    )

    if not set(unique_class3).issubset(
        {"normal", "attack"}
    ):
        raise ValueError(
            "Unexpected class3 values: "
            + str(unique_class3)
        )

    y = (
        class3
        .map({
            "normal": 0,
            "attack": 1,
        })
        .astype(np.int64)
        .to_numpy()
    )

    attack_class1 = (
        df.pop("class1")
        .fillna("unknown")
        .astype(str)
        .to_numpy()
    )

    attack_class2 = (
        df.pop("class2")
        .fillna("unknown")
        .astype(str)
        .to_numpy()
    )

    # --------------------------------------------------------
    # Remove identifiers / timestamps
    #
    # These are not treated as model features.
    # --------------------------------------------------------

    explicit_drop = [
        "Date",
        "Timestamp",
        "Scr_IP",
        "Des_IP",
    ]

    explicit_drop = [
        c
        for c in explicit_drop
        if c in df.columns
    ]

    df.drop(
        columns=explicit_drop,
        inplace=True
    )

    # --------------------------------------------------------
    # Convert booleans to numeric explicitly
    # --------------------------------------------------------

    bool_columns = [
        c
        for c in df.columns
        if pd.api.types.is_bool_dtype(
            df[c]
        )
    ]

    for col in bool_columns:
        df[col] = (
            df[col]
            .astype(np.uint8)
        )

    # --------------------------------------------------------
    # Identify categorical fields
    # --------------------------------------------------------

    categorical = [
        c
        for c in df.columns
        if (
            df[c].dtype == "object"
            or str(df[c].dtype).startswith("string")
        )
    ]

    category_counts = {}

    high_cardinality_drop = []

    for col in categorical:

        n_unique = int(
            df[col]
            .fillna("__MISSING__")
            .astype(str)
            .nunique()
        )

        category_counts[col] = n_unique

        if (
            n_unique >
            high_cardinality_threshold
        ):
            high_cardinality_drop.append(
                col
            )

    if high_cardinality_drop:

        df.drop(
            columns=high_cardinality_drop,
            inplace=True
        )

    # --------------------------------------------------------
    # Recompute remaining categorical columns
    # --------------------------------------------------------

    categorical = [
        c
        for c in df.columns
        if (
            df[c].dtype == "object"
            or str(df[c].dtype).startswith("string")
        )
    ]

    numeric = [
        c
        for c in df.columns
        if c not in categorical
    ]

    # --------------------------------------------------------
    # Numeric cleaning
    # --------------------------------------------------------

    for col in numeric:

        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )

        df[col] = df[col].replace(
            [np.inf, -np.inf],
            np.nan
        )

        if df[col].isna().any():

            median = df[col].median()

            if pd.isna(median):
                median = 0.0

            df[col] = (
                df[col]
                .fillna(median)
            )

    # --------------------------------------------------------
    # Categorical cleaning
    # --------------------------------------------------------

    for col in categorical:

        df[col] = (
            df[col]
            .fillna("__MISSING__")
            .astype(str)
        )

    # --------------------------------------------------------
    # One-hot encode low-cardinality categorical fields
    # --------------------------------------------------------

    X_df = pd.get_dummies(
        df,
        columns=categorical,
        drop_first=False,
        dtype=np.uint8,
    )

    del df

    X_df = X_df.replace(
        [np.inf, -np.inf],
        np.nan
    )

    if X_df.isna().any().any():

        bad = (
            X_df.columns[
                X_df.isna().any()
            ]
            .tolist()
        )

        raise ValueError(
            "NaNs remain after preprocessing: "
            + str(bad[:20])
        )

    # --------------------------------------------------------
    # Convert to float32 feature matrix
    # --------------------------------------------------------

    X = X_df.to_numpy(
        dtype=np.float32
    )

    feature_names = list(
        X_df.columns
    )

    del X_df

    # --------------------------------------------------------
    # Deterministic shuffle
    # --------------------------------------------------------

    rng = np.random.default_rng(
        seed
    )

    order = rng.permutation(
        len(y)
    )

    X = X[order]
    y = y[order]

    attack_class1 = (
        attack_class1[order]
    )

    attack_class2 = (
        attack_class2[order]
    )

    # --------------------------------------------------------
    # Save encoded dataset
    # --------------------------------------------------------

    np.savez_compressed(
        out_npz,
        X=X,
        y=y,
        attack_class1=attack_class1,
        attack_class2=attack_class2,
    )

    # --------------------------------------------------------
    # Save preprocessing metadata
    # --------------------------------------------------------

    meta = {
        "dataset":
            "X-IIoTID",

        "source_file":
            str(csv_path),

        "original_rows":
            original_rows,

        "final_rows":
            int(len(y)),

        "original_columns":
            original_columns,

        "n_features":
            int(X.shape[1]),

        "label_column":
            "class3",

        "label_mapping": {
            "Normal": 0,
            "Attack": 1,
        },

        "benign_count":
            int(
                np.sum(
                    y == 0
                )
            ),

        "intrusion_count":
            int(
                np.sum(
                    y == 1
                )
            ),

        "intrusion_fraction":
            float(
                np.mean(y)
            ),

        "attack_metadata_columns": [
            "class1",
            "class2",
        ],

        "explicit_dropped_columns":
            explicit_drop,

        "high_cardinality_dropped_columns":
            high_cardinality_drop,

        "categorical_unique_counts":
            category_counts,

        "boolean_columns":
            bool_columns,

        "feature_names":
            feature_names,

        "seed":
            int(seed),
    }

    Path(out_meta).write_text(
        json.dumps(
            meta,
            indent=2
        )
    )

    return meta
'''

PREPROCESS_FILE.write_text(
    preprocess_code
)

print(
    "\n✅ Created reusable X-IIoTID preprocessor:"
)

print(
    PREPROCESS_FILE
)


# ============================================================
# 3. Import and execute preprocessing
# ============================================================

if str(REPO) not in sys.path:

    sys.path.insert(
        0,
        str(REPO)
    )

# Avoid stale notebook import if cell is rerun.
if "src.data.preprocess_xiiotid" in sys.modules:
    del sys.modules[
        "src.data.preprocess_xiiotid"
    ]

from src.data.preprocess_xiiotid import (
    preprocess_xiiotid
)


ENCODED_FILE = (
    PROC_DIR /
    "xiiotid_encoded.npz"
)

META_FILE = (
    PROC_DIR /
    "xiiotid_encoded_meta.json"
)


print(
    "\nPreprocessing X-IIoTID..."
)

print(
    "This file is large; this step may take several minutes."
)

meta = preprocess_xiiotid(
    csv_path=RAW_FILE,
    out_npz=ENCODED_FILE,
    out_meta=META_FILE,
    seed=42,
    high_cardinality_threshold=100,
)

print(
    "✅ X-IIoTID encoding complete."
)


# ============================================================
# 4. Validate encoded artifact
# ============================================================

data = np.load(
    ENCODED_FILE,
    allow_pickle=True
)

X = data["X"]

y = (
    data["y"]
    .astype(np.int64)
)

attack_class1 = (
    data["attack_class1"]
)

attack_class2 = (
    data["attack_class2"]
)


assert X.ndim == 2

assert y.ndim == 1

assert len(X) == len(y)

assert set(
    np.unique(y).tolist()
) == {
    0,
    1,
}

assert np.isfinite(
    X
).all()


print(
    "\n========== X-IIoTID ENCODED DATA =========="
)

print(
    "records:",
    len(y)
)

print(
    "features:",
    X.shape[1]
)

print(
    "benign:",
    int(
        (y == 0).sum()
    )
)

print(
    "intrusion:",
    int(
        (y == 1).sum()
    )
)

print(
    "intrusion fraction:",
    round(
        float(
            y.mean()
        ),
        4
    )
)

print(
    "class1 categories:",
    len(
        np.unique(
            attack_class1
        )
    )
)

print(
    "class2 categories:",
    len(
        np.unique(
            attack_class2
        )
    )
)

print(
    "dropped identifiers:",
    meta[
        "explicit_dropped_columns"
    ]
)

print(
    "dropped high-cardinality:",
    meta[
        "high_cardinality_dropped_columns"
    ]
)

print(
    "============================================"
)


# ============================================================
# 5. Create K=10 Dirichlet alpha=0.3 partition
# ============================================================

K = 10

ALPHA = 0.3

SEED = 42

TRAIN_RATIO = 0.70

VAL_RATIO = 0.15

TEST_RATIO = 0.15


rng = np.random.default_rng(
    SEED
)

silo_indices = [
    []
    for _ in range(K)
]


# ------------------------------------------------------------
# Partition benign and intrusion classes independently
# ------------------------------------------------------------

for cls in [
    0,
    1,
]:

    cls_idx = np.where(
        y == cls
    )[0]

    rng.shuffle(
        cls_idx
    )

    proportions = rng.dirichlet(
        np.full(
            K,
            ALPHA
        )
    )

    counts = rng.multinomial(
        len(cls_idx),
        proportions
    )

    start = 0

    for k, count in enumerate(
        counts
    ):

        end = (
            start +
            count
        )

        silo_indices[
            k
        ].extend(
            cls_idx[
                start:end
            ].tolist()
        )

        start = end


# ============================================================
# 6. Within-silo 70/15/15 splits
# ============================================================

train_splits = []

val_splits = []

test_splits = []


for k in range(K):

    idx = np.asarray(
        silo_indices[k],
        dtype=np.int64
    )

    rng.shuffle(
        idx
    )

    n = len(
        idx
    )

    if n < 10:

        raise RuntimeError(
            f"Silo {k} contains only {n} samples."
        )

    n_train = int(
        np.floor(
            TRAIN_RATIO *
            n
        )
    )

    n_val = int(
        np.floor(
            VAL_RATIO *
            n
        )
    )

    train_splits.append(
        idx[
            :n_train
        ]
    )

    val_splits.append(
        idx[
            n_train:
            n_train + n_val
        ]
    )

    test_splits.append(
        idx[
            n_train + n_val:
        ]
    )


# ============================================================
# 7. Save split artifact
# ============================================================

SPLIT_FILE = (
    PROC_DIR /
    "splits_xiiotid_dirichlet_alpha_0.3.npz"
)


np.savez_compressed(
    SPLIT_FILE,

    train=np.array(
        train_splits,
        dtype=object
    ),

    val=np.array(
        val_splits,
        dtype=object
    ),

    test=np.array(
        test_splits,
        dtype=object
    ),

    K=np.array(
        [K],
        dtype=np.int64
    ),

    alpha=np.array(
        [ALPHA],
        dtype=np.float32
    ),

    seed=np.array(
        [SEED],
        dtype=np.int64
    ),
)


# ============================================================
# 8. Audit silos
# ============================================================

print(
    "\n========== X-IIoTID SILO AUDIT =========="
)


all_assigned = []


for k in range(K):

    combined = np.concatenate(
        [
            train_splits[k],
            val_splits[k],
            test_splits[k],
        ]
    )

    all_assigned.extend(
        combined.tolist()
    )

    silo_y = y[
        combined
    ]

    print(
        f"Silo {k:2d} | "
        f"total={len(combined):7d} | "
        f"train={len(train_splits[k]):7d} | "
        f"val={len(val_splits[k]):7d} | "
        f"test={len(test_splits[k]):7d} | "
        f"intrusion_rate={silo_y.mean():.4f}"
    )


all_assigned = np.asarray(
    all_assigned,
    dtype=np.int64
)


assert len(
    all_assigned
) == len(y), (
    "Some X-IIoTID records were not assigned."
)


assert len(
    np.unique(
        all_assigned
    )
) == len(y), (
    "Duplicate X-IIoTID assignments detected."
)


print(
    "\n✅ Every X-IIoTID record assigned exactly once."
)


# ============================================================
# 9. Save split summary
# ============================================================

summary = {
    "dataset":
        "X-IIoTID",

    "K":
        K,

    "alpha":
        ALPHA,

    "seed":
        SEED,

    "split_ratios": {
        "train":
            TRAIN_RATIO,

        "validation":
            VAL_RATIO,

        "test":
            TEST_RATIO,
    },

    "silos": [],
}


for k in range(K):

    combined = np.concatenate(
        [
            train_splits[k],
            val_splits[k],
            test_splits[k],
        ]
    )

    summary[
        "silos"
    ].append(
        {
            "silo":
                k,

            "total":
                int(
                    len(combined)
                ),

            "train":
                int(
                    len(
                        train_splits[k]
                    )
                ),

            "validation":
                int(
                    len(
                        val_splits[k]
                    )
                ),

            "test":
                int(
                    len(
                        test_splits[k]
                    )
                ),

            "intrusion_fraction":
                float(
                    y[
                        combined
                    ].mean()
                ),
        }
    )


SUMMARY_FILE = (
    PROC_DIR /
    "splits_xiiotid_dirichlet_alpha_0.3_summary.json"
)


SUMMARY_FILE.write_text(
    json.dumps(
        summary,
        indent=2
    )
)


# ============================================================
# 10. Final status
# ============================================================

print(
    "\n============================================"
)

print(
    "✅ STEP 20.3 X-IIoTID PREPROCESSING PASSED"
)

print(
    "============================================"
)


print(
    "\nEncoded dataset:"
)

print(
    ENCODED_FILE
)


print(
    "\nMetadata:"
)

print(
    META_FILE
)


print(
    "\nDirichlet split:"
)

print(
    SPLIT_FILE
)


print(
    "\nSplit summary:"
)

print(
    SUMMARY_FILE
)


print(
    "\nReusable preprocessor:"
)

print(
    PREPROCESS_FILE
)


print(
    "\nFinal:"
)

print(
    f"  records   = {len(y)}"
)

print(
    f"  input_dim = {X.shape[1]}"
)

print(
    f"  K         = {K}"
)

print(
    f"  alpha     = {ALPHA}"
)

✅ X-IIoTID raw file found:
/content/fl_ids_repo_real/data/raw/xiiotid/X-IIoTID dataset.csv

========== X-IIoTID LABEL AUDIT ==========
class3 values:
class3
Normal    421417
Attack    399417
Name: count, dtype: int64

class2 top categories:
class2
Normal               421417
RDOS                 141261
Reconnaissance       127590
Weaponization         67260
Lateral _movement     31596
Exfiltration          22134
Tampering              5122
C&C                    2863
Exploitation           1133
crypto-ransomware       458
Name: count, dtype: int64

✅ class3 confirmed as binary Normal/Attack.

✅ Created reusable X-IIoTID preprocessor:
/content/fl_ids_repo_real/src/data/preprocess_xiiotid.py

Preprocessing X-IIoTID...
This file is large; this step may take several minutes.
✅ X-IIoTID encoding complete.

========== X-IIoTID ENCODED DATA ==========
records: 820834
features: 40
benign: 421417
intrusion: 399417
intrusion fraction: 0.4866
class1 categories: 19
class2 categories: 10
dropped id

**Step 20.3A — Correct X-IIoTID feature typing**

In [10]:
# ============================================================
# STEP 20.3A — Correct X-IIoTID preprocessing
#
# Fix:
#   Numeric fields loaded as object/string must NOT be dropped
#   as high-cardinality categorical variables.
#
# Strategy:
#   1. Remove labels and identifiers.
#   2. Detect boolean fields.
#   3. Convert numeric-like object columns to numeric.
#   4. One-hot encode only genuine categorical columns.
#   5. Recreate K=10, alpha=0.3 Dirichlet splits.
# ============================================================

from pathlib import Path
import json
import sys
import gc
import numpy as np
import pandas as pd

REPO = Path("/content/fl_ids_repo_real")

RAW_FILE = (
    REPO /
    "data/raw/xiiotid/X-IIoTID dataset.csv"
)

PROC_DIR = (
    REPO /
    "data/processed"
)

SRC_DATA_DIR = (
    REPO /
    "src/data"
)

assert RAW_FILE.exists()

ENCODED_FILE = (
    PROC_DIR /
    "xiiotid_encoded.npz"
)

META_FILE = (
    PROC_DIR /
    "xiiotid_encoded_meta.json"
)

SPLIT_FILE = (
    PROC_DIR /
    "splits_xiiotid_dirichlet_alpha_0.3.npz"
)

SUMMARY_FILE = (
    PROC_DIR /
    "splits_xiiotid_dirichlet_alpha_0.3_summary.json"
)

PREPROCESS_FILE = (
    SRC_DATA_DIR /
    "preprocess_xiiotid.py"
)


# ============================================================
# 1. Create corrected reusable preprocessing module
# ============================================================

preprocess_code = r'''
from pathlib import Path
import json
import numpy as np
import pandas as pd


def preprocess_xiiotid(
    csv_path,
    out_npz,
    out_meta,
    seed=42,
    numeric_parse_threshold=0.95,
    categorical_cardinality_threshold=100,
):
    """
    X-IIoTID binary preprocessing.

    Target:
        class3 == Normal -> 0
        class3 == Attack -> 1

    class1/class2 are metadata only.

    Numeric-like string/object columns are converted to numeric
    before categorical-cardinality filtering.
    """

    csv_path = Path(csv_path)

    df = pd.read_csv(
        csv_path,
        low_memory=False
    )

    original_rows = len(df)
    original_columns = list(df.columns)

    # --------------------------------------------------------
    # Labels
    # --------------------------------------------------------

    for col in ["class1", "class2", "class3"]:

        if col not in df.columns:

            raise ValueError(
                f"Missing required column: {col}"
            )

    class3 = (
        df.pop("class3")
        .astype(str)
        .str.strip()
        .str.lower()
    )

    unique_target = sorted(
        class3.unique().tolist()
    )

    if not set(unique_target).issubset(
        {"normal", "attack"}
    ):

        raise ValueError(
            f"Unexpected class3 labels: {unique_target}"
        )

    y = (
        class3
        .map({
            "normal": 0,
            "attack": 1
        })
        .astype(np.int64)
        .to_numpy()
    )

    attack_class1 = (
        df.pop("class1")
        .fillna("unknown")
        .astype(str)
        .to_numpy()
    )

    attack_class2 = (
        df.pop("class2")
        .fillna("unknown")
        .astype(str)
        .to_numpy()
    )


    # --------------------------------------------------------
    # Remove identifiers / timestamps
    # --------------------------------------------------------

    explicit_drop = [
        "Date",
        "Timestamp",
        "Scr_IP",
        "Des_IP",
    ]

    explicit_drop = [
        c
        for c in explicit_drop
        if c in df.columns
    ]

    df.drop(
        columns=explicit_drop,
        inplace=True
    )


    # --------------------------------------------------------
    # Classify columns properly
    # --------------------------------------------------------

    numeric_columns = []
    boolean_columns = []
    categorical_columns = []

    numeric_conversion_ratio = {}


    for col in list(df.columns):

        series = df[col]

        # Native numeric
        if pd.api.types.is_numeric_dtype(series):

            df[col] = pd.to_numeric(
                series,
                errors="coerce"
            )

            numeric_columns.append(col)

            continue


        # Native boolean
        if pd.api.types.is_bool_dtype(series):

            df[col] = (
                series
                .astype(np.uint8)
            )

            boolean_columns.append(col)

            continue


        # Normalize textual representation
        stripped = (
            series
            .astype(str)
            .str.strip()
        )

        lowercase = (
            stripped
            .str.lower()
        )

        unique_nonmissing = set(
            lowercase[
                ~series.isna()
            ].unique().tolist()
        )


        # Boolean stored as text
        if (
            len(unique_nonmissing) > 0
            and unique_nonmissing.issubset(
                {"true", "false"}
            )
        ):

            df[col] = (
                lowercase
                .map({
                    "true": 1,
                    "false": 0
                })
                .fillna(0)
                .astype(np.uint8)
            )

            boolean_columns.append(col)

            continue


        # ----------------------------------------------------
        # Try numeric conversion BEFORE declaring categorical
        # ----------------------------------------------------

        converted = pd.to_numeric(
            series,
            errors="coerce"
        )

        nonmissing_original = int(
            series.notna().sum()
        )

        successfully_parsed = int(
            converted.notna().sum()
        )

        ratio = (
            successfully_parsed /
            max(nonmissing_original, 1)
        )

        numeric_conversion_ratio[col] = float(
            ratio
        )


        if ratio >= numeric_parse_threshold:

            df[col] = converted

            numeric_columns.append(col)

        else:

            df[col] = (
                series
                .fillna("__MISSING__")
                .astype(str)
            )

            categorical_columns.append(col)


    # --------------------------------------------------------
    # Numeric cleaning
    # --------------------------------------------------------

    for col in numeric_columns:

        df[col] = df[col].replace(
            [np.inf, -np.inf],
            np.nan
        )

        if df[col].isna().any():

            median = df[col].median()

            if pd.isna(median):

                median = 0.0

            df[col] = (
                df[col]
                .fillna(median)
            )


    # --------------------------------------------------------
    # High-cardinality filtering applies ONLY to true
    # categorical variables.
    # --------------------------------------------------------

    category_counts = {}

    high_cardinality_drop = []


    for col in categorical_columns:

        count = int(
            df[col].nunique(
                dropna=False
            )
        )

        category_counts[col] = count

        if (
            count >
            categorical_cardinality_threshold
        ):

            high_cardinality_drop.append(
                col
            )


    if high_cardinality_drop:

        df.drop(
            columns=high_cardinality_drop,
            inplace=True
        )


    categorical_columns = [
        c
        for c in categorical_columns
        if c not in high_cardinality_drop
    ]


    # --------------------------------------------------------
    # One-hot encode genuine categorical variables only
    # --------------------------------------------------------

    X_df = pd.get_dummies(
        df,
        columns=categorical_columns,
        drop_first=False,
        dtype=np.uint8
    )


    X_df = X_df.replace(
        [np.inf, -np.inf],
        np.nan
    )


    if X_df.isna().any().any():

        bad = (
            X_df.columns[
                X_df.isna().any()
            ]
            .tolist()
        )

        raise ValueError(
            f"NaNs remain: {bad[:20]}"
        )


    feature_names = list(
        X_df.columns
    )


    X = X_df.to_numpy(
        dtype=np.float32
    )


    # --------------------------------------------------------
    # Deterministic shuffle
    # --------------------------------------------------------

    rng = np.random.default_rng(seed)

    order = rng.permutation(
        len(y)
    )

    X = X[order]
    y = y[order]

    attack_class1 = (
        attack_class1[order]
    )

    attack_class2 = (
        attack_class2[order]
    )


    # --------------------------------------------------------
    # Save
    # --------------------------------------------------------

    np.savez_compressed(
        out_npz,
        X=X,
        y=y,
        attack_class1=attack_class1,
        attack_class2=attack_class2,
    )


    meta = {
        "dataset":
            "X-IIoTID",

        "source_file":
            str(csv_path),

        "original_rows":
            int(original_rows),

        "final_rows":
            int(len(y)),

        "original_columns":
            original_columns,

        "n_features":
            int(X.shape[1]),

        "label_column":
            "class3",

        "label_mapping": {
            "Normal": 0,
            "Attack": 1
        },

        "benign_count":
            int(
                np.sum(y == 0)
            ),

        "intrusion_count":
            int(
                np.sum(y == 1)
            ),

        "intrusion_fraction":
            float(
                np.mean(y)
            ),

        "numeric_columns":
            numeric_columns,

        "boolean_columns":
            boolean_columns,

        "categorical_columns":
            categorical_columns,

        "high_cardinality_dropped_columns":
            high_cardinality_drop,

        "categorical_unique_counts":
            category_counts,

        "numeric_conversion_ratio":
            numeric_conversion_ratio,

        "explicit_dropped_columns":
            explicit_drop,

        "feature_names":
            feature_names,

        "seed":
            int(seed),
    }


    Path(out_meta).write_text(
        json.dumps(
            meta,
            indent=2
        )
    )


    return meta
'''

PREPROCESS_FILE.write_text(
    preprocess_code
)

print(
    "✅ Corrected X-IIoTID preprocessor written:"
)

print(
    PREPROCESS_FILE
)


# ============================================================
# 2. Force fresh import
# ============================================================

if str(REPO) not in sys.path:

    sys.path.insert(
        0,
        str(REPO)
    )


if "src.data.preprocess_xiiotid" in sys.modules:

    del sys.modules[
        "src.data.preprocess_xiiotid"
    ]


from src.data.preprocess_xiiotid import (
    preprocess_xiiotid
)


# ============================================================
# 3. Regenerate encoded artifact
# ============================================================

print(
    "\nReprocessing X-IIoTID with corrected typing..."
)


meta = preprocess_xiiotid(
    csv_path=RAW_FILE,
    out_npz=ENCODED_FILE,
    out_meta=META_FILE,
    seed=42,
    numeric_parse_threshold=0.95,
    categorical_cardinality_threshold=100,
)


print(
    "✅ Corrected encoding complete."
)


# ============================================================
# 4. Inspect corrected feature typing
# ============================================================

data = np.load(
    ENCODED_FILE,
    allow_pickle=True
)

X = data["X"]

y = data["y"].astype(
    np.int64
)


print(
    "\n========== X-IIoTID CORRECTED FEATURE AUDIT =========="
)

print(
    "records:",
    len(y)
)

print(
    "features:",
    X.shape[1]
)

print(
    "numeric columns:",
    len(
        meta["numeric_columns"]
    )
)

print(
    "boolean columns:",
    len(
        meta["boolean_columns"]
    )
)

print(
    "categorical columns:",
    meta["categorical_columns"]
)

print(
    "high-cardinality categorical dropped:",
    meta[
        "high_cardinality_dropped_columns"
    ]
)

print(
    "benign:",
    int(
        (y == 0).sum()
    )
)

print(
    "intrusion:",
    int(
        (y == 1).sum()
    )
)

print(
    "intrusion fraction:",
    round(
        float(y.mean()),
        4
    )
)

print(
    "====================================================="
)


# ------------------------------------------------------------
# Critical validation:
# numeric traffic/system fields must survive preprocessing
# ------------------------------------------------------------

feature_names = set(
    meta[
        "feature_names"
    ]
)


required_numeric_features = [
    "Duration",
    "Scr_bytes",
    "Des_bytes",
    "Scr_pkts",
    "Des_pkts",
    "total_bytes",
    "total_packet",
    "paket_rate",
    "byte_rate",
    "Avg_user_time",
    "Avg_system_time",
    "Avg_kbmemused",
]


missing_required = [
    c
    for c in required_numeric_features
    if c not in feature_names
]


if missing_required:

    raise RuntimeError(
        "Important numeric features are still missing: "
        + str(
            missing_required
        )
    )


print(
    "\n✅ Important numeric traffic/system features retained."
)


assert X.ndim == 2

assert len(X) == len(y)

assert set(
    np.unique(y).tolist()
) == {
    0,
    1,
}

assert np.isfinite(
    X
).all()


# ============================================================
# 5. Recreate K=10, alpha=0.3 Dirichlet partition
# ============================================================

K = 10

ALPHA = 0.3

SEED = 42

TRAIN_RATIO = 0.70

VAL_RATIO = 0.15

TEST_RATIO = 0.15


rng = np.random.default_rng(
    SEED
)


silo_indices = [
    []
    for _ in range(K)
]


for cls in [
    0,
    1,
]:

    cls_idx = np.where(
        y == cls
    )[0]

    rng.shuffle(
        cls_idx
    )

    proportions = rng.dirichlet(
        np.full(
            K,
            ALPHA
        )
    )

    counts = rng.multinomial(
        len(cls_idx),
        proportions
    )

    start = 0

    for k, count in enumerate(
        counts
    ):

        end = (
            start +
            count
        )

        silo_indices[k].extend(
            cls_idx[
                start:end
            ].tolist()
        )

        start = end


# ============================================================
# 6. 70 / 15 / 15 split within each silo
# ============================================================

train_splits = []

val_splits = []

test_splits = []


for k in range(K):

    idx = np.asarray(
        silo_indices[k],
        dtype=np.int64
    )

    rng.shuffle(
        idx
    )

    n = len(idx)

    if n < 10:

        raise RuntimeError(
            f"Silo {k} has only {n} records."
        )


    n_train = int(
        np.floor(
            TRAIN_RATIO * n
        )
    )

    n_val = int(
        np.floor(
            VAL_RATIO * n
        )
    )


    train_splits.append(
        idx[
            :n_train
        ]
    )

    val_splits.append(
        idx[
            n_train:
            n_train + n_val
        ]
    )

    test_splits.append(
        idx[
            n_train + n_val:
        ]
    )


# ============================================================
# 7. Save corrected split
# ============================================================

np.savez_compressed(
    SPLIT_FILE,

    train=np.array(
        train_splits,
        dtype=object
    ),

    val=np.array(
        val_splits,
        dtype=object
    ),

    test=np.array(
        test_splits,
        dtype=object
    ),

    K=np.array(
        [K],
        dtype=np.int64
    ),

    alpha=np.array(
        [ALPHA],
        dtype=np.float32
    ),

    seed=np.array(
        [SEED],
        dtype=np.int64
    ),
)


# ============================================================
# 8. Audit corrected split
# ============================================================

print(
    "\n========== X-IIoTID CORRECTED SILO AUDIT =========="
)


all_assigned = []


for k in range(K):

    combined = np.concatenate(
        [
            train_splits[k],
            val_splits[k],
            test_splits[k],
        ]
    )

    all_assigned.extend(
        combined.tolist()
    )

    silo_y = y[
        combined
    ]


    print(
        f"Silo {k:2d} | "
        f"total={len(combined):7d} | "
        f"train={len(train_splits[k]):7d} | "
        f"val={len(val_splits[k]):7d} | "
        f"test={len(test_splits[k]):7d} | "
        f"intrusion_rate={silo_y.mean():.4f}"
    )


all_assigned = np.asarray(
    all_assigned,
    dtype=np.int64
)


assert len(
    all_assigned
) == len(y)


assert len(
    np.unique(
        all_assigned
    )
) == len(y)


print(
    "\n✅ Every X-IIoTID sample assigned exactly once."
)


# ============================================================
# 9. Save corrected split summary
# ============================================================

summary = {
    "dataset":
        "X-IIoTID",

    "K":
        K,

    "alpha":
        ALPHA,

    "seed":
        SEED,

    "split_ratios": {
        "train":
            TRAIN_RATIO,

        "validation":
            VAL_RATIO,

        "test":
            TEST_RATIO,
    },

    "silos": [],
}


for k in range(K):

    combined = np.concatenate(
        [
            train_splits[k],
            val_splits[k],
            test_splits[k],
        ]
    )


    summary[
        "silos"
    ].append(
        {
            "silo":
                k,

            "total":
                int(
                    len(combined)
                ),

            "train":
                int(
                    len(
                        train_splits[k]
                    )
                ),

            "validation":
                int(
                    len(
                        val_splits[k]
                    )
                ),

            "test":
                int(
                    len(
                        test_splits[k]
                    )
                ),

            "intrusion_fraction":
                float(
                    y[
                        combined
                    ].mean()
                ),
        }
    )


SUMMARY_FILE.write_text(
    json.dumps(
        summary,
        indent=2
    )
)


# ============================================================
# 10. Final
# ============================================================

print(
    "\n============================================"
)

print(
    "✅ STEP 20.3A X-IIoTID CORRECTION PASSED"
)

print(
    "============================================"
)


print(
    "\nFinal:"
)

print(
    f"  records   = {len(y)}"
)

print(
    f"  input_dim = {X.shape[1]}"
)

print(
    f"  K         = {K}"
)

print(
    f"  alpha     = {ALPHA}"
)

✅ Corrected X-IIoTID preprocessor written:
/content/fl_ids_repo_real/src/data/preprocess_xiiotid.py

Reprocessing X-IIoTID with corrected typing...
✅ Corrected encoding complete.

========== X-IIoTID CORRECTED FEATURE AUDIT ==========
records: 820834
features: 67
numeric columns: 43
boolean columns: 0
categorical columns: ['Protocol', 'Service', 'anomaly_alert']
high-cardinality categorical dropped: ['Duration', 'Scr_bytes', 'Des_bytes', 'Scr_pkts', 'Scr_ip_bytes', 'Des_pkts', 'Des_ip_bytes', 'total_bytes', 'total_packet', 'paket_rate', 'byte_rate', 'Scr_packts_ratio', 'Des_pkts_ratio', 'Scr_bytes_ratio', 'Des_bytes_ratio']
benign: 421417
intrusion: 399417
intrusion fraction: 0.4866


RuntimeError: Important numeric features are still missing: ['Duration', 'Scr_bytes', 'Des_bytes', 'Scr_pkts', 'Des_pkts', 'total_bytes', 'total_packet', 'paket_rate', 'byte_rate']

In [11]:
# ============================================================
# STEP 20.3B — Final correction of X-IIoTID feature typing
#
# Problem:
#   Several true numeric traffic fields contain placeholders
#   such as "-", so heuristic parsing classified them as
#   high-cardinality categorical columns.
#
# Fix:
#   Explicitly force known traffic/rate fields to numeric.
# ============================================================

from pathlib import Path
import json
import sys
import gc
import numpy as np
import pandas as pd

REPO = Path("/content/fl_ids_repo_real")

RAW_FILE = (
    REPO /
    "data/raw/xiiotid/X-IIoTID dataset.csv"
)

PROC_DIR = (
    REPO /
    "data/processed"
)

SRC_DATA_DIR = (
    REPO /
    "src/data"
)

ENCODED_FILE = (
    PROC_DIR /
    "xiiotid_encoded.npz"
)

META_FILE = (
    PROC_DIR /
    "xiiotid_encoded_meta.json"
)

SPLIT_FILE = (
    PROC_DIR /
    "splits_xiiotid_dirichlet_alpha_0.3.npz"
)

SUMMARY_FILE = (
    PROC_DIR /
    "splits_xiiotid_dirichlet_alpha_0.3_summary.json"
)

PREPROCESS_FILE = (
    SRC_DATA_DIR /
    "preprocess_xiiotid.py"
)

assert RAW_FILE.exists()

print("✅ Raw X-IIoTID found:")
print(RAW_FILE)


# ============================================================
# 1. Write final reusable preprocessor
# ============================================================

preprocess_code = r'''
from pathlib import Path
import json
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# Fields known from the X-IIoTID schema to be quantitative.
# Some contain textual placeholders in the CSV, therefore
# they must be explicitly coerced to numeric.
# ------------------------------------------------------------

FORCE_NUMERIC = [
    "Scr_port",
    "Des_port",
    "Duration",
    "Scr_bytes",
    "Des_bytes",
    "Conn_state",
    "missed_bytes",

    "Scr_pkts",
    "Scr_ip_bytes",
    "Des_pkts",
    "Des_ip_bytes",

    "total_bytes",
    "total_packet",
    "paket_rate",
    "byte_rate",

    "Scr_packts_ratio",
    "Des_pkts_ratio",
    "Scr_bytes_ratio",
    "Des_bytes_ratio",

    "Avg_user_time",
    "Std_user_time",
    "Avg_nice_time",
    "Std_nice_time",
    "Avg_system_time",
    "Std_system_time",
    "Avg_iowait_time",
    "Std_iowait_time",
    "Avg_ideal_time",
    "Std_ideal_time",

    "Avg_tps",
    "Std_tps",
    "Avg_rtps",
    "Std_rtps",
    "Avg_wtps",
    "Std_wtps",

    "Avg_ldavg_1",
    "Std_ldavg_1",
    "Avg_kbmemused",
    "Std_kbmemused",

    "Avg_num_Proc/s",
    "Std_num_proc/s",
    "Avg_num_cswch/s",
    "std_num_cswch/s",

    "OSSEC_alert",
    "OSSEC_alert_level",
    "Login_attempt",
    "Succesful_login",
    "File_activity",
    "Process_activity",
    "read_write_physical.process",
    "is_privileged",
]


BOOL_LIKE = [
    "is_syn_only",
    "Is_SYN_ACK",
    "is_pure_ack",
    "is_with_payload",
    "FIN or RST",
    "Bad_checksum",
    "is_SYN_with_RST",
]


TRUE_CATEGORICAL = [
    "Protocol",
    "Service",
    "anomaly_alert",
]


def _force_numeric(series):
    """
    Convert mixed numeric/string column to numeric.

    Known textual placeholders are treated as missing.
    """
    s = (
        series
        .replace(
            [
                "-",
                "--",
                "",
                " ",
                "None",
                "none",
                "NULL",
                "null",
                "NaN",
                "nan",
                "inf",
                "-inf",
                "Infinity",
                "-Infinity",
            ],
            np.nan
        )
    )

    return pd.to_numeric(
        s,
        errors="coerce"
    )


def preprocess_xiiotid(
    csv_path,
    out_npz,
    out_meta,
    seed=42,
):
    """
    Prepare X-IIoTID for binary federated IDS experiments.

    Target:
        class3 = Normal -> 0
        class3 = Attack -> 1

    class1 and class2 are attack metadata only.
    """

    csv_path = Path(csv_path)

    df = pd.read_csv(
        csv_path,
        low_memory=False
    )

    original_rows = int(len(df))
    original_columns = list(df.columns)


    # ========================================================
    # Target and attack metadata
    # ========================================================

    required_labels = [
        "class1",
        "class2",
        "class3",
    ]

    for col in required_labels:

        if col not in df.columns:

            raise ValueError(
                f"Missing required label column: {col}"
            )


    class3 = (
        df.pop("class3")
        .astype(str)
        .str.strip()
        .str.lower()
    )


    unique_target = sorted(
        class3.unique().tolist()
    )


    if not set(
        unique_target
    ).issubset(
        {
            "normal",
            "attack",
        }
    ):

        raise ValueError(
            f"Unexpected class3 labels: {unique_target}"
        )


    y = (
        class3
        .map(
            {
                "normal": 0,
                "attack": 1,
            }
        )
        .astype(np.int64)
        .to_numpy()
    )


    attack_class1 = (
        df.pop("class1")
        .fillna("unknown")
        .astype(str)
        .to_numpy()
    )


    attack_class2 = (
        df.pop("class2")
        .fillna("unknown")
        .astype(str)
        .to_numpy()
    )


    # ========================================================
    # Remove identifiers and timestamps
    # ========================================================

    explicit_drop = [
        "Date",
        "Timestamp",
        "Scr_IP",
        "Des_IP",
    ]


    explicit_drop = [
        col
        for col in explicit_drop
        if col in df.columns
    ]


    df.drop(
        columns=explicit_drop,
        inplace=True
    )


    # ========================================================
    # Force known quantitative fields to numeric
    # ========================================================

    forced_numeric_present = []

    for col in FORCE_NUMERIC:

        if col not in df.columns:
            continue

        forced_numeric_present.append(
            col
        )

        converted = _force_numeric(
            df[col]
        )

        median = converted.median()

        if pd.isna(median):
            median = 0.0

        df[col] = (
            converted
            .fillna(median)
        )


    # ========================================================
    # Explicitly encode boolean-like fields
    # ========================================================

    boolean_present = []

    for col in BOOL_LIKE:

        if col not in df.columns:
            continue

        boolean_present.append(
            col
        )

        s = (
            df[col]
            .astype(str)
            .str.strip()
            .str.lower()
        )


        mapping = {
            "true": 1,
            "false": 0,
            "1": 1,
            "0": 0,
        }


        converted = s.map(
            mapping
        )


        # If unusual values exist, coerce them numerically.
        if converted.isna().any():

            numeric_fallback = pd.to_numeric(
                df[col],
                errors="coerce"
            )

            converted = (
                converted
                .fillna(
                    numeric_fallback
                )
                .fillna(0)
            )


        df[col] = (
            converted
            .astype(np.uint8)
        )


    # ========================================================
    # Genuine categorical variables only
    # ========================================================

    categorical_present = [
        col
        for col in TRUE_CATEGORICAL
        if col in df.columns
    ]


    for col in categorical_present:

        df[col] = (
            df[col]
            .fillna("__MISSING__")
            .astype(str)
            .str.strip()
        )


    # ========================================================
    # Remaining unclassified object columns
    #
    # We do NOT silently discard them.
    # Attempt numeric conversion first.
    # ========================================================

    remaining_objects = [
        col
        for col in df.columns
        if (
            df[col].dtype == "object"
            and col not in categorical_present
        )
    ]


    auto_numeric = []
    unresolved_object = []


    for col in remaining_objects:

        converted = _force_numeric(
            df[col]
        )

        original_nonmissing = int(
            df[col].notna().sum()
        )

        converted_nonmissing = int(
            converted.notna().sum()
        )

        parse_ratio = (
            converted_nonmissing /
            max(
                original_nonmissing,
                1
            )
        )


        if parse_ratio >= 0.80:

            median = converted.median()

            if pd.isna(median):
                median = 0.0

            df[col] = (
                converted
                .fillna(median)
            )

            auto_numeric.append(
                col
            )

        else:

            unresolved_object.append(
                col
            )


    # Any unresolved text field is categorical.
    for col in unresolved_object:

        df[col] = (
            df[col]
            .fillna("__MISSING__")
            .astype(str)
        )


    final_categorical = (
        categorical_present +
        unresolved_object
    )


    # ========================================================
    # Clean all numeric columns
    # ========================================================

    numeric_columns = [
        col
        for col in df.columns
        if col not in final_categorical
    ]


    for col in numeric_columns:

        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )

        df[col] = df[col].replace(
            [
                np.inf,
                -np.inf,
            ],
            np.nan
        )

        if df[col].isna().any():

            median = df[col].median()

            if pd.isna(median):
                median = 0.0

            df[col] = (
                df[col]
                .fillna(median)
            )


    # ========================================================
    # One-hot encode real categorical fields
    # ========================================================

    X_df = pd.get_dummies(
        df,
        columns=final_categorical,
        drop_first=False,
        dtype=np.uint8,
    )


    X_df = X_df.replace(
        [
            np.inf,
            -np.inf,
        ],
        np.nan
    )


    if X_df.isna().any().any():

        bad = (
            X_df.columns[
                X_df.isna().any()
            ]
            .tolist()
        )

        raise ValueError(
            f"NaNs remain after preprocessing: {bad[:20]}"
        )


    feature_names = list(
        X_df.columns
    )


    X = X_df.to_numpy(
        dtype=np.float32
    )


    # ========================================================
    # Deterministic shuffle
    # ========================================================

    rng = np.random.default_rng(
        seed
    )

    order = rng.permutation(
        len(y)
    )


    X = X[order]

    y = y[order]

    attack_class1 = (
        attack_class1[order]
    )

    attack_class2 = (
        attack_class2[order]
    )


    # ========================================================
    # Save
    # ========================================================

    np.savez_compressed(
        out_npz,
        X=X,
        y=y,
        attack_class1=attack_class1,
        attack_class2=attack_class2,
    )


    meta = {
        "dataset":
            "X-IIoTID",

        "source_file":
            str(csv_path),

        "original_rows":
            original_rows,

        "final_rows":
            int(len(y)),

        "n_features":
            int(
                X.shape[1]
            ),

        "label_column":
            "class3",

        "label_mapping": {
            "Normal": 0,
            "Attack": 1,
        },

        "benign_count":
            int(
                np.sum(
                    y == 0
                )
            ),

        "intrusion_count":
            int(
                np.sum(
                    y == 1
                )
            ),

        "intrusion_fraction":
            float(
                np.mean(y)
            ),

        "explicit_dropped_columns":
            explicit_drop,

        "forced_numeric_columns":
            forced_numeric_present,

        "boolean_columns":
            boolean_present,

        "auto_numeric_columns":
            auto_numeric,

        "categorical_columns":
            final_categorical,

        "feature_names":
            feature_names,

        "seed":
            int(seed),
    }


    Path(
        out_meta
    ).write_text(
        json.dumps(
            meta,
            indent=2
        )
    )


    return meta
'''

PREPROCESS_FILE.write_text(
    preprocess_code
)

print(
    "✅ Final X-IIoTID preprocessor written:"
)

print(
    PREPROCESS_FILE
)


# ============================================================
# 2. Fresh import
# ============================================================

if str(REPO) not in sys.path:

    sys.path.insert(
        0,
        str(REPO)
    )


if (
    "src.data.preprocess_xiiotid"
    in sys.modules
):

    del sys.modules[
        "src.data.preprocess_xiiotid"
    ]


from src.data.preprocess_xiiotid import (
    preprocess_xiiotid
)


# ============================================================
# 3. Regenerate encoded dataset
# ============================================================

print(
    "\nReprocessing X-IIoTID..."
)


meta = preprocess_xiiotid(
    csv_path=RAW_FILE,
    out_npz=ENCODED_FILE,
    out_meta=META_FILE,
    seed=42,
)


print(
    "✅ X-IIoTID encoding complete."
)


# ============================================================
# 4. Validate corrected features
# ============================================================

data = np.load(
    ENCODED_FILE,
    allow_pickle=True
)


X = data["X"]

y = (
    data["y"]
    .astype(np.int64)
)


feature_names = set(
    meta[
        "feature_names"
    ]
)


required_numeric_features = [
    "Duration",
    "Scr_bytes",
    "Des_bytes",
    "Scr_pkts",
    "Des_pkts",
    "total_bytes",
    "total_packet",
    "paket_rate",
    "byte_rate",
    "Avg_user_time",
    "Avg_system_time",
    "Avg_kbmemused",
]


missing = [
    col
    for col
    in required_numeric_features
    if col not in feature_names
]


if missing:

    raise RuntimeError(
        "Numeric features still missing: "
        + str(missing)
    )


assert X.ndim == 2

assert len(
    X
) == len(y)

assert set(
    np.unique(
        y
    ).tolist()
) == {
    0,
    1,
}

assert np.isfinite(
    X
).all()


print(
    "\n========== X-IIoTID FINAL FEATURE AUDIT =========="
)

print(
    "records:",
    len(y)
)

print(
    "features:",
    X.shape[1]
)

print(
    "benign:",
    int(
        np.sum(
            y == 0
        )
    )
)

print(
    "intrusion:",
    int(
        np.sum(
            y == 1
        )
    )
)

print(
    "intrusion fraction:",
    round(
        float(
            y.mean()
        ),
        4
    )
)

print(
    "\nforced numeric columns:",
    len(
        meta[
            "forced_numeric_columns"
        ]
    )
)

print(
    "auto numeric columns:",
    meta[
        "auto_numeric_columns"
    ]
)

print(
    "categorical columns:",
    meta[
        "categorical_columns"
    ]
)

print(
    "\nImportant numeric features retained:"
)

for col in required_numeric_features:

    print(
        "  ✅",
        col
    )

print(
    "================================================="
)


# ============================================================
# 5. Recreate Dirichlet partition
# ============================================================

K = 10
ALPHA = 0.3
SEED = 42

TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15


rng = np.random.default_rng(
    SEED
)


silo_indices = [
    []
    for _ in range(K)
]


for cls in [
    0,
    1,
]:

    cls_idx = np.where(
        y == cls
    )[0]


    rng.shuffle(
        cls_idx
    )


    proportions = rng.dirichlet(
        np.full(
            K,
            ALPHA
        )
    )


    counts = rng.multinomial(
        len(
            cls_idx
        ),
        proportions
    )


    start = 0


    for k, count in enumerate(
        counts
    ):

        end = (
            start +
            count
        )


        silo_indices[
            k
        ].extend(
            cls_idx[
                start:end
            ].tolist()
        )


        start = end


# ============================================================
# 6. 70 / 15 / 15 split
# ============================================================

train_splits = []
val_splits = []
test_splits = []


for k in range(K):

    idx = np.asarray(
        silo_indices[k],
        dtype=np.int64
    )


    rng.shuffle(
        idx
    )


    n = len(
        idx
    )


    if n < 10:

        raise RuntimeError(
            f"Silo {k} has only {n} records."
        )


    n_train = int(
        np.floor(
            TRAIN_RATIO *
            n
        )
    )


    n_val = int(
        np.floor(
            VAL_RATIO *
            n
        )
    )


    train_splits.append(
        idx[
            :n_train
        ]
    )


    val_splits.append(
        idx[
            n_train:
            n_train + n_val
        ]
    )


    test_splits.append(
        idx[
            n_train + n_val:
        ]
    )


# ============================================================
# 7. Save split
# ============================================================

np.savez_compressed(
    SPLIT_FILE,

    train=np.array(
        train_splits,
        dtype=object
    ),

    val=np.array(
        val_splits,
        dtype=object
    ),

    test=np.array(
        test_splits,
        dtype=object
    ),

    K=np.array(
        [K],
        dtype=np.int64
    ),

    alpha=np.array(
        [ALPHA],
        dtype=np.float32
    ),

    seed=np.array(
        [SEED],
        dtype=np.int64
    ),
)


# ============================================================
# 8. Audit split
# ============================================================

print(
    "\n========== X-IIoTID FINAL SILO AUDIT =========="
)


all_assigned = []


for k in range(K):

    combined = np.concatenate(
        [
            train_splits[k],
            val_splits[k],
            test_splits[k],
        ]
    )


    all_assigned.extend(
        combined.tolist()
    )


    silo_y = y[
        combined
    ]


    print(
        f"Silo {k:2d} | "
        f"total={len(combined):7d} | "
        f"train={len(train_splits[k]):7d} | "
        f"val={len(val_splits[k]):7d} | "
        f"test={len(test_splits[k]):7d} | "
        f"intrusion_rate={silo_y.mean():.4f}"
    )


all_assigned = np.asarray(
    all_assigned,
    dtype=np.int64
)


assert len(
    all_assigned
) == len(y)


assert len(
    np.unique(
        all_assigned
    )
) == len(y)


print(
    "\n✅ Every X-IIoTID record assigned exactly once."
)


# ============================================================
# 9. Save summary
# ============================================================

summary = {
    "dataset":
        "X-IIoTID",

    "K":
        K,

    "alpha":
        ALPHA,

    "seed":
        SEED,

    "split_ratios": {
        "train":
            TRAIN_RATIO,

        "validation":
            VAL_RATIO,

        "test":
            TEST_RATIO,
    },

    "silos":
        []
}


for k in range(K):

    combined = np.concatenate(
        [
            train_splits[k],
            val_splits[k],
            test_splits[k],
        ]
    )


    summary[
        "silos"
    ].append(
        {
            "silo":
                k,

            "total":
                int(
                    len(
                        combined
                    )
                ),

            "train":
                int(
                    len(
                        train_splits[k]
                    )
                ),

            "validation":
                int(
                    len(
                        val_splits[k]
                    )
                ),

            "test":
                int(
                    len(
                        test_splits[k]
                    )
                ),

            "intrusion_fraction":
                float(
                    y[
                        combined
                    ].mean()
                )
        }
    )


SUMMARY_FILE.write_text(
    json.dumps(
        summary,
        indent=2
    )
)


# ============================================================
# 10. Final
# ============================================================

print(
    "\n============================================"
)

print(
    "✅ STEP 20.3B X-IIoTID FINAL PREPROCESSING PASSED"
)

print(
    "============================================"
)


print(
    "\nFinal:"
)

print(
    f"  records   = {len(y)}"
)

print(
    f"  input_dim = {X.shape[1]}"
)

print(
    f"  K         = {K}"
)

print(
    f"  alpha     = {ALPHA}"
)

✅ Raw X-IIoTID found:
/content/fl_ids_repo_real/data/raw/xiiotid/X-IIoTID dataset.csv
✅ Final X-IIoTID preprocessor written:
/content/fl_ids_repo_real/src/data/preprocess_xiiotid.py

Reprocessing X-IIoTID...
✅ X-IIoTID encoding complete.

========== X-IIoTID FINAL FEATURE AUDIT ==========
records: 820834
features: 82
benign: 421417
intrusion: 399417
intrusion fraction: 0.4866

forced numeric columns: 51
auto numeric columns: []
categorical columns: ['Protocol', 'Service', 'anomaly_alert']

Important numeric features retained:
  ✅ Duration
  ✅ Scr_bytes
  ✅ Des_bytes
  ✅ Scr_pkts
  ✅ Des_pkts
  ✅ total_bytes
  ✅ total_packet
  ✅ paket_rate
  ✅ byte_rate
  ✅ Avg_user_time
  ✅ Avg_system_time
  ✅ Avg_kbmemused

========== X-IIoTID FINAL SILO AUDIT ==========
Silo  0 | total= 270386 | train= 189270 | val=  40557 | test=  40559 | intrusion_rate=0.7629
Silo  1 | total=  12756 | train=   8929 | val=   1913 | test=   1914 | intrusion_rate=1.0000
Silo  2 | total= 109052 | train=  76336 | val=  

**Step 20.4 — Verify all three datasets use the same FL artifact interface**

In [12]:
# ============================================================
# STEP 20.4 — Three-dataset compatibility audit
#
# Verify:
#   Edge-IIoTset
#   ToN-IoT
#   X-IIoTID
#
# All must expose:
#   X, y
#   K = 10
#   train / val / test indices
#
# No training is performed here.
# ============================================================

from pathlib import Path
import numpy as np
import json

REPO = Path("/content/fl_ids_repo_real")
PROC = REPO / "data" / "processed"

DATASETS = {
    "Edge-IIoTset": {
        "encoded": PROC / "edgeiiotset_encoded.npz",
        "split": PROC / "splits_edgeiiotset_dirichlet_alpha_0.3.npz",
    },

    "ToN-IoT": {
        "encoded": PROC / "ton_iot_encoded.npz",
        "split": PROC / "splits_ton_iot_dirichlet_alpha_0.3.npz",
    },

    "X-IIoTID": {
        "encoded": PROC / "xiiotid_encoded.npz",
        "split": PROC / "splits_xiiotid_dirichlet_alpha_0.3.npz",
    },
}


print(
    "========== THREE-DATASET COMPATIBILITY AUDIT =========="
)


summary = {}


for dataset_name, paths in DATASETS.items():

    print(
        f"\n--- {dataset_name} ---"
    )

    encoded_path = paths["encoded"]
    split_path = paths["split"]

    if not encoded_path.exists():

        print(
            f"❌ Missing encoded file:\n{encoded_path}"
        )

        continue

    if not split_path.exists():

        print(
            f"❌ Missing split file:\n{split_path}"
        )

        continue


    # --------------------------------------------------------
    # Load encoded data
    # --------------------------------------------------------

    enc = np.load(
        encoded_path,
        allow_pickle=True
    )

    assert "X" in enc.files
    assert "y" in enc.files

    X = enc["X"]
    y = enc["y"].astype(
        np.int64
    )

    assert X.ndim == 2
    assert y.ndim == 1
    assert len(X) == len(y)
    assert set(
        np.unique(y).tolist()
    ) == {0, 1}
    assert np.isfinite(X).all()


    # --------------------------------------------------------
    # Load split
    # --------------------------------------------------------

    spl = np.load(
        split_path,
        allow_pickle=True
    )

    required_split_keys = {
        "train",
        "val",
        "test",
    }

    missing_keys = (
        required_split_keys -
        set(spl.files)
    )

    if missing_keys:

        raise RuntimeError(
            f"{dataset_name}: missing split keys "
            f"{sorted(missing_keys)}"
        )


    train = spl["train"]
    val = spl["val"]
    test = spl["test"]


    # --------------------------------------------------------
    # Determine K
    # --------------------------------------------------------

    if "K" in spl.files:

        K = int(
            np.asarray(
                spl["K"]
            ).reshape(-1)[0]
        )

    else:

        K = len(train)


    assert K == 10

    assert len(train) == 10
    assert len(val) == 10
    assert len(test) == 10


    # --------------------------------------------------------
    # Determine alpha when available
    # --------------------------------------------------------

    if "alpha" in spl.files:

        alpha = float(
            np.asarray(
                spl["alpha"]
            ).reshape(-1)[0]
        )

    else:

        alpha = None


    # --------------------------------------------------------
    # Verify assignment / no overlap
    # --------------------------------------------------------

    all_indices = []

    for k in range(K):

        tr = np.asarray(
            train[k],
            dtype=np.int64
        )

        va = np.asarray(
            val[k],
            dtype=np.int64
        )

        te = np.asarray(
            test[k],
            dtype=np.int64
        )

        # No train/val/test overlap within silo.
        assert len(
            np.intersect1d(
                tr,
                va
            )
        ) == 0

        assert len(
            np.intersect1d(
                tr,
                te
            )
        ) == 0

        assert len(
            np.intersect1d(
                va,
                te
            )
        ) == 0

        all_indices.extend(
            tr.tolist()
        )

        all_indices.extend(
            va.tolist()
        )

        all_indices.extend(
            te.tolist()
        )


    all_indices = np.asarray(
        all_indices,
        dtype=np.int64
    )

    assert len(
        all_indices
    ) == len(y), (
        f"{dataset_name}: not all records assigned."
    )

    assert len(
        np.unique(
            all_indices
        )
    ) == len(y), (
        f"{dataset_name}: duplicate split assignments."
    )


    # --------------------------------------------------------
    # Dataset summary
    # --------------------------------------------------------

    benign = int(
        np.sum(
            y == 0
        )
    )

    intrusion = int(
        np.sum(
            y == 1
        )
    )

    print(
        f"✅ records     = {len(y)}"
    )

    print(
        f"✅ input_dim   = {X.shape[1]}"
    )

    print(
        f"✅ benign      = {benign}"
    )

    print(
        f"✅ intrusion   = {intrusion}"
    )

    print(
        f"✅ K           = {K}"
    )

    print(
        f"✅ alpha       = {alpha}"
    )

    print(
        "✅ split integrity passed"
    )


    summary[
        dataset_name
    ] = {
        "records":
            int(
                len(y)
            ),

        "input_dim":
            int(
                X.shape[1]
            ),

        "benign":
            benign,

        "intrusion":
            intrusion,

        "K":
            K,

        "alpha":
            alpha,
    }


# ============================================================
# Final validation
# ============================================================

if len(summary) != 3:

    print(
        "\n⚠️ Not all three datasets are currently available."
    )

    print(
        "Do not proceed to the final three-dataset run yet."
    )

else:

    print(
        "\n============================================"
    )

    print(
        "✅ STEP 20.4 THREE-DATASET AUDIT PASSED"
    )

    print(
        "============================================"
    )

    print(
        "\nAll three datasets now use the same interface:"
    )

    print(
        "  X + binary y"
    )

    print(
        "  K = 10 silos"
    )

    print(
        "  train / validation / test splits"
    )

    print(
        "  Dirichlet non-IID partitioning"
    )


# ------------------------------------------------------------
# Save compact reproducibility manifest
# ------------------------------------------------------------

MANIFEST = (
    PROC /
    "three_dataset_manifest.json"
)

MANIFEST.write_text(
    json.dumps(
        summary,
        indent=2
    )
)

print(
    "\nManifest:"
)

print(
    MANIFEST
)

========== THREE-DATASET COMPATIBILITY AUDIT ==========

--- Edge-IIoTset ---
❌ Missing encoded file:
/content/fl_ids_repo_real/data/processed/edgeiiotset_encoded.npz

--- ToN-IoT ---
✅ records     = 211043
✅ input_dim   = 115
✅ benign      = 50000
✅ intrusion   = 161043
✅ K           = 10
✅ alpha       = 0.30000001192092896
✅ split integrity passed

--- X-IIoTID ---
✅ records     = 820834
✅ input_dim   = 82
✅ benign      = 421417
✅ intrusion   = 399417
✅ K           = 10
✅ alpha       = 0.30000001192092896
✅ split integrity passed

⚠️ Not all three datasets are currently available.
Do not proceed to the final three-dataset run yet.

Manifest:
/content/fl_ids_repo_real/data/processed/three_dataset_manifest.json


**Step 20.4A — Restore Edge-IIoTset from your Drive archive**

In [13]:
# ============================================================
# STEP 20.4A — Restore Edge-IIoTset into persistent repository
#
# Uses:
#   /content/drive/MyDrive/archive.zip
#
# Saves permanently under:
#   /content/fl_ids_repo_real/data/raw/edgeiiotset/
#   /content/fl_ids_repo_real/data/processed/
#
# Recreates:
#   src/data/preprocess_edgeiiotset.py
#   src/data/splits.py
#
# Outputs:
#   edgeiiotset_encoded.npz
#   edgeiiotset_encoded_meta.json
#   splits_edgeiiotset_dirichlet_alpha_0.3.npz
#   splits_edgeiiotset_dirichlet_alpha_0.3_summary.json
# ============================================================

from pathlib import Path
import zipfile
import json
import sys
import numpy as np
import pandas as pd

REPO = Path("/content/fl_ids_repo_real")

ZIP_PATH = Path(
    "/content/drive/MyDrive/archive.zip"
)

RAW_DIR = (
    REPO /
    "data/raw/edgeiiotset"
)

PROC_DIR = (
    REPO /
    "data/processed"
)

SRC_DATA_DIR = (
    REPO /
    "src/data"
)

RAW_DIR.mkdir(
    parents=True,
    exist_ok=True
)

PROC_DIR.mkdir(
    parents=True,
    exist_ok=True
)

SRC_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)


print(
    "✅ Repository:"
)
print(
    REPO
)


# ============================================================
# 1. Find existing Edge-IIoTset CSV
# ============================================================

def find_edge_csv(root):

    csvs = sorted(
        [
            p
            for p in root.rglob("*.csv")
            if p.is_file()
        ]
    )

    if not csvs:
        return None

    preferred_name = (
        "DNN-EdgeIIoT-dataset.csv"
    )

    for p in csvs:

        if (
            preferred_name.lower()
            in p.name.lower()
        ):
            return p

    # fallback:
    # choose largest CSV if preferred file name is absent
    return max(
        csvs,
        key=lambda p: p.stat().st_size
    )


EDGE_CSV = find_edge_csv(
    RAW_DIR
)


# ============================================================
# 2. Extract archive only if dataset is not already restored
# ============================================================

if EDGE_CSV is None:

    print(
        "\nEdge-IIoTset raw CSV not yet in repository."
    )

    assert ZIP_PATH.exists(), (
        "Edge-IIoTset archive not found at:\n"
        f"{ZIP_PATH}\n\n"
        "If your archive has another name, send me the "
        "Drive filename."
    )

    print(
        "Extracting Edge-IIoTset archive..."
    )

    with zipfile.ZipFile(
        ZIP_PATH,
        "r"
    ) as zf:

        zf.extractall(
            RAW_DIR
        )

    print(
        "✅ Archive extracted."
    )

    EDGE_CSV = find_edge_csv(
        RAW_DIR
    )


assert EDGE_CSV is not None, (
    "No Edge-IIoTset CSV found after extraction."
)


print(
    "\n✅ Selected Edge-IIoTset CSV:"
)

print(
    EDGE_CSV
)

print(
    "Size:",
    round(
        EDGE_CSV.stat().st_size /
        (1024 ** 2),
        2
    ),
    "MB"
)


# ============================================================
# 3. Create __init__.py files if missing
# ============================================================

(REPO / "src").mkdir(
    parents=True,
    exist_ok=True
)

(REPO / "src/__init__.py").touch(
    exist_ok=True
)

(SRC_DATA_DIR / "__init__.py").touch(
    exist_ok=True
)


# ============================================================
# 4. Restore original Edge-IIoTset preprocessing module
# ============================================================

PREPROCESS_FILE = (
    SRC_DATA_DIR /
    "preprocess_edgeiiotset.py"
)

preprocess_code = r'''
import json
from pathlib import Path

import numpy as np
import pandas as pd


DROP_COLUMNS_DEFAULT = [
    "frame.time",
    "ip.src_host",
    "ip.dst_host",
    "arp.src.proto_ipv4",
    "arp.dst.proto_ipv4",
    "http.file_data",
    "http.request.full_uri",
    "icmp.transmit_timestamp",
    "http.request.uri.query",
    "tcp.options",
    "tcp.payload",
    "tcp.srcport",
    "tcp.dstport",
    "udp.port",
    "mqtt.msg",
]


PREFERRED_CAT_COLS = [
    "http.request.method",
    "http.referer",
    "http.request.version",
    "dns.qry.name.len",
    "mqtt.conack.flags",
    "mqtt.protoname",
    "mqtt.topic",
]


def preprocess_edgeiiotset(
    csv_path,
    out_npz,
    out_meta,
    drop_columns=None,
):

    csv_path = Path(
        csv_path
    )

    out_npz = Path(
        out_npz
    )

    out_meta = Path(
        out_meta
    )

    out_npz.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    out_meta.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    # --------------------------------------------------------
    # Load
    # --------------------------------------------------------

    df = pd.read_csv(
        csv_path,
        low_memory=False
    )

    original_rows = int(
        len(df)
    )


    # --------------------------------------------------------
    # Drop identifier / raw payload fields
    # --------------------------------------------------------

    if drop_columns is None:

        drop_columns = (
            DROP_COLUMNS_DEFAULT
        )

    existing = [
        c
        for c in drop_columns
        if c in df.columns
    ]

    df.drop(
        columns=existing,
        inplace=True
    )


    # --------------------------------------------------------
    # Clean
    # --------------------------------------------------------

    before_nan = len(df)

    df.dropna(
        axis=0,
        how="any",
        inplace=True
    )

    dropped_nan = (
        before_nan -
        len(df)
    )


    before_dup = len(df)

    df.drop_duplicates(
        keep="first",
        inplace=True
    )

    dropped_duplicates = (
        before_dup -
        len(df)
    )


    # Deterministic shuffle
    df = (
        df.sample(
            frac=1.0,
            random_state=42
        )
        .reset_index(
            drop=True
        )
    )


    # --------------------------------------------------------
    # Binary target
    # --------------------------------------------------------

    if (
        "Attack_type"
        not in df.columns
    ):

        raise ValueError(
            "Expected 'Attack_type' column "
            "in Edge-IIoTset CSV."
        )


    attack_type = (
        df["Attack_type"]
        .astype(str)
        .to_numpy()
    )


    y = (
        attack_type != "Normal"
    ).astype(
        np.int64
    )


    # --------------------------------------------------------
    # Features
    # --------------------------------------------------------

    feature_df = df.drop(
        columns=[
            "Attack_type"
        ]
    )


    preferred_present = [
        c
        for c in PREFERRED_CAT_COLS
        if c in feature_df.columns
    ]


    object_cols = [
        c
        for c in feature_df.columns
        if feature_df[c].dtype
        == "object"
    ]


    if preferred_present:

        cat_cols = sorted(
            set(
                preferred_present +
                object_cols
            )
        )

    else:

        cat_cols = object_cols


    X_df = pd.get_dummies(
        feature_df,
        columns=cat_cols,
        drop_first=False
    )


    if X_df.isna().any().any():

        raise ValueError(
            "NaNs found after Edge-IIoTset encoding."
        )


    X = X_df.to_numpy(
        dtype=np.float32
    )


    # --------------------------------------------------------
    # Save encoded artifact
    # --------------------------------------------------------

    np.savez_compressed(
        out_npz,
        X=X,
        y=y,
        attack_type=attack_type,
    )


    meta = {
        "dataset":
            "Edge-IIoTset",

        "source_csv":
            str(csv_path),

        "original_rows":
            original_rows,

        "final_rows":
            int(
                X.shape[0]
            ),

        "n_features":
            int(
                X.shape[1]
            ),

        "benign_count":
            int(
                np.sum(
                    y == 0
                )
            ),

        "intrusion_count":
            int(
                np.sum(
                    y == 1
                )
            ),

        "dropped_nan_rows":
            int(
                dropped_nan
            ),

        "dropped_duplicate_rows":
            int(
                dropped_duplicates
            ),

        "feature_names":
            list(
                X_df.columns
            ),

        "encoded_categorical_columns":
            cat_cols,

        "label_definition":
            "Label=1 if Attack_type!='Normal' else 0",

        "dropped_columns":
            existing,

        "seed":
            42,
    }


    out_meta.write_text(
        json.dumps(
            meta,
            indent=2
        )
    )


    return (
        str(out_npz),
        str(out_meta)
    )
'''

PREPROCESS_FILE.write_text(
    preprocess_code
)

print(
    "\n✅ Restored:"
)

print(
    PREPROCESS_FILE
)


# ============================================================
# 5. Restore shared Dirichlet split module
# ============================================================

SPLITS_FILE = (
    SRC_DATA_DIR /
    "splits.py"
)

splits_code = r'''
import json
from pathlib import Path

import numpy as np
from sklearn.model_selection import train_test_split


def dirichlet_partition_indices(
    y,
    K,
    alpha,
    seed=42,
    min_per_silo=10,
    max_tries=500,
):

    rng = np.random.default_rng(
        seed
    )

    classes = np.unique(
        y
    )

    best_silos = None
    best_min_size = -1


    for _ in range(
        max_tries
    ):

        silos = [
            []
            for _ in range(K)
        ]


        for c in classes:

            idx_c = np.where(
                y == c
            )[0]

            rng.shuffle(
                idx_c
            )


            props = rng.dirichlet(
                alpha=np.full(
                    K,
                    alpha
                )
            )


            counts = rng.multinomial(
                len(idx_c),
                props
            )


            start = 0


            for k in range(K):

                cnt = counts[k]

                if cnt > 0:

                    silos[k].extend(
                        idx_c[
                            start:
                            start + cnt
                        ].tolist()
                    )

                start += cnt


        sizes = np.array(
            [
                len(s)
                for s in silos
            ],
            dtype=int
        )


        min_size = int(
            sizes.min()
        )


        if (
            min_size >
            best_min_size
        ):

            best_min_size = (
                min_size
            )

            best_silos = [
                np.array(
                    s,
                    dtype=np.int64
                )
                for s in silos
            ]


        if (
            min_size >=
            min_per_silo
        ):

            for k in range(K):

                rng.shuffle(
                    silos[k]
                )


            return (
                [
                    np.array(
                        s,
                        dtype=np.int64
                    )
                    for s in silos
                ],
                True,
                best_min_size,
            )


    return (
        best_silos,
        False,
        best_min_size,
    )


def split_silo_indices(
    idx,
    y,
    seed=42,
    ratios=(
        0.70,
        0.15,
        0.15,
    ),
):

    idx = np.array(
        idx,
        dtype=np.int64
    )

    y_s = y[
        idx
    ]


    if (
        len(idx) < 30
        or len(
            np.unique(
                y_s
            )
        ) < 2
    ):

        n = len(
            idx
        )

        n_train = max(
            1,
            int(
                ratios[0] *
                n
            )
        )

        n_val = max(
            0,
            int(
                ratios[1] *
                n
            )
        )


        perm = (
            np.random.default_rng(
                seed
            )
            .permutation(
                idx
            )
        )


        tr = perm[
            :n_train
        ]

        va = perm[
            n_train:
            n_train + n_val
        ]

        te = perm[
            n_train + n_val:
        ]


        return (
            tr,
            va,
            te,
        )


    test_val_frac = (
        ratios[1] +
        ratios[2]
    )


    strat_ok = (
        len(
            np.unique(
                y_s
            )
        ) > 1
        and
        np.min(
            np.bincount(
                y_s
            )
        ) >= 2
    )


    tr, temp = train_test_split(
        idx,
        test_size=test_val_frac,
        random_state=seed,
        shuffle=True,
        stratify=(
            y_s
            if strat_ok
            else None
        ),
    )


    y_temp = y[
        temp
    ]


    val_frac_of_temp = (
        ratios[1] /
        (
            ratios[1] +
            ratios[2]
        )
    )


    strat_ok2 = (
        len(
            np.unique(
                y_temp
            )
        ) > 1
        and
        np.min(
            np.bincount(
                y_temp
            )
        ) >= 2
    )


    va, te = train_test_split(
        temp,
        test_size=(
            1.0 -
            val_frac_of_temp
        ),
        random_state=seed,
        shuffle=True,
        stratify=(
            y_temp
            if strat_ok2
            else None
        ),
    )


    return (
        tr,
        va,
        te,
    )


def make_splits_and_save(
    encoded_npz,
    out_npz,
    out_summary_json,
    K=10,
    alpha=0.3,
    seed=42,
    ratios=(
        0.70,
        0.15,
        0.15,
    ),
    min_per_silo=10,
):

    data = np.load(
        encoded_npz,
        allow_pickle=True
    )

    y = data[
        "y"
    ].astype(
        int
    )


    silos, met, best_min = (
        dirichlet_partition_indices(
            y,
            K=K,
            alpha=alpha,
            seed=seed,
            min_per_silo=
                min_per_silo,
        )
    )


    splits = {
        "train": [],
        "val": [],
        "test": [],
    }


    stats = []


    for k in range(K):

        tr, va, te = (
            split_silo_indices(
                silos[k],
                y,
                seed=seed,
                ratios=ratios,
            )
        )


        splits[
            "train"
        ].append(
            tr
        )

        splits[
            "val"
        ].append(
            va
        )

        splits[
            "test"
        ].append(
            te
        )


        yk = y[
            silos[k]
        ]


        stats.append(
            {
                "silo":
                    k,

                "n_total":
                    int(
                        len(
                            silos[k]
                        )
                    ),

                "n0":
                    int(
                        (
                            yk == 0
                        ).sum()
                    ),

                "n1":
                    int(
                        (
                            yk == 1
                        ).sum()
                    ),

                "intrusion_rate":
                    float(
                        (
                            yk == 1
                        ).mean()
                    )
                    if len(yk)
                    else 0.0,

                "n_train":
                    int(
                        len(tr)
                    ),

                "n_val":
                    int(
                        len(va)
                    ),

                "n_test":
                    int(
                        len(te)
                    ),
            }
        )


    out_npz = Path(
        out_npz
    )

    out_npz.parent.mkdir(
        parents=True,
        exist_ok=True
    )


    np.savez_compressed(
        out_npz,

        train=np.array(
            splits[
                "train"
            ],
            dtype=object
        ),

        val=np.array(
            splits[
                "val"
            ],
            dtype=object
        ),

        test=np.array(
            splits[
                "test"
            ],
            dtype=object
        ),

        alpha=np.array(
            [
                alpha
            ],
            dtype=np.float32
        ),

        K=np.array(
            [
                K
            ],
            dtype=np.int32
        ),

        seed=np.array(
            [
                seed
            ],
            dtype=np.int32
        ),
    )


    out_summary_json = Path(
        out_summary_json
    )


    out_summary_json.write_text(
        json.dumps(
            {
                "alpha":
                    alpha,

                "K":
                    K,

                "seed":
                    seed,

                "ratios":
                    ratios,

                "min_per_silo_target":
                    min_per_silo,

                "met_min_per_silo_target":
                    bool(
                        met
                    ),

                "best_min_silo_size_seen":
                    int(
                        best_min
                    ),

                "silo_stats":
                    stats,
            },
            indent=2
        )
    )


    return (
        str(
            out_npz
        ),
        str(
            out_summary_json
        ),
    )
'''

SPLITS_FILE.write_text(
    splits_code
)

print(
    "✅ Restored:"
)

print(
    SPLITS_FILE
)


# ============================================================
# 6. Fresh imports
# ============================================================

if str(REPO) not in sys.path:

    sys.path.insert(
        0,
        str(REPO)
    )


for module_name in [
    "src.data.preprocess_edgeiiotset",
    "src.data.splits",
]:

    if module_name in sys.modules:

        del sys.modules[
            module_name
        ]


from src.data.preprocess_edgeiiotset import (
    preprocess_edgeiiotset
)

from src.data.splits import (
    make_splits_and_save
)


# ============================================================
# 7. Preprocess Edge-IIoTset
# ============================================================

ENCODED_FILE = (
    PROC_DIR /
    "edgeiiotset_encoded.npz"
)

META_FILE = (
    PROC_DIR /
    "edgeiiotset_encoded_meta.json"
)


print(
    "\nPreprocessing Edge-IIoTset..."
)

print(
    "This is the large dataset, so this may take a few minutes."
)


preprocess_edgeiiotset(
    csv_path=str(
        EDGE_CSV
    ),

    out_npz=str(
        ENCODED_FILE
    ),

    out_meta=str(
        META_FILE
    ),
)


print(
    "✅ Edge-IIoTset encoding completed."
)


# ============================================================
# 8. Inspect encoded data
# ============================================================

edge = np.load(
    ENCODED_FILE,
    allow_pickle=True
)

X = edge[
    "X"
]

y = edge[
    "y"
].astype(
    np.int64
)


assert X.ndim == 2

assert len(
    X
) == len(y)

assert set(
    np.unique(
        y
    ).tolist()
) == {
    0,
    1,
}

assert np.isfinite(
    X
).all()


print(
    "\n========== EDGE-IIoTset ENCODED DATA =========="
)

print(
    "records:",
    len(y)
)

print(
    "features:",
    X.shape[1]
)

print(
    "benign:",
    int(
        np.sum(
            y == 0
        )
    )
)

print(
    "intrusion:",
    int(
        np.sum(
            y == 1
        )
    )
)

print(
    "intrusion fraction:",
    round(
        float(
            y.mean()
        ),
        4
    )
)

print(
    "================================================"
)


# ============================================================
# 9. Recreate alpha=0.3, K=10 split
# ============================================================

SPLIT_FILE = (
    PROC_DIR /
    "splits_edgeiiotset_dirichlet_alpha_0.3.npz"
)

SUMMARY_FILE = (
    PROC_DIR /
    "splits_edgeiiotset_dirichlet_alpha_0.3_summary.json"
)


print(
    "\nCreating Edge-IIoTset Dirichlet split..."
)


make_splits_and_save(
    encoded_npz=str(
        ENCODED_FILE
    ),

    out_npz=str(
        SPLIT_FILE
    ),

    out_summary_json=str(
        SUMMARY_FILE
    ),

    K=10,

    alpha=0.3,

    seed=42,

    ratios=(
        0.70,
        0.15,
        0.15,
    ),

    min_per_silo=10,
)


print(
    "✅ Edge-IIoTset split created."
)


# ============================================================
# 10. Final split audit
# ============================================================

spl = np.load(
    SPLIT_FILE,
    allow_pickle=True
)


train = spl[
    "train"
]

val = spl[
    "val"
]

test = spl[
    "test"
]


K = int(
    spl[
        "K"
    ][0]
)


alpha = float(
    spl[
        "alpha"
    ][0]
)


all_idx = []


print(
    "\n========== EDGE-IIoTset SILO AUDIT =========="
)


for k in range(K):

    tr = np.asarray(
        train[k],
        dtype=np.int64
    )

    va = np.asarray(
        val[k],
        dtype=np.int64
    )

    te = np.asarray(
        test[k],
        dtype=np.int64
    )


    combined = np.concatenate(
        [
            tr,
            va,
            te,
        ]
    )


    all_idx.extend(
        combined.tolist()
    )


    yk = y[
        combined
    ]


    print(
        f"Silo {k:2d} | "
        f"total={len(combined):7d} | "
        f"train={len(tr):7d} | "
        f"val={len(va):7d} | "
        f"test={len(te):7d} | "
        f"intrusion_rate={yk.mean():.4f}"
    )


all_idx = np.asarray(
    all_idx,
    dtype=np.int64
)


assert len(
    all_idx
) == len(y)


assert len(
    np.unique(
        all_idx
    )
) == len(y)


print(
    "\n✅ Every Edge-IIoTset record assigned exactly once."
)


# ============================================================
# 11. Reproducibility check against previous pipeline
# ============================================================

EXPECTED_ROWS = 1909671
EXPECTED_FEATURES = 96


print(
    "\n========== REPRODUCIBILITY CHECK =========="
)


if (
    len(y) ==
    EXPECTED_ROWS
):

    print(
        f"✅ record count matches previous run: {EXPECTED_ROWS}"
    )

else:

    print(
        f"⚠️ record count differs: "
        f"current={len(y)}, expected={EXPECTED_ROWS}"
    )


if (
    X.shape[1] ==
    EXPECTED_FEATURES
):

    print(
        f"✅ feature dimension matches previous run: {EXPECTED_FEATURES}"
    )

else:

    print(
        f"⚠️ feature dimension differs: "
        f"current={X.shape[1]}, expected={EXPECTED_FEATURES}"
    )


print(
    "==========================================="
)


# ============================================================
# 12. Final
# ============================================================

print(
    "\n============================================"
)

print(
    "✅ STEP 20.4A EDGE-IIoTset RESTORE PASSED"
)

print(
    "============================================"
)


print(
    "\nFinal:"
)

print(
    f"  records   = {len(y)}"
)

print(
    f"  input_dim = {X.shape[1]}"
)

print(
    f"  K         = {K}"
)

print(
    f"  alpha     = {alpha}"
)

print(
    "\nArtifacts saved persistently in Google Drive."
)

✅ Repository:
/content/fl_ids_repo_real

Edge-IIoTset raw CSV not yet in repository.
Extracting Edge-IIoTset archive...
✅ Archive extracted.

✅ Selected Edge-IIoTset CSV:
/content/fl_ids_repo_real/data/raw/edgeiiotset/Edge-IIoTset dataset/Selected dataset for ML and DL/DNN-EdgeIIoT-dataset.csv
Size: 1161.02 MB

✅ Restored:
/content/fl_ids_repo_real/src/data/preprocess_edgeiiotset.py
✅ Restored:
/content/fl_ids_repo_real/src/data/splits.py

Preprocessing Edge-IIoTset...
This is the large dataset, so this may take a few minutes.
✅ Edge-IIoTset encoding completed.

========== EDGE-IIoTset ENCODED DATA ==========
records: 1909671
features: 96
benign: 1363998
intrusion: 545673
intrusion fraction: 0.2857

Creating Edge-IIoTset Dirichlet split...
✅ Edge-IIoTset split created.

========== EDGE-IIoTset SILO AUDIT ==========
Silo  0 | total=  44448 | train=  31113 | val=   6667 | test=   6668 | intrusion_rate=0.4814
Silo  1 | total=  68101 | train=  47670 | val=  10215 | test=  10216 | intrusion

**Step 20.5A — Inspect the current training **

In [14]:
# ============================================================
# STEP 20.5A — Inspect current FL training interface
#
# Purpose:
#   Identify the exact existing client-training, model,
#   aggregation, clipping, and evaluation functions before
#   adding the targeted backdoor implementation.
#
# No files are modified.
# ============================================================

from pathlib import Path

REPO = Path("/content/fl_ids_repo_real")

print("========== REPOSITORY TRAINING AUDIT ==========\n")

# ------------------------------------------------------------
# Candidate implementation files
# ------------------------------------------------------------

keywords = [
    "train",
    "client",
    "federat",
    "aggregate",
    "fedavg",
    "fedprox",
    "trim",
    "model",
    "eval",
    "metric",
    "runner",
    "experiment",
]

candidate_files = []

for root_name in ["src", "scripts"]:

    root = REPO / root_name

    if not root.exists():
        continue

    for p in root.rglob("*.py"):

        name_lower = str(p.relative_to(REPO)).lower()

        if any(
            k in name_lower
            for k in keywords
        ):
            candidate_files.append(p)


candidate_files = sorted(
    set(candidate_files)
)


print("Candidate implementation files:")

for p in candidate_files:
    print(
        " ",
        p.relative_to(REPO)
    )


# ------------------------------------------------------------
# Search source for the exact mechanisms we need to extend
# ------------------------------------------------------------

search_terms = [
    "fedprox",
    "trimmean",
    "trim_mean",
    "clip",
    "local_train",
    "train_local",
    "train_client",
    "evaluate",
    "threshold",
    "weighted_bce",
    "bcewithlogits",
]


print(
    "\n========== RELEVANT DEFINITIONS =========="
)


for p in candidate_files:

    try:
        text = p.read_text(
            errors="ignore"
        )

    except Exception:
        continue

    lines = text.splitlines()

    hits = []

    for i, line in enumerate(lines):

        lower = line.lower()

        if (
            line.lstrip().startswith(
                ("def ", "class ")
            )
            or any(
                term in lower
                for term in search_terms
            )
        ):

            hits.append(
                (i + 1, line)
            )

    if hits:

        print(
            f"\n--- {p.relative_to(REPO)} ---"
        )

        for line_no, line in hits[:100]:

            print(
                f"{line_no:4d}: {line}"
            )


# ------------------------------------------------------------
# Show configs because backdoor experiment must inherit the
# actual model/FedProx/clipping settings rather than inventing
# a second training pipeline.
# ------------------------------------------------------------

print(
    "\n========== CONFIG FILES =========="
)

config_dir = REPO / "configs"

if config_dir.exists():

    for p in sorted(
        config_dir.glob("*.json")
    ):

        print(
            f"\n--- {p.relative_to(REPO)} ---"
        )

        try:

            text = p.read_text(
                errors="ignore"
            )

            print(
                text[:6000]
            )

        except Exception as e:

            print(
                "Could not read:",
                e
            )


print(
    "\n============================================"
)

print(
    "✅ STEP 20.5A TRAINING INTERFACE AUDIT COMPLETE"
)

print(
    "============================================"
)

========== REPOSITORY TRAINING AUDIT ==========

Candidate implementation files:

========== RELEVANT DEFINITIONS ==========

========== CONFIG FILES ==========

✅ STEP 20.5A TRAINING INTERFACE AUDIT COMPLETE


**Step 20.5B — Extract the actual training definitions from the notebook**

In [15]:
# ============================================================
# STEP 20.5B — Locate and inspect the actual notebook FL code
#
# Purpose:
#   - find the current notebook
#   - extract definitions related to:
#       model
#       FedProx
#       local training
#       clipping
#       FedAvg
#       TrimMean
#       threshold calibration
#       evaluation
#
# No files are modified.
# ============================================================

from pathlib import Path
import json
import re

SEARCH_ROOTS = [
    Path("/content"),
    Path("/content/drive/MyDrive"),
]

# ------------------------------------------------------------
# 1. Find notebooks
# ------------------------------------------------------------

notebooks = []

for root in SEARCH_ROOTS:

    if not root.exists():
        continue

    try:

        for p in root.rglob("*.ipynb"):

            if p.is_file():
                notebooks.append(p)

    except Exception:
        pass


print(
    "========== NOTEBOOK CANDIDATES =========="
)

for i, p in enumerate(
    notebooks,
    start=1
):

    print(
        f"[{i}] {p}"
    )


# ------------------------------------------------------------
# 2. Prefer FL / IIoT notebook
# ------------------------------------------------------------

preferred = []

for p in notebooks:

    name = p.name.lower()

    if (
        "fl" in name
        or "iiot" in name
        or "intrusion" in name
    ):

        preferred.append(p)


if preferred:

    NOTEBOOK = preferred[0]

elif notebooks:

    NOTEBOOK = notebooks[0]

else:

    raise FileNotFoundError(
        "No .ipynb notebook found in /content or Google Drive."
    )


print(
    "\n✅ Selected notebook:"
)

print(
    NOTEBOOK
)


# ------------------------------------------------------------
# 3. Load notebook JSON
# ------------------------------------------------------------

with open(
    NOTEBOOK,
    "r",
    encoding="utf-8"
) as f:

    nb = json.load(f)


# ------------------------------------------------------------
# 4. Search for relevant code
# ------------------------------------------------------------

KEYWORDS = [
    "class MLP",
    "class Net",
    "def local",
    "def train",
    "fedprox",
    "mu",
    "prox",
    "clip",
    "fedavg",
    "trimmean",
    "trim_mean",
    "aggregate",
    "threshold",
    "roc_curve",
    "precision_recall",
    "evaluate",
    "f1_score",
    "false_positive",
    "bcewithlogits",
    "pos_weight",
]


print(
    "\n========== RELEVANT NOTEBOOK CELLS =========="
)


relevant_cells = []


for idx, cell in enumerate(
    nb.get("cells", [])
):

    if cell.get("cell_type") != "code":
        continue

    source = "".join(
        cell.get(
            "source",
            []
        )
    )

    source_lower = source.lower()

    matched = [
        k
        for k in KEYWORDS
        if k.lower() in source_lower
    ]

    if matched:

        relevant_cells.append(
            idx
        )

        print(
            "\n"
            + "=" * 70
        )

        print(
            f"CELL {idx}"
        )

        print(
            "Matched:",
            matched
        )

        print(
            "=" * 70
        )

        # Print at most first 250 lines
        lines = source.splitlines()

        for line_no, line in enumerate(
            lines[:250],
            start=1
        ):

            print(
                f"{line_no:03d}: {line}"
            )


print(
    "\n============================================"
)

print(
    "Relevant cells found:",
    relevant_cells
)

print(
    "✅ STEP 20.5B NOTEBOOK TRAINING AUDIT COMPLETE"
)

print(
    "============================================"
)

Streaming output truncated to the last 5000 lines.
066: "import numpy as np",
067: "import pandas as pd",
068: "",
069: "DROP_COLUMNS_DEFAULT = [",
070: "    \"frame.time\", \"ip.src_host\", \"ip.dst_host\", \"arp.src.proto_ipv4\", \"arp.dst.proto_ipv4\",",
071: "    \"http.file_data\", \"http.request.full_uri\", \"icmp.transmit_timestamp\",",
072: "    \"http.request.uri.query\", \"tcp.options\", \"tcp.payload\", \"tcp.srcport\",",
073: "    \"tcp.dstport\", \"udp.port\", \"mqtt.msg\"",
074: "]",
075: "",
076: "PREFERRED_CAT_COLS = [",
077: "    \"http.request.method\", \"http.referer\", \"http.request.version\", \"dns.qry.name.len\",",
078: "    \"mqtt.conack.flags\", \"mqtt.protoname\", \"mqtt.topic\"",
079: "]",
080: "",
081: "def preprocess_edgeiiotset(csv_path: str, out_npz: str, out_meta: str, drop_columns=None):",
082: "    csv_path = Path(csv_path)",
083: "    out_npz = Path(out_npz); out_npz.parent.mkdir(parents=True, exist_ok=True)",
084: "    out_meta = Path(out_meta); out_

**Step 20.5 — Implement real targeted backdoor + explicit ASR**

In [16]:
# ============================================================
# STEP 20.5 — IMPLEMENT TARGETED BACKDOOR + ASR
#
# Implements:
#   ✓ existing MLP pipeline
#   ✓ local FedProx training
#   ✓ update clipping
#   ✓ FedAvg / TrimMean
#   ✓ targeted data-poisoning backdoor
#   ✓ fixed tabular trigger
#   ✓ clean per-silo threshold calibration
#   ✓ explicit ASR on triggered ATTACK test samples
#
# Important:
#   This is a REAL targeted backdoor experiment.
#   It is separate from the previous sign-flip experiment.
# ============================================================

from pathlib import Path
import json
import py_compile
import sys

REPO = Path("/content/fl_ids_repo_real")

assert REPO.exists()

# ------------------------------------------------------------
# Directories
# ------------------------------------------------------------

for d in [
    REPO / "src",
    REPO / "src/data",
    REPO / "src/models",
    REPO / "src/fl",
    REPO / "src/eval",
    REPO / "scripts",
    REPO / "configs",
    REPO / "configs/backdoor",
]:

    d.mkdir(
        parents=True,
        exist_ok=True
    )

    if "src" in d.parts:
        (d / "__init__.py").touch(
            exist_ok=True
        )


# ============================================================
# 1. MLP — same implementation as current notebook
# ============================================================

MLP_FILE = (
    REPO /
    "src/models/mlp.py"
)

MLP_FILE.write_text(
r'''
import torch
import torch.nn as nn


class MLPIDS(nn.Module):

    def __init__(
        self,
        input_dim,
        hidden=(256, 128, 64),
        dropout=0.30,
        batch_norm=False,
    ):

        super().__init__()

        layers = []

        in_dim = input_dim

        for h in hidden:

            layers.append(
                nn.Linear(
                    in_dim,
                    h
                )
            )

            if batch_norm:

                layers.append(
                    nn.BatchNorm1d(
                        h
                    )
                )

            layers.append(
                nn.ReLU()
            )

            layers.append(
                nn.Dropout(
                    dropout
                )
            )

            in_dim = h

        self.net = nn.Sequential(
            *layers
        )

        self.out = nn.Linear(
            in_dim,
            1
        )


    def forward(self, x):

        return self.out(
            self.net(x)
        ).squeeze(-1)
'''
)


# ============================================================
# 2. Data loaders — existing local-normalization protocol
# ============================================================

LOADERS_FILE = (
    REPO /
    "src/data/loaders.py"
)

LOADERS_FILE.write_text(
r'''
import numpy as np
import torch

from torch.utils.data import (
    Dataset,
    DataLoader,
)


class TabularDataset(Dataset):

    def __init__(
        self,
        X,
        y,
    ):

        self.X = torch.from_numpy(
            X
        ).float()

        self.y = torch.from_numpy(
            y
        ).long()


    def __len__(self):

        return self.X.shape[0]


    def __getitem__(
        self,
        i,
    ):

        return (
            self.X[i],
            self.y[i],
        )


def fit_local_norm(
    X_train
):

    mu = X_train.mean(
        axis=0,
        dtype=np.float64
    )

    sigma = X_train.std(
        axis=0,
        dtype=np.float64
    )

    return (
        mu.astype(
            np.float32
        ),
        sigma.astype(
            np.float32
        ),
    )


def apply_local_norm(
    X,
    mu,
    sigma,
    eps=1e-8,
):

    return (
        X - mu
    ) / (
        sigma + eps
    )


def build_silo_loaders(
    encoded_npz,
    splits_npz,
    batch_size=256,
    eps=1e-8,
):

    data = np.load(
        encoded_npz,
        allow_pickle=True
    )

    X = data[
        "X"
    ].astype(
        np.float32
    )

    y = data[
        "y"
    ].astype(
        np.int64
    )


    spl = np.load(
        splits_npz,
        allow_pickle=True
    )

    train_idxs = spl[
        "train"
    ]

    val_idxs = spl[
        "val"
    ]

    test_idxs = spl[
        "test"
    ]

    K = int(
        spl["K"][0]
    )


    silo = []


    for k in range(K):

        tr = np.asarray(
            train_idxs[k],
            dtype=np.int64
        )

        va = np.asarray(
            val_idxs[k],
            dtype=np.int64
        )

        te = np.asarray(
            test_idxs[k],
            dtype=np.int64
        )


        X_tr = X[tr]
        y_tr = y[tr]

        X_va = X[va]
        y_va = y[va]

        X_te = X[te]
        y_te = y[te]


        # IMPORTANT:
        # normalization parameters are learned from
        # local TRAINING data only.
        mu_k, sigma_k = fit_local_norm(
            X_tr
        )


        X_tr = apply_local_norm(
            X_tr,
            mu_k,
            sigma_k,
            eps
        )

        X_va = apply_local_norm(
            X_va,
            mu_k,
            sigma_k,
            eps
        )

        X_te = apply_local_norm(
            X_te,
            mu_k,
            sigma_k,
            eps
        )


        silo.append(
            {
                "train_loader":
                    DataLoader(
                        TabularDataset(
                            X_tr,
                            y_tr
                        ),
                        batch_size=batch_size,
                        shuffle=True,
                    ),

                "val_loader":
                    DataLoader(
                        TabularDataset(
                            X_va,
                            y_va
                        ),
                        batch_size=batch_size,
                        shuffle=False,
                    ),

                "test_loader":
                    DataLoader(
                        TabularDataset(
                            X_te,
                            y_te
                        ),
                        batch_size=batch_size,
                        shuffle=False,
                    ),

                # Keep normalized arrays available for
                # reproducible poisoning / trigger evaluation.
                "X_train":
                    X_tr,

                "y_train":
                    y_tr,

                "X_val":
                    X_va,

                "y_val":
                    y_va,

                "X_test":
                    X_te,

                "y_test":
                    y_te,

                "n_train":
                    int(
                        len(tr)
                    ),

                "n_val":
                    int(
                        len(va)
                    ),

                "n_test":
                    int(
                        len(te)
                    ),
            }
        )


    return (
        silo,
        X.shape[1],
        K,
    )
'''
)


# ============================================================
# 3. FL aggregation
# ============================================================

AGG_FILE = (
    REPO /
    "src/fl/aggregators.py"
)

AGG_FILE.write_text(
r'''
import numpy as np
import torch


def agg_fedavg(
    deltas,
    weights,
):

    W = float(
        np.sum(
            weights
        )
    )

    out = torch.zeros_like(
        deltas[0]
    )

    for delta, weight in zip(
        deltas,
        weights
    ):

        out += (
            delta *
            (
                float(weight) /
                W
            )
        )

    return out


def agg_trimmed_mean(
    deltas,
    beta=0.10,
):

    X = torch.stack(
        deltas,
        dim=0
    )

    m = X.shape[0]

    trim = int(
        np.floor(
            beta * m
        )
    )

    if (
        2 * trim >= m
    ):

        return torch.mean(
            X,
            dim=0
        )


    sorted_X, _ = torch.sort(
        X,
        dim=0
    )


    kept = sorted_X[
        trim:
        m - trim,
        :
    ]


    return torch.mean(
        kept,
        dim=0
    )
'''
)


# ============================================================
# 4. FedProx client training
# ============================================================

CLIENT_FILE = (
    REPO /
    "src/fl/client.py"
)

CLIENT_FILE.write_text(
r'''
import copy
import torch


def get_param_vector(
    model
):

    return torch.cat(
        [
            p.view(-1)
            for p
            in model.parameters()
        ]
    )


def fedprox_penalty(
    model,
    w0_vec,
):

    w_vec = get_param_vector(
        model
    )

    return 0.5 * torch.sum(
        (
            w_vec -
            w0_vec
        ) ** 2
    )


def train_one_client_fedprox(
    global_model,
    train_loader,
    criterion,
    device,
    cfg,
    seed_offset=0,
):

    torch.manual_seed(
        int(
            cfg["seed"] +
            seed_offset
        )
    )


    model = copy.deepcopy(
        global_model
    ).to(
        device
    )


    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=cfg["lr"],
        weight_decay=
            cfg["weight_decay"],
    )


    with torch.no_grad():

        w0 = get_param_vector(
            model
        ).detach()


    model.train()


    for _ in range(
        cfg["local_epochs"]
    ):

        for xb, yb in train_loader:

            xb = xb.to(
                device
            )

            yb = (
                yb.float()
                .to(device)
            )


            optimizer.zero_grad()


            logits = model(
                xb
            )


            base_loss = criterion(
                logits,
                yb
            )


            prox_loss = (
                cfg["fedprox_mu"] *
                fedprox_penalty(
                    model,
                    w0
                )
            )


            loss = (
                base_loss +
                prox_loss
            )


            loss.backward()


            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                cfg["grad_clip"],
            )


            optimizer.step()


    return model
'''
)


# ============================================================
# 5. FL vector utilities
# ============================================================

UTIL_FILE = (
    REPO /
    "src/fl/utils.py"
)

UTIL_FILE.write_text(
r'''
import torch


def state_dict_to_vector(
    state_dict
):

    return torch.cat(
        [
            value.detach()
            .reshape(-1)
            .float()
            .cpu()

            for value
            in state_dict.values()

            if torch.is_tensor(
                value
            )
        ]
    )


def vector_to_state_dict_like(
    vector,
    reference_state,
):

    out = {}

    offset = 0


    for key, value in (
        reference_state.items()
    ):

        if torch.is_tensor(
            value
        ):

            n = value.numel()

            out[key] = (
                vector[
                    offset:
                    offset + n
                ]
                .view_as(value)
                .to(value.dtype)
            )

            offset += n

        else:

            out[key] = value


    return out


def compute_delta(
    global_model,
    local_model,
):

    global_state = (
        global_model.state_dict()
    )

    local_state = (
        local_model.state_dict()
    )


    delta = (
        state_dict_to_vector(
            local_state
        ) -
        state_dict_to_vector(
            global_state
        )
    )


    return (
        delta,
        global_state,
    )


def clip_and_noise(
    delta_vec,
    C=1.0,
    sigma=0.0,
    seed=42,
):

    delta = (
        delta_vec
        .clone()
    )


    norm = torch.linalg.norm(
        delta,
        ord=2
    )


    scale = max(
        1.0,
        float(
            norm.item()
        ) /
        float(C)
    )


    delta = (
        delta /
        scale
    )


    if (
        sigma is not None
        and sigma > 0.0
    ):

        generator = (
            torch.Generator(
                device="cpu"
            )
            .manual_seed(
                int(seed)
            )
        )


        delta = (
            delta +
            torch.normal(
                mean=0.0,
                std=float(sigma),
                size=delta.shape,
                generator=generator,
            )
        )


    return delta


def apply_delta_to_model(
    global_model,
    delta_vec,
    reference_state,
    gamma=1.0,
):

    current_state = (
        global_model.state_dict()
    )


    delta_state = (
        vector_to_state_dict_like(
            delta_vec,
            reference_state,
        )
    )


    new_state = {}


    for key, value in (
        current_state.items()
    ):

        if torch.is_tensor(
            value
        ):

            new_state[key] = (
                value +
                gamma *
                delta_state[key].to(
                    value.device
                )
            )

        else:

            new_state[key] = (
                value
            )


    global_model.load_state_dict(
        new_state
    )


    return global_model
'''
)


# ============================================================
# 6. Targeted backdoor implementation
# ============================================================

BACKDOOR_FILE = (
    REPO /
    "src/fl/backdoor.py"
)

BACKDOOR_FILE.write_text(
r'''
import numpy as np
import torch

from torch.utils.data import (
    DataLoader,
)

from src.data.loaders import (
    TabularDataset,
)


def select_trigger_features(
    silo_data,
    trigger_dims=5,
):
    """
    Select trigger dimensions using TRAINING DATA ONLY.

    The same feature indices are used for all silos.

    Features with the largest pooled training variance are
    selected. No validation/test information is used.
    """

    d = (
        silo_data[0][
            "X_train"
        ].shape[1]
    )

    sum_x = np.zeros(
        d,
        dtype=np.float64
    )

    sum_x2 = np.zeros(
        d,
        dtype=np.float64
    )

    n_total = 0


    for site in silo_data:

        X = site[
            "X_train"
        ].astype(
            np.float64,
            copy=False
        )

        sum_x += X.sum(
            axis=0
        )

        sum_x2 += (
            X * X
        ).sum(
            axis=0
        )

        n_total += (
            X.shape[0]
        )


    mean = (
        sum_x /
        max(
            n_total,
            1
        )
    )


    variance = (
        sum_x2 /
        max(
            n_total,
            1
        )
        -
        mean ** 2
    )


    trigger_dims = min(
        int(trigger_dims),
        d
    )


    selected = np.argsort(
        variance
    )[
        -trigger_dims:
    ]


    return np.sort(
        selected.astype(
            np.int64
        )
    )


def apply_trigger(
    X,
    trigger_indices,
    trigger_value=3.0,
):
    """
    Fixed tabular trigger in LOCAL NORMALIZED feature space.

    A value of +3 corresponds to a strong but bounded
    positive standardized feature value.
    """

    X_triggered = np.array(
        X,
        dtype=np.float32,
        copy=True
    )


    X_triggered[
        :,
        trigger_indices
    ] = float(
        trigger_value
    )


    return X_triggered


def make_poisoned_train_loader(
    X_train,
    y_train,
    trigger_indices,
    poison_fraction,
    trigger_value,
    batch_size,
    seed,
):
    """
    Targeted attack -> benign backdoor.

    Among local ATTACK samples (y=1):
      1. select poison_fraction
      2. inject fixed trigger
      3. change their training target to benign (0)

    Clean samples remain unchanged.
    """

    X_poison = np.array(
        X_train,
        dtype=np.float32,
        copy=True
    )

    y_poison = np.array(
        y_train,
        dtype=np.int64,
        copy=True
    )


    attack_idx = np.where(
        y_poison == 1
    )[0]


    if (
        len(attack_idx) == 0
        or poison_fraction <= 0
    ):

        loader = DataLoader(
            TabularDataset(
                X_poison,
                y_poison
            ),
            batch_size=batch_size,
            shuffle=True,
        )

        return (
            loader,
            0
        )


    rng = np.random.default_rng(
        int(seed)
    )


    n_poison = max(
        1,
        int(
            np.floor(
                poison_fraction *
                len(attack_idx)
            )
        )
    )


    selected = rng.choice(
        attack_idx,
        size=n_poison,
        replace=False,
    )


    X_poison[
        selected
    ] = apply_trigger(
        X_poison[
            selected
        ],
        trigger_indices,
        trigger_value,
    )


    # Target class = benign.
    y_poison[
        selected
    ] = 0


    generator = (
        torch.Generator()
        .manual_seed(
            int(seed)
        )
    )


    loader = DataLoader(
        TabularDataset(
            X_poison,
            y_poison,
        ),
        batch_size=batch_size,
        shuffle=True,
        generator=generator,
    )


    return (
        loader,
        int(
            n_poison
        )
    )


@torch.no_grad()
def triggered_attack_success_rate(
    model,
    X_test,
    y_test,
    threshold,
    trigger_indices,
    trigger_value,
    device,
    batch_size=4096,
):
    """
    ASR =
        triggered attack samples classified as BENIGN
        ------------------------------------------------
        total triggered attack samples

    Only clean test samples with true label y=1 are used.
    """

    attack_mask = (
        np.asarray(
            y_test
        ) == 1
    )


    X_attack = np.asarray(
        X_test
    )[
        attack_mask
    ]


    if len(
        X_attack
    ) == 0:

        return {
            "asr": None,
            "n_triggered_attack":
                0,
            "n_predicted_benign":
                0,
        }


    X_triggered = apply_trigger(
        X_attack,
        trigger_indices,
        trigger_value,
    )


    dataset = TabularDataset(
        X_triggered,
        np.ones(
            len(X_triggered),
            dtype=np.int64
        )
    )


    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
    )


    model.eval()

    predictions = []


    for xb, _ in loader:

        xb = xb.to(
            device
        )

        probs = torch.sigmoid(
            model(
                xb
            )
        )


        pred = (
            probs >=
            float(
                threshold
            )
        ).long()


        predictions.append(
            pred.cpu().numpy()
        )


    predictions = np.concatenate(
        predictions
    )


    n_benign = int(
        np.sum(
            predictions == 0
        )
    )


    asr = (
        n_benign /
        len(predictions)
    )


    return {
        "asr":
            float(asr),

        "n_triggered_attack":
            int(
                len(predictions)
            ),

        "n_predicted_benign":
            n_benign,
    }
'''
)


# ============================================================
# 7. Threshold/evaluation functions
# ============================================================

EVAL_FILE = (
    REPO /
    "src/eval/thresholds.py"
)

EVAL_FILE.write_text(
r'''
import numpy as np
import torch


def metrics_at_threshold(
    probs,
    y_true,
    threshold,
):

    y_pred = (
        probs >=
        threshold
    ).astype(
        int
    )


    tp = int(
        (
            (y_pred == 1) &
            (y_true == 1)
        ).sum()
    )

    tn = int(
        (
            (y_pred == 0) &
            (y_true == 0)
        ).sum()
    )

    fp = int(
        (
            (y_pred == 1) &
            (y_true == 0)
        ).sum()
    )

    fn = int(
        (
            (y_pred == 0) &
            (y_true == 1)
        ).sum()
    )


    precision = (
        tp /
        (
            tp +
            fp +
            1e-12
        )
    )

    recall = (
        tp /
        (
            tp +
            fn +
            1e-12
        )
    )

    f1 = (
        2 *
        precision *
        recall /
        (
            precision +
            recall +
            1e-12
        )
    )

    accuracy = (
        (
            tp +
            tn
        ) /
        max(
            1,
            tp +
            tn +
            fp +
            fn
        )
    )

    fpr = (
        fp /
        (
            fp +
            tn +
            1e-12
        )
    )


    return {
        "acc":
            float(
                accuracy
            ),

        "precision":
            float(
                precision
            ),

        "recall":
            float(
                recall
            ),

        "f1":
            float(
                f1
            ),

        "fpr":
            float(
                fpr
            ),

        "tp":
            tp,

        "tn":
            tn,

        "fp":
            fp,

        "fn":
            fn,

        "pred_pos_rate":
            float(
                y_pred.mean()
            ),
    }


@torch.no_grad()
def collect_probs(
    model,
    loader,
    device,
):

    probs = []
    labels = []


    model.eval()


    for xb, yb in loader:

        xb = xb.to(
            device
        )


        p = torch.sigmoid(
            model(
                xb
            )
        )


        probs.append(
            p.cpu().numpy()
        )

        labels.append(
            yb.numpy()
        )


    return (
        np.concatenate(
            probs
        ).astype(
            np.float64
        ),

        np.concatenate(
            labels
        ).astype(
            int
        ),
    )


def robust_theta_from_fpr(
    probs_val,
    y_val,
    default_theta,
    target_fpr=0.05,
    theta_min=0.05,
    theta_max=0.95,
    min_benign_val=200,
):

    benign = probs_val[
        y_val == 0
    ]


    if (
        benign.size <
        min_benign_val
    ):

        return (
            float(
                default_theta
            ),
            "fallback_default"
        )


    threshold = float(
        np.quantile(
            benign,
            1.0 -
            target_fpr
        )
    )


    threshold = float(
        np.clip(
            threshold,
            theta_min,
            theta_max,
        )
    )


    return (
        threshold,
        "quantile_benign"
    )
'''
)


# ============================================================
# 8. Backdoor experiment runner
# ============================================================

RUNNER_FILE = (
    REPO /
    "scripts/run_backdoor_experiment.py"
)

RUNNER_FILE.write_text(
r'''
import argparse
import json
import time
import random

from pathlib import Path

import numpy as np
import torch
import torch.nn as nn


from src.data.loaders import (
    build_silo_loaders,
)

from src.models.mlp import (
    MLPIDS,
)

from src.fl.client import (
    train_one_client_fedprox,
)

from src.fl.aggregators import (
    agg_fedavg,
    agg_trimmed_mean,
)

from src.fl.utils import (
    compute_delta,
    clip_and_noise,
    apply_delta_to_model,
)

from src.fl.backdoor import (
    select_trigger_features,
    make_poisoned_train_loader,
    triggered_attack_success_rate,
)

from src.eval.thresholds import (
    collect_probs,
    metrics_at_threshold,
    robust_theta_from_fpr,
)


def set_seed(
    seed
):

    random.seed(
        seed
    )

    np.random.seed(
        seed
    )

    torch.manual_seed(
        seed
    )

    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(
            seed
        )


def global_clean_evaluation(
    model,
    silo_data,
    device,
    cfg,
):

    per_silo = []

    thresholds = []


    for k, site in enumerate(
        silo_data
    ):

        probs_val, y_val = (
            collect_probs(
                model,
                site[
                    "val_loader"
                ],
                device,
            )
        )


        threshold, note = (
            robust_theta_from_fpr(
                probs_val,
                y_val,
                cfg[
                    "threshold"
                ],

                target_fpr=
                    cfg[
                        "calibration"
                    ][
                        "target_fpr"
                    ],

                theta_min=
                    cfg[
                        "calibration"
                    ][
                        "theta_min"
                    ],

                theta_max=
                    cfg[
                        "calibration"
                    ][
                        "theta_max"
                    ],

                min_benign_val=
                    cfg[
                        "calibration"
                    ][
                        "min_benign_val"
                    ],
            )
        )


        # Preserve the notebook's threshold guard.
        val_metrics = (
            metrics_at_threshold(
                probs_val,
                y_val,
                threshold,
            )
        )


        if (
            val_metrics[
                "pred_pos_rate"
            ] >
            cfg[
                "calibration"
            ][
                "max_pred_pos"
            ]
        ):

            threshold = min(
                cfg[
                    "calibration"
                ][
                    "theta_max"
                ],
                max(
                    threshold,
                    0.80
                )
            )


        elif (
            val_metrics[
                "pred_pos_rate"
            ] <
            cfg[
                "calibration"
            ][
                "min_pred_pos"
            ]
        ):

            threshold = max(
                cfg[
                    "calibration"
                ][
                    "theta_min"
                ],
                min(
                    threshold,
                    0.20
                )
            )


        probs_test, y_test = (
            collect_probs(
                model,
                site[
                    "test_loader"
                ],
                device,
            )
        )


        metrics = (
            metrics_at_threshold(
                probs_test,
                y_test,
                threshold,
            )
        )


        metrics[
            "silo"
        ] = int(k)

        metrics[
            "threshold"
        ] = float(
            threshold
        )

        metrics[
            "threshold_note"
        ] = note


        per_silo.append(
            metrics
        )

        thresholds.append(
            float(
                threshold
            )
        )


    tp = sum(
        x["tp"]
        for x in per_silo
    )

    tn = sum(
        x["tn"]
        for x in per_silo
    )

    fp = sum(
        x["fp"]
        for x in per_silo
    )

    fn = sum(
        x["fn"]
        for x in per_silo
    )


    precision = (
        tp /
        (
            tp +
            fp +
            1e-12
        )
    )

    recall = (
        tp /
        (
            tp +
            fn +
            1e-12
        )
    )

    f1 = (
        2 *
        precision *
        recall /
        (
            precision +
            recall +
            1e-12
        )
    )

    fpr = (
        fp /
        (
            fp +
            tn +
            1e-12
        )
    )


    return (
        {
            "f1":
                float(
                    f1
                ),

            "fpr":
                float(
                    fpr
                ),

            "precision":
                float(
                    precision
                ),

            "recall":
                float(
                    recall
                ),

            "tp":
                int(
                    tp
                ),

            "tn":
                int(
                    tn
                ),

            "fp":
                int(
                    fp
                ),

            "fn":
                int(
                    fn
                ),
        },

        per_silo,
        thresholds,
    )


def evaluate_backdoor_asr(
    model,
    silo_data,
    thresholds,
    trigger_indices,
    trigger_value,
    device,
    batch_size,
):

    per_silo = []

    total_triggered = 0
    total_success = 0


    for k, site in enumerate(
        silo_data
    ):

        result = (
            triggered_attack_success_rate(
                model=model,

                X_test=
                    site[
                        "X_test"
                    ],

                y_test=
                    site[
                        "y_test"
                    ],

                threshold=
                    thresholds[k],

                trigger_indices=
                    trigger_indices,

                trigger_value=
                    trigger_value,

                device=device,

                batch_size=
                    batch_size,
            )
        )


        result[
            "silo"
        ] = int(k)

        result[
            "threshold"
        ] = float(
            thresholds[k]
        )


        per_silo.append(
            result
        )


        total_triggered += (
            result[
                "n_triggered_attack"
            ]
        )

        total_success += (
            result[
                "n_predicted_benign"
            ]
        )


    global_asr = (
        total_success /
        total_triggered
        if total_triggered > 0
        else None
    )


    return (
        {
            "asr":
                (
                    float(
                        global_asr
                    )
                    if global_asr
                    is not None
                    else None
                ),

            "n_triggered_attack":
                int(
                    total_triggered
                ),

            "n_predicted_benign":
                int(
                    total_success
                ),
        },

        per_silo,
    )


def main():

    parser = (
        argparse.ArgumentParser()
    )

    parser.add_argument(
        "--config",
        required=True,
    )

    args = parser.parse_args()


    cfg = json.loads(
        Path(
            args.config
        ).read_text()
    )


    set_seed(
        int(
            cfg[
                "seed"
            ]
        )
    )


    device = torch.device(
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )


    print(
        "Device:",
        device
    )


    # --------------------------------------------------------
    # Load FL data
    # --------------------------------------------------------

    silo_data, input_dim, K = (
        build_silo_loaders(
            cfg[
                "encoded_npz"
            ],

            cfg[
                "splits_npz"
            ],

            batch_size=
                cfg[
                    "batch_size"
                ],
        )
    )


    assert K == 10


    # --------------------------------------------------------
    # Trigger chosen ONLY from training data
    # --------------------------------------------------------

    trigger_indices = (
        select_trigger_features(
            silo_data,
            trigger_dims=
                cfg[
                    "backdoor"
                ][
                    "trigger_dims"
                ],
        )
    )


    trigger_value = float(
        cfg[
            "backdoor"
        ][
            "trigger_value"
        ]
    )


    print(
        "Trigger indices:",
        trigger_indices.tolist()
    )

    print(
        "Trigger value:",
        trigger_value
    )


    # --------------------------------------------------------
    # Model
    # --------------------------------------------------------

    model = MLPIDS(
        input_dim,

        hidden=tuple(
            cfg[
                "hidden"
            ]
        ),

        dropout=
            cfg[
                "dropout"
            ],

        batch_norm=
            cfg[
                "batch_norm"
            ],
    ).to(
        device
    )


    pos_weight = torch.tensor(
        [
            cfg[
                "pos_weight"
            ]
        ],
        device=device,
    )


    criterion = (
        nn.BCEWithLogitsLoss(
            pos_weight=
                pos_weight
        )
    )


    rng = np.random.default_rng(
        cfg[
            "seed"
        ]
    )


    round_audit = []


    # --------------------------------------------------------
    # FL rounds
    # --------------------------------------------------------

    for round_id in range(
        cfg[
            "rounds"
        ]
    ):

        # Security experiment:
        # full participation keeps exactly K=10 and
        # exactly two malicious silos at alpha_mal=0.20.
        participants = list(
            range(K)
        )


        malicious_count = int(
            np.floor(
                cfg[
                    "malicious_frac"
                ] *
                len(
                    participants
                )
            )
        )


        malicious = set(
            rng.choice(
                participants,
                size=
                    malicious_count,
                replace=False,
            ).tolist()
        )


        deltas = []
        weights = []

        ref_state = None

        poisoned_samples_round = 0


        for client_id in participants:

            site = silo_data[
                client_id
            ]


            if (
                client_id
                in malicious
            ):

                train_loader, n_poison = (
                    make_poisoned_train_loader(
                        X_train=
                            site[
                                "X_train"
                            ],

                        y_train=
                            site[
                                "y_train"
                            ],

                        trigger_indices=
                            trigger_indices,

                        poison_fraction=
                            cfg[
                                "backdoor"
                            ][
                                "poison_fraction"
                            ],

                        trigger_value=
                            trigger_value,

                        batch_size=
                            cfg[
                                "batch_size"
                            ],

                        seed=
                            cfg[
                                "seed"
                            ]
                            +
                            1000 *
                            round_id
                            +
                            client_id,
                    )
                )


                poisoned_samples_round += (
                    n_poison
                )

            else:

                train_loader = site[
                    "train_loader"
                ]


            local_model = (
                train_one_client_fedprox(
                    model,

                    train_loader,

                    criterion,

                    device,

                    cfg,

                    seed_offset=
                        1000 *
                        round_id
                        +
                        client_id,
                )
            )


            delta_vec, ref_state = (
                compute_delta(
                    model,
                    local_model,
                )
            )


            # IMPORTANT:
            # Backdoor poisoning occurs during LOCAL TRAINING.
            # Update protection therefore happens afterwards.
            if cfg[
                "use_update_clipping"
            ]:

                protected_delta = (
                    clip_and_noise(
                        delta_vec,

                        C=
                            cfg[
                                "clip_C"
                            ],

                        sigma=
                            cfg[
                                "dp_sigma"
                            ],

                        seed=
                            cfg[
                                "seed"
                            ]
                            +
                            1000 *
                            round_id
                            +
                            client_id,
                    )
                )

            else:

                protected_delta = (
                    delta_vec.clone()
                )


            deltas.append(
                protected_delta
            )


            weights.append(
                site[
                    "n_train"
                ]
            )


        # ----------------------------------------------------
        # Server aggregation
        # ----------------------------------------------------

        if (
            cfg[
                "aggregator"
            ] ==
            "fedavg"
        ):

            aggregate = (
                agg_fedavg(
                    deltas,
                    weights,
                )
            )

        elif (
            cfg[
                "aggregator"
            ] ==
            "trimmean"
        ):

            aggregate = (
                agg_trimmed_mean(
                    deltas,
                    beta=
                        cfg[
                            "trim_beta"
                        ],
                )
            )

        else:

            raise ValueError(
                "aggregator must be "
                "'fedavg' or 'trimmean'"
            )


        model = (
            apply_delta_to_model(
                model,

                aggregate,

                ref_state,

                gamma=
                    cfg[
                        "server_gamma"
                    ],
            )
        )


        audit_row = {
            "round":
                int(
                    round_id
                ),

            "participants":
                [
                    int(x)
                    for x
                    in participants
                ],

            "malicious_clients":
                [
                    int(x)
                    for x
                    in sorted(
                        malicious
                    )
                ],

            "poisoned_training_samples":
                int(
                    poisoned_samples_round
                ),
        }


        round_audit.append(
            audit_row
        )


        print(
            f"round={round_id + 1:02d} "
            f"| malicious={sorted(malicious)} "
            f"| poisoned_samples="
            f"{poisoned_samples_round}"
        )


    # --------------------------------------------------------
    # Clean evaluation under poisoned training
    # --------------------------------------------------------

    (
        clean_global,
        clean_per_silo,
        thresholds,
    ) = global_clean_evaluation(
        model,
        silo_data,
        device,
        cfg,
    )


    # --------------------------------------------------------
    # Triggered-test ASR
    # --------------------------------------------------------

    (
        asr_global,
        asr_per_silo,
    ) = evaluate_backdoor_asr(
        model,
        silo_data,
        thresholds,
        trigger_indices,
        trigger_value,
        device,
        cfg[
            "eval_batch_size"
        ],
    )


    # --------------------------------------------------------
    # Save everything
    # --------------------------------------------------------

    timestamp = time.strftime(
        "%Y%m%d_%H%M%S",
        time.gmtime()
    )


    run_dir = (
        Path(
            "runs/backdoor"
        )
        /
        (
            cfg[
                "dataset"
            ]
            +
            "_"
            +
            cfg[
                "name"
            ]
            +
            "_seed"
            +
            str(
                cfg[
                    "seed"
                ]
            )
            +
            "_"
            +
            timestamp
        )
    )


    run_dir.mkdir(
        parents=True,
        exist_ok=True
    )


    result = {
        "dataset":
            cfg[
                "dataset"
            ],

        "experiment":
            cfg[
                "name"
            ],

        "seed":
            int(
                cfg[
                    "seed"
                ]
            ),

        "K":
            int(K),

        "input_dim":
            int(
                input_dim
            ),

        "malicious_fraction":
            float(
                cfg[
                    "malicious_frac"
                ]
            ),

        "poison_fraction":
            float(
                cfg[
                    "backdoor"
                ][
                    "poison_fraction"
                ]
            ),

        "trigger_indices":
            [
                int(x)
                for x
                in trigger_indices
            ],

        "trigger_value":
            float(
                trigger_value
            ),

        "target_class":
            "benign",

        "clean_metrics_under_attack":
            clean_global,

        "backdoor":
            asr_global,

        "clean_per_silo":
            clean_per_silo,

        "backdoor_per_silo":
            asr_per_silo,

        "round_audit":
            round_audit,
    }


    (
        run_dir /
        "results.json"
    ).write_text(
        json.dumps(
            result,
            indent=2
        )
    )


    (
        run_dir /
        "config.json"
    ).write_text(
        json.dumps(
            cfg,
            indent=2
        )
    )


    torch.save(
        model.state_dict(),
        run_dir /
        "model_final.pt",
    )


    print(
        "\n============================================"
    )

    print(
        "FINAL CLEAN METRICS UNDER BACKDOOR TRAINING"
    )

    print(
        json.dumps(
            clean_global,
            indent=2
        )
    )


    print(
        "\nBACKDOOR ASR"
    )

    print(
        json.dumps(
            asr_global,
            indent=2
        )
    )


    print(
        "\nSaved:"
    )

    print(
        run_dir
    )


if __name__ == "__main__":

    main()
'''
)


# ============================================================
# 9. Create three dataset configs
# ============================================================

BASE = {
    "seed": 42,

    "batch_size": 256,

    # Existing notebook architecture
    "hidden": [
        256,
        128,
        64
    ],

    "dropout": 0.30,

    # IMPORTANT:
    # current executable notebook uses False
    "batch_norm": False,

    "pos_weight": 2.0,

    # Security experiment
    "rounds": 10,

    "local_epochs": 2,

    "fedprox_mu": 1e-3,

    # Current executable notebook value
    "lr": 3e-4,

    "weight_decay": 1e-4,

    "grad_clip": 1.0,

    "server_gamma": 1.0,

    "malicious_frac": 0.20,

    "use_update_clipping": True,

    "clip_C": 1.0,

    "dp_sigma": 0.0,

    "aggregator": "trimmean",

    "trim_beta": 0.10,

    "threshold": 0.50,

    "eval_batch_size": 4096,

    "backdoor": {
        # Poison 20% of malicious-client attack-class
        # training samples.
        "poison_fraction": 0.20,

        # Fixed 5-dimensional trigger.
        "trigger_dims": 5,

        # Applied after local normalization.
        "trigger_value": 3.0,
    },

    "calibration": {
        "target_fpr": 0.05,
        "theta_min": 0.05,
        "theta_max": 0.95,
        "min_benign_val": 200,
        "max_pred_pos": 0.90,
        "min_pred_pos": 0.01,
    },
}


datasets = {
    "edgeiiotset": {
        "encoded_npz":
            "data/processed/edgeiiotset_encoded.npz",

        "splits_npz":
            "data/processed/splits_edgeiiotset_dirichlet_alpha_0.3.npz",
    },

    "ton_iot": {
        "encoded_npz":
            "data/processed/ton_iot_encoded.npz",

        "splits_npz":
            "data/processed/splits_ton_iot_dirichlet_alpha_0.3.npz",
    },

    "xiiotid": {
        "encoded_npz":
            "data/processed/xiiotid_encoded.npz",

        "splits_npz":
            "data/processed/splits_xiiotid_dirichlet_alpha_0.3.npz",
    },
}


CONFIG_DIR = (
    REPO /
    "configs/backdoor"
)


for dataset_name, paths in (
    datasets.items()
):

    cfg = dict(
        BASE
    )

    cfg["backdoor"] = dict(
        BASE[
            "backdoor"
        ]
    )

    cfg["calibration"] = dict(
        BASE[
            "calibration"
        ]
    )

    cfg[
        "dataset"
    ] = dataset_name

    cfg[
        "name"
    ] = (
        "modeA_targeted_backdoor"
    )

    cfg.update(
        paths
    )


    out = (
        CONFIG_DIR /
        (
            dataset_name +
            "_modeA.json"
        )
    )


    out.write_text(
        json.dumps(
            cfg,
            indent=2
        )
    )


    print(
        "✅ Config:",
        out
    )


# ============================================================
# 10. Syntax check
# ============================================================

files_to_check = [
    MLP_FILE,
    LOADERS_FILE,
    AGG_FILE,
    CLIENT_FILE,
    UTIL_FILE,
    BACKDOOR_FILE,
    EVAL_FILE,
    RUNNER_FILE,
]


for p in files_to_check:

    py_compile.compile(
        str(p),
        doraise=True
    )

    print(
        "✅ syntax:",
        p.relative_to(REPO)
    )


# ============================================================
# 11. Artifact checks
# ============================================================

print(
    "\n========== DATASET ARTIFACT CHECK =========="
)


for dataset_name, paths in datasets.items():

    enc = (
        REPO /
        paths[
            "encoded_npz"
        ]
    )

    spl = (
        REPO /
        paths[
            "splits_npz"
        ]
    )


    print(
        dataset_name,
        "| encoded:",
        enc.exists(),
        "| split:",
        spl.exists()
    )


    assert enc.exists()

    assert spl.exists()


# ============================================================
# Final
# ============================================================

print(
    "\n============================================"
)

print(
    "✅ STEP 20.5 BACKDOOR IMPLEMENTATION COMPLETE"
)

print(
    "============================================"
)

print(
    "\nImplemented attack:"
)

print(
    "  true attack sample (y=1)"
)

print(
    "      ↓"
)

print(
    "  fixed 5-feature trigger"
)

print(
    "      ↓"
)

print(
    "  poisoned training target = benign (y=0)"
)

print(
    "\nASR definition:"
)

print(
    "  triggered attack test samples predicted benign"
)

print(
    "  -----------------------------------------------"
)

print(
    "          all triggered attack test samples"
)

print(
    "\nNo sign-flip poisoning is used in this experiment."
)

✅ Config: /content/fl_ids_repo_real/configs/backdoor/edgeiiotset_modeA.json
✅ Config: /content/fl_ids_repo_real/configs/backdoor/ton_iot_modeA.json
✅ Config: /content/fl_ids_repo_real/configs/backdoor/xiiotid_modeA.json
✅ syntax: src/models/mlp.py
✅ syntax: src/data/loaders.py
✅ syntax: src/fl/aggregators.py
✅ syntax: src/fl/client.py
✅ syntax: src/fl/utils.py
✅ syntax: src/fl/backdoor.py
✅ syntax: src/eval/thresholds.py
✅ syntax: scripts/run_backdoor_experiment.py

========== DATASET ARTIFACT CHECK ==========
edgeiiotset | encoded: True | split: True
ton_iot | encoded: True | split: True
xiiotid | encoded: True | split: True

✅ STEP 20.5 BACKDOOR IMPLEMENTATION COMPLETE

Implemented attack:
  true attack sample (y=1)
      ↓
  fixed 5-feature trigger
      ↓
  poisoned training target = benign (y=0)

ASR definition:
  triggered attack test samples predicted benign
  -----------------------------------------------
          all triggered attack test samples

No sign-flip poisoning is u

In [17]:
# ============================================================
# STEP 20.6 — FAST TARGETED-BACKDOOR SMOKE TEST
#
# Dataset: ToN-IoT
# Purpose: execution validation only
#
# NOT FOR MANUSCRIPT RESULTS
# ============================================================

from pathlib import Path
import json
import subprocess
import os
import sys

REPO = Path("/content/fl_ids_repo_real")

BASE_CONFIG = (
    REPO /
    "configs/backdoor/ton_iot_modeA.json"
)

SMOKE_CONFIG = (
    REPO /
    "configs/backdoor/ton_iot_modeA_SMOKE.json"
)

assert BASE_CONFIG.exists()

# ------------------------------------------------------------
# 1. Create temporary smoke-test config
# ------------------------------------------------------------

cfg = json.loads(
    BASE_CONFIG.read_text()
)

cfg["name"] = (
    "modeA_targeted_backdoor_SMOKE"
)

# Fast execution only
cfg["rounds"] = 1
cfg["local_epochs"] = 1

# Preserve actual security mechanisms
cfg["malicious_frac"] = 0.20
cfg["use_update_clipping"] = True
cfg["clip_C"] = 1.0
cfg["aggregator"] = "trimmean"
cfg["trim_beta"] = 0.10
cfg["dp_sigma"] = 0.0

# Keep real targeted backdoor active
cfg["backdoor"]["poison_fraction"] = 0.20
cfg["backdoor"]["trigger_dims"] = 5
cfg["backdoor"]["trigger_value"] = 3.0

SMOKE_CONFIG.write_text(
    json.dumps(
        cfg,
        indent=2
    )
)

print("✅ Smoke config:")
print(SMOKE_CONFIG)


# ------------------------------------------------------------
# 2. Ensure repository package is importable
# ------------------------------------------------------------

env = os.environ.copy()

existing_pythonpath = env.get(
    "PYTHONPATH",
    ""
)

env["PYTHONPATH"] = (
    str(REPO)
    if not existing_pythonpath
    else
    str(REPO) + ":" + existing_pythonpath
)


# ------------------------------------------------------------
# 3. Run end-to-end smoke test
# ------------------------------------------------------------

print(
    "\n============================================"
)

print(
    "RUNNING TARGETED-BACKDOOR SMOKE TEST"
)

print(
    "============================================"
)

print(
    "Dataset       : ToN-IoT"
)

print(
    "Rounds        : 1"
)

print(
    "Local epochs  : 1"
)

print(
    "Malicious frac: 0.20"
)

print(
    "Poison frac   : 0.20"
)

print(
    "Aggregator    : TrimMean"
)

print(
    "Clip C        : 1.0"
)

print(
    "\n⚠️ This is NOT a manuscript result.\n"
)


cmd = [
    sys.executable,
    str(
        REPO /
        "scripts/run_backdoor_experiment.py"
    ),
    "--config",
    str(SMOKE_CONFIG),
]


result = subprocess.run(
    cmd,
    cwd=str(REPO),
    env=env,
)


if result.returncode != 0:

    raise RuntimeError(
        "Backdoor smoke test failed. "
        "Send me the traceback."
    )


print(
    "\n============================================"
)

print(
    "✅ STEP 20.6 BACKDOOR SMOKE TEST PASSED"
)

print(
    "============================================"
)

print(
    "\nThe following mechanisms executed together:"
)

print(
    "  ✓ targeted attack→benign poisoning"
)

print(
    "  ✓ fixed trigger injection"
)

print(
    "  ✓ FedProx local training"
)

print(
    "  ✓ clipping C=1.0"
)

print(
    "  ✓ TrimMean beta=0.1"
)

print(
    "  ✓ local validation threshold calibration"
)

print(
    "  ✓ clean F1/FPR evaluation"
)

print(
    "  ✓ explicit triggered-test ASR"
)

print(
    "\nAgain: do NOT use smoke-test numbers in the paper."
)

✅ Smoke config:
/content/fl_ids_repo_real/configs/backdoor/ton_iot_modeA_SMOKE.json

RUNNING TARGETED-BACKDOOR SMOKE TEST
Dataset       : ToN-IoT
Rounds        : 1
Local epochs  : 1
Malicious frac: 0.20
Poison frac   : 0.20
Aggregator    : TrimMean
Clip C        : 1.0

⚠️ This is NOT a manuscript result.


✅ STEP 20.6 BACKDOOR SMOKE TEST PASSED

The following mechanisms executed together:
  ✓ targeted attack→benign poisoning
  ✓ fixed trigger injection
  ✓ FedProx local training
  ✓ clipping C=1.0
  ✓ TrimMean beta=0.1
  ✓ local validation threshold calibration
  ✓ clean F1/FPR evaluation
  ✓ explicit triggered-test ASR

Again: do NOT use smoke-test numbers in the paper.


In [18]:
# ============================================================
# STEP 20.7 — FINAL MODE-A TARGETED BACKDOOR RUN
#
# Runs:
#   1. Edge-IIoTset
#   2. ToN-IoT
#   3. X-IIoTID
#
# Paper experiment:
#   K              = 10
#   rounds         = 10
#   local epochs   = 2
#   FedProx mu     = 1e-3
#   malicious frac = 0.20
#   clipping C     = 1.0
#   TrimMean beta  = 0.10
#   DP sigma       = 0
#
# Saves:
#   runs/backdoor/...
#   results/security_backdoor_modeA_results.csv
#   results/security_backdoor_modeA_results.json
# ============================================================

from pathlib import Path
import subprocess
import os
import sys
import json
import pandas as pd
import time

REPO = Path("/content/fl_ids_repo_real")

CONFIG_DIR = (
    REPO /
    "configs/backdoor"
)

RESULT_DIR = (
    REPO /
    "results"
)

RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 1. Verify / enforce final experimental settings
# ============================================================

config_files = {
    "Edge-IIoTset":
        CONFIG_DIR /
        "edgeiiotset_modeA.json",

    "ToN-IoT":
        CONFIG_DIR /
        "ton_iot_modeA.json",

    "X-IIoTID":
        CONFIG_DIR /
        "xiiotid_modeA.json",
}


for dataset_name, config_path in (
    config_files.items()
):

    assert config_path.exists(), (
        f"Missing config: {config_path}"
    )

    cfg = json.loads(
        config_path.read_text()
    )

    # --------------------------------------------------------
    # Final reproducible Mode-A configuration
    # --------------------------------------------------------

    cfg["seed"] = 42

    cfg["rounds"] = 10

    cfg["local_epochs"] = 2

    cfg["fedprox_mu"] = 1e-3

    cfg["malicious_frac"] = 0.20

    cfg["use_update_clipping"] = True

    cfg["clip_C"] = 1.0

    cfg["dp_sigma"] = 0.0

    cfg["aggregator"] = "trimmean"

    cfg["trim_beta"] = 0.10

    cfg["server_gamma"] = 1.0

    # Current executable implementation
    cfg["lr"] = 3e-4

    cfg["batch_norm"] = False

    # Targeted backdoor
    cfg["backdoor"]["poison_fraction"] = 0.20

    cfg["backdoor"]["trigger_dims"] = 5

    cfg["backdoor"]["trigger_value"] = 3.0

    cfg["name"] = (
        "modeA_targeted_backdoor_FINAL"
    )

    config_path.write_text(
        json.dumps(
            cfg,
            indent=2
        )
    )

    print(
        f"✅ Final config verified: {dataset_name}"
    )


# ============================================================
# 2. Python environment
# ============================================================

env = os.environ.copy()

existing_pythonpath = env.get(
    "PYTHONPATH",
    ""
)

env["PYTHONPATH"] = (
    str(REPO)
    if not existing_pythonpath
    else
    str(REPO) + ":" + existing_pythonpath
)


RUNNER = (
    REPO /
    "scripts/run_backdoor_experiment.py"
)

assert RUNNER.exists()


# ============================================================
# 3. Helper: locate newest result produced for a dataset
# ============================================================

def newest_result_dir(
    dataset_key,
):

    root = (
        REPO /
        "runs/backdoor"
    )

    if not root.exists():
        return None

    matches = [
        p
        for p in root.iterdir()
        if (
            p.is_dir()
            and dataset_key.lower()
            in p.name.lower()
            and "final"
            in p.name.lower()
            and (
                p /
                "results.json"
            ).exists()
        )
    ]

    if not matches:
        return None

    return max(
        matches,
        key=lambda p:
            (
                p /
                "results.json"
            ).stat().st_mtime
    )


# ============================================================
# 4. Run datasets sequentially
# ============================================================

dataset_keys = {
    "Edge-IIoTset":
        "edgeiiotset",

    "ToN-IoT":
        "ton_iot",

    "X-IIoTID":
        "xiiotid",
}


all_results = []

total_start = time.time()


for run_number, (
    dataset_name,
    config_path
) in enumerate(
    config_files.items(),
    start=1,
):

    print(
        "\n"
        + "=" * 70
    )

    print(
        f"FINAL RUN {run_number}/3: {dataset_name}"
    )

    print(
        "=" * 70
    )

    print(
        "Config:",
        config_path
    )

    print(
        "Rounds       : 10"
    )

    print(
        "Local epochs : 2"
    )

    print(
        "Malicious    : 20%"
    )

    print(
        "Aggregator   : TrimMean"
    )

    print(
        "Clip C       : 1.0"
    )

    print(
        "Target       : Attack -> Benign"
    )

    print(
        "Seed         : 42"
    )

    start = time.time()


    cmd = [
        sys.executable,
        str(RUNNER),
        "--config",
        str(config_path),
    ]


    result = subprocess.run(
        cmd,
        cwd=str(REPO),
        env=env,
    )


    elapsed = (
        time.time() -
        start
    )


    if result.returncode != 0:

        raise RuntimeError(
            f"{dataset_name} final run failed."
        )


    print(
        f"\n✅ {dataset_name} completed "
        f"in {elapsed / 60:.1f} minutes."
    )


    # --------------------------------------------------------
    # Locate generated result
    # --------------------------------------------------------

    result_dir = newest_result_dir(
        dataset_keys[
            dataset_name
        ]
    )


    if result_dir is None:

        raise RuntimeError(
            f"Could not locate saved result for "
            f"{dataset_name}"
        )


    result_json = (
        result_dir /
        "results.json"
    )


    result_data = json.loads(
        result_json.read_text()
    )


    clean = result_data[
        "clean_metrics_under_attack"
    ]

    backdoor = result_data[
        "backdoor"
    ]


    row = {
        "dataset":
            dataset_name,

        "seed":
            result_data[
                "seed"
            ],

        "K":
            result_data[
                "K"
            ],

        "input_dim":
            result_data[
                "input_dim"
            ],

        "malicious_fraction":
            result_data[
                "malicious_fraction"
            ],

        "poison_fraction":
            result_data[
                "poison_fraction"
            ],

        "clean_f1_under_attack":
            clean[
                "f1"
            ],

        "clean_fpr_under_attack":
            clean[
                "fpr"
            ],

        "clean_precision_under_attack":
            clean[
                "precision"
            ],

        "clean_recall_under_attack":
            clean[
                "recall"
            ],

        "backdoor_asr":
            backdoor[
                "asr"
            ],

        "triggered_attack_samples":
            backdoor[
                "n_triggered_attack"
            ],

        "successful_backdoor_samples":
            backdoor[
                "n_predicted_benign"
            ],

        "trigger_indices":
            result_data[
                "trigger_indices"
            ],

        "trigger_value":
            result_data[
                "trigger_value"
            ],

        "runtime_minutes":
            elapsed / 60.0,

        "result_directory":
            str(
                result_dir
            ),
    }


    all_results.append(
        row
    )


    print(
        "\nRESULT SUMMARY"
    )

    print(
        f"F1  = {row['clean_f1_under_attack']:.4f}"
    )

    print(
        f"FPR = {row['clean_fpr_under_attack']:.4f}"
    )

    print(
        f"ASR = {row['backdoor_asr']:.4f}"
    )

    print(
        "Triggered attacks =",
        row[
            "triggered_attack_samples"
        ]
    )


# ============================================================
# 5. Consolidate results
# ============================================================

results_df = pd.DataFrame(
    all_results
)


CSV_OUT = (
    RESULT_DIR /
    "security_backdoor_modeA_results.csv"
)

JSON_OUT = (
    RESULT_DIR /
    "security_backdoor_modeA_results.json"
)


results_df.to_csv(
    CSV_OUT,
    index=False
)


JSON_OUT.write_text(
    json.dumps(
        all_results,
        indent=2
    )
)


# ============================================================
# 6. Cross-dataset descriptive mean
#
# This is only a descriptive mean across the three datasets.
# It is NOT a statistical replication mean across random seeds.
# ============================================================

mean_f1 = float(
    results_df[
        "clean_f1_under_attack"
    ].mean()
)

mean_fpr = float(
    results_df[
        "clean_fpr_under_attack"
    ].mean()
)

mean_asr = float(
    results_df[
        "backdoor_asr"
    ].mean()
)


total_elapsed = (
    time.time() -
    total_start
)


print(
    "\n"
    + "=" * 70
)

print(
    "FINAL MODE-A TARGETED BACKDOOR RESULTS"
)

print(
    "=" * 70
)


print(
    results_df[
        [
            "dataset",
            "clean_f1_under_attack",
            "clean_fpr_under_attack",
            "backdoor_asr",
            "triggered_attack_samples",
        ]
    ].to_string(
        index=False
    )
)


print(
    "\nDescriptive mean across datasets:"
)

print(
    f"F1  = {mean_f1:.4f}"
)

print(
    f"FPR = {mean_fpr:.4f}"
)

print(
    f"ASR = {mean_asr:.4f}"
)


print(
    "\nTotal runtime:"
)

print(
    f"{total_elapsed / 60:.1f} minutes"
)


print(
    "\nSaved:"
)

print(
    CSV_OUT
)

print(
    JSON_OUT
)


print(
    "\n============================================"
)

print(
    "✅ STEP 20.7 FINAL MODE-A RUN COMPLETE"
)

print(
    "============================================"
)

✅ Final config verified: Edge-IIoTset
✅ Final config verified: ToN-IoT
✅ Final config verified: X-IIoTID

FINAL RUN 1/3: Edge-IIoTset
Config: /content/fl_ids_repo_real/configs/backdoor/edgeiiotset_modeA.json
Rounds       : 10
Local epochs : 2
Malicious    : 20%
Aggregator   : TrimMean
Clip C       : 1.0
Target       : Attack -> Benign
Seed         : 42

✅ Edge-IIoTset completed in 13.1 minutes.

RESULT SUMMARY
F1  = 0.9441
FPR = 0.0421
ASR = 0.5273
Triggered attacks = 81853

FINAL RUN 2/3: ToN-IoT
Config: /content/fl_ids_repo_real/configs/backdoor/ton_iot_modeA.json
Rounds       : 10
Local epochs : 2
Malicious    : 20%
Aggregator   : TrimMean
Clip C       : 1.0
Target       : Attack -> Benign
Seed         : 42

✅ ToN-IoT completed in 1.6 minutes.

RESULT SUMMARY
F1  = 0.8673
FPR = 0.0901
ASR = 0.9876
Triggered attacks = 24129

FINAL RUN 3/3: X-IIoTID
Config: /content/fl_ids_repo_real/configs/backdoor/xiiotid_modeA.json
Rounds       : 10
Local epochs : 2
Malicious    : 20%
Aggregator   

**Step 20.8 — Implement Mode B secure aggregation + smoke test**

In [19]:
# ============================================================
# STEP 20.8 — MODE B SECURE-AGGREGATION IMPLEMENTATION
#
# Mode B pipeline:
#
# local targeted-backdoor training
#        ↓
# FedProx
#        ↓
# clipping C=1
#        ↓
# weighted client contribution
#        ↓
# pairwise additive masking
#        ↓
# coordinator sees ONLY masked contributions
#        ↓
# masks cancel in aggregate
#        ↓
# weighted FedAvg aggregate
#
# IMPORTANT:
#   Mode B cannot perform TrimMean because individual
#   protected updates are not visible to the coordinator.
#
# This implements a reproducible coordinator-blind
# secure-aggregation SIMULATION, not production cryptography.
# ============================================================

from pathlib import Path
import json
import sys
import subprocess
import os
import py_compile

REPO = Path("/content/fl_ids_repo_real")

assert REPO.exists()


# ============================================================
# 1. Secure aggregation module
# ============================================================

SECAGG_FILE = (
    REPO /
    "src/fl/secure_aggregation.py"
)


SECAGG_FILE.write_text(
r'''
import hashlib

import numpy as np
import torch


def _pair_seed(
    base_seed,
    round_id,
    client_i,
    client_j,
):
    """
    Deterministic pair-specific seed.

    Used only for reproducibility of the secure-aggregation
    simulation.
    """

    i = min(
        int(client_i),
        int(client_j)
    )

    j = max(
        int(client_i),
        int(client_j)
    )

    token = (
        f"{int(base_seed)}|"
        f"{int(round_id)}|"
        f"{i}|{j}"
    ).encode(
        "utf-8"
    )


    digest = hashlib.sha256(
        token
    ).digest()


    return int.from_bytes(
        digest[:8],
        byteorder="little",
        signed=False,
    ) % (
        2**63 - 1
    )


def secure_aggregate_weighted(
    deltas,
    weights,
    client_ids,
    base_seed=42,
    round_id=0,
    mask_scale=1.0,
):
    """
    Pairwise additive-mask secure-aggregation simulation.

    Each client contributes:

        weight_k * delta_k + pairwise masks

    For every pair (i,j):

        client i adds     +r_ij
        client j adds     -r_ij

    Therefore:

        sum(masked contributions)
        =
        sum(weight_k * delta_k)

    The coordinator only needs the masked contributions.

    Returns:
        weighted aggregate
        audit dictionary
    """

    if len(deltas) == 0:

        raise ValueError(
            "No client updates supplied."
        )


    if not (
        len(deltas)
        ==
        len(weights)
        ==
        len(client_ids)
    ):

        raise ValueError(
            "deltas, weights, and client_ids must "
            "have equal length."
        )


    n_clients = len(
        deltas
    )


    # --------------------------------------------------------
    # Weighted contributions
    # --------------------------------------------------------

    weighted = []

    for delta, weight in zip(
        deltas,
        weights
    ):

        weighted.append(
            delta.detach()
            .clone()
            .cpu()
            *
            float(weight)
        )


    # --------------------------------------------------------
    # Create masked client messages
    # --------------------------------------------------------

    masked = [
        x.clone()
        for x in weighted
    ]


    pair_count = 0


    for a in range(
        n_clients
    ):

        for b in range(
            a + 1,
            n_clients
        ):

            client_a = int(
                client_ids[a]
            )

            client_b = int(
                client_ids[b]
            )


            seed = _pair_seed(
                base_seed=
                    base_seed,

                round_id=
                    round_id,

                client_i=
                    client_a,

                client_j=
                    client_b,
            )


            generator = (
                torch.Generator(
                    device="cpu"
                )
                .manual_seed(
                    seed
                )
            )


            mask = torch.randn(
                weighted[a].shape,
                generator=generator,
                dtype=weighted[a].dtype,
                device="cpu",
            ) * float(
                mask_scale
            )


            # Pairwise cancellation
            masked[a] += mask
            masked[b] -= mask

            pair_count += 1


    # --------------------------------------------------------
    # Coordinator aggregation
    #
    # Only masked messages are summed here.
    # --------------------------------------------------------

    masked_sum = torch.zeros_like(
        masked[0]
    )


    for contribution in masked:

        masked_sum += contribution


    total_weight = float(
        np.sum(
            weights
        )
    )


    if total_weight <= 0:

        raise ValueError(
            "Total client weight must be positive."
        )


    aggregate = (
        masked_sum /
        total_weight
    )


    # --------------------------------------------------------
    # Audit only:
    # verify mathematically that pairwise masks cancel.
    #
    # Raw unmasked sum is NOT used to construct the returned
    # aggregate.
    # --------------------------------------------------------

    raw_sum = torch.zeros_like(
        weighted[0]
    )


    for contribution in weighted:

        raw_sum += contribution


    residual = (
        masked_sum -
        raw_sum
    )


    max_abs_residual = float(
        torch.max(
            torch.abs(
                residual
            )
        ).item()
    )


    l2_residual = float(
        torch.linalg.norm(
            residual
        ).item()
    )


    # Confirm that transmitted masked contributions differ
    # from the underlying weighted client contribution.
    masking_distances = []

    for raw, protected in zip(
        weighted,
        masked
    ):

        masking_distances.append(
            float(
                torch.linalg.norm(
                    protected -
                    raw
                ).item()
            )
        )


    audit = {
        "protocol":
            "pairwise_additive_masking_simulation",

        "n_clients":
            int(
                n_clients
            ),

        "pair_count":
            int(
                pair_count
            ),

        "total_weight":
            float(
                total_weight
            ),

        "mask_scale":
            float(
                mask_scale
            ),

        "max_abs_mask_cancellation_residual":
            max_abs_residual,

        "l2_mask_cancellation_residual":
            l2_residual,

        "min_masking_distance":
            float(
                min(
                    masking_distances
                )
            ),

        "mean_masking_distance":
            float(
                np.mean(
                    masking_distances
                )
            ),

        "coordinator_visibility":
            "masked_weighted_client_contributions_only",

        "robust_aggregation":
            False,
    }


    return (
        aggregate,
        audit
    )
'''
)


print(
    "✅ Created:"
)

print(
    SECAGG_FILE
)


# ============================================================
# 2. Patch existing backdoor runner
# ============================================================

RUNNER = (
    REPO /
    "scripts/run_backdoor_experiment.py"
)

assert RUNNER.exists()


runner_text = RUNNER.read_text()


# ------------------------------------------------------------
# Add secure-aggregation import
# ------------------------------------------------------------

secure_import = '''
from src.fl.secure_aggregation import (
    secure_aggregate_weighted,
)
'''


if (
    "secure_aggregate_weighted"
    not in runner_text
):

    marker = '''
from src.fl.aggregators import (
    agg_fedavg,
    agg_trimmed_mean,
)
'''


    assert marker in runner_text, (
        "Could not locate aggregation import block."
    )


    runner_text = runner_text.replace(
        marker,
        marker + "\n" + secure_import,
        1
    )


# ------------------------------------------------------------
# Add secure-FedAvg branch
# ------------------------------------------------------------

old_aggregation = '''
        if (
            cfg[
                "aggregator"
            ] ==
            "fedavg"
        ):

            aggregate = (
                agg_fedavg(
                    deltas,
                    weights,
                )
            )

        elif (
            cfg[
                "aggregator"
            ] ==
            "trimmean"
        ):

            aggregate = (
                agg_trimmed_mean(
                    deltas,
                    beta=
                        cfg[
                            "trim_beta"
                        ],
                )
            )

        else:

            raise ValueError(
                "aggregator must be "
                "'fedavg' or 'trimmean'"
            )
'''


new_aggregation = '''
        secagg_audit = None

        if (
            cfg[
                "aggregator"
            ] ==
            "fedavg"
        ):

            aggregate = (
                agg_fedavg(
                    deltas,
                    weights,
                )
            )

        elif (
            cfg[
                "aggregator"
            ] ==
            "trimmean"
        ):

            aggregate = (
                agg_trimmed_mean(
                    deltas,
                    beta=
                        cfg[
                            "trim_beta"
                        ],
                )
            )

        elif (
            cfg[
                "aggregator"
            ] ==
            "secure_fedavg"
        ):

            aggregate, secagg_audit = (
                secure_aggregate_weighted(
                    deltas=
                        deltas,

                    weights=
                        weights,

                    client_ids=
                        participants,

                    base_seed=
                        cfg[
                            "seed"
                        ],

                    round_id=
                        round_id,

                    mask_scale=
                        cfg.get(
                            "secure_aggregation",
                            {}
                        ).get(
                            "mask_scale",
                            1.0
                        ),
                )
            )

        else:

            raise ValueError(
                "aggregator must be "
                "'fedavg', 'trimmean', "
                "or 'secure_fedavg'"
            )
'''


if (
    "secure_fedavg"
    not in runner_text
):

    assert old_aggregation in runner_text, (
        "Could not locate aggregation block."
    )


    runner_text = runner_text.replace(
        old_aggregation,
        new_aggregation,
        1
    )


# ------------------------------------------------------------
# Add secure-aggregation audit to each round
# ------------------------------------------------------------

old_audit = '''
            "poisoned_training_samples":
                int(
                    poisoned_samples_round
                ),
        }
'''


new_audit = '''
            "poisoned_training_samples":
                int(
                    poisoned_samples_round
                ),

            "secure_aggregation":
                secagg_audit,
        }
'''


if (
    '"secure_aggregation":'
    not in runner_text
):

    assert old_audit in runner_text, (
        "Could not locate round audit block."
    )


    runner_text = runner_text.replace(
        old_audit,
        new_audit,
        1
    )


RUNNER.write_text(
    runner_text
)


print(
    "✅ Patched runner:"
)

print(
    RUNNER
)


# ============================================================
# 3. Syntax check
# ============================================================

py_compile.compile(
    str(
        SECAGG_FILE
    ),
    doraise=True
)

py_compile.compile(
    str(
        RUNNER
    ),
    doraise=True
)


print(
    "✅ Mode-B implementation syntax passed."
)


# ============================================================
# 4. Create Mode-B configs
# ============================================================

MODE_A_CONFIGS = {
    "edgeiiotset":
        REPO /
        "configs/backdoor/edgeiiotset_modeA.json",

    "ton_iot":
        REPO /
        "configs/backdoor/ton_iot_modeA.json",

    "xiiotid":
        REPO /
        "configs/backdoor/xiiotid_modeA.json",
}


MODE_B_CONFIGS = {}


for dataset, mode_a_path in (
    MODE_A_CONFIGS.items()
):

    assert mode_a_path.exists()


    cfg = json.loads(
        mode_a_path.read_text()
    )


    # --------------------------------------------------------
    # Mode B configuration
    # --------------------------------------------------------

    cfg["name"] = (
        "modeB_secureagg_targeted_backdoor_FINAL"
    )


    # Coordinator-blind secure aggregation.
    cfg["aggregator"] = (
        "secure_fedavg"
    )


    # Retain protection before masking.
    cfg["use_update_clipping"] = True

    cfg["clip_C"] = 1.0

    cfg["dp_sigma"] = 0.0


    # TrimMean is deliberately unavailable in Mode B.
    # Keep the field only for config compatibility;
    # the secure_fedavg branch does not use it.
    cfg["trim_beta"] = 0.10


    cfg[
        "secure_aggregation"
    ] = {
        "protocol":
            "pairwise_additive_masking_simulation",

        "mask_scale":
            1.0,

        "coordinator_sees":
            "masked_weighted_contributions_only",

        "supports_trimmean":
            False,
    }


    out = (
        REPO /
        "configs/backdoor" /
        f"{dataset}_modeB.json"
    )


    out.write_text(
        json.dumps(
            cfg,
            indent=2
        )
    )


    MODE_B_CONFIGS[
        dataset
    ] = out


    print(
        "✅ Mode-B config:",
        out
    )


# ============================================================
# 5. Create one-round ToN-IoT smoke config
# ============================================================

TON_MODE_B = (
    MODE_B_CONFIGS[
        "ton_iot"
    ]
)


smoke_cfg = json.loads(
    TON_MODE_B.read_text()
)


smoke_cfg["name"] = (
    "modeB_secureagg_targeted_backdoor_SMOKE"
)

smoke_cfg["rounds"] = 1

smoke_cfg["local_epochs"] = 1


SMOKE_CONFIG = (
    REPO /
    "configs/backdoor/"
    "ton_iot_modeB_SMOKE.json"
)


SMOKE_CONFIG.write_text(
    json.dumps(
        smoke_cfg,
        indent=2
    )
)


print(
    "\n✅ Smoke config:"
)

print(
    SMOKE_CONFIG
)


# ============================================================
# 6. Execute Mode-B smoke test
# ============================================================

env = os.environ.copy()


existing_pythonpath = env.get(
    "PYTHONPATH",
    ""
)


env["PYTHONPATH"] = (
    str(REPO)
    if not existing_pythonpath
    else
    str(REPO)
    + ":"
    + existing_pythonpath
)


print(
    "\n============================================"
)

print(
    "RUNNING MODE-B SECURE-AGGREGATION SMOKE TEST"
)

print(
    "============================================"
)

print(
    "Dataset       : ToN-IoT"
)

print(
    "Rounds        : 1"
)

print(
    "Local epochs  : 1"
)

print(
    "Malicious frac: 0.20"
)

print(
    "Clipping C    : 1.0"
)

print(
    "Aggregation   : secure weighted FedAvg"
)

print(
    "TrimMean      : DISABLED by design"
)

print(
    "Attack        : targeted Attack -> Benign"
)

print(
    "\n⚠️ Smoke-test values are NOT paper results.\n"
)


cmd = [
    sys.executable,

    str(
        RUNNER
    ),

    "--config",

    str(
        SMOKE_CONFIG
    ),
]


result = subprocess.run(
    cmd,
    cwd=str(REPO),
    env=env,
)


if (
    result.returncode != 0
):

    raise RuntimeError(
        "Mode-B smoke test failed. "
        "Send me the traceback."
    )


# ============================================================
# 7. Find smoke-test result and verify secure aggregation
# ============================================================

RUN_ROOT = (
    REPO /
    "runs/backdoor"
)


matches = [
    p
    for p in RUN_ROOT.iterdir()
    if (
        p.is_dir()
        and
        "ton_iot" in p.name.lower()
        and
        "modeb" in p.name.lower()
        and
        "smoke" in p.name.lower()
        and
        (
            p /
            "results.json"
        ).exists()
    )
]


assert matches, (
    "Could not locate Mode-B smoke-test result."
)


latest = max(
    matches,
    key=lambda p:
        (
            p /
            "results.json"
        ).stat().st_mtime
)


smoke_result = json.loads(
    (
        latest /
        "results.json"
    ).read_text()
)


round_audit = (
    smoke_result[
        "round_audit"
    ][0]
)


secagg = round_audit[
    "secure_aggregation"
]


assert secagg is not None


print(
    "\n========== SECURE-AGGREGATION AUDIT =========="
)

print(
    "protocol:",
    secagg[
        "protocol"
    ]
)

print(
    "clients:",
    secagg[
        "n_clients"
    ]
)

print(
    "pairwise masks:",
    secagg[
        "pair_count"
    ]
)

print(
    "coordinator visibility:",
    secagg[
        "coordinator_visibility"
    ]
)

print(
    "robust aggregation:",
    secagg[
        "robust_aggregation"
    ]
)

print(
    "max cancellation residual:",
    secagg[
        "max_abs_mask_cancellation_residual"
    ]
)

print(
    "L2 cancellation residual:",
    secagg[
        "l2_mask_cancellation_residual"
    ]
)

print(
    "minimum masking distance:",
    secagg[
        "min_masking_distance"
    ]
)


# Numerical masking cancellation tolerance.
#
# Because masks and weighted model contributions are float32,
# tiny roundoff residuals are expected.
assert (
    secagg[
        "max_abs_mask_cancellation_residual"
    ]
    < 1e-2
), (
    "Pairwise masks did not cancel within numerical tolerance."
)


assert (
    secagg[
        "min_masking_distance"
    ]
    > 0
), (
    "Masked contributions unexpectedly equal raw contributions."
)


assert (
    secagg[
        "n_clients"
    ]
    == 10
)


assert (
    secagg[
        "pair_count"
    ]
    == 45
)


print(
    "\n✅ Pairwise masks cancel in aggregate."
)

print(
    "✅ Individual transmitted contributions are masked."
)

print(
    "✅ Coordinator receives no individual unmasked update "
    "in the Mode-B aggregation path."
)


print(
    "\n============================================"
)

print(
    "✅ STEP 20.8 MODE-B IMPLEMENTATION PASSED"
)

print(
    "============================================"
)

print(
    "\nThis smoke run is NOT a manuscript result."
)

✅ Created:
/content/fl_ids_repo_real/src/fl/secure_aggregation.py
✅ Patched runner:
/content/fl_ids_repo_real/scripts/run_backdoor_experiment.py
✅ Mode-B implementation syntax passed.
✅ Mode-B config: /content/fl_ids_repo_real/configs/backdoor/edgeiiotset_modeB.json
✅ Mode-B config: /content/fl_ids_repo_real/configs/backdoor/ton_iot_modeB.json
✅ Mode-B config: /content/fl_ids_repo_real/configs/backdoor/xiiotid_modeB.json

✅ Smoke config:
/content/fl_ids_repo_real/configs/backdoor/ton_iot_modeB_SMOKE.json

RUNNING MODE-B SECURE-AGGREGATION SMOKE TEST
Dataset       : ToN-IoT
Rounds        : 1
Local epochs  : 1
Malicious frac: 0.20
Clipping C    : 1.0
Aggregation   : secure weighted FedAvg
TrimMean      : DISABLED by design
Attack        : targeted Attack -> Benign

⚠️ Smoke-test values are NOT paper results.


========== SECURE-AGGREGATION AUDIT ==========
protocol: pairwise_additive_masking_simulation
clients: 10
pairwise masks: 45
coordinator visibility: masked_weighted_client_contribu

**Step 20.9 — Final Mode-B runs + Mode-A/Mode-B comparison**

In [20]:
# ============================================================
# STEP 20.9 — FINAL MODE-B RUN + MODE-A vs MODE-B COMPARISON
#
# Runs Mode B on:
#   1. Edge-IIoTset
#   2. ToN-IoT
#   3. X-IIoTID
#
# Mode B:
#   targeted backdoor
#   FedProx mu = 1e-3
#   clipping C = 1
#   pairwise additive-mask secure aggregation
#   weighted FedAvg after mask cancellation
#   NO TrimMean (coordinator-blind)
#
# Saves:
#   results/security_backdoor_modeB_results.csv
#   results/security_backdoor_modeB_results.json
#   results/security_modeA_vs_modeB.csv
#   results/security_modeA_vs_modeB.json
# ============================================================

from pathlib import Path
import subprocess
import sys
import os
import json
import time
import pandas as pd
import numpy as np

REPO = Path(
    "/content/fl_ids_repo_real"
)

RESULT_DIR = (
    REPO /
    "results"
)

RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

RUNNER = (
    REPO /
    "scripts/run_backdoor_experiment.py"
)

assert RUNNER.exists()


# ============================================================
# 1. Final Mode-B configurations
# ============================================================

CONFIGS = {
    "Edge-IIoTset":
        REPO /
        "configs/backdoor/edgeiiotset_modeB.json",

    "ToN-IoT":
        REPO /
        "configs/backdoor/ton_iot_modeB.json",

    "X-IIoTID":
        REPO /
        "configs/backdoor/xiiotid_modeB.json",
}


dataset_keys = {
    "Edge-IIoTset":
        "edgeiiotset",

    "ToN-IoT":
        "ton_iot",

    "X-IIoTID":
        "xiiotid",
}


for dataset_name, config_path in CONFIGS.items():

    assert config_path.exists(), (
        f"Missing Mode-B config: {config_path}"
    )

    cfg = json.loads(
        config_path.read_text()
    )

    # --------------------------------------------------------
    # Explicitly enforce final experiment
    # --------------------------------------------------------

    cfg["name"] = (
        "modeB_secureagg_targeted_backdoor_FINAL"
    )

    cfg["seed"] = 42

    cfg["rounds"] = 10

    cfg["local_epochs"] = 2

    cfg["fedprox_mu"] = 1e-3

    cfg["lr"] = 3e-4

    cfg["batch_norm"] = False

    cfg["malicious_frac"] = 0.20

    cfg["use_update_clipping"] = True

    cfg["clip_C"] = 1.0

    cfg["dp_sigma"] = 0.0

    # --------------------------------------------------------
    # Mode B = secure weighted FedAvg
    # --------------------------------------------------------

    cfg["aggregator"] = (
        "secure_fedavg"
    )

    cfg["secure_aggregation"] = {
        "protocol":
            "pairwise_additive_masking_simulation",

        "mask_scale":
            1.0,

        "coordinator_sees":
            "masked_weighted_contributions_only",

        "supports_trimmean":
            False,
    }

    # --------------------------------------------------------
    # Same targeted backdoor as Mode A
    # --------------------------------------------------------

    cfg["backdoor"]["poison_fraction"] = 0.20

    cfg["backdoor"]["trigger_dims"] = 5

    cfg["backdoor"]["trigger_value"] = 3.0


    config_path.write_text(
        json.dumps(
            cfg,
            indent=2
        )
    )

    print(
        f"✅ Mode-B final config verified: {dataset_name}"
    )


# ============================================================
# 2. Runtime environment
# ============================================================

env = os.environ.copy()

existing_pythonpath = env.get(
    "PYTHONPATH",
    ""
)

env["PYTHONPATH"] = (
    str(REPO)
    if not existing_pythonpath
    else
    str(REPO) +
    ":" +
    existing_pythonpath
)


# ============================================================
# 3. Helper: locate newest Mode-B result
# ============================================================

def newest_modeB_result(
    dataset_key
):

    root = (
        REPO /
        "runs/backdoor"
    )

    if not root.exists():
        return None


    candidates = [
        p
        for p in root.iterdir()

        if (
            p.is_dir()

            and dataset_key.lower()
            in p.name.lower()

            and "modeb" in p.name.lower()

            and "final" in p.name.lower()

            and (
                p /
                "results.json"
            ).exists()
        )
    ]


    if not candidates:
        return None


    return max(
        candidates,
        key=lambda p:
            (
                p /
                "results.json"
            ).stat().st_mtime
    )


# ============================================================
# 4. Execute all three final Mode-B runs
# ============================================================

modeB_results = []

total_start = time.time()


for run_number, (
    dataset_name,
    config_path
) in enumerate(
    CONFIGS.items(),
    start=1
):

    print(
        "\n"
        + "=" * 72
    )

    print(
        f"FINAL MODE-B RUN {run_number}/3: {dataset_name}"
    )

    print(
        "=" * 72
    )

    print(
        "Rounds        : 10"
    )

    print(
        "Local epochs  : 2"
    )

    print(
        "FedProx mu    : 0.001"
    )

    print(
        "Malicious frac: 0.20"
    )

    print(
        "Poison frac   : 0.20"
    )

    print(
        "Clip C        : 1.0"
    )

    print(
        "Aggregation   : Secure weighted FedAvg"
    )

    print(
        "TrimMean      : disabled"
    )

    print(
        "Target        : Attack -> Benign"
    )

    print(
        "Seed          : 42"
    )


    start = time.time()


    cmd = [
        sys.executable,
        str(RUNNER),

        "--config",
        str(config_path),
    ]


    result = subprocess.run(
        cmd,
        cwd=str(REPO),
        env=env,
    )


    runtime_minutes = (
        time.time() -
        start
    ) / 60.0


    if result.returncode != 0:

        raise RuntimeError(
            f"Mode-B run failed for {dataset_name}."
        )


    result_dir = newest_modeB_result(
        dataset_keys[
            dataset_name
        ]
    )


    if result_dir is None:

        raise RuntimeError(
            f"Could not locate Mode-B result "
            f"for {dataset_name}."
        )


    result_json = (
        result_dir /
        "results.json"
    )


    result_data = json.loads(
        result_json.read_text()
    )


    clean = (
        result_data[
            "clean_metrics_under_attack"
        ]
    )

    attack = (
        result_data[
            "backdoor"
        ]
    )


    # --------------------------------------------------------
    # Secure-aggregation audit across ALL rounds
    # --------------------------------------------------------

    secagg_rounds = []

    for r in result_data[
        "round_audit"
    ]:

        audit = r.get(
            "secure_aggregation"
        )

        if audit is None:

            raise RuntimeError(
                f"{dataset_name}: missing secure "
                "aggregation audit."
            )

        secagg_rounds.append(
            audit
        )


    max_cancellation_residual = max(
        x[
            "max_abs_mask_cancellation_residual"
        ]
        for x in secagg_rounds
    )


    mean_cancellation_residual = float(
        np.mean(
            [
                x[
                    "max_abs_mask_cancellation_residual"
                ]
                for x in secagg_rounds
            ]
        )
    )


    min_masking_distance = min(
        x[
            "min_masking_distance"
        ]
        for x in secagg_rounds
    )


    pair_counts = set(
        x[
            "pair_count"
        ]
        for x in secagg_rounds
    )


    client_counts = set(
        x[
            "n_clients"
        ]
        for x in secagg_rounds
    )


    assert pair_counts == {45}

    assert client_counts == {10}

    assert (
        min_masking_distance >
        0
    )


    row = {
        "dataset":
            dataset_name,

        "mode":
            "Mode B",

        "seed":
            int(
                result_data[
                    "seed"
                ]
            ),

        "K":
            int(
                result_data[
                    "K"
                ]
            ),

        "input_dim":
            int(
                result_data[
                    "input_dim"
                ]
            ),

        "clean_f1_under_attack":
            float(
                clean[
                    "f1"
                ]
            ),

        "clean_fpr_under_attack":
            float(
                clean[
                    "fpr"
                ]
            ),

        "clean_precision_under_attack":
            float(
                clean[
                    "precision"
                ]
            ),

        "clean_recall_under_attack":
            float(
                clean[
                    "recall"
                ]
            ),

        "backdoor_asr":
            float(
                attack[
                    "asr"
                ]
            ),

        "triggered_attack_samples":
            int(
                attack[
                    "n_triggered_attack"
                ]
            ),

        "successful_backdoor_samples":
            int(
                attack[
                    "n_predicted_benign"
                ]
            ),

        "secure_aggregation":
            True,

        "robust_aggregation":
            False,

        "pairwise_masks_per_round":
            45,

        "max_mask_cancellation_residual":
            float(
                max_cancellation_residual
            ),

        "mean_mask_cancellation_residual":
            float(
                mean_cancellation_residual
            ),

        "min_masking_distance":
            float(
                min_masking_distance
            ),

        "runtime_minutes":
            float(
                runtime_minutes
            ),

        "result_directory":
            str(
                result_dir
            ),
    }


    modeB_results.append(
        row
    )


    print(
        f"\n✅ {dataset_name} Mode B complete."
    )

    print(
        f"Runtime = {runtime_minutes:.1f} min"
    )

    print(
        f"F1      = {row['clean_f1_under_attack']:.4f}"
    )

    print(
        f"FPR     = {row['clean_fpr_under_attack']:.4f}"
    )

    print(
        f"ASR     = {row['backdoor_asr']:.4f}"
    )

    print(
        "Max mask residual =",
        row[
            "max_mask_cancellation_residual"
        ]
    )


# ============================================================
# 5. Save Mode-B-only results
# ============================================================

modeB_df = pd.DataFrame(
    modeB_results
)


MODEB_CSV = (
    RESULT_DIR /
    "security_backdoor_modeB_results.csv"
)

MODEB_JSON = (
    RESULT_DIR /
    "security_backdoor_modeB_results.json"
)


modeB_df.to_csv(
    MODEB_CSV,
    index=False
)


MODEB_JSON.write_text(
    json.dumps(
        modeB_results,
        indent=2
    )
)


# ============================================================
# 6. Load previously executed Mode-A results
# ============================================================

MODEA_CSV = (
    RESULT_DIR /
    "security_backdoor_modeA_results.csv"
)


assert MODEA_CSV.exists(), (
    "Mode-A result file missing: "
    f"{MODEA_CSV}"
)


modeA_df = pd.read_csv(
    MODEA_CSV
)


# ============================================================
# 7. Build clean Mode-A versus Mode-B table
# ============================================================

comparison_rows = []


for dataset_name in [
    "Edge-IIoTset",
    "ToN-IoT",
    "X-IIoTID",
]:

    a = modeA_df[
        modeA_df[
            "dataset"
        ] == dataset_name
    ].iloc[0]


    b = modeB_df[
        modeB_df[
            "dataset"
        ] == dataset_name
    ].iloc[0]


    comparison_rows.append(
        {
            "dataset":
                dataset_name,

            "modeA_f1":
                float(
                    a[
                        "clean_f1_under_attack"
                    ]
                ),

            "modeA_fpr":
                float(
                    a[
                        "clean_fpr_under_attack"
                    ]
                ),

            "modeA_asr":
                float(
                    a[
                        "backdoor_asr"
                    ]
                ),

            "modeB_f1":
                float(
                    b[
                        "clean_f1_under_attack"
                    ]
                ),

            "modeB_fpr":
                float(
                    b[
                        "clean_fpr_under_attack"
                    ]
                ),

            "modeB_asr":
                float(
                    b[
                        "backdoor_asr"
                    ]
                ),

            # Positive value means Mode B has higher ASR.
            "delta_asr_B_minus_A":
                float(
                    b[
                        "backdoor_asr"
                    ]
                    -
                    a[
                        "backdoor_asr"
                    ]
                ),
        }
    )


comparison_df = pd.DataFrame(
    comparison_rows
)


COMPARE_CSV = (
    RESULT_DIR /
    "security_modeA_vs_modeB.csv"
)

COMPARE_JSON = (
    RESULT_DIR /
    "security_modeA_vs_modeB.json"
)


comparison_df.to_csv(
    COMPARE_CSV,
    index=False
)


COMPARE_JSON.write_text(
    json.dumps(
        comparison_rows,
        indent=2
    )
)


# ============================================================
# 8. Descriptive cross-dataset means
# ============================================================

mean_A = {
    "f1":
        float(
            comparison_df[
                "modeA_f1"
            ].mean()
        ),

    "fpr":
        float(
            comparison_df[
                "modeA_fpr"
            ].mean()
        ),

    "asr":
        float(
            comparison_df[
                "modeA_asr"
            ].mean()
        ),
}


mean_B = {
    "f1":
        float(
            comparison_df[
                "modeB_f1"
            ].mean()
        ),

    "fpr":
        float(
            comparison_df[
                "modeB_fpr"
            ].mean()
        ),

    "asr":
        float(
            comparison_df[
                "modeB_asr"
            ].mean()
        ),
}


# ============================================================
# 9. Final output
# ============================================================

total_runtime = (
    time.time() -
    total_start
) / 60.0


print(
    "\n"
    + "=" * 80
)

print(
    "FINAL MODE-A vs MODE-B TARGETED BACKDOOR RESULTS"
)

print(
    "=" * 80
)


print(
    comparison_df[
        [
            "dataset",
            "modeA_f1",
            "modeA_fpr",
            "modeA_asr",
            "modeB_f1",
            "modeB_fpr",
            "modeB_asr",
            "delta_asr_B_minus_A",
        ]
    ].to_string(
        index=False
    )
)


print(
    "\nMODE A descriptive mean across datasets:"
)

print(
    f"F1  = {mean_A['f1']:.4f}"
)

print(
    f"FPR = {mean_A['fpr']:.4f}"
)

print(
    f"ASR = {mean_A['asr']:.4f}"
)


print(
    "\nMODE B descriptive mean across datasets:"
)

print(
    f"F1  = {mean_B['f1']:.4f}"
)

print(
    f"FPR = {mean_B['fpr']:.4f}"
)

print(
    f"ASR = {mean_B['asr']:.4f}"
)


print(
    "\nASR difference (Mode B - Mode A):"
)

print(
    f"{mean_B['asr'] - mean_A['asr']:+.4f}"
)


print(
    "\nTotal Mode-B runtime:"
)

print(
    f"{total_runtime:.1f} minutes"
)


print(
    "\nSaved:"
)

print(
    MODEB_CSV
)

print(
    MODEB_JSON
)

print(
    COMPARE_CSV
)

print(
    COMPARE_JSON
)


print(
    "\n============================================"
)

print(
    "✅ STEP 20.9 FINAL MODE-B RUN COMPLETE"
)

print(
    "============================================"
)

✅ Mode-B final config verified: Edge-IIoTset
✅ Mode-B final config verified: ToN-IoT
✅ Mode-B final config verified: X-IIoTID

FINAL MODE-B RUN 1/3: Edge-IIoTset
Rounds        : 10
Local epochs  : 2
FedProx mu    : 0.001
Malicious frac: 0.20
Poison frac   : 0.20
Clip C        : 1.0
Aggregation   : Secure weighted FedAvg
TrimMean      : disabled
Target        : Attack -> Benign
Seed          : 42

✅ Edge-IIoTset Mode B complete.
Runtime = 13.0 min
F1      = 0.5526
FPR     = 0.0000
ASR     = 0.9728
Max mask residual = 0.0234375

FINAL MODE-B RUN 2/3: ToN-IoT
Rounds        : 10
Local epochs  : 2
FedProx mu    : 0.001
Malicious frac: 0.20
Poison frac   : 0.20
Clip C        : 1.0
Aggregation   : Secure weighted FedAvg
TrimMean      : disabled
Target        : Attack -> Benign
Seed          : 42

✅ ToN-IoT Mode B complete.
Runtime = 1.6 min
F1      = 0.9221
FPR     = 0.4675
ASR     = 0.9815
Max mask residual = 0.000732421875

FINAL MODE-B RUN 3/3: X-IIoTID
Rounds        : 10
Local epochs  : 2

**Step 20.10 — Clean FedProx vs Proposed + real Wilcoxon test**

In [21]:
# ============================================================
# STEP 20.10 — CLEAN FEDPROX vs PROPOSED + WILCOXON
#
# Reviewer requirement:
#   actual paired Wilcoxon signed-rank test on K=10 site-level
#   F1 values.
#
# Clean comparison:
#
# FedProx-MLP:
#   FedProx local optimization
#   weighted FedAvg
#   no clipping
#   no TrimMean
#   no attack
#
# Proposed:
#   FedProx local optimization
#   clipping C=1
#   TrimMean beta=0.1
#   no attack
#
# Saves:
#   results/per_site_f1_pairs.csv
#   results/wilcoxon_results.csv
#   results/wilcoxon_results.json
# ============================================================

from pathlib import Path
import json
import subprocess
import os
import sys
import time
import numpy as np
import pandas as pd

from scipy.stats import wilcoxon


REPO = Path(
    "/content/fl_ids_repo_real"
)

RESULT_DIR = (
    REPO /
    "results"
)

CONFIG_DIR = (
    REPO /
    "configs/wilcoxon"
)

CONFIG_DIR.mkdir(
    parents=True,
    exist_ok=True
)

RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

RUNNER = (
    REPO /
    "scripts/run_backdoor_experiment.py"
)

assert RUNNER.exists()


# ============================================================
# 1. Dataset definitions
# ============================================================

DATASETS = {
    "Edge-IIoTset": {
        "key":
            "edgeiiotset",

        "encoded_npz":
            "data/processed/edgeiiotset_encoded.npz",

        "splits_npz":
            "data/processed/splits_edgeiiotset_dirichlet_alpha_0.3.npz",
    },

    "ToN-IoT": {
        "key":
            "ton_iot",

        "encoded_npz":
            "data/processed/ton_iot_encoded.npz",

        "splits_npz":
            "data/processed/splits_ton_iot_dirichlet_alpha_0.3.npz",
    },

    "X-IIoTID": {
        "key":
            "xiiotid",

        "encoded_npz":
            "data/processed/xiiotid_encoded.npz",

        "splits_npz":
            "data/processed/splits_xiiotid_dirichlet_alpha_0.3.npz",
    },
}


# ============================================================
# 2. Common experimental configuration
# ============================================================

COMMON = {
    "seed": 42,

    "batch_size": 256,

    "hidden": [
        256,
        128,
        64
    ],

    "dropout": 0.30,

    # Match executable implementation.
    "batch_norm": False,

    "pos_weight": 2.0,

    "rounds": 10,

    "local_epochs": 2,

    "fedprox_mu": 1e-3,

    "lr": 3e-4,

    "weight_decay": 1e-4,

    "grad_clip": 1.0,

    "server_gamma": 1.0,

    # CLEAN EXPERIMENT
    "malicious_frac": 0.0,

    "dp_sigma": 0.0,

    "threshold": 0.50,

    "eval_batch_size": 4096,

    # Backdoor code remains available in the runner,
    # but is completely disabled here.
    "backdoor": {
        "poison_fraction": 0.0,
        "trigger_dims": 5,
        "trigger_value": 3.0,
    },

    "calibration": {
        "target_fpr": 0.05,
        "theta_min": 0.05,
        "theta_max": 0.95,
        "min_benign_val": 200,
        "max_pred_pos": 0.90,
        "min_pred_pos": 0.01,
    },
}


# ============================================================
# 3. Build FedProx and Proposed configs
# ============================================================

CONFIGS = {}


for dataset_name, info in DATASETS.items():

    # --------------------------------------------------------
    # FedProx-MLP baseline
    # --------------------------------------------------------

    baseline = json.loads(
        json.dumps(
            COMMON
        )
    )

    baseline.update(
        {
            "dataset":
                info["key"],

            "encoded_npz":
                info["encoded_npz"],

            "splits_npz":
                info["splits_npz"],

            "name":
                "wilcoxon_clean_fedprox",

            "aggregator":
                "fedavg",

            "use_update_clipping":
                False,

            # retained only because runner expects field
            "clip_C":
                1.0,

            "trim_beta":
                0.10,
        }
    )


    baseline_path = (
        CONFIG_DIR /
        f"{info['key']}_fedprox.json"
    )


    baseline_path.write_text(
        json.dumps(
            baseline,
            indent=2
        )
    )


    # --------------------------------------------------------
    # Proposed clean method
    # --------------------------------------------------------

    proposed = json.loads(
        json.dumps(
            COMMON
        )
    )

    proposed.update(
        {
            "dataset":
                info["key"],

            "encoded_npz":
                info["encoded_npz"],

            "splits_npz":
                info["splits_npz"],

            "name":
                "wilcoxon_clean_proposed",

            "aggregator":
                "trimmean",

            "use_update_clipping":
                True,

            "clip_C":
                1.0,

            "trim_beta":
                0.10,
        }
    )


    proposed_path = (
        CONFIG_DIR /
        f"{info['key']}_proposed.json"
    )


    proposed_path.write_text(
        json.dumps(
            proposed,
            indent=2
        )
    )


    CONFIGS[
        dataset_name
    ] = {
        "key":
            info["key"],

        "fedprox":
            baseline_path,

        "proposed":
            proposed_path,
    }


print(
    "✅ Clean Wilcoxon configurations created."
)


# ============================================================
# 4. Runtime environment
# ============================================================

env = os.environ.copy()

existing_pythonpath = env.get(
    "PYTHONPATH",
    ""
)

env["PYTHONPATH"] = (
    str(REPO)
    if not existing_pythonpath
    else
    str(REPO)
    + ":"
    + existing_pythonpath
)


# ============================================================
# 5. Helper to locate exact result
# ============================================================

RUN_ROOT = (
    REPO /
    "runs/backdoor"
)


def find_newest_result(
    dataset_key,
    experiment_fragment,
):

    if not RUN_ROOT.exists():
        return None


    candidates = [
        p
        for p in RUN_ROOT.iterdir()

        if (
            p.is_dir()

            and dataset_key.lower()
            in p.name.lower()

            and experiment_fragment.lower()
            in p.name.lower()

            and (
                p /
                "results.json"
            ).exists()
        )
    ]


    if not candidates:
        return None


    return max(
        candidates,
        key=lambda p:
            (
                p /
                "results.json"
            ).stat().st_mtime
    )


# ============================================================
# 6. Run clean baseline + proposed for each dataset
# ============================================================

stored_results = {}

start_all = time.time()


for dataset_name, info in CONFIGS.items():

    stored_results[
        dataset_name
    ] = {}


    for method_name in [
        "fedprox",
        "proposed",
    ]:

        config_path = info[
            method_name
        ]


        print(
            "\n"
            + "=" * 72
        )

        print(
            f"{dataset_name} — {method_name.upper()}"
        )

        print(
            "=" * 72
        )


        print(
            "Clean training: malicious_frac = 0"
        )

        print(
            "Rounds        : 10"
        )

        print(
            "Local epochs  : 2"
        )

        print(
            "FedProx mu    : 0.001"
        )

        print(
            "Seed          : 42"
        )


        if method_name == "fedprox":

            print(
                "Aggregation   : weighted FedAvg"
            )

            print(
                "Clipping      : OFF"
            )

        else:

            print(
                "Aggregation   : TrimMean beta=0.1"
            )

            print(
                "Clipping      : C=1.0"
            )


        start = time.time()


        cmd = [
            sys.executable,
            str(RUNNER),

            "--config",
            str(config_path),
        ]


        result = subprocess.run(
            cmd,
            cwd=str(REPO),
            env=env,
        )


        elapsed = (
            time.time() -
            start
        ) / 60.0


        if result.returncode != 0:

            raise RuntimeError(
                f"{dataset_name} / {method_name} failed."
            )


        fragment = (
            "wilcoxon_clean_fedprox"
            if method_name == "fedprox"
            else
            "wilcoxon_clean_proposed"
        )


        result_dir = find_newest_result(
            info[
                "key"
            ],
            fragment,
        )


        if result_dir is None:

            raise RuntimeError(
                "Could not locate result directory for "
                f"{dataset_name}/{method_name}"
            )


        result_data = json.loads(
            (
                result_dir /
                "results.json"
            ).read_text()
        )


        stored_results[
            dataset_name
        ][
            method_name
        ] = result_data


        global_metrics = (
            result_data[
                "clean_metrics_under_attack"
            ]
        )


        print(
            f"\n✅ runtime = {elapsed:.1f} min"
        )

        print(
            f"global F1  = "
            f"{global_metrics['f1']:.4f}"
        )

        print(
            f"global FPR = "
            f"{global_metrics['fpr']:.4f}"
        )


# ============================================================
# 7. Extract the paired K=10 site F1 values
# ============================================================

pair_rows = []

wilcoxon_rows = []


for dataset_name in DATASETS:

    baseline_result = (
        stored_results[
            dataset_name
        ][
            "fedprox"
        ]
    )

    proposed_result = (
        stored_results[
            dataset_name
        ][
            "proposed"
        ]
    )


    baseline_sites = sorted(
        baseline_result[
            "clean_per_silo"
        ],
        key=lambda x:
            int(
                x["silo"]
            )
    )


    proposed_sites = sorted(
        proposed_result[
            "clean_per_silo"
        ],
        key=lambda x:
            int(
                x["silo"]
            )
    )


    assert len(
        baseline_sites
    ) == 10

    assert len(
        proposed_sites
    ) == 10


    baseline_f1 = np.asarray(
        [
            x[
                "f1"
            ]
            for x
            in baseline_sites
        ],
        dtype=float
    )


    proposed_f1 = np.asarray(
        [
            x[
                "f1"
            ]
            for x
            in proposed_sites
        ],
        dtype=float
    )


    # --------------------------------------------------------
    # Store exact paired observations
    # --------------------------------------------------------

    for site in range(
        10
    ):

        pair_rows.append(
            {
                "dataset":
                    dataset_name,

                "site":
                    site,

                "fedprox_f1":
                    float(
                        baseline_f1[
                            site
                        ]
                    ),

                "proposed_f1":
                    float(
                        proposed_f1[
                            site
                        ]
                    ),

                "difference":
                    float(
                        proposed_f1[
                            site
                        ]
                        -
                        baseline_f1[
                            site
                        ]
                    ),
            }
        )


    # --------------------------------------------------------
    # Paired Wilcoxon signed-rank test
    # --------------------------------------------------------

    differences = (
        proposed_f1 -
        baseline_f1
    )


    nonzero_count = int(
        np.sum(
            differences != 0
        )
    )


    if nonzero_count == 0:

        statistic = 0.0
        p_value = 1.0
        method_note = (
            "all paired differences equal zero"
        )

    else:

        test = wilcoxon(
            proposed_f1,
            baseline_f1,

            alternative=
                "two-sided",

            zero_method=
                "wilcox",

            method=
                "auto",
        )


        statistic = float(
            test.statistic
        )

        p_value = float(
            test.pvalue
        )

        method_note = (
            "paired two-sided Wilcoxon signed-rank"
        )


    wilcoxon_rows.append(
        {
            "dataset":
                dataset_name,

            "n_sites":
                10,

            "fedprox_mean_f1":
                float(
                    baseline_f1.mean()
                ),

            "fedprox_std_f1":
                float(
                    baseline_f1.std(
                        ddof=1
                    )
                ),

            "proposed_mean_f1":
                float(
                    proposed_f1.mean()
                ),

            "proposed_std_f1":
                float(
                    proposed_f1.std(
                        ddof=1
                    )
                ),

            "mean_paired_difference":
                float(
                    differences.mean()
                ),

            "median_paired_difference":
                float(
                    np.median(
                        differences
                    )
                ),

            "positive_differences":
                int(
                    np.sum(
                        differences > 0
                    )
                ),

            "negative_differences":
                int(
                    np.sum(
                        differences < 0
                    )
                ),

            "zero_differences":
                int(
                    np.sum(
                        differences == 0
                    )
                ),

            "wilcoxon_statistic":
                statistic,

            "p_value":
                p_value,

            "test":
                method_note,
        }
    )


# ============================================================
# 8. Save reproducibility artifacts
# ============================================================

pairs_df = pd.DataFrame(
    pair_rows
)

wilcoxon_df = pd.DataFrame(
    wilcoxon_rows
)


PAIRS_CSV = (
    RESULT_DIR /
    "per_site_f1_pairs.csv"
)

WILCOXON_CSV = (
    RESULT_DIR /
    "wilcoxon_results.csv"
)

WILCOXON_JSON = (
    RESULT_DIR /
    "wilcoxon_results.json"
)


pairs_df.to_csv(
    PAIRS_CSV,
    index=False
)


wilcoxon_df.to_csv(
    WILCOXON_CSV,
    index=False
)


WILCOXON_JSON.write_text(
    json.dumps(
        wilcoxon_rows,
        indent=2
    )
)


# ============================================================
# 9. Final output
# ============================================================

print(
    "\n"
    + "=" * 92
)

print(
    "FINAL PAIRED WILCOXON RESULTS — CLEAN K=10 SITE F1"
)

print(
    "=" * 92
)


display_df = wilcoxon_df[
    [
        "dataset",
        "fedprox_mean_f1",
        "fedprox_std_f1",
        "proposed_mean_f1",
        "proposed_std_f1",
        "positive_differences",
        "negative_differences",
        "wilcoxon_statistic",
        "p_value",
    ]
]


print(
    display_df.to_string(
        index=False
    )
)


print(
    "\nExact paired site-level values:"
)


for dataset_name in DATASETS:

    subset = pairs_df[
        pairs_df[
            "dataset"
        ] == dataset_name
    ]


    print(
        f"\n{dataset_name}"
    )


    print(
        subset[
            [
                "site",
                "fedprox_f1",
                "proposed_f1",
                "difference",
            ]
        ].to_string(
            index=False
        )
    )


total_minutes = (
    time.time() -
    start_all
) / 60.0


print(
    "\nSaved:"
)

print(
    PAIRS_CSV
)

print(
    WILCOXON_CSV
)

print(
    WILCOXON_JSON
)


print(
    f"\nTotal runtime: "
    f"{total_minutes:.1f} minutes"
)


print(
    "\n============================================"
)

print(
    "✅ STEP 20.10 WILCOXON IMPLEMENTATION COMPLETE"
)

print(
    "============================================"
)

✅ Clean Wilcoxon configurations created.

Edge-IIoTset — FEDPROX
Clean training: malicious_frac = 0
Rounds        : 10
Local epochs  : 2
FedProx mu    : 0.001
Seed          : 42
Aggregation   : weighted FedAvg
Clipping      : OFF

✅ runtime = 12.6 min
global F1  = 0.7089
global FPR = 0.0000

Edge-IIoTset — PROPOSED
Clean training: malicious_frac = 0
Rounds        : 10
Local epochs  : 2
FedProx mu    : 0.001
Seed          : 42
Aggregation   : TrimMean beta=0.1
Clipping      : C=1.0

✅ runtime = 11.8 min
global F1  = 0.9439
global FPR = 0.0423

ToN-IoT — FEDPROX
Clean training: malicious_frac = 0
Rounds        : 10
Local epochs  : 2
FedProx mu    : 0.001
Seed          : 42
Aggregation   : weighted FedAvg
Clipping      : OFF

✅ runtime = 1.6 min
global F1  = 0.9284
global FPR = 0.3936

ToN-IoT — PROPOSED
Clean training: malicious_frac = 0
Rounds        : 10
Local epochs  : 2
FedProx mu    : 0.001
Seed          : 42
Aggregation   : TrimMean beta=0.1
Clipping      : C=1.0

✅ runtime = 1.5 m

**Step 20.11 — Build the reproducibility package**

In [22]:
# ============================================================
# STEP 20.11 — REPRODUCIBILITY PACKAGE FOR REVIEWER 2 COMMENT 3
#
# Creates:
#   reproducibility/
#       environment.json
#       artifact_manifest.json
#       README_REPRODUCIBILITY.md
#
#   results/
#       reviewer2_comment3_evidence.json
#
# Also attempts to copy the executed notebook into:
#   notebooks/FL_and_IIoT_updated.ipynb
#
# NO TRAINING.
# ============================================================

from pathlib import Path
import json
import shutil
import subprocess
import sys
import platform
import datetime
import os

import numpy as np
import pandas as pd
import torch
import sklearn
import scipy


REPO = Path(
    "/content/fl_ids_repo_real"
)

assert REPO.exists()


REPRO_DIR = (
    REPO /
    "reproducibility"
)

RESULT_DIR = (
    REPO /
    "results"
)

NOTEBOOK_DIR = (
    REPO /
    "notebooks"
)


for p in [
    REPRO_DIR,
    RESULT_DIR,
    NOTEBOOK_DIR,
]:

    p.mkdir(
        parents=True,
        exist_ok=True
    )


# ============================================================
# 1. Verify the required reviewer artifacts
# ============================================================

required_artifacts = {

    # --------------------------------------------------------
    # Three datasets
    # --------------------------------------------------------

    "Edge encoded dataset":
        REPO /
        "data/processed/edgeiiotset_encoded.npz",

    "Edge split":
        REPO /
        "data/processed/splits_edgeiiotset_dirichlet_alpha_0.3.npz",

    "ToN encoded dataset":
        REPO /
        "data/processed/ton_iot_encoded.npz",

    "ToN split":
        REPO /
        "data/processed/splits_ton_iot_dirichlet_alpha_0.3.npz",

    "X-IIoTID encoded dataset":
        REPO /
        "data/processed/xiiotid_encoded.npz",

    "X-IIoTID split":
        REPO /
        "data/processed/splits_xiiotid_dirichlet_alpha_0.3.npz",

    # --------------------------------------------------------
    # Implementations
    # --------------------------------------------------------

    "Targeted backdoor implementation":
        REPO /
        "src/fl/backdoor.py",

    "Secure aggregation implementation":
        REPO /
        "src/fl/secure_aggregation.py",

    "FedProx client training":
        REPO /
        "src/fl/client.py",

    "Aggregation implementation":
        REPO /
        "src/fl/aggregators.py",

    "Experiment runner":
        REPO /
        "scripts/run_backdoor_experiment.py",

    # --------------------------------------------------------
    # Executed results
    # --------------------------------------------------------

    "Mode A targeted-backdoor results":
        RESULT_DIR /
        "security_backdoor_modeA_results.csv",

    "Mode B targeted-backdoor results":
        RESULT_DIR /
        "security_backdoor_modeB_results.csv",

    "Mode A vs B comparison":
        RESULT_DIR /
        "security_modeA_vs_modeB.csv",

    "Paired site F1 observations":
        RESULT_DIR /
        "per_site_f1_pairs.csv",

    "Wilcoxon results":
        RESULT_DIR /
        "wilcoxon_results.csv",
}


print(
    "========== REQUIRED ARTIFACT CHECK =========="
)


missing = []


for name, path in required_artifacts.items():

    exists = path.exists()

    print(
        f"{'✅' if exists else '❌'} "
        f"{name}: "
        f"{path.relative_to(REPO)}"
    )

    if not exists:
        missing.append(
            str(path)
        )


if missing:

    raise FileNotFoundError(
        "Missing required artifacts:\n"
        +
        "\n".join(
            missing
        )
    )


print(
    "\n✅ All required implementation/result artifacts exist."
)


# ============================================================
# 2. Environment record
# ============================================================

environment = {

    "created_utc":
        datetime.datetime.now(
            datetime.timezone.utc
        ).isoformat(),

    "python":
        sys.version,

    "platform":
        platform.platform(),

    "numpy":
        np.__version__,

    "pandas":
        pd.__version__,

    "torch":
        torch.__version__,

    "scikit_learn":
        sklearn.__version__,

    "scipy":
        scipy.__version__,

    "cuda_available":
        bool(
            torch.cuda.is_available()
        ),

    "cuda_version":
        torch.version.cuda,

    "device":
        (
            torch.cuda.get_device_name(0)
            if torch.cuda.is_available()
            else "CPU"
        ),

    "seed":
        42,

    "K":
        10,

    "dirichlet_alpha":
        0.3,

    "train_validation_test":
        [
            0.70,
            0.15,
            0.15,
        ],

    "local_epochs":
        2,

    "communication_rounds":
        10,

    "fedprox_mu":
        1e-3,

    "learning_rate":
        3e-4,

    "batch_norm":
        False,

    "dropout":
        0.30,

    "clip_C":
        1.0,

    "trim_beta":
        0.10,

    "malicious_fraction":
        0.20,

    "backdoor_poison_fraction":
        0.20,

    "trigger_dimensions":
        5,

    "trigger_value_normalized":
        3.0,
}


ENV_FILE = (
    REPRO_DIR /
    "environment.json"
)


ENV_FILE.write_text(
    json.dumps(
        environment,
        indent=2
    )
)


print(
    "\n✅ Environment record:"
)

print(
    ENV_FILE
)


# ============================================================
# 3. Compact artifact manifest
#
# Avoid expensive hashing of multi-GB dataset files.
# Record exact path and size instead.
# ============================================================

manifest = {}


for name, path in required_artifacts.items():

    manifest[
        name
    ] = {

        "relative_path":
            str(
                path.relative_to(
                    REPO
                )
            ),

        "size_bytes":
            int(
                path.stat().st_size
            ),

        "exists":
            True,
    }


MANIFEST_FILE = (
    REPRO_DIR /
    "artifact_manifest.json"
)


MANIFEST_FILE.write_text(
    json.dumps(
        manifest,
        indent=2
    )
)


print(
    "✅ Artifact manifest:"
)

print(
    MANIFEST_FILE
)


# ============================================================
# 4. Load final executed security results
# ============================================================

modeA = pd.read_csv(
    RESULT_DIR /
    "security_backdoor_modeA_results.csv"
)

modeB = pd.read_csv(
    RESULT_DIR /
    "security_backdoor_modeB_results.csv"
)

comparison = pd.read_csv(
    RESULT_DIR /
    "security_modeA_vs_modeB.csv"
)

wilcoxon_df = pd.read_csv(
    RESULT_DIR /
    "wilcoxon_results.csv"
)


# ============================================================
# 5. Reviewer-specific evidence summary
# ============================================================

reviewer_evidence = {

    "reviewer_comment":
        "Reviewer 2 Comment 3",

    "status":
        "implementation artifacts added and executed",

    "three_dataset_support": {

        "Edge-IIoTset": {
            "records":
                1909671,

            "input_dim":
                96,
        },

        "ToN-IoT": {
            "records":
                211043,

            "input_dim":
                115,
        },

        "X-IIoTID": {
            "records":
                820834,

            "input_dim":
                82,
        },
    },

    "targeted_backdoor": {

        "attack":
            "attack-class sample with fixed trigger relabeled to benign during malicious-client local training",

        "asr_definition":
            "fraction of triggered attack-class test samples predicted as benign",

        "modeA_results":
            modeA[
                [
                    "dataset",
                    "clean_f1_under_attack",
                    "clean_fpr_under_attack",
                    "backdoor_asr",
                ]
            ].to_dict(
                orient="records"
            ),
    },

    "secure_aggregation": {

        "implementation":
            "pairwise additive masking simulation",

        "coordinator_visibility":
            "masked weighted client contributions only",

        "trimmean_available":
            False,

        "modeB_results":
            modeB[
                [
                    "dataset",
                    "clean_f1_under_attack",
                    "clean_fpr_under_attack",
                    "backdoor_asr",
                ]
            ].to_dict(
                orient="records"
            ),
    },

    "wilcoxon": {

        "test":
            "paired two-sided Wilcoxon signed-rank test on K=10 site-level F1 values",

        "results":
            wilcoxon_df[
                [
                    "dataset",
                    "fedprox_mean_f1",
                    "fedprox_std_f1",
                    "proposed_mean_f1",
                    "proposed_std_f1",
                    "wilcoxon_statistic",
                    "p_value",
                ]
            ].to_dict(
                orient="records"
            ),
    },

    "important_interpretation":
        (
            "The executed Wilcoxon tests do not establish "
            "statistical significance at alpha=0.05. "
            "The manuscript must not retain the previous "
            "claim of p<0.01 across all datasets."
        ),
}


EVIDENCE_FILE = (
    RESULT_DIR /
    "reviewer2_comment3_evidence.json"
)


EVIDENCE_FILE.write_text(
    json.dumps(
        reviewer_evidence,
        indent=2
    )
)


print(
    "✅ Reviewer evidence summary:"
)

print(
    EVIDENCE_FILE
)


# ============================================================
# 6. Try to locate the executed notebook
# ============================================================

notebook_candidates = []


search_roots = [
    Path("/content"),
    Path("/content/drive/MyDrive"),
]


for root in search_roots:

    if not root.exists():
        continue

    # Limit to likely notebook names rather than scanning
    # the entire Drive unnecessarily.
    patterns = [
        "*FL*IIoT*.ipynb",
        "*FL*IIOT*.ipynb",
        "*fl*iiot*.ipynb",
        "*updated*.ipynb",
    ]


    for pattern in patterns:

        try:

            notebook_candidates.extend(
                root.rglob(
                    pattern
                )
            )

        except Exception:
            pass


# Deduplicate
notebook_candidates = sorted(
    set(
        p.resolve()
        for p in notebook_candidates
        if p.is_file()
    ),
    key=lambda p:
        p.stat().st_mtime,
    reverse=True,
)


EXECUTED_NOTEBOOK = None


# ============================================================
# 7. Prefer notebook containing stored outputs
# ============================================================

for candidate in notebook_candidates:

    try:

        notebook_json = json.loads(
            candidate.read_text(
                encoding="utf-8"
            )
        )


        code_cells = [
            cell
            for cell
            in notebook_json.get(
                "cells",
                []
            )
            if cell.get(
                "cell_type"
            ) == "code"
        ]


        cells_with_outputs = sum(
            1
            for cell
            in code_cells
            if len(
                cell.get(
                    "outputs",
                    []
                )
            ) > 0
        )


        if (
            len(code_cells) > 0
            and cells_with_outputs > 0
        ):

            EXECUTED_NOTEBOOK = candidate

            print(
                "\nCandidate executed notebook:"
            )

            print(
                candidate
            )

            print(
                "Code cells:",
                len(
                    code_cells
                )
            )

            print(
                "Cells with stored outputs:",
                cells_with_outputs
            )

            break

    except Exception:
        continue


# ============================================================
# 8. Copy executed notebook into repository
# ============================================================

NOTEBOOK_STATUS = {
    "found":
        False
}


if EXECUTED_NOTEBOOK is not None:

    notebook_target = (
        NOTEBOOK_DIR /
        "FL_and_IIoT_updated.ipynb"
    )


    if (
        EXECUTED_NOTEBOOK.resolve()
        != notebook_target.resolve()
    ):

        shutil.copy2(
            EXECUTED_NOTEBOOK,
            notebook_target
        )


    notebook_json = json.loads(
        notebook_target.read_text(
            encoding="utf-8"
        )
    )


    code_cells = [
        cell
        for cell
        in notebook_json[
            "cells"
        ]
        if cell.get(
            "cell_type"
        ) == "code"
    ]


    output_cells = [
        cell
        for cell
        in code_cells
        if len(
            cell.get(
                "outputs",
                []
            )
        ) > 0
    ]


    NOTEBOOK_STATUS = {

        "found":
            True,

        "repository_path":
            str(
                notebook_target.relative_to(
                    REPO
                )
            ),

        "code_cells":
            len(
                code_cells
            ),

        "code_cells_with_outputs":
            len(
                output_cells
            ),
    }


    print(
        "\n✅ Executed notebook copied into repository:"
    )

    print(
        notebook_target
    )

    print(
        f"Stored outputs: "
        f"{len(output_cells)}/{len(code_cells)} "
        f"code cells"
    )


else:

    print(
        "\n⚠️ No executed notebook file was automatically found."
    )

    print(
        "The implementation/results are safe in Drive, "
        "but the current Colab notebook should also be saved "
        "into the repository before GitHub publication."
    )


# ============================================================
# 9. Reproducibility README
# ============================================================

README = f"""# Reproducibility

This repository contains the implementation and executed result
artifacts used for the revised federated IIoT intrusion-detection
experiments.

## Datasets

The experiments use:

- Edge-IIoTset
- ToN-IoT Network
- X-IIoTID

Each dataset is represented by an encoded NPZ artifact and a
K=10 Dirichlet non-IID split with alpha=0.3 and a 70/15/15
within-silo train/validation/test division.

## Executed configuration

- K: 10 silos
- Dirichlet alpha: 0.3
- Seed: 42
- FL rounds: 10
- Local epochs: 2
- FedProx mu: 1e-3
- Learning rate: 3e-4
- MLP: [256, 128, 64]
- Dropout: 0.30
- Batch normalization: disabled in the executable implementation
- Update clipping: C=1.0
- TrimMean beta: 0.10

## Targeted backdoor

Malicious clients inject a fixed five-feature trigger into a subset
of local attack-class samples and relabel those samples to the benign
target class.

ASR is computed as:

triggered attack test samples classified as benign /
all triggered attack test samples.

The targeted backdoor experiment is distinct from the sign-flip
poisoning sensitivity experiment.

## Mode A

Mode A exposes protected individual client updates to the coordinator.

Pipeline:

FedProx -> clipping -> TrimMean.

## Mode B

Mode B uses a pairwise additive-masking secure-aggregation simulation.

Pipeline:

FedProx -> clipping -> weighted contribution -> pairwise masking
-> aggregate-only weighted FedAvg.

The coordinator-side aggregation path operates on masked client
contributions. TrimMean is unavailable because individual updates
are hidden.

This implementation is a simulation-level secure-aggregation model
for studying coordinator visibility. It is not presented as a
production cryptographic deployment.

## Wilcoxon analysis

The repository stores the ten paired site-level F1 observations for
FedProx-MLP and the proposed visible-update configuration.

The test is a paired, two-sided Wilcoxon signed-rank test.

The executed p-values are:

- Edge-IIoTset: 0.195312
- ToN-IoT: 0.193359
- X-IIoTID: 0.083984

These results do not establish statistical significance at the
0.05 level.

## Result artifacts

- results/security_backdoor_modeA_results.csv
- results/security_backdoor_modeB_results.csv
- results/security_modeA_vs_modeB.csv
- results/per_site_f1_pairs.csv
- results/wilcoxon_results.csv
- results/reviewer2_comment3_evidence.json

## Environment

See:

reproducibility/environment.json

## Notebook

Executed-notebook discovery status:

{json.dumps(NOTEBOOK_STATUS, indent=2)}
"""


README_FILE = (
    REPRO_DIR /
    "README_REPRODUCIBILITY.md"
)


README_FILE.write_text(
    README
)


print(
    "\n✅ Reproducibility README:"
)

print(
    README_FILE
)


# ============================================================
# 10. Git status — confirms what must be committed
# ============================================================

print(
    "\n========== GIT STATUS =========="
)


git_result = subprocess.run(
    [
        "git",
        "status",
        "--short",
    ],
    cwd=str(REPO),
    capture_output=True,
    text=True,
)


print(
    git_result.stdout
    if git_result.stdout.strip()
    else "Working tree clean."
)


print(
    "================================"
)


# ============================================================
# Final
# ============================================================

print(
    "\n============================================"
)

print(
    "✅ STEP 20.11 REPRODUCIBILITY PACKAGE COMPLETE"
)

print(
    "============================================"
)


print(
    "\nEvidence:"
)

print(
    EVIDENCE_FILE
)

print(
    "\nEnvironment:"
)

print(
    ENV_FILE
)

print(
    "\nManifest:"
)

print(
    MANIFEST_FILE
)

print(
    "\nREADME:"
)

print(
    README_FILE
)


if NOTEBOOK_STATUS["found"]:

    print(
        "\n✅ Executed notebook with stored outputs is "
        "present in the repository."
    )

else:

    print(
        "\n⚠️ Save/copy the current executed Colab notebook "
        "into the repository before publishing GitHub."
    )

========== REQUIRED ARTIFACT CHECK ==========
✅ Edge encoded dataset: data/processed/edgeiiotset_encoded.npz
✅ Edge split: data/processed/splits_edgeiiotset_dirichlet_alpha_0.3.npz
✅ ToN encoded dataset: data/processed/ton_iot_encoded.npz
✅ ToN split: data/processed/splits_ton_iot_dirichlet_alpha_0.3.npz
✅ X-IIoTID encoded dataset: data/processed/xiiotid_encoded.npz
✅ X-IIoTID split: data/processed/splits_xiiotid_dirichlet_alpha_0.3.npz
✅ Targeted backdoor implementation: src/fl/backdoor.py
✅ Secure aggregation implementation: src/fl/secure_aggregation.py
✅ FedProx client training: src/fl/client.py
✅ Aggregation implementation: src/fl/aggregators.py
✅ Experiment runner: scripts/run_backdoor_experiment.py
✅ Mode A targeted-backdoor results: results/security_backdoor_modeA_results.csv
✅ Mode B targeted-backdoor results: results/security_backdoor_modeB_results.csv
✅ Mode A vs B comparison: results/security_modeA_vs_modeB.csv
✅ Paired site F1 observations: results/per_site_f1_pairs.csv
✅ W

In [23]:
# ============================================================
# CREATE COMPLETE ZIP OF fl_ids_repo_real
#
# Source:
#   /content/fl_ids_repo_real
#
# Output:
#   /content/fl_ids_repo_real_complete.zip
#
# Includes EVERYTHING in the repository folder.
# ============================================================

from pathlib import Path
import zipfile
import os
import time

SOURCE = Path("/content/fl_ids_repo_real")
ZIP_PATH = Path("/content/fl_ids_repo_real_complete.zip")

assert SOURCE.exists(), (
    f"Source folder not found: {SOURCE}"
)

# Remove previous ZIP if it exists
if ZIP_PATH.exists():
    ZIP_PATH.unlink()

print("Source:")
print(SOURCE)

print("\nCreating ZIP:")
print(ZIP_PATH)

print(
    "\n⚠️ This includes raw datasets, processed datasets, "
    "results, runs, notebook, source code, and Git files."
)

start_time = time.time()

file_count = 0
total_bytes = 0


with zipfile.ZipFile(
    ZIP_PATH,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6,
    allowZip64=True,
) as zf:

    for root, dirs, files in os.walk(SOURCE):

        root_path = Path(root)

        for filename in files:

            file_path = (
                root_path /
                filename
            )

            # Skip broken symbolic links only
            if not file_path.exists():
                continue

            relative_path = (
                file_path.relative_to(
                    SOURCE
                )
            )

            # Make ZIP contain one top-level folder:
            #
            # fl_ids_repo_real/
            #     src/
            #     data/
            #     results/
            #     ...
            archive_name = (
                Path("fl_ids_repo_real")
                /
                relative_path
            )

            zf.write(
                file_path,
                arcname=str(
                    archive_name
                )
            )

            file_count += 1

            try:
                total_bytes += (
                    file_path.stat().st_size
                )
            except Exception:
                pass

            if file_count % 100 == 0:
                print(
                    f"Archived {file_count} files..."
                )


elapsed = (
    time.time() -
    start_time
)

zip_size_gb = (
    ZIP_PATH.stat().st_size
    / (1024 ** 3)
)

source_size_gb = (
    total_bytes
    / (1024 ** 3)
)


print(
    "\n============================================"
)

print(
    "✅ COMPLETE ZIP CREATED"
)

print(
    "============================================"
)

print(
    f"Files archived : {file_count}"
)

print(
    f"Source size    : {source_size_gb:.2f} GB"
)

print(
    f"ZIP size       : {zip_size_gb:.2f} GB"
)

print(
    f"Time           : {elapsed / 60:.1f} minutes"
)

print(
    "\nZIP location:"
)

print(
    ZIP_PATH
)

print(
    "\nYou can now download this single file "
    "from the Colab Files panel."
)

Source:
/content/fl_ids_repo_real

Creating ZIP:
/content/fl_ids_repo_real_complete.zip

⚠️ This includes raw datasets, processed datasets, results, runs, notebook, source code, and Git files.
Archived 100 files...
Archived 200 files...

✅ COMPLETE ZIP CREATED
Files archived : 201
Source size    : 9.02 GB
ZIP size       : 1.28 GB
Time           : 3.7 minutes

ZIP location:
/content/fl_ids_repo_real_complete.zip

You can now download this single file from the Colab Files panel.


In [25]:
# ============================================================
# STEP 20.14 — FULL CLASS-BALANCED LOSS RERUN
#
# Scientific correction:
#   Replace fixed pos_weight = 2.0 with
#
#       pos_weight =
#           N_benign_train / N_attack_train
#
#   computed from TRAINING PARTITIONS ONLY.
#
# Runs BOTH methods fairly:
#
#   Baseline:
#       FedProx + weighted FedAvg
#
#   Proposed:
#       FedProx + clipping C=1 + TrimMean beta=0.1
#
# Fixed:
#   K            = 10
#   alpha        = 0.3
#   rounds       = 10
#   local epochs = 2
#   mu           = 1e-3
#   lr           = 3e-4
#   seed         = 42
#
# No test information is used to compute class weights.
#
# Original results are NOT overwritten.
# ============================================================

from pathlib import Path
import json
import os
import sys
import time
import subprocess

import numpy as np
import pandas as pd

from scipy.stats import wilcoxon


# ============================================================
# 1. Paths
# ============================================================

REPO = Path(
    "/content/fl_ids_repo_real"
)

RESULT_DIR = (
    REPO /
    "results"
)

CONFIG_DIR = (
    REPO /
    "configs/class_balanced"
)

RUNNER = (
    REPO /
    "scripts/run_backdoor_experiment.py"
)


RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CONFIG_DIR.mkdir(
    parents=True,
    exist_ok=True
)

assert REPO.exists()
assert RUNNER.exists()


# ============================================================
# 2. Dataset definitions
# ============================================================

DATASETS = {

    "Edge-IIoTset": {
        "key":
            "edgeiiotset",

        "encoded":
            REPO /
            "data/processed/edgeiiotset_encoded.npz",

        "split":
            REPO /
            "data/processed/splits_edgeiiotset_dirichlet_alpha_0.3.npz",
    },

    "ToN-IoT": {
        "key":
            "ton_iot",

        "encoded":
            REPO /
            "data/processed/ton_iot_encoded.npz",

        "split":
            REPO /
            "data/processed/splits_ton_iot_dirichlet_alpha_0.3.npz",
    },

    "X-IIoTID": {
        "key":
            "xiiotid",

        "encoded":
            REPO /
            "data/processed/xiiotid_encoded.npz",

        "split":
            REPO /
            "data/processed/splits_xiiotid_dirichlet_alpha_0.3.npz",
    },
}


# ============================================================
# 3. Compute DATASET-SPECIFIC class weights
#    using TRAINING INDICES ONLY
# ============================================================

class_weight_rows = []

POS_WEIGHTS = {}


print(
    "=" * 78
)

print(
    "TRAINING-ONLY CLASS-WEIGHT CALCULATION"
)

print(
    "=" * 78
)


for dataset_name, info in DATASETS.items():

    assert info["encoded"].exists()
    assert info["split"].exists()


    data = np.load(
        info["encoded"],
        allow_pickle=True
    )

    splits = np.load(
        info["split"],
        allow_pickle=True
    )


    y = np.asarray(
        data["y"],
        dtype=np.int64
    )


    train_groups = (
        splits["train"]
    )


    # --------------------------------------------------------
    # Combine TRAIN indices only.
    # No validation/test labels are used.
    # --------------------------------------------------------

    all_train_idx = np.concatenate(
        [
            np.asarray(
                idx,
                dtype=np.int64
            )
            for idx
            in train_groups
        ]
    )


    y_train = y[
        all_train_idx
    ]


    n_benign = int(
        np.sum(
            y_train == 0
        )
    )

    n_attack = int(
        np.sum(
            y_train == 1
        )
    )


    if n_attack == 0:

        raise RuntimeError(
            f"{dataset_name}: "
            "no attack samples in training data."
        )


    pos_weight = (
        n_benign /
        n_attack
    )


    POS_WEIGHTS[
        dataset_name
    ] = float(
        pos_weight
    )


    class_weight_rows.append(
        {
            "dataset":
                dataset_name,

            "train_samples":
                int(
                    len(
                        y_train
                    )
                ),

            "train_benign":
                n_benign,

            "train_attack":
                n_attack,

            "train_attack_fraction":
                float(
                    n_attack /
                    len(
                        y_train
                    )
                ),

            "pos_weight":
                float(
                    pos_weight
                ),
        }
    )


    print(
        f"\n{dataset_name}"
    )

    print(
        f"  training records = "
        f"{len(y_train):,}"
    )

    print(
        f"  benign           = "
        f"{n_benign:,}"
    )

    print(
        f"  attack           = "
        f"{n_attack:,}"
    )

    print(
        f"  pos_weight       = "
        f"{pos_weight:.6f}"
    )


class_weight_df = pd.DataFrame(
    class_weight_rows
)


CLASS_WEIGHT_FILE = (
    RESULT_DIR /
    "training_only_class_weights.csv"
)


class_weight_df.to_csv(
    CLASS_WEIGHT_FILE,
    index=False
)


# ============================================================
# 4. Common FINAL configuration
# ============================================================

COMMON = {

    "seed":
        42,

    "batch_size":
        256,

    "hidden": [
        256,
        128,
        64
    ],

    "dropout":
        0.30,

    "batch_norm":
        False,

    "rounds":
        10,

    "local_epochs":
        2,

    "fedprox_mu":
        1e-3,

    "lr":
        3e-4,

    "weight_decay":
        1e-4,

    "grad_clip":
        1.0,

    "server_gamma":
        1.0,

    # Clean comparison
    "malicious_frac":
        0.0,

    "dp_sigma":
        0.0,

    "threshold":
        0.50,

    "eval_batch_size":
        4096,

    "backdoor": {
        "poison_fraction":
            0.0,

        "trigger_dims":
            5,

        "trigger_value":
            3.0,
    },

    "calibration": {

        "target_fpr":
            0.05,

        "theta_min":
            0.05,

        "theta_max":
            0.95,

        "min_benign_val":
            200,

        "max_pred_pos":
            0.90,

        "min_pred_pos":
            0.01,
    },
}


# ============================================================
# 5. Build paired baseline/proposed configs
# ============================================================

CONFIGS = {}


for dataset_name, info in DATASETS.items():

    key = info["key"]

    pos_weight = (
        POS_WEIGHTS[
            dataset_name
        ]
    )


    # --------------------------------------------------------
    # FedProx baseline
    # --------------------------------------------------------

    fedprox_cfg = json.loads(
        json.dumps(
            COMMON
        )
    )


    fedprox_cfg.update(
        {
            "dataset":
                key,

            "encoded_npz":
                str(
                    info[
                        "encoded"
                    ].relative_to(
                        REPO
                    )
                ),

            "splits_npz":
                str(
                    info[
                        "split"
                    ].relative_to(
                        REPO
                    )
                ),

            "name":
                "classbalanced_clean_fedprox_FINAL",

            # Dataset-level training-only weight
            "pos_weight":
                float(
                    pos_weight
                ),

            "aggregator":
                "fedavg",

            "use_update_clipping":
                False,

            "clip_C":
                1.0,

            "trim_beta":
                0.10,
        }
    )


    fedprox_path = (
        CONFIG_DIR /
        f"{key}_fedprox_classbalanced_FINAL.json"
    )


    fedprox_path.write_text(
        json.dumps(
            fedprox_cfg,
            indent=2
        )
    )


    # --------------------------------------------------------
    # Proposed method
    # --------------------------------------------------------

    proposed_cfg = json.loads(
        json.dumps(
            COMMON
        )
    )


    proposed_cfg.update(
        {
            "dataset":
                key,

            "encoded_npz":
                str(
                    info[
                        "encoded"
                    ].relative_to(
                        REPO
                    )
                ),

            "splits_npz":
                str(
                    info[
                        "split"
                    ].relative_to(
                        REPO
                    )
                ),

            "name":
                "classbalanced_clean_proposed_FINAL",

            # EXACT SAME dataset-level class weight
            "pos_weight":
                float(
                    pos_weight
                ),

            "aggregator":
                "trimmean",

            "use_update_clipping":
                True,

            # Keep original selected bound.
            "clip_C":
                1.0,

            "trim_beta":
                0.10,
        }
    )


    proposed_path = (
        CONFIG_DIR /
        f"{key}_proposed_classbalanced_FINAL.json"
    )


    proposed_path.write_text(
        json.dumps(
            proposed_cfg,
            indent=2
        )
    )


    CONFIGS[
        dataset_name
    ] = {

        "key":
            key,

        "fedprox":
            fedprox_path,

        "proposed":
            proposed_path,
    }


print(
    "\n✅ Six class-balanced configurations created."
)


# ============================================================
# 6. Environment
# ============================================================

env = os.environ.copy()


existing_pythonpath = env.get(
    "PYTHONPATH",
    ""
)


env["PYTHONPATH"] = (
    str(REPO)
    if not existing_pythonpath
    else
    str(REPO)
    + ":"
    + existing_pythonpath
)


RUN_ROOT = (
    REPO /
    "runs/backdoor"
)


# ============================================================
# 7. Locate result helper
# ============================================================

def newest_result(
    dataset_key,
    experiment_name,
):

    if not RUN_ROOT.exists():

        return None


    candidates = [
        p

        for p in RUN_ROOT.iterdir()

        if (
            p.is_dir()

            and
            dataset_key.lower()
            in p.name.lower()

            and
            experiment_name.lower()
            in p.name.lower()

            and
            (
                p /
                "results.json"
            ).exists()
        )
    ]


    if not candidates:

        return None


    return max(
        candidates,

        key=lambda p:
            (
                p /
                "results.json"
            ).stat().st_mtime
    )


# ============================================================
# 8. Execute all six full experiments
# ============================================================

stored_results = {}

runtime_rows = []

start_all = time.time()


for dataset_name, info in CONFIGS.items():

    stored_results[
        dataset_name
    ] = {}


    print(
        "\n"
        + "#" * 78
    )

    print(
        f"{dataset_name}"
    )

    print(
        f"TRAINING-ONLY POS_WEIGHT = "
        f"{POS_WEIGHTS[dataset_name]:.6f}"
    )

    print(
        "#" * 78
    )


    for method in [
        "fedprox",
        "proposed",
    ]:

        config_path = (
            info[
                method
            ]
        )


        print(
            "\n"
            + "=" * 72
        )

        print(
            f"{dataset_name} — "
            f"{method.upper()}"
        )

        print(
            "=" * 72
        )

        print(
            "Rounds        : 10"
        )

        print(
            "Local epochs  : 2"
        )

        print(
            "FedProx mu    : 0.001"
        )

        print(
            "LR            : 0.0003"
        )

        print(
            f"pos_weight    : "
            f"{POS_WEIGHTS[dataset_name]:.6f}"
        )


        if method == "fedprox":

            print(
                "Aggregation   : weighted FedAvg"
            )

            print(
                "Clipping      : OFF"
            )

            experiment_name = (
                "classbalanced_clean_fedprox_final"
            )

        else:

            print(
                "Aggregation   : TrimMean beta=0.10"
            )

            print(
                "Clipping      : C=1.0"
            )

            experiment_name = (
                "classbalanced_clean_proposed_final"
            )


        start = time.time()


        result = subprocess.run(
            [
                sys.executable,
                str(
                    RUNNER
                ),

                "--config",
                str(
                    config_path
                ),
            ],

            cwd=str(
                REPO
            ),

            env=env,
        )


        elapsed = (
            time.time() -
            start
        ) / 60.0


        if result.returncode != 0:

            raise RuntimeError(
                f"Experiment failed: "
                f"{dataset_name} / {method}"
            )


        result_dir = newest_result(
            info[
                "key"
            ],

            experiment_name,
        )


        if result_dir is None:

            raise RuntimeError(
                "Could not locate result for "
                f"{dataset_name}/{method}"
            )


        result_data = json.loads(
            (
                result_dir /
                "results.json"
            ).read_text()
        )


        stored_results[
            dataset_name
        ][
            method
        ] = result_data


        global_metrics = (
            result_data[
                "clean_metrics_under_attack"
            ]
        )


        runtime_rows.append(
            {
                "dataset":
                    dataset_name,

                "method":
                    method,

                "runtime_minutes":
                    elapsed,

                "global_f1":
                    float(
                        global_metrics[
                            "f1"
                        ]
                    ),

                "global_fpr":
                    float(
                        global_metrics[
                            "fpr"
                        ]
                    ),
            }
        )


        print(
            f"\n✅ runtime = "
            f"{elapsed:.1f} min"
        )

        print(
            f"Global F1  = "
            f"{global_metrics['f1']:.4f}"
        )

        print(
            f"Global FPR = "
            f"{global_metrics['fpr']:.4f}"
        )


# ============================================================
# 9. Paired silo-level F1 + Wilcoxon
# ============================================================

pair_rows = []

summary_rows = []


for dataset_name in DATASETS:

    fedprox_sites = sorted(
        stored_results[
            dataset_name
        ][
            "fedprox"
        ][
            "clean_per_silo"
        ],

        key=lambda x:
            int(
                x[
                    "silo"
                ]
            )
    )


    proposed_sites = sorted(
        stored_results[
            dataset_name
        ][
            "proposed"
        ][
            "clean_per_silo"
        ],

        key=lambda x:
            int(
                x[
                    "silo"
                ]
            )
    )


    assert len(
        fedprox_sites
    ) == 10

    assert len(
        proposed_sites
    ) == 10


    fedprox_f1 = np.asarray(
        [
            float(
                x[
                    "f1"
                ]
            )

            for x
            in fedprox_sites
        ],
        dtype=float,
    )


    proposed_f1 = np.asarray(
        [
            float(
                x[
                    "f1"
                ]
            )

            for x
            in proposed_sites
        ],
        dtype=float,
    )


    differences = (
        proposed_f1 -
        fedprox_f1
    )


    # --------------------------------------------------------
    # Wilcoxon
    # --------------------------------------------------------

    if np.allclose(
        differences,
        0
    ):

        statistic = 0.0

        p_value = 1.0

    else:

        test = wilcoxon(
            proposed_f1,
            fedprox_f1,

            alternative=
                "two-sided",

            zero_method=
                "wilcox",

            method=
                "auto",
        )


        statistic = float(
            test.statistic
        )

        p_value = float(
            test.pvalue
        )


    # --------------------------------------------------------
    # Per-site storage
    # --------------------------------------------------------

    for site in range(
        10
    ):

        pair_rows.append(
            {
                "dataset":
                    dataset_name,

                "site":
                    site,

                "fedprox_f1":
                    float(
                        fedprox_f1[
                            site
                        ]
                    ),

                "proposed_f1":
                    float(
                        proposed_f1[
                            site
                        ]
                    ),

                "difference":
                    float(
                        differences[
                            site
                        ]
                    ),
            }
        )


    fedprox_global = (
        stored_results[
            dataset_name
        ][
            "fedprox"
        ][
            "clean_metrics_under_attack"
        ]
    )


    proposed_global = (
        stored_results[
            dataset_name
        ][
            "proposed"
        ][
            "clean_metrics_under_attack"
        ]
    )


    summary_rows.append(
        {
            "dataset":
                dataset_name,

            "pos_weight":
                float(
                    POS_WEIGHTS[
                        dataset_name
                    ]
                ),

            "fedprox_global_f1":
                float(
                    fedprox_global[
                        "f1"
                    ]
                ),

            "proposed_global_f1":
                float(
                    proposed_global[
                        "f1"
                    ]
                ),

            "fedprox_global_fpr":
                float(
                    fedprox_global[
                        "fpr"
                    ]
                ),

            "proposed_global_fpr":
                float(
                    proposed_global[
                        "fpr"
                    ]
                ),

            "fedprox_mean_f1":
                float(
                    fedprox_f1.mean()
                ),

            "fedprox_std_f1":
                float(
                    fedprox_f1.std(
                        ddof=1
                    )
                ),

            "proposed_mean_f1":
                float(
                    proposed_f1.mean()
                ),

            "proposed_std_f1":
                float(
                    proposed_f1.std(
                        ddof=1
                    )
                ),

            "mean_difference":
                float(
                    differences.mean()
                ),

            "positive_sites":
                int(
                    np.sum(
                        differences > 0
                    )
                ),

            "negative_sites":
                int(
                    np.sum(
                        differences < 0
                    )
                ),

            "zero_sites":
                int(
                    np.sum(
                        differences == 0
                    )
                ),

            "wilcoxon_statistic":
                statistic,

            "p_value":
                p_value,
        }
    )


pairs_df = pd.DataFrame(
    pair_rows
)


summary_df = pd.DataFrame(
    summary_rows
)


runtime_df = pd.DataFrame(
    runtime_rows
)


# ============================================================
# 10. Save new results
# ============================================================

PAIRS_OUT = (
    RESULT_DIR /
    "per_site_f1_pairs_classbalanced.csv"
)

SUMMARY_OUT = (
    RESULT_DIR /
    "wilcoxon_results_classbalanced.csv"
)

SUMMARY_JSON = (
    RESULT_DIR /
    "wilcoxon_results_classbalanced.json"
)

RUNTIME_OUT = (
    RESULT_DIR /
    "classbalanced_global_results.csv"
)


pairs_df.to_csv(
    PAIRS_OUT,
    index=False
)


summary_df.to_csv(
    SUMMARY_OUT,
    index=False
)


SUMMARY_JSON.write_text(
    json.dumps(
        summary_rows,
        indent=2
    )
)


runtime_df.to_csv(
    RUNTIME_OUT,
    index=False
)


# ============================================================
# 11. Compare old fixed-weight vs corrected weighting
# ============================================================

OLD_SUMMARY_FILE = (
    RESULT_DIR /
    "wilcoxon_results.csv"
)


comparison_rows = []


if OLD_SUMMARY_FILE.exists():

    old_df = pd.read_csv(
        OLD_SUMMARY_FILE
    )


    for dataset_name in DATASETS:

        old = old_df[
            old_df[
                "dataset"
            ] == dataset_name
        ].iloc[0]


        new = summary_df[
            summary_df[
                "dataset"
            ] == dataset_name
        ].iloc[0]


        comparison_rows.append(
            {
                "dataset":
                    dataset_name,

                "old_proposed_mean_f1":
                    float(
                        old[
                            "proposed_mean_f1"
                        ]
                    ),

                "new_proposed_mean_f1":
                    float(
                        new[
                            "proposed_mean_f1"
                        ]
                    ),

                "delta_proposed_mean_f1":
                    float(
                        new[
                            "proposed_mean_f1"
                        ]
                        -
                        old[
                            "proposed_mean_f1"
                        ]
                    ),

                "old_p_value":
                    float(
                        old[
                            "p_value"
                        ]
                    ),

                "new_p_value":
                    float(
                        new[
                            "p_value"
                        ]
                    ),
            }
        )


comparison_df = pd.DataFrame(
    comparison_rows
)


COMPARE_OUT = (
    RESULT_DIR /
    "fixed_vs_classbalanced_comparison.csv"
)


if len(
    comparison_df
) > 0:

    comparison_df.to_csv(
        COMPARE_OUT,
        index=False
    )


# ============================================================
# 12. Final results
# ============================================================

print(
    "\n"
    + "=" * 110
)

print(
    "FINAL CLASS-BALANCED FEDPROX vs PROPOSED RESULTS"
)

print(
    "=" * 110
)


print(
    summary_df[
        [
            "dataset",
            "pos_weight",
            "fedprox_mean_f1",
            "proposed_mean_f1",
            "proposed_std_f1",
            "positive_sites",
            "negative_sites",
            "wilcoxon_statistic",
            "p_value",
        ]
    ].to_string(
        index=False
    )
)


print(
    "\n"
    + "=" * 110
)

print(
    "GLOBAL CLEAN PERFORMANCE"
)

print(
    "=" * 110
)


print(
    summary_df[
        [
            "dataset",
            "fedprox_global_f1",
            "fedprox_global_fpr",
            "proposed_global_f1",
            "proposed_global_fpr",
        ]
    ].to_string(
        index=False
    )
)


if len(
    comparison_df
) > 0:

    print(
        "\n"
        + "=" * 110
    )

    print(
        "OLD FIXED-WEIGHT vs NEW CLASS-BALANCED PROPOSED"
    )

    print(
        "=" * 110
    )


    print(
        comparison_df.to_string(
            index=False
        )
    )


print(
    "\nSaved:"
)

print(
    CLASS_WEIGHT_FILE
)

print(
    PAIRS_OUT
)

print(
    SUMMARY_OUT
)

print(
    SUMMARY_JSON
)

print(
    RUNTIME_OUT
)

if len(
    comparison_df
) > 0:

    print(
        COMPARE_OUT
    )


print(
    "\nTotal runtime:"
)

print(
    f"{(time.time() - start_all) / 60:.1f} minutes"
)


print(
    "\n============================================"
)

print(
    "✅ STEP 20.14 CLASS-BALANCED FULL RUN COMPLETE"
)

print(
    "============================================"
)

TRAINING-ONLY CLASS-WEIGHT CALCULATION

Edge-IIoTset
  training records = 1,336,765
  benign           = 954,796
  attack           = 381,969
  pos_weight       = 2.499669

ToN-IoT
  training records = 147,726
  benign           = 34,985
  attack           = 112,741
  pos_weight       = 0.310313

X-IIoTID
  training records = 574,580
  benign           = 295,083
  attack           = 279,497
  pos_weight       = 1.055764

✅ Six class-balanced configurations created.

##############################################################################
Edge-IIoTset
TRAINING-ONLY POS_WEIGHT = 2.499669
##############################################################################

Edge-IIoTset — FEDPROX
Rounds        : 10
Local epochs  : 2
FedProx mu    : 0.001
LR            : 0.0003
pos_weight    : 2.499669
Aggregation   : weighted FedAvg
Clipping      : OFF

✅ runtime = 13.1 min
Global F1  = 0.6740
Global FPR = 0.0000

Edge-IIoTset — PROPOSED
Rounds        : 10
Local epochs  : 2
FedProx mu    :

In [26]:
# ============================================================
# STEP 20.15 — VALIDATION-CONSTRAINED THRESHOLD RECALIBRATION
#
# NO RETRAINING.
#
# Uses the already-trained class-balanced models from Step 20.14.
#
# For every dataset/method/site:
#
#   theta* = argmax validation F1
#            subject to validation FPR <= 0.05
#
# If a local validation silo contains only one class:
#   use dataset-level pooled-validation threshold.
#
# IMPORTANT:
#   - threshold chosen from VALIDATION only
#   - TEST used once for final evaluation
#   - same rule for FedProx and Proposed
# ============================================================

from pathlib import Path
import sys
import json
import gc

import numpy as np
import pandas as pd
import torch

from scipy.stats import wilcoxon


# ============================================================
# 1. Setup
# ============================================================

REPO = Path(
    "/content/fl_ids_repo_real"
)

if str(REPO) not in sys.path:
    sys.path.insert(
        0,
        str(REPO)
    )


from src.data.loaders import (
    build_silo_loaders,
)

from src.models.mlp import (
    MLPIDS,
)

from src.eval.thresholds import (
    collect_probs,
    metrics_at_threshold,
)


DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(
    "Device:",
    DEVICE
)


RUN_ROOT = (
    REPO /
    "runs/backdoor"
)

RESULT_DIR = (
    REPO /
    "results"
)


# ============================================================
# 2. Locate latest Step 20.14 trained models
# ============================================================

DATASETS = {

    "Edge-IIoTset":
        "edgeiiotset",

    "ToN-IoT":
        "ton_iot",

    "X-IIoTID":
        "xiiotid",
}


EXPERIMENTS = {

    "FedProx":
        "classbalanced_clean_fedprox_final",

    "Proposed":
        "classbalanced_clean_proposed_final",
}


def find_latest_run(
    dataset_key,
    experiment_fragment,
):

    candidates = [
        p

        for p in RUN_ROOT.iterdir()

        if (
            p.is_dir()

            and
            dataset_key.lower()
            in p.name.lower()

            and
            experiment_fragment.lower()
            in p.name.lower()

            and
            (p / "model_final.pt").exists()

            and
            (p / "config.json").exists()
        )
    ]


    if not candidates:

        raise FileNotFoundError(
            f"No trained run found for "
            f"{dataset_key} / {experiment_fragment}"
        )


    return max(
        candidates,

        key=lambda p:
            (
                p /
                "model_final.pt"
            ).stat().st_mtime
    )


RUNS = {}


print(
    "\n========== TRAINED MODEL CHECK =========="
)


for dataset_name, dataset_key in DATASETS.items():

    RUNS[
        dataset_name
    ] = {}


    for method, fragment in EXPERIMENTS.items():

        run_dir = find_latest_run(
            dataset_key,
            fragment,
        )


        RUNS[
            dataset_name
        ][
            method
        ] = run_dir


        print(
            f"✅ {dataset_name:14s} | "
            f"{method:8s} | "
            f"{run_dir.name}"
        )


# ============================================================
# 3. Exact validation-threshold search
# ============================================================

def find_best_f1_threshold(
    probs,
    labels,
    max_fpr=0.05,
):
    """
    Exact search over unique validation probabilities.

    Primary:
        maximize validation F1

    Constraint:
        validation FPR <= max_fpr

    Tie breaking:
        1. lower FPR
        2. higher threshold

    No test information is involved.
    """

    probs = np.asarray(
        probs,
        dtype=np.float64
    )

    labels = np.asarray(
        labels,
        dtype=np.int64
    )


    P = int(
        np.sum(
            labels == 1
        )
    )

    N = int(
        np.sum(
            labels == 0
        )
    )


    if (
        P == 0
        or N == 0
    ):

        return None


    # Sort scores descending.
    order = np.argsort(
        -probs,
        kind="mergesort"
    )


    scores = probs[
        order
    ]

    y = labels[
        order
    ]


    tp_cum = np.cumsum(
        y == 1
    )

    fp_cum = np.cumsum(
        y == 0
    )


    # End index of each equal-score block.
    block_ends = np.r_[
        np.where(
            scores[:-1]
            !=
            scores[1:]
        )[0],

        len(scores) - 1,
    ]


    thresholds = scores[
        block_ends
    ]

    tp = tp_cum[
        block_ends
    ].astype(
        np.float64
    )

    fp = fp_cum[
        block_ends
    ].astype(
        np.float64
    )


    fn = (
        P -
        tp
    )


    fpr = (
        fp /
        float(N)
    )


    f1 = (
        2.0 *
        tp
        /
        (
            2.0 *
            tp
            +
            fp
            +
            fn
            +
            1e-12
        )
    )


    feasible = (
        fpr <=
        float(
            max_fpr
        )
        +
        1e-12
    )


    if not np.any(
        feasible
    ):

        # Numerically unlikely because a sufficiently high
        # threshold generally provides FPR near zero.
        best = int(
            np.argmin(
                fpr
            )
        )

        return {
            "threshold":
                float(
                    thresholds[
                        best
                    ]
                ),

            "val_f1":
                float(
                    f1[
                        best
                    ]
                ),

            "val_fpr":
                float(
                    fpr[
                        best
                    ]
                ),

            "constraint_met":
                False,
        }


    indices = np.where(
        feasible
    )[0]


    # Highest F1 first.
    best_f1 = np.max(
        f1[
            indices
        ]
    )


    indices = indices[
        np.isclose(
            f1[
                indices
            ],
            best_f1,
            rtol=0,
            atol=1e-12,
        )
    ]


    # Then lowest FPR.
    best_fpr = np.min(
        fpr[
            indices
        ]
    )


    indices = indices[
        np.isclose(
            fpr[
                indices
            ],
            best_fpr,
            rtol=0,
            atol=1e-12,
        )
    ]


    # Finally choose higher threshold.
    best = indices[
        np.argmax(
            thresholds[
                indices
            ]
        )
    ]


    return {
        "threshold":
            float(
                thresholds[
                    best
                ]
            ),

        "val_f1":
            float(
                f1[
                    best
                ]
            ),

        "val_fpr":
            float(
                fpr[
                    best
                ]
            ),

        "constraint_met":
            True,
    }


# ============================================================
# 4. Load trained model
# ============================================================

def load_trained_model(
    run_dir,
    input_dim,
):

    cfg = json.loads(
        (
            run_dir /
            "config.json"
        ).read_text()
    )


    model = MLPIDS(
        input_dim=input_dim,

        hidden=tuple(
            cfg[
                "hidden"
            ]
        ),

        dropout=float(
            cfg[
                "dropout"
            ]
        ),

        batch_norm=bool(
            cfg[
                "batch_norm"
            ]
        ),
    ).to(
        DEVICE
    )


    model_file = (
        run_dir /
        "model_final.pt"
    )


    try:

        state = torch.load(
            model_file,
            map_location=DEVICE,
            weights_only=True,
        )

    except TypeError:

        state = torch.load(
            model_file,
            map_location=DEVICE,
        )


    model.load_state_dict(
        state
    )

    model.eval()


    return (
        model,
        cfg,
    )


# ============================================================
# 5. Evaluation
# ============================================================

site_rows = []

summary_rows = []


for dataset_name, dataset_key in DATASETS.items():

    print(
        "\n"
        + "=" * 84
    )

    print(
        dataset_name
    )

    print(
        "=" * 84
    )


    # Both methods use identical data/splits.
    reference_run = (
        RUNS[
            dataset_name
        ][
            "Proposed"
        ]
    )


    reference_cfg = json.loads(
        (
            reference_run /
            "config.json"
        ).read_text()
    )


    encoded_path = (
        REPO /
        reference_cfg[
            "encoded_npz"
        ]
    )


    split_path = (
        REPO /
        reference_cfg[
            "splits_npz"
        ]
    )


    silo_data, input_dim, K = (
        build_silo_loaders(
            encoded_path,
            split_path,
            batch_size=4096,
        )
    )


    assert K == 10


    for method in [
        "FedProx",
        "Proposed",
    ]:

        print(
            f"\n--- {method} ---"
        )


        model, cfg = (
            load_trained_model(
                RUNS[
                    dataset_name
                ][
                    method
                ],
                input_dim,
            )
        )


        # ====================================================
        # First collect ALL validation predictions.
        # This gives a validation-only fallback threshold.
        # ====================================================

        val_cache = []

        pooled_probs = []

        pooled_labels = []


        for k, site in enumerate(
            silo_data
        ):

            probs_val, y_val = (
                collect_probs(
                    model,
                    site[
                        "val_loader"
                    ],
                    DEVICE,
                )
            )


            val_cache.append(
                (
                    probs_val,
                    y_val,
                )
            )


            pooled_probs.append(
                probs_val
            )

            pooled_labels.append(
                y_val
            )


        pooled_probs = np.concatenate(
            pooled_probs
        )

        pooled_labels = np.concatenate(
            pooled_labels
        )


        pooled_selection = (
            find_best_f1_threshold(
                pooled_probs,
                pooled_labels,
                max_fpr=0.05,
            )
        )


        if pooled_selection is None:

            raise RuntimeError(
                f"{dataset_name}/{method}: "
                "pooled validation data does not "
                "contain both classes."
            )


        pooled_threshold = (
            pooled_selection[
                "threshold"
            ]
        )


        print(
            f"Pooled validation fallback theta = "
            f"{pooled_threshold:.6f}"
        )


        # ====================================================
        # Site thresholds and TEST evaluation
        # ====================================================

        total_tp = 0
        total_tn = 0
        total_fp = 0
        total_fn = 0

        method_site_f1 = []


        for k, site in enumerate(
            silo_data
        ):

            probs_val, y_val = (
                val_cache[
                    k
                ]
            )


            n_val_attack = int(
                np.sum(
                    y_val == 1
                )
            )

            n_val_benign = int(
                np.sum(
                    y_val == 0
                )
            )


            local_selection = (
                find_best_f1_threshold(
                    probs_val,
                    y_val,
                    max_fpr=0.05,
                )
            )


            if local_selection is None:

                threshold = float(
                    pooled_threshold
                )

                threshold_source = (
                    "pooled_validation_fallback"
                )

                # Calculate validation metrics for audit.
                local_val_metrics = (
                    metrics_at_threshold(
                        probs_val,
                        y_val,
                        threshold,
                    )
                )


                val_f1 = float(
                    local_val_metrics[
                        "f1"
                    ]
                )

                val_fpr = float(
                    local_val_metrics[
                        "fpr"
                    ]
                )

            else:

                threshold = float(
                    local_selection[
                        "threshold"
                    ]
                )

                threshold_source = (
                    "local_validation"
                )

                val_f1 = float(
                    local_selection[
                        "val_f1"
                    ]
                )

                val_fpr = float(
                    local_selection[
                        "val_fpr"
                    ]
                )


            # -----------------------------------------------
            # TEST is touched only after threshold fixed.
            # -----------------------------------------------

            probs_test, y_test = (
                collect_probs(
                    model,
                    site[
                        "test_loader"
                    ],
                    DEVICE,
                )
            )


            test_metrics = (
                metrics_at_threshold(
                    probs_test,
                    y_test,
                    threshold,
                )
            )


            method_site_f1.append(
                float(
                    test_metrics[
                        "f1"
                    ]
                )
            )


            total_tp += int(
                test_metrics[
                    "tp"
                ]
            )

            total_tn += int(
                test_metrics[
                    "tn"
                ]
            )

            total_fp += int(
                test_metrics[
                    "fp"
                ]
            )

            total_fn += int(
                test_metrics[
                    "fn"
                ]
            )


            site_rows.append(
                {
                    "dataset":
                        dataset_name,

                    "method":
                        method,

                    "site":
                        k,

                    "threshold":
                        threshold,

                    "threshold_source":
                        threshold_source,

                    "n_val_benign":
                        n_val_benign,

                    "n_val_attack":
                        n_val_attack,

                    "val_f1":
                        val_f1,

                    "val_fpr":
                        val_fpr,

                    "test_f1":
                        float(
                            test_metrics[
                                "f1"
                            ]
                        ),

                    "test_fpr":
                        float(
                            test_metrics[
                                "fpr"
                            ]
                        ),

                    "test_precision":
                        float(
                            test_metrics[
                                "precision"
                            ]
                        ),

                    "test_recall":
                        float(
                            test_metrics[
                                "recall"
                            ]
                        ),
                }
            )


        precision = (
            total_tp /
            (
                total_tp +
                total_fp +
                1e-12
            )
        )


        recall = (
            total_tp /
            (
                total_tp +
                total_fn +
                1e-12
            )
        )


        global_f1 = (
            2.0 *
            precision *
            recall /
            (
                precision +
                recall +
                1e-12
            )
        )


        global_fpr = (
            total_fp /
            (
                total_fp +
                total_tn +
                1e-12
            )
        )


        summary_rows.append(
            {
                "dataset":
                    dataset_name,

                "method":
                    method,

                "macro_test_f1":
                    float(
                        np.mean(
                            method_site_f1
                        )
                    ),

                "std_test_f1":
                    float(
                        np.std(
                            method_site_f1,
                            ddof=1
                        )
                    ),

                "global_test_f1":
                    float(
                        global_f1
                    ),

                "global_test_fpr":
                    float(
                        global_fpr
                    ),

                "pooled_validation_threshold":
                    float(
                        pooled_threshold
                    ),
            }
        )


        print(
            f"Macro test F1  = "
            f"{np.mean(method_site_f1):.4f}"
        )

        print(
            f"Global test F1 = "
            f"{global_f1:.4f}"
        )

        print(
            f"Global test FPR = "
            f"{global_fpr:.4f}"
        )


        del model

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        gc.collect()


    del silo_data

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    gc.collect()


# ============================================================
# 6. Create paired FedProx / Proposed site table
# ============================================================

site_df = pd.DataFrame(
    site_rows
)

summary_df = pd.DataFrame(
    summary_rows
)


pair_rows = []

wilcoxon_rows = []


for dataset_name in DATASETS:

    base = (
        site_df[
            (
                site_df[
                    "dataset"
                ]
                ==
                dataset_name
            )
            &
            (
                site_df[
                    "method"
                ]
                ==
                "FedProx"
            )
        ]
        .sort_values(
            "site"
        )
    )


    proposed = (
        site_df[
            (
                site_df[
                    "dataset"
                ]
                ==
                dataset_name
            )
            &
            (
                site_df[
                    "method"
                ]
                ==
                "Proposed"
            )
        ]
        .sort_values(
            "site"
        )
    )


    assert len(base) == 10
    assert len(proposed) == 10


    base_f1 = base[
        "test_f1"
    ].to_numpy(
        dtype=float
    )


    prop_f1 = proposed[
        "test_f1"
    ].to_numpy(
        dtype=float
    )


    diff = (
        prop_f1 -
        base_f1
    )


    for site in range(
        10
    ):

        pair_rows.append(
            {
                "dataset":
                    dataset_name,

                "site":
                    site,

                "fedprox_f1":
                    float(
                        base_f1[
                            site
                        ]
                    ),

                "proposed_f1":
                    float(
                        prop_f1[
                            site
                        ]
                    ),

                "difference":
                    float(
                        diff[
                            site
                        ]
                    ),
            }
        )


    if np.allclose(
        diff,
        0
    ):

        W = 0.0
        p = 1.0

    else:

        test = wilcoxon(
            prop_f1,
            base_f1,

            alternative=
                "two-sided",

            zero_method=
                "wilcox",

            method=
                "auto",
        )


        W = float(
            test.statistic
        )

        p = float(
            test.pvalue
        )


    wilcoxon_rows.append(
        {
            "dataset":
                dataset_name,

            "fedprox_mean_f1":
                float(
                    base_f1.mean()
                ),

            "fedprox_std_f1":
                float(
                    base_f1.std(
                        ddof=1
                    )
                ),

            "proposed_mean_f1":
                float(
                    prop_f1.mean()
                ),

            "proposed_std_f1":
                float(
                    prop_f1.std(
                        ddof=1
                    )
                ),

            "positive_sites":
                int(
                    np.sum(
                        diff > 0
                    )
                ),

            "negative_sites":
                int(
                    np.sum(
                        diff < 0
                    )
                ),

            "wilcoxon_statistic":
                W,

            "p_value":
                p,
        }
    )


pairs_df = pd.DataFrame(
    pair_rows
)

wilcoxon_df = pd.DataFrame(
    wilcoxon_rows
)


# ============================================================
# 7. Save
# ============================================================

SITE_OUT = (
    RESULT_DIR /
    "threshold_calibrated_site_metrics.csv"
)

PAIR_OUT = (
    RESULT_DIR /
    "per_site_f1_pairs_classbalanced_valcal.csv"
)

WILCOXON_OUT = (
    RESULT_DIR /
    "wilcoxon_results_classbalanced_valcal.csv"
)

GLOBAL_OUT = (
    RESULT_DIR /
    "threshold_calibrated_global_results.csv"
)


site_df.to_csv(
    SITE_OUT,
    index=False
)

pairs_df.to_csv(
    PAIR_OUT,
    index=False
)

wilcoxon_df.to_csv(
    WILCOXON_OUT,
    index=False
)

summary_df.to_csv(
    GLOBAL_OUT,
    index=False
)


# ============================================================
# 8. Final output
# ============================================================

print(
    "\n"
    + "=" * 104
)

print(
    "FINAL VALIDATION-CALIBRATED PAIRED RESULTS"
)

print(
    "=" * 104
)


print(
    wilcoxon_df[
        [
            "dataset",
            "fedprox_mean_f1",
            "fedprox_std_f1",
            "proposed_mean_f1",
            "proposed_std_f1",
            "positive_sites",
            "negative_sites",
            "wilcoxon_statistic",
            "p_value",
        ]
    ].to_string(
        index=False
    )
)


print(
    "\n"
    + "=" * 104
)

print(
    "VALIDATION-CALIBRATED GLOBAL TEST PERFORMANCE"
)

print(
    "=" * 104
)


pivot = summary_df.pivot(
    index="dataset",
    columns="method",
    values=[
        "macro_test_f1",
        "global_test_f1",
        "global_test_fpr",
    ]
)


print(
    pivot.to_string()
)


print(
    "\n"
    + "=" * 104
)

print(
    "THRESHOLD SOURCE AUDIT"
)

print(
    "=" * 104
)


audit = (
    site_df.groupby(
        [
            "dataset",
            "method",
            "threshold_source",
        ]
    )
    .size()
    .reset_index(
        name="sites"
    )
)


print(
    audit.to_string(
        index=False
    )
)


print(
    "\nSaved:"
)

print(
    SITE_OUT
)

print(
    PAIR_OUT
)

print(
    WILCOXON_OUT
)

print(
    GLOBAL_OUT
)


print(
    "\n============================================"
)

print(
    "✅ STEP 20.15 THRESHOLD RECALIBRATION COMPLETE"
)

print(
    "============================================"
)

Device: cpu

========== TRAINED MODEL CHECK ==========
✅ Edge-IIoTset   | FedProx  | edgeiiotset_classbalanced_clean_fedprox_FINAL_seed42_20261002_110512
✅ Edge-IIoTset   | Proposed | edgeiiotset_classbalanced_clean_proposed_FINAL_seed42_20261002_111822
✅ ToN-IoT        | FedProx  | ton_iot_classbalanced_clean_fedprox_FINAL_seed42_20261002_111953
✅ ToN-IoT        | Proposed | ton_iot_classbalanced_clean_proposed_FINAL_seed42_20261002_112124
✅ X-IIoTID       | FedProx  | xiiotid_classbalanced_clean_fedprox_FINAL_seed42_20261002_112711
✅ X-IIoTID       | Proposed | xiiotid_classbalanced_clean_proposed_FINAL_seed42_20261002_113239

Edge-IIoTset

--- FedProx ---
Pooled validation fallback theta = 0.003303
Macro test F1  = 0.6997
Global test F1 = 0.9991
Global test FPR = 0.0005

--- Proposed ---
Pooled validation fallback theta = 0.953797
Macro test F1  = 0.7166
Global test F1 = 0.9982
Global test FPR = 0.0014

ToN-IoT

--- FedProx ---
Pooled validation fallback theta = 0.999911
Macro test 

In [27]:
# ============================================================
# STEP 20.16 — FINAL EDGE-IIoTset ABLATION STUDY
#
# Dataset:
#   Edge-IIoTset only
#
# Final configuration:
#   K              = 10
#   alpha          = 0.3
#   rounds         = 10
#   local epochs   = 2
#   lr             = 3e-4
#   weight decay   = 1e-4
#   FedProx mu     = 1e-3
#   clipping C     = 1.0
#   TrimMean beta  = 0.10
#   pos_weight     = training-only class balance
#
# Ablations:
#   1. Full method
#   2. FedAvg instead of TrimMean
#   3. mu = 0 (no FedProx)
#   4. No clipping
#   5. No threshold recalibration
#
# IMPORTANT:
#   - Full model from Step 20.14 is reused.
#   - Only three new models are trained.
#   - "No threshold recalibration" reuses the full model
#     but evaluates it with theta = 0.50.
#   - Other variants use the SAME validation-only
#     threshold calibration rule.
#   - Test labels are never used to select thresholds.
# ============================================================


from pathlib import Path
import os
import sys
import json
import time
import gc
import subprocess

import numpy as np
import pandas as pd
import torch

from sklearn.metrics import roc_auc_score


# ============================================================
# 1. Repository
# ============================================================

REPO = Path(
    "/content/fl_ids_repo_real"
)

if str(REPO) not in sys.path:
    sys.path.insert(
        0,
        str(REPO)
    )


from src.data.loaders import build_silo_loaders
from src.models.mlp import MLPIDS
from src.eval.thresholds import (
    collect_probs,
    metrics_at_threshold,
)


RUNNER = (
    REPO /
    "scripts/run_backdoor_experiment.py"
)

RUN_ROOT = (
    REPO /
    "runs/backdoor"
)

RESULT_DIR = (
    REPO /
    "results"
)

CONFIG_DIR = (
    REPO /
    "configs/ablation_edgeiiotset"
)


CONFIG_DIR.mkdir(
    parents=True,
    exist_ok=True
)

RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


assert RUNNER.exists()


# ============================================================
# 2. Start from FINAL class-balanced proposed configuration
# ============================================================

BASE_CONFIG = (
    REPO /
    "configs/class_balanced/"
    "edgeiiotset_proposed_classbalanced_FINAL.json"
)

assert BASE_CONFIG.exists(), (
    f"Missing final proposed configuration:\n"
    f"{BASE_CONFIG}"
)


base_cfg = json.loads(
    BASE_CONFIG.read_text()
)


# Audit final configuration.
print(
    "=" * 80
)

print(
    "FINAL EDGE-IIoTset CONFIGURATION"
)

print(
    "=" * 80
)

for key in [
    "seed",
    "rounds",
    "local_epochs",
    "lr",
    "weight_decay",
    "fedprox_mu",
    "pos_weight",
    "aggregator",
    "use_update_clipping",
    "clip_C",
    "trim_beta",
]:

    print(
        f"{key:24s}: "
        f"{base_cfg.get(key)}"
    )


# ============================================================
# 3. Locate existing FULL METHOD model
# ============================================================

FULL_FRAGMENT = (
    "classbalanced_clean_proposed_final"
)


def latest_run(
    fragment,
):

    candidates = [
        p

        for p in RUN_ROOT.iterdir()

        if (
            p.is_dir()

            and
            "edgeiiotset"
            in p.name.lower()

            and
            fragment.lower()
            in p.name.lower()

            and
            (p / "model_final.pt").exists()

            and
            (p / "config.json").exists()
        )
    ]


    if not candidates:

        return None


    return max(
        candidates,

        key=lambda p:
            (
                p /
                "model_final.pt"
            ).stat().st_mtime
    )


full_run = latest_run(
    FULL_FRAGMENT
)


if full_run is None:

    raise FileNotFoundError(
        "Could not locate the final Edge-IIoTset "
        "proposed model from Step 20.14."
    )


print(
    "\n✅ Full method model:"
)

print(
    full_run
)


# ============================================================
# 4. Create only THREE training ablations
# ============================================================

VARIANT_SPECS = {

    "FedAvg instead of TrimMean": {
        "name":
            "ablation_edge_fedavg_FINAL",

        # Keep FedProx and clipping.
        "aggregator":
            "fedavg",

        "fedprox_mu":
            1e-3,

        "use_update_clipping":
            True,

        "clip_C":
            1.0,
    },


    "No FedProx ($\\mu=0$)": {
        "name":
            "ablation_edge_no_fedprox_FINAL",

        "aggregator":
            "trimmean",

        "fedprox_mu":
            0.0,

        "use_update_clipping":
            True,

        "clip_C":
            1.0,
    },


    "No clipping": {
        "name":
            "ablation_edge_no_clipping_FINAL",

        "aggregator":
            "trimmean",

        "fedprox_mu":
            1e-3,

        "use_update_clipping":
            False,

        # Retained but inactive.
        "clip_C":
            1.0,
    },
}


variant_configs = {}


for label, changes in VARIANT_SPECS.items():

    cfg = json.loads(
        json.dumps(
            base_cfg
        )
    )


    # --------------------------------------------------------
    # Preserve FINAL experiment settings
    # --------------------------------------------------------

    cfg["seed"] = 42
    cfg["rounds"] = 10
    cfg["local_epochs"] = 2

    cfg["lr"] = 3e-4
    cfg["weight_decay"] = 1e-4

    cfg["batch_norm"] = False

    cfg["pos_weight"] = float(
        base_cfg[
            "pos_weight"
        ]
    )

    cfg["malicious_frac"] = 0.0
    cfg["dp_sigma"] = 0.0

    cfg["trim_beta"] = 0.10


    if "backdoor" in cfg:

        cfg[
            "backdoor"
        ][
            "poison_fraction"
        ] = 0.0


    # --------------------------------------------------------
    # Apply ONE ablation
    # --------------------------------------------------------

    for key, value in changes.items():

        cfg[
            key
        ] = value


    config_path = (
        CONFIG_DIR /
        (
            changes["name"]
            + ".json"
        )
    )


    config_path.write_text(
        json.dumps(
            cfg,
            indent=2
        )
    )


    variant_configs[
        label
    ] = {
        "config":
            config_path,

        "fragment":
            changes[
                "name"
            ].lower(),
    }


    print(
        f"✅ Config created: {label}"
    )


# ============================================================
# 5. Run the three training ablations
# ============================================================

env = os.environ.copy()


existing_pythonpath = (
    env.get(
        "PYTHONPATH",
        ""
    )
)


env["PYTHONPATH"] = (
    str(REPO)

    if not existing_pythonpath

    else
    str(REPO)
    +
    ":"
    +
    existing_pythonpath
)


trained_runs = {
    "Full method":
        full_run
}


runtime_rows = []


start_all = time.time()


for i, (
    label,
    info
) in enumerate(
    variant_configs.items(),
    start=1,
):

    print(
        "\n"
        + "=" * 82
    )

    print(
        f"ABLATION TRAINING {i}/3"
    )

    print(
        label
    )

    print(
        "=" * 82
    )


    t0 = time.time()


    result = subprocess.run(
        [
            sys.executable,

            str(
                RUNNER
            ),

            "--config",

            str(
                info[
                    "config"
                ]
            ),
        ],

        cwd=str(
            REPO
        ),

        env=env,
    )


    elapsed = (
        time.time() -
        t0
    ) / 60.0


    if result.returncode != 0:

        raise RuntimeError(
            f"Ablation failed: {label}"
        )


    run_dir = latest_run(
        info[
            "fragment"
        ]
    )


    if run_dir is None:

        raise RuntimeError(
            f"Could not locate trained model: {label}"
        )


    trained_runs[
        label
    ] = run_dir


    runtime_rows.append(
        {
            "configuration":
                label,

            "training_minutes":
                elapsed,
        }
    )


    print(
        f"\n✅ {label}"
    )

    print(
        f"Runtime = {elapsed:.1f} min"
    )

    print(
        f"Model   = {run_dir.name}"
    )


# ============================================================
# 6. Load common Edge-IIoTset silo data
# ============================================================

encoded_path = (
    REPO /
    base_cfg[
        "encoded_npz"
    ]
)

split_path = (
    REPO /
    base_cfg[
        "splits_npz"
    ]
)


assert encoded_path.exists()
assert split_path.exists()


silo_data, input_dim, K = (
    build_silo_loaders(
        encoded_path,
        split_path,
        batch_size=4096,
    )
)


assert K == 10


print(
    "\nInput dimension:",
    input_dim
)

print(
    "Silos:",
    K
)


# ============================================================
# 7. Validation-only threshold calibration
#
# theta* = maximum validation F1
#          subject to validation FPR <= 0.05
#
# This is exactly the calibration rule used in Step 20.15.
# ============================================================

def select_threshold(
    probs,
    labels,
    max_fpr=0.05,
):

    probs = np.asarray(
        probs,
        dtype=np.float64
    )

    labels = np.asarray(
        labels,
        dtype=np.int64
    )


    positives = int(
        np.sum(
            labels == 1
        )
    )

    negatives = int(
        np.sum(
            labels == 0
        )
    )


    # Cannot calibrate locally if one class is absent.
    if (
        positives == 0
        or negatives == 0
    ):

        return None


    order = np.argsort(
        -probs,
        kind="mergesort",
    )


    scores = probs[
        order
    ]

    y = labels[
        order
    ]


    tp_cum = np.cumsum(
        y == 1
    )

    fp_cum = np.cumsum(
        y == 0
    )


    block_ends = np.r_[
        np.where(
            scores[:-1]
            !=
            scores[1:]
        )[0],

        len(scores) - 1,
    ]


    thresholds = scores[
        block_ends
    ]


    tp = tp_cum[
        block_ends
    ].astype(
        float
    )


    fp = fp_cum[
        block_ends
    ].astype(
        float
    )


    fn = (
        positives -
        tp
    )


    fpr = (
        fp /
        float(
            negatives
        )
    )


    f1 = (
        2.0 *
        tp
        /
        (
            2.0 *
            tp
            +
            fp
            +
            fn
            +
            1e-12
        )
    )


    feasible = (
        fpr <=
        max_fpr
        +
        1e-12
    )


    if not np.any(
        feasible
    ):

        return None


    idx = np.where(
        feasible
    )[0]


    best_f1 = np.max(
        f1[
            idx
        ]
    )


    idx = idx[
        np.isclose(
            f1[
                idx
            ],
            best_f1,
            atol=1e-12,
            rtol=0,
        )
    ]


    # Tie break: lower FPR.
    best_fpr = np.min(
        fpr[
            idx
        ]
    )


    idx = idx[
        np.isclose(
            fpr[
                idx
            ],
            best_fpr,
            atol=1e-12,
            rtol=0,
        )
    ]


    # Final tie break: higher threshold.
    best = idx[
        np.argmax(
            thresholds[
                idx
            ]
        )
    ]


    return float(
        thresholds[
            best
        ]
    )


# ============================================================
# 8. Model loader
# ============================================================

def load_model(
    run_dir,
):

    cfg = json.loads(
        (
            run_dir /
            "config.json"
        ).read_text()
    )


    model = MLPIDS(
        input_dim=input_dim,

        hidden=tuple(
            cfg[
                "hidden"
            ]
        ),

        dropout=float(
            cfg[
                "dropout"
            ]
        ),

        batch_norm=bool(
            cfg[
                "batch_norm"
            ]
        ),
    )


    model_file = (
        run_dir /
        "model_final.pt"
    )


    try:

        state = torch.load(
            model_file,
            map_location="cpu",
            weights_only=True,
        )

    except TypeError:

        state = torch.load(
            model_file,
            map_location="cpu",
        )


    model.load_state_dict(
        state
    )

    model.eval()


    return model


# ============================================================
# 9. Evaluation function
# ============================================================

def evaluate_model(
    model,
    use_calibration=True,
):

    # --------------------------------------------------------
    # Validation predictions first
    # --------------------------------------------------------

    val_cache = []

    pooled_val_probs = []
    pooled_val_labels = []


    for site in silo_data:

        probs_val, y_val = (
            collect_probs(
                model,
                site[
                    "val_loader"
                ],
                torch.device(
                    "cpu"
                ),
            )
        )


        val_cache.append(
            (
                probs_val,
                y_val
            )
        )


        pooled_val_probs.append(
            probs_val
        )

        pooled_val_labels.append(
            y_val
        )


    pooled_val_probs = np.concatenate(
        pooled_val_probs
    )

    pooled_val_labels = np.concatenate(
        pooled_val_labels
    )


    pooled_theta = select_threshold(
        pooled_val_probs,
        pooled_val_labels,
        max_fpr=0.05,
    )


    if pooled_theta is None:

        pooled_theta = 0.50


    # --------------------------------------------------------
    # Test evaluation
    # --------------------------------------------------------

    per_site = []

    all_probs = []
    all_labels = []

    TP = TN = FP = FN = 0


    for k, site in enumerate(
        silo_data
    ):

        probs_val, y_val = (
            val_cache[
                k
            ]
        )


        if use_calibration:

            theta = select_threshold(
                probs_val,
                y_val,
                max_fpr=0.05,
            )


            if theta is None:

                theta = float(
                    pooled_theta
                )

                threshold_source = (
                    "pooled-validation"
                )

            else:

                threshold_source = (
                    "local-validation"
                )

        else:

            # -----------------------------------------------
            # No threshold recalibration ablation
            # -----------------------------------------------

            theta = 0.50

            threshold_source = (
                "fixed-0.50"
            )


        probs_test, y_test = (
            collect_probs(
                model,
                site[
                    "test_loader"
                ],
                torch.device(
                    "cpu"
                ),
            )
        )


        m = metrics_at_threshold(
            probs_test,
            y_test,
            theta,
        )


        per_site.append(
            {
                "site":
                    k,

                "threshold":
                    theta,

                "threshold_source":
                    threshold_source,

                "precision":
                    float(
                        m[
                            "precision"
                        ]
                    ),

                "recall":
                    float(
                        m[
                            "recall"
                        ]
                    ),

                "f1":
                    float(
                        m[
                            "f1"
                        ]
                    ),

                "fpr":
                    float(
                        m[
                            "fpr"
                        ]
                    ),
            }
        )


        TP += int(
            m[
                "tp"
            ]
        )

        TN += int(
            m[
                "tn"
            ]
        )

        FP += int(
            m[
                "fp"
            ]
        )

        FN += int(
            m[
                "fn"
            ]
        )


        all_probs.append(
            np.asarray(
                probs_test
            )
        )

        all_labels.append(
            np.asarray(
                y_test
            )
        )


    # --------------------------------------------------------
    # Macro per-site metrics
    # --------------------------------------------------------

    macro_precision = float(
        np.mean(
            [
                x[
                    "precision"
                ]
                for x
                in per_site
            ]
        )
    )


    macro_recall = float(
        np.mean(
            [
                x[
                    "recall"
                ]
                for x
                in per_site
            ]
        )
    )


    macro_f1 = float(
        np.mean(
            [
                x[
                    "f1"
                ]
                for x
                in per_site
            ]
        )
    )


    macro_fpr = float(
        np.mean(
            [
                x[
                    "fpr"
                ]
                for x
                in per_site
            ]
        )
    )


    # --------------------------------------------------------
    # Global pooled confusion metrics
    # --------------------------------------------------------

    global_precision = (
        TP /
        (
            TP +
            FP +
            1e-12
        )
    )


    global_recall = (
        TP /
        (
            TP +
            FN +
            1e-12
        )
    )


    global_f1 = (
        2.0 *
        global_precision *
        global_recall
        /
        (
            global_precision
            +
            global_recall
            +
            1e-12
        )
    )


    global_fpr = (
        FP /
        (
            FP +
            TN +
            1e-12
        )
    )


    # --------------------------------------------------------
    # Threshold-independent ROC-AUC
    # --------------------------------------------------------

    all_probs = np.concatenate(
        all_probs
    )

    all_labels = np.concatenate(
        all_labels
    )


    global_auc = float(
        roc_auc_score(
            all_labels,
            all_probs,
        )
    )


    return {

        "macro_precision":
            macro_precision,

        "macro_recall":
            macro_recall,

        "macro_f1":
            macro_f1,

        "macro_fpr":
            macro_fpr,

        "global_precision":
            float(
                global_precision
            ),

        "global_recall":
            float(
                global_recall
            ),

        "global_f1":
            float(
                global_f1
            ),

        "global_fpr":
            float(
                global_fpr
            ),

        "global_roc_auc":
            global_auc,

        "per_site":
            per_site,
    }


# ============================================================
# 10. Evaluate the four TRAINING configurations
# ============================================================

final_rows = []

site_rows = []


for label in [
    "Full method",
    "FedAvg instead of TrimMean",
    "No FedProx ($\\mu=0$)",
    "No clipping",
]:

    print(
        "\n"
        + "=" * 82
    )

    print(
        f"EVALUATING: {label}"
    )

    print(
        "=" * 82
    )


    model = load_model(
        trained_runs[
            label
        ]
    )


    metrics = evaluate_model(
        model,
        use_calibration=True,
    )


    final_rows.append(
        {
            "configuration":
                label,

            "macro_precision":
                metrics[
                    "macro_precision"
                ],

            "macro_recall":
                metrics[
                    "macro_recall"
                ],

            "macro_f1":
                metrics[
                    "macro_f1"
                ],

            "macro_fpr":
                metrics[
                    "macro_fpr"
                ],

            "global_precision":
                metrics[
                    "global_precision"
                ],

            "global_recall":
                metrics[
                    "global_recall"
                ],

            "global_f1":
                metrics[
                    "global_f1"
                ],

            "global_fpr":
                metrics[
                    "global_fpr"
                ],

            "global_roc_auc":
                metrics[
                    "global_roc_auc"
                ],
        }
    )


    for row in metrics[
        "per_site"
    ]:

        site_rows.append(
            {
                "configuration":
                    label,

                **row,
            }
        )


    print(
        f"Macro F1   = "
        f"{metrics['macro_f1']:.4f}"
    )

    print(
        f"Macro FPR  = "
        f"{metrics['macro_fpr']:.4f}"
    )

    print(
        f"Global F1  = "
        f"{metrics['global_f1']:.4f}"
    )

    print(
        f"Global FPR = "
        f"{metrics['global_fpr']:.4f}"
    )

    print(
        f"ROC-AUC    = "
        f"{metrics['global_roc_auc']:.4f}"
    )


    del model
    gc.collect()


# ============================================================
# 11. No-threshold-recalibration ablation
#
# SAME trained full model.
# Only decision threshold changes to theta=0.50.
# ============================================================

print(
    "\n"
    + "=" * 82
)

print(
    "EVALUATING: No threshold recalibration"
)

print(
    "=" * 82
)


full_model = load_model(
    full_run
)


no_threshold = evaluate_model(
    full_model,
    use_calibration=False,
)


final_rows.append(
    {
        "configuration":
            "No threshold recalibration",

        "macro_precision":
            no_threshold[
                "macro_precision"
            ],

        "macro_recall":
            no_threshold[
                "macro_recall"
            ],

        "macro_f1":
            no_threshold[
                "macro_f1"
            ],

        "macro_fpr":
            no_threshold[
                "macro_fpr"
            ],

        "global_precision":
            no_threshold[
                "global_precision"
            ],

        "global_recall":
            no_threshold[
                "global_recall"
            ],

        "global_f1":
            no_threshold[
                "global_f1"
            ],

        "global_fpr":
            no_threshold[
                "global_fpr"
            ],

        "global_roc_auc":
            no_threshold[
                "global_roc_auc"
            ],
    }
)


for row in no_threshold[
    "per_site"
]:

    site_rows.append(
        {
            "configuration":
                "No threshold recalibration",

            **row,
        }
    )


print(
    f"Macro F1   = "
    f"{no_threshold['macro_f1']:.4f}"
)

print(
    f"Macro FPR  = "
    f"{no_threshold['macro_fpr']:.4f}"
)

print(
    f"Global F1  = "
    f"{no_threshold['global_f1']:.4f}"
)

print(
    f"Global FPR = "
    f"{no_threshold['global_fpr']:.4f}"
)

print(
    f"ROC-AUC    = "
    f"{no_threshold['global_roc_auc']:.4f}"
)


# ============================================================
# 12. Results
# ============================================================

ablation_df = pd.DataFrame(
    final_rows
)

site_df = pd.DataFrame(
    site_rows
)


# Preserve intended row order.
ORDER = [
    "Full method",
    "FedAvg instead of TrimMean",
    "No FedProx ($\\mu=0$)",
    "No clipping",
    "No threshold recalibration",
]


ablation_df[
    "order"
] = (
    ablation_df[
        "configuration"
    ]
    .map(
        {
            x: i
            for i, x
            in enumerate(
                ORDER
            )
        }
    )
)


ablation_df = (
    ablation_df
    .sort_values(
        "order"
    )
    .drop(
        columns=[
            "order"
        ]
    )
)


# ============================================================
# 13. Deltas relative to full method
# ============================================================

full = (
    ablation_df[
        ablation_df[
            "configuration"
        ]
        ==
        "Full method"
    ]
    .iloc[0]
)


ablation_df[
    "delta_macro_f1"
] = (
    ablation_df[
        "macro_f1"
    ]
    -
    full[
        "macro_f1"
    ]
)


ablation_df[
    "delta_macro_fpr"
] = (
    ablation_df[
        "macro_fpr"
    ]
    -
    full[
        "macro_fpr"
    ]
)


ablation_df[
    "delta_global_f1"
] = (
    ablation_df[
        "global_f1"
    ]
    -
    full[
        "global_f1"
    ]
)


ablation_df[
    "delta_global_fpr"
] = (
    ablation_df[
        "global_fpr"
    ]
    -
    full[
        "global_fpr"
    ]
)


# ============================================================
# 14. Save
# ============================================================

OUT = (
    RESULT_DIR /
    "edgeiiotset_final_ablation.csv"
)

SITE_OUT = (
    RESULT_DIR /
    "edgeiiotset_final_ablation_per_site.csv"
)

JSON_OUT = (
    RESULT_DIR /
    "edgeiiotset_final_ablation.json"
)


ablation_df.to_csv(
    OUT,
    index=False,
)


site_df.to_csv(
    SITE_OUT,
    index=False,
)


JSON_OUT.write_text(
    json.dumps(
        ablation_df.to_dict(
            orient="records"
        ),
        indent=2,
    )
)


# ============================================================
# 15. Final printout
# ============================================================

print(
    "\n"
    + "=" * 118
)

print(
    "FINAL EDGE-IIoTset ABLATION — MACRO SITE METRICS"
)

print(
    "=" * 118
)


print(
    ablation_df[
        [
            "configuration",
            "macro_precision",
            "macro_recall",
            "macro_f1",
            "macro_fpr",
            "delta_macro_f1",
            "delta_macro_fpr",
        ]
    ].to_string(
        index=False
    )
)


print(
    "\n"
    + "=" * 118
)

print(
    "FINAL EDGE-IIoTset ABLATION — GLOBAL TEST METRICS"
)

print(
    "=" * 118
)


print(
    ablation_df[
        [
            "configuration",
            "global_precision",
            "global_recall",
            "global_f1",
            "global_fpr",
            "global_roc_auc",
            "delta_global_f1",
            "delta_global_fpr",
        ]
    ].to_string(
        index=False
    )
)


print(
    "\nSaved:"
)

print(
    OUT
)

print(
    SITE_OUT
)

print(
    JSON_OUT
)


print(
    "\nNew training runtime:"
)

print(
    f"{(time.time() - start_all) / 60:.1f} minutes"
)


print(
    "\n============================================"
)

print(
    "✅ STEP 20.16 EDGE-IIoTset ABLATION COMPLETE"
)

print(
    "============================================"
)

FINAL EDGE-IIoTset CONFIGURATION
seed                    : 42
rounds                  : 10
local_epochs            : 2
lr                      : 0.0003
weight_decay            : 0.0001
fedprox_mu              : 0.001
pos_weight              : 2.49966882129178
aggregator              : trimmean
use_update_clipping     : True
clip_C                  : 1.0
trim_beta               : 0.1

✅ Full method model:
/content/fl_ids_repo_real/runs/backdoor/edgeiiotset_classbalanced_clean_proposed_FINAL_seed42_20261002_111822
✅ Config created: FedAvg instead of TrimMean
✅ Config created: No FedProx ($\mu=0$)
✅ Config created: No clipping

ABLATION TRAINING 1/3
FedAvg instead of TrimMean

✅ FedAvg instead of TrimMean
Runtime = 11.9 min
Model   = edgeiiotset_ablation_edge_fedavg_FINAL_seed42_20261002_144644

ABLATION TRAINING 2/3
No FedProx ($\mu=0$)

✅ No FedProx ($\mu=0$)
Runtime = 11.9 min
Model   = edgeiiotset_ablation_edge_no_fedprox_FINAL_seed42_20261002_145841

ABLATION TRAINING 3/3
No clipping